<a href="https://colab.research.google.com/github/kankun0174/Graduation-Paper/blob/colab/%E3%83%94%E3%82%A2%E3%83%8E%E7%B7%B4%E7%BF%92%E6%94%AF%E6%8F%B4%E3%83%97%E3%83%AD%E3%82%B0%E3%83%A9%E3%83%A0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# このセルを先に実行して、qtaをアップロード→変換
from google.colab import files
import subprocess

print("qtaファイルをアップロードしてください")
uploaded = files.upload()

for name, content in uploaded.items():
    with open(name, 'wb') as f:
        f.write(content)
    out_name = name.replace('.qta', '.mp3')
    subprocess.run(['ffmpeg', '-y', '-i', name, out_name])
    print(f"✓ 変換完了: {out_name}")
    files.download(out_name)

In [ ]:
# ==============================================================================
# 🎹🧠 Piano Performance Scorer v4.4 - Offline Brain Wave Integration
# ==============================================================================
# Mind MonitorのCSVをアップロードして、録音と時間軸を合わせて脳波を小節に割り当て
#
# 使い方:
# 1. Mind Monitorで脳波を記録（CSVエクスポート）
# 2. 録音開始時刻と終了時刻をメモ（Mind Monitorの画面で確認）
# 3. Colabで録音 or 音声ファイルをアップロード
# 4. 脳波CSVをアップロード + 録音開始/終了時刻を入力
# 5. 採点実行 → 脳波と演奏が時間軸で統合される
# ==============================================================================

import numpy as np
import warnings
warnings.filterwarnings('ignore')

def install_dependencies():
    import subprocess, sys
    packages = [
        'pretty_midi', 'music21', 'librosa', 'matplotlib', 'pandas',
        'ipywidgets', 'transkun', 'dtw-python', 'verovio'
    ]
    for pkg in packages:
        try:
            __import__(pkg.replace('-', '_').replace('dtw-python', 'dtw'))
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    print("✓ 依存ライブラリ準備完了")

from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any
from datetime import datetime, timedelta
import threading
import time

@dataclass
class NoteEvent:
    pitch: int
    start_time: float
    end_time: float
    velocity: int = 64
    matched: bool = False
    @property
    def duration(self): return self.end_time - self.start_time
    def pitch_name(self):
        names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
        return f"{names[self.pitch % 12]}{(self.pitch // 12) - 1}"

@dataclass
class MeasureData:
    number: int
    index: int
    start_time: float
    end_time: float
    tempo: float
    time_signature: Tuple[int, int]
    ref_notes: List[NoteEvent] = field(default_factory=list)
    perf_notes: List[NoteEvent] = field(default_factory=list)
    pitch_score: float = 0.0
    rhythm_score: float = 0.0
    tempo_score: float = 0.0
    perf_start_time: float = 0.0
    perf_end_time: float = 0.0
    brain_state: str = ""
    brain_metrics: Dict = field(default_factory=dict)
    @property
    def duration(self): return self.end_time - self.start_time

@dataclass
class ScoringParams:
    pitch_tolerance: int = 0
    ignore_octave: bool = False
    chord_time_window: float = 0.05
    allow_arpeggio: bool = True
    arpeggio_max_time: float = 0.2
    rhythm_tolerance: float = 0.1
    tempo_tolerance: float = 0.1

# -----------------------------
# Brain Wave CSV Parser
# -----------------------------
class BrainWaveCSVParser:
    """Mind MonitorのCSVを解析"""

    def __init__(self):
        self.data = []
        self.start_time = None
        self.end_time = None
        self.recording_start = None  # 録音開始時刻（絶対時刻）
        self.recording_end = None    # 録音終了時刻（絶対時刻）
        self.aligned_data = []       # 録音に合わせた相対時刻データ

    def parse_csv(self, csv_path: str) -> bool:
        """Mind Monitor CSVを解析"""
        import pandas as pd

        try:
            df = pd.read_csv(csv_path)
            print(f"📊 CSV読み込み: {len(df)}行")
            print(f"   列: {list(df.columns)[:10]}...")

            # タイムスタンプ列を探す
            time_col = None
            for col in ['TimeStamp', 'Timestamp', 'timestamp', 'Time', 'time']:
                if col in df.columns:
                    time_col = col
                    break

            if time_col is None:
                print("⚠️ タイムスタンプ列が見つかりません")
                return False

            # α/β/θ列を探す
            alpha_cols = [c for c in df.columns if 'Alpha' in c or 'alpha' in c]
            beta_cols = [c for c in df.columns if 'Beta' in c or 'beta' in c]
            theta_cols = [c for c in df.columns if 'Theta' in c or 'theta' in c]

            print(f"   Alpha列: {alpha_cols[:2]}")
            print(f"   Beta列: {beta_cols[:2]}")
            print(f"   Theta列: {theta_cols[:2]}")

            self.data = []
            for _, row in df.iterrows():
                try:
                    # タイムスタンプをパース
                    ts_str = str(row[time_col])
                    # 複数のフォーマットを試す
                    ts = None
                    for fmt in ['%Y-%m-%d %H:%M:%S.%f', '%Y-%m-%d %H:%M:%S',
                               '%H:%M:%S.%f', '%H:%M:%S',
                               '%Y/%m/%d %H:%M:%S.%f', '%Y/%m/%d %H:%M:%S']:
                        try:
                            ts = datetime.strptime(ts_str, fmt)
                            break
                        except:
                            continue

                    if ts is None:
                        # Unix timestampかも
                        try:
                            ts = datetime.fromtimestamp(float(ts_str))
                        except:
                            continue

                    # α/β/θの平均を計算
                    def get_avg(cols):
                        vals = [float(row[c]) for c in cols if c in row and pd.notna(row[c])]
                        return sum(vals) / len(vals) if vals else 0.0

                    alpha = get_avg(alpha_cols)
                    beta = get_avg(beta_cols)
                    theta = get_avg(theta_cols)

                    self.data.append({
                        'timestamp': ts,
                        'alpha': alpha,
                        'beta': beta,
                        'theta': theta
                    })
                except Exception as e:
                    continue

            if not self.data:
                print("⚠️ 有効なデータがありません")
                return False

            self.start_time = self.data[0]['timestamp']
            self.end_time = self.data[-1]['timestamp']
            duration = (self.end_time - self.start_time).total_seconds()

            print(f"✓ 脳波データ: {len(self.data)}サンプル")
            print(f"   期間: {self.start_time.strftime('%H:%M:%S')} - {self.end_time.strftime('%H:%M:%S')} ({duration:.1f}秒)")

            return True

        except Exception as e:
            print(f"❌ CSV解析エラー: {e}")
            import traceback
            traceback.print_exc()
            return False

    def set_recording_time(self, start_str: str, end_str: str) -> bool:
        """録音の開始/終了時刻を設定（HH:MM:SS形式）"""
        try:
            # 今日の日付を使用
            base_date = self.start_time.date() if self.start_time else datetime.now().date()

            # 時刻をパース
            for fmt in ['%H:%M:%S.%f', '%H:%M:%S', '%H:%M']:
                try:
                    start_time = datetime.strptime(start_str, fmt).time()
                    break
                except:
                    continue
            else:
                print(f"⚠️ 開始時刻のフォーマットエラー: {start_str}")
                return False

            for fmt in ['%H:%M:%S.%f', '%H:%M:%S', '%H:%M']:
                try:
                    end_time = datetime.strptime(end_str, fmt).time()
                    break
                except:
                    continue
            else:
                print(f"⚠️ 終了時刻のフォーマットエラー: {end_str}")
                return False

            self.recording_start = datetime.combine(base_date, start_time)
            self.recording_end = datetime.combine(base_date, end_time)

            duration = (self.recording_end - self.recording_start).total_seconds()
            print(f"✓ 録音時間設定: {start_str} - {end_str} ({duration:.1f}秒)")

            return True

        except Exception as e:
            print(f"❌ 時刻設定エラー: {e}")
            return False

    def align_to_recording(self) -> bool:
        """脳波データを録音時間に合わせて相対時刻に変換（Z-score方式で状態判定）"""
        if not self.data or not self.recording_start or not self.recording_end:
            print("⚠️ データまたは録音時間が設定されていません")
            return False

        self.aligned_data = []
        rec_duration = (self.recording_end - self.recording_start).total_seconds()

        # まず録音範囲内のデータを抽出して相対値を計算
        temp_data = []
        for d in self.data:
            ts = d['timestamp']
            if self.recording_start <= ts <= self.recording_end:
                alpha, beta, theta = d['alpha'], d['beta'], d['theta']
                total = alpha + beta + theta
                if total > 0:
                    a_rel = alpha / total
                    b_rel = beta / total
                    t_rel = theta / total
                    engagement = b_rel / (a_rel + t_rel + 1e-9)
                    temp_data.append({
                        'timestamp': ts,
                        'relative_time': (ts - self.recording_start).total_seconds(),
                        'alpha': alpha, 'beta': beta, 'theta': theta,
                        'alpha_rel': a_rel, 'beta_rel': b_rel, 'theta_rel': t_rel,
                        'engagement': engagement
                    })

        if not temp_data:
            print("⚠️ 録音範囲内にデータがありません")
            return False

        # ベースライン統計を計算（Z-score用）
        import statistics
        alpha_vals = [d['alpha_rel'] for d in temp_data]
        eng_vals = [d['engagement'] for d in temp_data]

        alpha_mean = statistics.fmean(alpha_vals)
        alpha_std = statistics.pstdev(alpha_vals) or 0.01
        eng_mean = statistics.fmean(eng_vals)
        eng_std = statistics.pstdev(eng_vals) or 0.01

        print(f"   ベースライン: α相対={alpha_mean:.3f}±{alpha_std:.3f}, eng={eng_mean:.3f}±{eng_std:.3f}")

        # Z-score方式で状態判定
        focus_count = relax_count = neutral_count = 0
        for d in temp_data:
            z_alpha = (d['alpha_rel'] - alpha_mean) / alpha_std
            z_eng = (d['engagement'] - eng_mean) / eng_std

            # Z-score判定：平均からの偏差で判断
            if z_eng > 0.5 and z_alpha < 0:
                state = 'FOCUSED'
                focus_count += 1
            elif z_alpha > 0.5 and z_eng < 0:
                state = 'RELAXED'
                relax_count += 1
            else:
                state = 'NEUTRAL'
                neutral_count += 1

            self.aligned_data.append({
                'relative_time': d['relative_time'],
                'alpha': d['alpha'], 'beta': d['beta'], 'theta': d['theta'],
                'alpha_rel': d['alpha_rel'], 'beta_rel': d['beta_rel'], 'theta_rel': d['theta_rel'],
                'engagement': d['engagement'],
                'z_alpha': z_alpha, 'z_eng': z_eng,
                'state': state
            })

        print(f"✓ アライメント完了: {len(self.aligned_data)}サンプル（{rec_duration:.1f}秒間）")
        print(f"   状態分布: FOCUSED={focus_count} ({focus_count/len(self.aligned_data)*100:.1f}%), RELAXED={relax_count} ({relax_count/len(self.aligned_data)*100:.1f}%), NEUTRAL={neutral_count} ({neutral_count/len(self.aligned_data)*100:.1f}%)")
        return len(self.aligned_data) > 0

    def get_states_for_time_range(self, start_sec: float, end_sec: float) -> Dict[str, Any]:
        """指定時間範囲の脳波状態を取得"""
        if not self.aligned_data:
            return {'available': False, 'dominant_state': 'NO_DATA', 'count': 0}

        states_in_range = [
            d for d in self.aligned_data
            if start_sec <= d['relative_time'] <= end_sec
        ]

        if not states_in_range:
            return {'available': True, 'dominant_state': 'NO_DATA', 'count': 0, 'focus_ratio': 0.0}

        counts = {'FOCUSED': 0, 'RELAXED': 0, 'NEUTRAL': 0, 'NO_DATA': 0}
        alpha_sum, beta_sum, theta_sum = 0.0, 0.0, 0.0

        for s in states_in_range:
            st = s.get('state', 'NEUTRAL')
            if st in counts:
                counts[st] += 1
            alpha_sum += s.get('alpha_rel', 0)
            beta_sum += s.get('beta_rel', 0)
            theta_sum += s.get('theta_rel', 0)

        total = len(states_in_range)
        dominant = max(counts, key=counts.get)

        return {
            'available': True,
            'dominant_state': dominant,
            'count': total,
            'focus_ratio': counts['FOCUSED'] / total if total > 0 else 0.0,
            'relax_ratio': counts['RELAXED'] / total if total > 0 else 0.0,
            'avg_alpha': alpha_sum / total if total > 0 else 0,
            'avg_beta': beta_sum / total if total > 0 else 0,
            'avg_theta': theta_sum / total if total > 0 else 0,
            'state_counts': counts
        }

    def get_session_summary(self) -> Dict[str, Any]:
        """セッション全体のサマリー"""
        if not self.aligned_data:
            return {'available': False}

        counts = {'FOCUSED': 0, 'RELAXED': 0, 'NEUTRAL': 0, 'NO_DATA': 0}
        for d in self.aligned_data:
            st = d.get('state', 'NEUTRAL')
            if st in counts:
                counts[st] += 1

        total = len(self.aligned_data)
        duration = self.aligned_data[-1]['relative_time'] if self.aligned_data else 0

        return {
            'available': True,
            'total_samples': total,
            'duration': duration,
            'focus_ratio': counts['FOCUSED'] / total if total > 0 else 0.0,
            'relax_ratio': counts['RELAXED'] / total if total > 0 else 0.0,
            'neutral_ratio': counts['NEUTRAL'] / total if total > 0 else 0.0,
            'state_counts': counts
        }

brain_csv_parser = BrainWaveCSVParser()

# -----------------------------
# Reference Parser
# -----------------------------
class ReferenceParser:
    def __init__(self, midi_path, mxl_path):
        self.midi_path, self.mxl_path = midi_path, mxl_path

    def parse(self):
        print("📖 参照データを解析中...")
        midi_notes, midi_dur = self._parse_midi()
        print(f"  ✓ MIDI: {len(midi_notes)}音符, {midi_dur:.1f}秒")
        mxl_measures = self._parse_mxl()
        print(f"  ✓ MXL: {len(mxl_measures)}小節")
        measures = self._create_measures(midi_notes, mxl_measures, midi_dur)
        return measures, midi_notes

    def _parse_midi(self):
        import pretty_midi
        pm = pretty_midi.PrettyMIDI(self.midi_path)
        notes = []
        for inst in pm.instruments:
            if not inst.is_drum:
                for n in inst.notes:
                    notes.append(NoteEvent(pitch=n.pitch, start_time=n.start, end_time=n.end, velocity=n.velocity))
        notes.sort(key=lambda n: (n.start_time, n.pitch))
        return notes, pm.get_end_time()

    def _parse_mxl(self):
        from music21 import converter, tempo as m21tempo, meter
        score = converter.parse(self.mxl_path)
        try:
            score = score.expandRepeats()
        except:
            pass
        measures, current_tempo, current_ts = [], 120.0, (4, 4)
        for t in score.flatten().getElementsByClass(m21tempo.MetronomeMark):
            if t.number:
                current_tempo = float(t.number)
                break
        part = score.parts[0] if score.parts else score
        for m in part.getElementsByClass('Measure'):
            for t in m.getElementsByClass(m21tempo.MetronomeMark):
                if t.number:
                    current_tempo = float(t.number)
            for ts in m.getElementsByClass(meter.TimeSignature):
                if ts.numerator:
                    current_ts = (ts.numerator, ts.denominator)
            measures.append({
                'number': m.measureNumber,
                'tempo': current_tempo or 120,
                'time_signature': current_ts,
                'quarter_length': float(m.quarterLength) if m.quarterLength else 4.0
            })
        return measures

    def _create_measures(self, midi_notes, mxl_measures, midi_dur):
        if not mxl_measures:
            return []
        measures = []
        total_ql = sum(m['quarter_length'] for m in mxl_measures)
        midi_span = (midi_notes[-1].start_time - midi_notes[0].start_time) if midi_notes else midi_dur
        midi_offset = midi_notes[0].start_time if midi_notes else 0
        current_time = midi_offset
        for i, mxl in enumerate(mxl_measures):
            ql = mxl['quarter_length']
            dur = (ql / total_ql) * midi_span if total_ql > 0 else 2.0
            end_time = current_time + dur
            ref_notes = [
                NoteEvent(pitch=n.pitch, start_time=n.start_time, end_time=n.end_time, velocity=n.velocity)
                for n in midi_notes
                if current_time - 0.05 <= n.start_time < end_time + 0.05
            ]
            measures.append(MeasureData(
                number=mxl['number'], index=i, start_time=current_time, end_time=end_time,
                tempo=mxl['tempo'], time_signature=mxl['time_signature'], ref_notes=ref_notes
            ))
            current_time = end_time
        return measures

# -----------------------------
# Audio Transcriber
# -----------------------------
class AudioTranscriber:
    def __init__(self, use_gpu=True):
        self.use_gpu = use_gpu

    def transcribe(self, audio_path):
        import subprocess, tempfile, os, pretty_midi, librosa
        print("🎵 音声を解析中...")
        device = 'cpu'
        if self.use_gpu:
            try:
                import torch
                if torch.cuda.is_available():
                    device = 'cuda'
                    print("  ✓ CUDA使用")
            except:
                pass
        y, sr = librosa.load(audio_path, sr=None)
        duration = len(y) / sr
        print(f"  - 音声長: {duration:.1f}秒")
        with tempfile.TemporaryDirectory() as td:
            out = os.path.join(td, "out.mid")
            subprocess.run(
                ['python3', '-m', 'transkun.transcribe', audio_path, out, '--device', device],
                check=True, capture_output=True, timeout=600
            )
            pm = pretty_midi.PrettyMIDI(out)
            notes = [
                NoteEvent(pitch=n.pitch, start_time=n.start, end_time=n.end, velocity=n.velocity)
                for inst in pm.instruments if not inst.is_drum for n in inst.notes
            ]
            notes.sort(key=lambda n: (n.start_time, n.pitch))
            print(f"  ✓ {len(notes)}音符を認識")
            return notes, duration

# -----------------------------
# DTW Aligner
# -----------------------------
class DTWAligner:
    def __init__(self, params):
        self.params = params

    def align(self, perf_notes, ref_notes, measures):
        print("🔗 アライメント中...")
        if not perf_notes or not ref_notes:
            return measures
        try:
            from dtw import dtw
            pp = np.array([n.pitch for n in perf_notes]).reshape(-1, 1)
            rp = np.array([n.pitch for n in ref_notes]).reshape(-1, 1)
            def dist(x, y):
                d = abs(x[0] - y[0])
                return d/12*2 if d % 12 == 0 and d > 0 else 0 if d <= self.params.pitch_tolerance else d
            alignment = dtw(pp, rp, dist_method=dist, keep_internals=True,
                          step_pattern='symmetric2', window_type='sakoechiba',
                          window_args={'window_size': 400})
            path = list(zip(alignment.index1, alignment.index2))
        except:
            path = [(i, min(i, len(ref_notes)-1)) for i in range(len(perf_notes))]

        pairs = [(perf_notes[p].start_time, ref_notes[r].start_time)
                 for p, r in path if p < len(perf_notes) and r < len(ref_notes)]
        if not pairs:
            return measures

        pt = np.array([p[0] for p in pairs])
        rt = np.array([p[1] for p in pairs])

        def time_map(t):
            if t <= pt[0]: return rt[0]
            if t >= pt[-1]: return rt[-1]
            idx = np.searchsorted(pt, t)
            if idx == 0: return rt[0]
            ratio = (t - pt[idx-1]) / (pt[idx] - pt[idx-1]) if pt[idx] != pt[idx-1] else 0
            return rt[idx-1] + ratio * (rt[idx] - rt[idx-1])

        for note in perf_notes:
            ref_t = time_map(note.start_time)
            for m in measures:
                if m.start_time <= ref_t < m.end_time:
                    m.perf_notes.append(NoteEvent(
                        pitch=note.pitch, start_time=note.start_time,
                        end_time=note.end_time, velocity=note.velocity
                    ))
                    if not m.perf_start_time or note.start_time < m.perf_start_time:
                        m.perf_start_time = note.start_time
                    if note.start_time > m.perf_end_time:
                        m.perf_end_time = note.start_time
                    break

        print(f"  ✓ {sum(len(m.perf_notes) for m in measures)}/{len(perf_notes)}音符をマッピング")
        return measures

# -----------------------------
# Scoring Engine
# -----------------------------
class ScoringEngine:
    def __init__(self, params, brain_parser=None):
        self.params = params
        self.brain_parser = brain_parser

    def score_all(self, measures, audio_duration=None):
        print("📊 採点中...")

        all_perf = [n for m in measures for n in m.perf_notes]
        if all_perf:
            perf_start = min(n.start_time for n in all_perf)
            perf_end = max(n.end_time for n in all_perf)
            perf_duration = perf_end - perf_start
        else:
            perf_start, perf_end, perf_duration = 0, 0, 0

        brain_duration = None
        if self.brain_parser and self.brain_parser.aligned_data:
            brain_duration = self.brain_parser.aligned_data[-1]['relative_time']
            print(f"  🧠 脳波: {len(self.brain_parser.aligned_data)}サンプル, {brain_duration:.1f}秒")

        for m in measures:
            m.pitch_score = self._score_pitch(m)
            m.rhythm_score = self._score_rhythm(m)
            m.tempo_score = self._score_tempo(m)

            # 脳波割り当て
            if self.brain_parser and brain_duration and brain_duration > 0 and m.perf_notes:
                if perf_duration > 0:
                    m_start = m.perf_start_time if m.perf_start_time else min(n.start_time for n in m.perf_notes)
                    m_end = m.perf_end_time if m.perf_end_time else max(n.end_time for n in m.perf_notes)

                    # 演奏時刻を脳波時刻に変換
                    brain_start = ((m_start - perf_start) / perf_duration) * brain_duration
                    brain_end = ((m_end - perf_start) / perf_duration) * brain_duration

                    summary = self.brain_parser.get_states_for_time_range(brain_start, brain_end)
                    m.brain_state = summary.get('dominant_state', '')
                    m.brain_metrics = {
                        'focus_ratio': summary.get('focus_ratio', 0.0),
                        'relax_ratio': summary.get('relax_ratio', 0.0),
                        'count': summary.get('count', 0),
                    }

        result = self._calc_total(measures)
        print(f"  ✓ 総合スコア: {result['total_score']:.1f}")

        if self.brain_parser and self.brain_parser.aligned_data:
            result['brain_summary'] = self.brain_parser.get_session_summary()

        return result

    def _score_pitch(self, m):
        if not m.ref_notes: return 1.0 if not m.perf_notes else 0.0
        if not m.perf_notes: return 0.0
        ref_groups = self._group_by_time(m.ref_notes, self.params.chord_time_window)
        perf_groups = self._group_by_time(m.perf_notes, self.params.arpeggio_max_time if self.params.allow_arpeggio else self.params.chord_time_window)
        total = sum(len(g['pitches']) for g in ref_groups)
        pool = set(p for g in perf_groups for p in g['pitches'])
        matched = sum(1 for g in ref_groups for rp in g['pitches'] if rp in pool or any(self._match(rp, pp) for pp in pool))
        return matched / total if total > 0 else 1.0

    def _group_by_time(self, notes, window):
        if not notes: return []
        notes = sorted(notes, key=lambda n: n.start_time)
        groups = [{'time': notes[0].start_time, 'pitches': [notes[0].pitch]}]
        for n in notes[1:]:
            if n.start_time - groups[-1]['time'] <= window:
                groups[-1]['pitches'].append(n.pitch)
            else:
                groups.append({'time': n.start_time, 'pitches': [n.pitch]})
        return groups

    def _match(self, rp, pp):
        d = abs(rp - pp)
        return d == 0 or d <= self.params.pitch_tolerance or (self.params.ignore_octave and d % 12 == 0)

    def _score_rhythm(self, m):
        if not m.ref_notes or not m.perf_notes: return 1.0 if not m.ref_notes else 0.0
        def pos(notes, s, e):
            d = e - s if e > s else 0.1
            return [(n.start_time - s) / d for n in notes]
        ref_pos = pos(m.ref_notes, m.start_time, m.end_time)
        perf_pos = pos(m.perf_notes, m.perf_start_time, m.perf_end_time) if m.perf_end_time > m.perf_start_time else pos(m.perf_notes, m.perf_notes[0].start_time, m.perf_notes[-1].start_time + 0.1)
        tol = self.params.rhythm_tolerance
        return sum(1 for rp in ref_pos if any(abs(rp - pp) <= tol for pp in perf_pos)) / len(ref_pos) if ref_pos else 1.0

    def _score_tempo(self, m):
        if not m.perf_notes or len(m.perf_notes) < 2: return 1.0
        pd, rd = m.perf_end_time - m.perf_start_time, m.duration
        if pd <= 0 or rd <= 0: return 1.0
        r = pd / rd
        return 1.0 if 1 - self.params.tempo_tolerance <= r <= 1 + self.params.tempo_tolerance else max(0, 1 - (abs(r - 1) - self.params.tempo_tolerance) * 2)

    def _calc_total(self, measures):
        played = [m for m in measures if m.perf_notes]
        if not played:
            return {'total_score': 0, 'pitch_score': 0, 'rhythm_score': 0, 'tempo_score': 0, 'measures_played': 0, 'measures_total': len(measures), 'measure_details': []}
        ap, ar, at = np.mean([m.pitch_score for m in played]), np.mean([m.rhythm_score for m in played]), np.mean([m.tempo_score for m in played])
        return {
            'total_score': (ap * 0.5 + ar * 0.3 + at * 0.2) * 100, 'pitch_score': ap * 100, 'rhythm_score': ar * 100, 'tempo_score': at * 100,
            'measures_played': len(played), 'measures_total': len(measures),
            'start_measure': measures[min(m.index for m in played)].number, 'end_measure': measures[max(m.index for m in played)].number,
            'measure_details': [{'number': m.number, 'index': m.index, 'pitch': m.pitch_score*100, 'rhythm': m.rhythm_score*100, 'tempo': m.tempo_score*100, 'ref_notes': len(m.ref_notes), 'perf_notes': len(m.perf_notes), 'brain_state': m.brain_state, 'brain_metrics': m.brain_metrics} for m in measures]
        }

# -----------------------------
# Visualizer
# -----------------------------
class Visualizer:
    def show_summary(self, result):
        from IPython.display import display, HTML
        r = result
        brain_html = ""
        if 'brain_summary' in r and r['brain_summary'].get('available'):
            bs = r['brain_summary']
            brain_html = f"""<div style="margin-top:15px;padding:10px;background:rgba(255,255,255,0.1);border-radius:8px;"><div style="font-size:14px;margin-bottom:5px;">🧠 脳波 ({bs['total_samples']}サンプル, {bs.get('duration', 0):.1f}秒)</div><div style="display:flex;gap:15px;flex-wrap:wrap;"><span>🟢集中: {bs['focus_ratio']*100:.0f}%</span><span>🔵リラックス: {bs['relax_ratio']*100:.0f}%</span><span>⚪中立: {bs['neutral_ratio']*100:.0f}%</span></div></div>"""
        html = f"""<div style="background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);padding:20px;border-radius:15px;color:white;margin:10px 0;"><h2 style="margin:0 0 15px 0;">🎹 採点結果</h2><div style="display:flex;justify-content:space-around;flex-wrap:wrap;"><div style="text-align:center;padding:10px;"><div style="font-size:36px;font-weight:bold;">{r['total_score']:.1f}</div><div>総合</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['pitch_score']:.1f}</div><div>🎵ピッチ</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['rhythm_score']:.1f}</div><div>🥁リズム</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['tempo_score']:.1f}</div><div>⏱️テンポ</div></div></div><div style="margin-top:10px;">演奏: {r['measures_played']}/{r['measures_total']}小節</div>{brain_html}</div>"""
        display(HTML(html))

    def show_table(self, result, only_played=True):
        import pandas as pd
        from IPython.display import display
        df = pd.DataFrame(result.get('measure_details', []))
        if df.empty: return
        if only_played: df = df[df['perf_notes'] > 0]
        has_brain = any(d.get('brain_state') for d in result.get('measure_details', []))
        if has_brain:
            df['brain'] = df.apply(lambda r: (r['brain_state'][:3] if r.get('brain_state') else '-'), axis=1)
            df = df[['number', 'index', 'pitch', 'rhythm', 'tempo', 'ref_notes', 'perf_notes', 'brain']]
            df.columns = ['小節', '通し', 'ピッチ', 'リズム', 'テンポ', '楽譜', '演奏', '脳波']
        else:
            df = df[['number', 'index', 'pitch', 'rhythm', 'tempo', 'ref_notes', 'perf_notes']]
            df.columns = ['小節', '通し', 'ピッチ', 'リズム', 'テンポ', '楽譜', '演奏']
        def color(v):
            try:
                v = float(v)
                return 'background-color:#90EE90' if v >= 80 else 'background-color:#FFE4B5' if v >= 50 else 'background-color:#FFB6C1' if v > 0 else ''
            except: return ''
        display(df.style.applymap(color, subset=['ピッチ', 'リズム', 'テンポ']))

    def show_measure_detail(self, m):
        """小節の詳細情報をテキストで表示（音符リスト付き）"""
        from IPython.display import display, HTML
        def p2n(p):
            names = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
            return f"{names[p%12]}{p//12-1}"

        ref_pitches = sorted(set(n.pitch for n in m.ref_notes))
        perf_pitches = sorted(set(n.pitch for n in m.perf_notes))
        matched = set(ref_pitches) & set(perf_pitches)
        missing = set(ref_pitches) - set(perf_pitches)
        extra = set(perf_pitches) - set(ref_pitches)

        brain_html = ""
        if m.brain_state:
            metrics = m.brain_metrics or {}
            brain_html = f"""<div style="background:#e9ecef;padding:10px;border-radius:4px;margin:10px 0;">
                🧠 <b>{m.brain_state}</b> | 集中率: {metrics.get('focus_ratio', 0)*100:.0f}% | サンプル数: {metrics.get('count', 0)}
            </div>"""

        html = f"""<div style="border:2px solid #333;padding:15px;border-radius:8px;background:#fafafa;">
            <h3>小節 {m.number} (通し番号: {m.index})</h3>
            <div style="display:flex;gap:10px;margin:10px 0;">
                <span style="padding:5px 12px;border-radius:4px;background:{'#90EE90' if m.pitch_score>=0.8 else '#FFE4B5' if m.pitch_score>=0.5 else '#FFB6C1'};">🎵 ピッチ: {m.pitch_score*100:.1f}%</span>
                <span style="padding:5px 12px;border-radius:4px;background:{'#90EE90' if m.rhythm_score>=0.8 else '#FFE4B5' if m.rhythm_score>=0.5 else '#FFB6C1'};">🥁 リズム: {m.rhythm_score*100:.1f}%</span>
                <span style="padding:5px 12px;border-radius:4px;background:{'#90EE90' if m.tempo_score>=0.8 else '#FFE4B5' if m.tempo_score>=0.5 else '#FFB6C1'};">⏱️ テンポ: {m.tempo_score*100:.1f}%</span>
            </div>
            {brain_html}
            <div style="background:#f0f0f0;padding:10px;border-radius:5px;margin-top:10px;">
                <div style="color:green;margin:3px 0;">✓ 一致: {', '.join(p2n(p) for p in sorted(matched)) or 'なし'}</div>
                <div style="color:red;margin:3px 0;">✗ 不足: {', '.join(p2n(p) for p in sorted(missing)) or 'なし'}</div>
                <div style="color:orange;margin:3px 0;">+ 余分: {', '.join(p2n(p) for p in sorted(extra)) or 'なし'}</div>
            </div>
            <div style="margin-top:10px;font-size:12px;color:#666;">
                楽譜: {len(m.ref_notes)}音 ({m.start_time:.2f}s - {m.end_time:.2f}s) |
                演奏: {len(m.perf_notes)}音 ({m.perf_start_time:.2f}s - {m.perf_end_time:.2f}s)
            </div>
        </div>"""
        display(HTML(html))

    def play_measure_audio(self, m, play_ref=True):
        """小節の音符をWebAudioで再生（JavaScript）"""
        from IPython.display import display, Javascript
        import json

        if play_ref:
            notes = [{'pitch': n.pitch, 'start': n.start_time - m.start_time, 'dur': min(n.duration, 2.0)} for n in m.ref_notes]
            label = "楽譜"
        else:
            if not m.perf_notes:
                print("演奏データがありません")
                return
            base = m.perf_notes[0].start_time
            notes = [{'pitch': n.pitch, 'start': n.start_time - base, 'dur': min(n.duration, 2.0)} for n in m.perf_notes]
            label = "演奏"

        print(f"🔊 小節{m.number}の{label}を再生中...")

        js_code = f"""
(function() {{
    var notes = {json.dumps(notes)};
    var ctx = new (window.AudioContext || window.webkitAudioContext)();
    var real = new Float32Array([0, 1.0, 0.5, 0.35, 0.2, 0.12, 0.08, 0.05, 0.03]);
    var imag = new Float32Array(real.length);
    var pianoWave = ctx.createPeriodicWave(real, imag);

    notes.forEach(function(n) {{
        var o = ctx.createOscillator();
        var g = ctx.createGain();
        o.connect(g);
        g.connect(ctx.destination);
        o.frequency.value = 440 * Math.pow(2, (n.pitch - 69) / 12);
        o.setPeriodicWave(pianoWave);
        var s = Math.max(0, n.start);
        var d = n.dur;
        g.gain.setValueAtTime(0, ctx.currentTime + s);
        g.gain.linearRampToValueAtTime(0.3, ctx.currentTime + s + 0.01);
        g.gain.exponentialRampToValueAtTime(0.001, ctx.currentTime + s + d + 0.3);
        o.start(ctx.currentTime + s);
        o.stop(ctx.currentTime + s + d + 0.4);
    }});
}})();
"""
        display(Javascript(js_code))

    def show_brain_timeline(self, measures):
        import matplotlib.pyplot as plt
        played = [m for m in measures if m.perf_notes]
        if not played: print("演奏小節がありません"); return
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
        indices = [m.index for m in played]
        scores = [(m.pitch_score * 0.5 + m.rhythm_score * 0.3 + m.tempo_score * 0.2) * 100 for m in played]
        colors = ['#90EE90' if s >= 80 else '#FFE4B5' if s >= 50 else '#FFB6C1' for s in scores]
        ax1.bar(indices, scores, color=colors, edgecolor='gray')
        ax1.axhline(y=70, color='red', linestyle='--', alpha=0.5, label='70%')
        ax1.set_ylabel('Score (%)')
        ax1.set_ylim(0, 100)
        ax1.legend()
        ax1.set_title('Score and Brain State by Measure')
        state_colors = {'FOCUSED': '#28a745', 'RELAXED': '#007bff', 'NEUTRAL': '#6c757d', 'NO_DATA': '#dc3545'}
        brain_colors = [state_colors.get(m.brain_state, '#999') for m in played]
        ax2.bar(indices, [1]*len(indices), color=brain_colors, edgecolor='gray')
        ax2.set_ylabel('Brain')
        ax2.set_xlabel('Measure Index')
        ax2.set_yticks([])
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=c, label=s) for s, c in state_colors.items()]
        ax2.legend(handles=legend_elements, loc='upper right', ncol=4)
        plt.tight_layout()
        plt.show()

        # 統計
        state_counts = {}
        for m in played:
            st = m.brain_state or 'NO_DATA'
            state_counts[st] = state_counts.get(st, 0) + 1
        print("\n脳波状態の統計:")
        for st, cnt in sorted(state_counts.items(), key=lambda x: -x[1]):
            print(f"  {st}: {cnt}小節 ({cnt/len(played)*100:.1f}%)")

    def show_measure_comparison(self, m):
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        def plot(ax, notes, off, c, title):
            if not notes: ax.text(0.5, 0.5, 'No notes', ha='center', va='center', transform=ax.transAxes); ax.set_title(title); return
            for n in notes: ax.barh(n.pitch, n.duration, left=n.start_time - off, height=0.6, color=c, alpha=0.6)
            ax.set_ylim(min(n.pitch for n in notes) - 2, max(n.pitch for n in notes) + 2)
            ax.set_xlabel('Time (s)'); ax.set_ylabel('Pitch'); ax.set_title(title); ax.grid(True, alpha=0.3)
        plot(axes[0], m.ref_notes, m.start_time, 'blue', f'Score ({len(m.ref_notes)} notes)')
        plot(axes[1], m.perf_notes, m.perf_notes[0].start_time if m.perf_notes else 0, 'red', f'Perf ({len(m.perf_notes)} notes)')
        brain = f" | 🧠{m.brain_state}" if m.brain_state else ""
        fig.suptitle(f"Measure {m.number} | P:{m.pitch_score*100:.1f}% | R:{m.rhythm_score*100:.1f}%{brain}", y=1.02)
        plt.tight_layout(); plt.show()

    def show_problem_measures(self, measures, threshold=0.7):
        from IPython.display import display, HTML
        problems = []
        for m in measures:
            issues = []
            if m.pitch_score < threshold: issues.append(f"P:{m.pitch_score*100:.0f}%")
            if m.rhythm_score < threshold: issues.append(f"R:{m.rhythm_score*100:.0f}%")
            if issues: problems.append({'num': m.number, 'idx': m.index, 'issues': issues, 'brain': m.brain_state or '-'})
        if not problems: print("✓ No problems (all >= 70%)"); return
        html = f"<h3>⚠️ Problems ({len(problems)})</h3><table style='border-collapse:collapse;'><tr style='background:#f0f0f0;'><th style='padding:5px;border:1px solid #ddd;'>M#</th><th style='padding:5px;border:1px solid #ddd;'>Idx</th><th style='padding:5px;border:1px solid #ddd;'>Issues</th><th style='padding:5px;border:1px solid #ddd;'>Brain</th></tr>"
        for p in problems[:30]: html += f"<tr><td style='padding:5px;border:1px solid #ddd;'>{p['num']}</td><td style='padding:5px;border:1px solid #ddd;'>{p['idx']}</td><td style='padding:5px;border:1px solid #ddd;color:red;'>{', '.join(p['issues'])}</td><td style='padding:5px;border:1px solid #ddd;'>{p['brain']}</td></tr>"
        html += "</table>"
        display(HTML(html))

    def show_pitch_histogram(self, measures):
        import matplotlib.pyplot as plt
        ref_p = [n.pitch for m in measures for n in m.ref_notes]
        perf_p = [n.pitch for m in measures for n in m.perf_notes]
        if not ref_p and not perf_p: print("No data"); return
        fig, ax = plt.subplots(figsize=(12, 4))
        bins = range(min(ref_p + perf_p) - 1, max(ref_p + perf_p) + 2)
        ax.hist(ref_p, bins=bins, alpha=0.5, label=f'Score({len(ref_p)})', color='blue')
        ax.hist(perf_p, bins=bins, alpha=0.5, label=f'Perf({len(perf_p)})', color='red')
        ax.legend(); ax.grid(True, alpha=0.3); plt.show()

    # ========== 卒論用エクスポート機能 ==========

    def export_thesis_data(self, measures, result, filename_prefix="thesis_data"):
        """卒論用のデータをCSVとしてエクスポート"""
        import pandas as pd
        from datetime import datetime

        # 1. 小節ごとの詳細データ
        measure_data = []
        for m in measures:
            if not m.perf_notes:
                continue
            measure_data.append({
                'measure_number': m.number,
                'measure_index': m.index,
                'pitch_score': m.pitch_score * 100,
                'rhythm_score': m.rhythm_score * 100,
                'tempo_score': m.tempo_score * 100,
                'total_score': (m.pitch_score * 0.5 + m.rhythm_score * 0.3 + m.tempo_score * 0.2) * 100,
                'ref_notes': len(m.ref_notes),
                'perf_notes': len(m.perf_notes),
                'brain_state': m.brain_state or 'NO_DATA',
                'focus_ratio': m.brain_metrics.get('focus_ratio', 0) if m.brain_metrics else 0,
                'brain_samples': m.brain_metrics.get('count', 0) if m.brain_metrics else 0,
            })

        df_measures = pd.DataFrame(measure_data)

        # 2. サマリーデータ
        summary_data = {
            'total_score': result['total_score'],
            'pitch_score': result['pitch_score'],
            'rhythm_score': result['rhythm_score'],
            'tempo_score': result['tempo_score'],
            'measures_played': result['measures_played'],
            'measures_total': result['measures_total'],
        }
        if 'brain_summary' in result and result['brain_summary'].get('available'):
            bs = result['brain_summary']
            summary_data.update({
                'brain_samples': bs['total_samples'],
                'brain_duration': bs['duration'],
                'focus_ratio': bs['focus_ratio'],
                'relax_ratio': bs['relax_ratio'],
                'neutral_ratio': bs['neutral_ratio'],
            })

        df_summary = pd.DataFrame([summary_data])

        # 3. 脳波状態別の統計
        if df_measures['brain_state'].notna().any():
            brain_stats = df_measures.groupby('brain_state').agg({
                'pitch_score': ['mean', 'std', 'count'],
                'rhythm_score': ['mean', 'std'],
                'total_score': ['mean', 'std'],
            }).round(2)
            brain_stats.columns = ['_'.join(col) for col in brain_stats.columns]
            brain_stats = brain_stats.reset_index()
        else:
            brain_stats = pd.DataFrame()

        # ファイル保存
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        measures_file = f"{filename_prefix}_measures_{timestamp}.csv"
        summary_file = f"{filename_prefix}_summary_{timestamp}.csv"
        brain_file = f"{filename_prefix}_brain_stats_{timestamp}.csv"

        df_measures.to_csv(measures_file, index=False, encoding='utf-8-sig')
        df_summary.to_csv(summary_file, index=False, encoding='utf-8-sig')
        if not brain_stats.empty:
            brain_stats.to_csv(brain_file, index=False, encoding='utf-8-sig')

        print(f"✓ エクスポート完了:")
        print(f"  - {measures_file} (小節ごとデータ)")
        print(f"  - {summary_file} (サマリー)")
        if not brain_stats.empty:
            print(f"  - {brain_file} (脳波状態別統計)")

        return df_measures, df_summary, brain_stats

    def show_thesis_analysis(self, measures, result):
        """卒論用の統計分析を表示"""
        import pandas as pd
        import matplotlib.pyplot as plt
        from IPython.display import display, HTML

        # データフレーム作成
        data = []
        for m in measures:
            if not m.perf_notes:
                continue
            data.append({
                'measure': m.number,
                'pitch': m.pitch_score * 100,
                'rhythm': m.rhythm_score * 100,
                'tempo': m.tempo_score * 100,
                'total': (m.pitch_score * 0.5 + m.rhythm_score * 0.3 + m.tempo_score * 0.2) * 100,
                'brain': m.brain_state or 'NO_DATA',
            })

        df = pd.DataFrame(data)

        # 1. 基本統計
        html = "<h2>📊 卒論用統計分析</h2>"
        html += "<h3>1. スコア基本統計</h3>"
        stats = df[['pitch', 'rhythm', 'tempo', 'total']].describe().round(2)
        html += stats.to_html()

        # 2. 脳波状態別統計
        if df['brain'].notna().any() and len(df['brain'].unique()) > 1:
            html += "<h3>2. 脳波状態別スコア</h3>"
            brain_stats = df.groupby('brain')[['pitch', 'rhythm', 'tempo', 'total']].agg(['mean', 'std', 'count']).round(2)
            html += brain_stats.to_html()

            # 3. 統計検定（FOCUSED vs NEUTRAL）
            from scipy import stats as scipy_stats
            focused = df[df['brain'] == 'FOCUSED']['total']
            neutral = df[df['brain'] == 'NEUTRAL']['total']
            relaxed = df[df['brain'] == 'RELAXED']['total']

            html += "<h3>3. 統計検定</h3>"
            if len(focused) >= 2 and len(neutral) >= 2:
                t_stat, p_val = scipy_stats.ttest_ind(focused, neutral)
                sig = "有意 (p<0.05)" if p_val < 0.05 else "有意差なし"
                html += f"<p><b>FOCUSED vs NEUTRAL:</b> t={t_stat:.3f}, p={p_val:.4f} → {sig}</p>"
                html += f"<p>  FOCUSED: M={focused.mean():.1f}, SD={focused.std():.1f}, N={len(focused)}</p>"
                html += f"<p>  NEUTRAL: M={neutral.mean():.1f}, SD={neutral.std():.1f}, N={len(neutral)}</p>"

            if len(relaxed) >= 2 and len(neutral) >= 2:
                t_stat, p_val = scipy_stats.ttest_ind(relaxed, neutral)
                sig = "有意 (p<0.05)" if p_val < 0.05 else "有意差なし"
                html += f"<p><b>RELAXED vs NEUTRAL:</b> t={t_stat:.3f}, p={p_val:.4f} → {sig}</p>"

        display(HTML(html))

        # 4. グラフ
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        # 4-1. スコア分布
        df[['pitch', 'rhythm', 'tempo', 'total']].hist(ax=axes[0], bins=20, alpha=0.7)
        axes[0].set_title('Score Distribution')

        # 4-2. 脳波状態別boxplot
        if len(df['brain'].unique()) > 1:
            df.boxplot(column='total', by='brain', ax=axes[1])
            axes[1].set_title('Total Score by Brain State')
            axes[1].set_xlabel('Brain State')
            axes[1].set_ylabel('Total Score')
            plt.suptitle('')

        # 4-3. スコア推移
        axes[2].plot(df['measure'], df['total'], 'b-', alpha=0.7, label='Total')
        axes[2].axhline(y=70, color='red', linestyle='--', alpha=0.5)
        axes[2].set_xlabel('Measure')
        axes[2].set_ylabel('Score')
        axes[2].set_title('Score Progression')
        axes[2].legend()

        plt.tight_layout()
        plt.show()

        return df

    def show_score_with_click(self, mxl_path, measures):
        from IPython.display import display, HTML
        import verovio, re, json
        try:
            tk = verovio.toolkit()
            tk.setOptions({"pageWidth": 1800, "pageHeight": 3000, "scale": 35, "adjustPageHeight": True})
            tk.loadFile(mxl_path)
            pc = tk.getPageCount()
            amd = [{'index': m.index, 'number': m.number, 'pitch': m.pitch_score*100, 'rhythm': m.rhythm_score*100, 'ref_notes': len(m.ref_notes), 'perf_notes': len(m.perf_notes), 'brain_state': m.brain_state, 'start_time': m.start_time, 'end_time': m.end_time, 'perf_start': m.perf_notes[0].start_time if m.perf_notes else 0, 'ref_note_list': [{'pitch': n.pitch, 'start': n.start_time, 'dur': n.duration} for n in m.ref_notes], 'perf_note_list': [{'pitch': n.pitch, 'start': n.start_time, 'dur': n.duration} for n in m.perf_notes]} for m in measures]
            mc = {i: "#90EE90" if (m.pitch_score + m.rhythm_score)/2*100 >= 90 else "#FFFF99" if (m.pitch_score + m.rhythm_score)/2*100 >= 70 else "#FFE4B5" if (m.pitch_score + m.rhythm_score)/2*100 >= 50 else "#FFB6C1" for i, m in enumerate(measures)}
            svgs = []
            measure_idx = 0
            for p in range(1, pc + 1):
                svg = tk.renderToSVG(p)
                def add_attr(match):
                    nonlocal measure_idx
                    result = match.group(0)
                    if 'data-measure-idx' not in result:
                        result = result[:-1] + f' data-measure-idx="{measure_idx}" style="cursor:pointer;">'
                        measure_idx += 1
                    return result
                svg = re.sub(r'<g[^>]*class="[^"]*measure[^"]*"[^>]*>', add_attr, svg)
                svgs.append(svg)
            html = f'''<style>.score-viewer{{display:grid;grid-template-columns:1.5fr 1fr;gap:15px;height:700px;}}.score-panel{{overflow:auto;border:1px solid #ccc;padding:10px;background:white;}}.detail-panel{{border:1px solid #ccc;border-radius:8px;background:#f8f9fa;padding:15px;overflow-y:auto;}}.measure-highlight{{cursor:pointer;}}.measure-highlight:hover{{filter:brightness(0.85);}}</style><div class="score-viewer"><div class="score-panel" id="score-panel">{"".join(svgs)}</div><div class="detail-panel" id="detail-panel"><h3>Click a measure</h3></div></div><script>(function(){{var mc={json.dumps(mc)};var am={json.dumps(amd)};var pn=['C','C#','D','D#','E','F','F#','G','G#','A','A#','B'];function p2n(p){{return pn[p%12]+(Math.floor(p/12)-1);}}function createPianoWave(ctx){{var real=new Float32Array([0,1.0,0.5,0.35,0.2,0.12,0.08,0.05,0.03]);var imag=new Float32Array(real.length);return ctx.createPeriodicWave(real,imag);}}function show(i){{var m=am[i];if(!m)return;var rp=new Set(m.ref_note_list.map(n=>n.pitch));var pp=new Set(m.perf_note_list.map(n=>n.pitch));var mt=[...rp].filter(p=>pp.has(p));var ms=[...rp].filter(p=>!pp.has(p));var h='<h3>M'+m.number+'</h3><div style="margin:10px 0;"><span style="padding:4px 8px;border-radius:4px;background:'+(m.pitch>=80?'#90EE90':'#FFB6C1')+';">P:'+m.pitch.toFixed(1)+'%</span> <span style="padding:4px 8px;border-radius:4px;background:'+(m.rhythm>=80?'#87CEEB':'#FFB6C1')+';">R:'+m.rhythm.toFixed(1)+'%</span></div>';if(m.brain_state)h+='<div style="background:#e9ecef;padding:5px;border-radius:4px;margin:5px 0;">🧠'+m.brain_state+'</div>';h+='<div style="margin:10px 0;"><button onclick="playRef('+i+')" style="padding:8px 16px;margin:5px;cursor:pointer;border:none;border-radius:4px;background:#4CAF50;color:white;">🔊楽譜</button>';if(m.perf_notes>0)h+='<button onclick="playPerf('+i+')" style="padding:8px 16px;margin:5px;cursor:pointer;border:none;border-radius:4px;background:#f44336;color:white;">🔊演奏</button>';h+='</div><div style="font-size:11px;"><span style="color:green;">✓'+mt.map(p=>p2n(p)).join(',')||'none'+'</span><br><span style="color:red;">✗'+ms.map(p=>p2n(p)).join(',')||'none'+'</span></div>';document.getElementById('detail-panel').innerHTML=h;}}function playRef(i){{var m=am[i];if(!m||!m.ref_note_list.length)return;var ctx=new(window.AudioContext||window.webkitAudioContext)();var pianoWave=createPianoWave(ctx);m.ref_note_list.forEach(n=>{{var o=ctx.createOscillator();var g=ctx.createGain();o.connect(g);g.connect(ctx.destination);o.frequency.value=440*Math.pow(2,(n.pitch-69)/12);o.setPeriodicWave(pianoWave);var s=Math.max(0,n.start-m.start_time);var d=Math.min(n.dur,2.0);g.gain.setValueAtTime(0,ctx.currentTime+s);g.gain.linearRampToValueAtTime(0.35,ctx.currentTime+s+0.008);g.gain.exponentialRampToValueAtTime(0.001,ctx.currentTime+s+d+0.5);o.start(ctx.currentTime+s);o.stop(ctx.currentTime+s+d+0.6);}});}}function playPerf(i){{var m=am[i];if(!m||!m.perf_note_list.length)return;var ctx=new(window.AudioContext||window.webkitAudioContext)();var pianoWave=createPianoWave(ctx);m.perf_note_list.forEach(n=>{{var o=ctx.createOscillator();var g=ctx.createGain();o.connect(g);g.connect(ctx.destination);o.frequency.value=440*Math.pow(2,(n.pitch-69)/12);o.setPeriodicWave(pianoWave);var s=Math.max(0,n.start-m.perf_start);var d=Math.min(n.dur,2.0);g.gain.setValueAtTime(0,ctx.currentTime+s);g.gain.linearRampToValueAtTime(0.3,ctx.currentTime+s+0.008);g.gain.exponentialRampToValueAtTime(0.001,ctx.currentTime+s+d+0.5);o.start(ctx.currentTime+s);o.stop(ctx.currentTime+s+d+0.6);}});}}window.playRef=playRef;window.playPerf=playPerf;setTimeout(function(){{var panel=document.getElementById('score-panel');if(!panel)return;var els=panel.querySelectorAll('[data-measure-idx]');if(!els.length){{els=panel.querySelectorAll('g[class*="measure"]');els.forEach(function(el,idx){{el.setAttribute('data-measure-idx',idx);}});}}els.forEach(function(el){{var idx=parseInt(el.getAttribute('data-measure-idx'));var color=mc[idx]||'#FFF';el.classList.add('measure-highlight');try{{var bbox=el.getBBox();if(bbox.width>0){{var rect=document.createElementNS('http://www.w3.org/2000/svg','rect');rect.setAttribute('x',bbox.x-5);rect.setAttribute('y',bbox.y-5);rect.setAttribute('width',bbox.width+10);rect.setAttribute('height',bbox.height+10);rect.setAttribute('fill',color);rect.setAttribute('opacity','0.4');rect.style.pointerEvents='none';el.insertBefore(rect,el.firstChild);}}}}catch(e){{}}el.addEventListener('click',function(e){{e.stopPropagation();if(idx<am.length)show(idx);}});}});}},500);}})();</script>'''
            display(HTML(html))
        except Exception as e:
            print(f"楽譜表示エラー: {e}")

# -----------------------------
# UI
# -----------------------------
class PianoScorerUI:
    def __init__(self):
        self.params = ScoringParams()
        self.measures, self.ref_notes, self.perf_notes, self.result = [], [], [], None
        self.visualizer = Visualizer()
        self.audio_path = self.midi_path = self.mxl_path = self.recorded_audio = self.brain_csv_path = None
        self.audio_duration = None
        self.is_recording = False
        # 録音時刻を自動記録
        self.recording_start_time: Optional[datetime] = None
        self.recording_end_time: Optional[datetime] = None

    def _receive_audio_callback(self, base64_data, mime_type='audio/webm'):
        import base64, tempfile
        try:
            raw = base64.b64decode(base64_data)
            suffix = '.ogg' if 'ogg' in str(mime_type) else '.webm'
            with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
                f.write(raw)
                self.recorded_audio = f.name
            if self.input_mode.value == '録音':
                self.audio_path = self.recorded_audio
            self.record_status.value = '<span style="color:green;">✓完了</span>'
        except Exception as e:
            self.record_status.value = f'<span style="color:red;">Error:{e}</span>'

    def create_ui(self):
        import ipywidgets as widgets
        from IPython.display import display, HTML

        display(HTML("""<div style="background:linear-gradient(135deg,#1a1a2e 0%,#16213e 100%);padding:20px;border-radius:15px;color:white;margin-bottom:20px;">
            <h1 style="margin:0;">🎹🧠 Piano Performance Scorer v4.4</h1>
            <p style="margin:5px 0 0 0;opacity:0.8;">オフライン脳波統合版（Mind Monitor CSVアップロード）</p>
        </div>"""))

        # ---- 脳波CSV ----
        self.brain_enabled = widgets.Checkbox(value=False, description='🧠脳波CSVを使用')
        self.brain_enabled.observe(self._on_brain_toggle, names='value')
        self.brain_csv_upload = widgets.FileUpload(accept='.csv', multiple=False, description='脳波CSV')
        self.rec_start_input = widgets.Text(placeholder='HH:MM:SS', description='録音開始:', layout=widgets.Layout(width='180px'))
        self.rec_end_input = widgets.Text(placeholder='HH:MM:SS', description='録音終了:', layout=widgets.Layout(width='180px'))
        self.parse_brain_btn = widgets.Button(description='🧠CSV解析', button_style='info', layout=widgets.Layout(width='100px'))
        self.parse_brain_btn.on_click(self._on_parse_brain)
        self.brain_status = widgets.HTML(value='<span style="color:gray;">CSVをアップロード</span>')
        self.brain_controls = widgets.VBox([
            widgets.HBox([widgets.Label("脳波CSV:"), self.brain_csv_upload]),
            widgets.HBox([self.rec_start_input, self.rec_end_input, self.parse_brain_btn]),
            self.brain_status
        ])
        self.brain_controls.layout.display = 'none'
        brain_info = widgets.HTML("""<div style="background:#e8f4f8;padding:10px;border-radius:5px;margin:5px 0;font-size:12px;">
            <b>使い方:</b> Mind Monitorで録音中に脳波を記録 → CSVエクスポート → 録音開始/終了時刻を入力 → CSV解析
        </div>""")
        brain_box = widgets.VBox([widgets.HTML("<h3>🧠脳波CSV</h3>"), self.brain_enabled, self.brain_controls, brain_info])

        # ---- ファイル/録音 ----
        self.input_mode = widgets.RadioButtons(options=['ファイル', '録音'], value='ファイル', layout=widgets.Layout(width='100px'))
        self.input_mode.observe(self._on_input_mode_change, names='value')
        self.audio_upload = widgets.FileUpload(accept='.mp3,.wav,.m4a,.qta,.webm,.ogg,.aac', multiple=False)
        self.midi_upload = widgets.FileUpload(accept='.mid,.midi', multiple=False)
        self.mxl_upload = widgets.FileUpload(accept='.mxl,.xml,.musicxml', multiple=False)
        self.record_btn = widgets.Button(description='🎤録音', button_style='info', layout=widgets.Layout(width='80px'))
        self.record_btn.on_click(self._on_record)
        self.gain_slider = widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description='Gain:', layout=widgets.Layout(width='180px'))
        self.record_status = widgets.HTML(value='<span style="color:gray;">待機</span>')
        self.audio_file_box = widgets.VBox([widgets.Label("音声"), self.audio_upload])
        self.audio_record_box = widgets.VBox([widgets.HBox([self.record_btn, self.record_status]), self.gain_slider])
        self.audio_record_box.layout.display = 'none'
        upload_box = widgets.VBox([
            widgets.HTML("<h3>📁入力</h3>"),
            widgets.HBox([
                widgets.VBox([self.input_mode, self.audio_file_box, self.audio_record_box]),
                widgets.VBox([widgets.Label("MIDI"), self.midi_upload]),
                widgets.VBox([widgets.Label("MXL"), self.mxl_upload]),
            ])
        ])

        # ---- パラメータ ----
        self.pitch_tol = widgets.IntSlider(value=0, min=0, max=3, description='Pitch:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        self.ignore_oct = widgets.Checkbox(value=False, description='Oct無視')
        self.arpeggio = widgets.Checkbox(value=True, description='アルペジオ')
        self.rhythm_tol = widgets.FloatSlider(value=0.1, min=0.05, max=0.3, step=0.01, description='Rhythm:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        self.tempo_tol = widgets.FloatSlider(value=0.1, min=0.05, max=0.3, step=0.05, description='Tempo:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        params_box = widgets.VBox([
            widgets.HTML("<h3>⚙️パラメータ</h3>"),
            widgets.HBox([self.pitch_tol, self.ignore_oct, self.arpeggio]),
            widgets.HBox([self.rhythm_tol, self.tempo_tol]),
        ])

        # ---- ボタン ----
        self.run_btn = widgets.Button(description='🎵採点', button_style='success', layout=widgets.Layout(width='100px', height='40px'))
        self.run_btn.on_click(self._on_run)
        self.play_btn = widgets.Button(description='▶️再生', button_style='info', layout=widgets.Layout(width='80px'))
        self.play_btn.on_click(self._on_play)
        self.reset_rec_btn = widgets.Button(description='🔄録音リセット', button_style='warning', layout=widgets.Layout(width='100px'))
        self.reset_rec_btn.on_click(self._on_reset_recording)
        self.reset_all_btn = widgets.Button(description='🗑️全リセット', button_style='danger', layout=widgets.Layout(width='100px'))
        self.reset_all_btn.on_click(self._on_reset_all)
        btn_box = widgets.HBox([self.run_btn, self.play_btn, self.reset_rec_btn, self.reset_all_btn],
                               layout=widgets.Layout(justify_content='center', margin='15px 0'))

        self.output = widgets.Output()

        # ---- 分析 ----
        self.measure_sel = widgets.IntSlider(value=1, min=1, max=100, description='M#:', layout=widgets.Layout(width='200px'))
        self.show_btn = widgets.Button(description='📊グラフ', button_style='info', layout=widgets.Layout(width='70px'))
        self.show_btn.on_click(self._on_show_measure)
        self.show_detail_btn = widgets.Button(description='📋詳細', button_style='info', layout=widgets.Layout(width='60px'))
        self.show_detail_btn.on_click(self._on_show_detail)
        self.play_ref_btn = widgets.Button(description='🔊楽譜', button_style='success', layout=widgets.Layout(width='60px'))
        self.play_ref_btn.on_click(self._on_play_ref)
        self.play_perf_btn = widgets.Button(description='🔊演奏', button_style='danger', layout=widgets.Layout(width='60px'))
        self.play_perf_btn.on_click(self._on_play_perf)
        self.show_score_btn = widgets.Button(description='🎼楽譜全体', button_style='success', layout=widgets.Layout(width='80px'))
        self.show_score_btn.on_click(self._on_show_score)
        self.show_problems_btn = widgets.Button(description='⚠️問題', button_style='warning', layout=widgets.Layout(width='70px'))
        self.show_problems_btn.on_click(self._on_show_problems)
        self.show_brain_btn = widgets.Button(description='🧠タイムライン', button_style='info', layout=widgets.Layout(width='100px'))
        self.show_brain_btn.on_click(self._on_show_brain)
        self.show_all_btn = widgets.Button(description='📋全表', layout=widgets.Layout(width='60px'))
        self.show_all_btn.on_click(self._on_show_all)

        # 卒論用
        self.thesis_btn = widgets.Button(description='📊卒論分析', button_style='primary', layout=widgets.Layout(width='90px'))
        self.thesis_btn.on_click(self._on_thesis_analysis)
        self.export_btn = widgets.Button(description='💾CSV出力', button_style='primary', layout=widgets.Layout(width='80px'))
        self.export_btn.on_click(self._on_export)

        debug_box = widgets.VBox([
            widgets.HTML("<h3>🔍分析</h3>"),
            widgets.HBox([self.measure_sel, self.show_btn, self.show_detail_btn, self.play_ref_btn, self.play_perf_btn]),
            widgets.HBox([self.show_score_btn, self.show_problems_btn, self.show_brain_btn, self.show_all_btn, self.thesis_btn, self.export_btn]),
        ])
        self.debug_output = widgets.Output()

        try:
            from google.colab import output as colab_output
            colab_output.register_callback('piano.receive_audio', self._receive_audio_callback)
        except: pass

        display(widgets.VBox([brain_box, upload_box, params_box, btn_box, self.output, debug_box, self.debug_output]))

    def _on_brain_toggle(self, c):
        self.brain_controls.layout.display = 'block' if c['new'] else 'none'

    def _on_parse_brain(self, b):
        import tempfile
        with self.output:
            self.output.clear_output()
            if not self.brain_csv_upload.value:
                print("⚠️ 脳波CSVをアップロードしてください")
                return

            # CSVを一時ファイルに保存
            name = list(self.brain_csv_upload.value.keys())[0]
            content = self.brain_csv_upload.value[name]['content']
            with tempfile.NamedTemporaryFile(suffix='.csv', delete=False) as f:
                f.write(content)
                self.brain_csv_path = f.name

            # CSV解析
            if not brain_csv_parser.parse_csv(self.brain_csv_path):
                self.brain_status.value = '<span style="color:red;">CSV解析失敗</span>'
                return

            # 時刻設定（自動入力された値を使用）
            start_str = self.rec_start_input.value.strip()
            end_str = self.rec_end_input.value.strip()

            if not start_str or not end_str:
                # 録音時刻がない場合はCSV全体を使う提案
                csv_start = brain_csv_parser.start_time.strftime("%H:%M:%S")
                csv_end = brain_csv_parser.end_time.strftime("%H:%M:%S")
                self.brain_status.value = f'''<span style="color:orange;">
                    録音時刻を入力してください<br>
                    CSV範囲: {csv_start} - {csv_end}<br>
                    <small>録音ボタンを使うと自動入力されます</small>
                </span>'''
                return

            if not brain_csv_parser.set_recording_time(start_str, end_str):
                self.brain_status.value = '<span style="color:red;">時刻設定失敗</span>'
                return

            # アライメント
            if brain_csv_parser.align_to_recording():
                duration = (brain_csv_parser.recording_end - brain_csv_parser.recording_start).total_seconds()
                self.brain_status.value = f'''<span style="color:green;">
                    ✓ {len(brain_csv_parser.aligned_data)}サンプル準備完了<br>
                    {start_str} - {end_str} ({duration:.1f}秒)
                </span>'''
            else:
                self.brain_status.value = '<span style="color:red;">アライメント失敗（録音時間がCSV範囲外？）</span>'

    def _on_input_mode_change(self, c):
        if c['new'] == 'ファイル':
            self.audio_file_box.layout.display = 'block'
            self.audio_record_box.layout.display = 'none'
        else:
            self.audio_file_box.layout.display = 'none'
            self.audio_record_box.layout.display = 'block'

    def _update_params(self):
        self.params.pitch_tolerance = self.pitch_tol.value
        self.params.ignore_octave = self.ignore_oct.value
        self.params.allow_arpeggio = self.arpeggio.value
        self.params.rhythm_tolerance = self.rhythm_tol.value
        self.params.tempo_tolerance = self.tempo_tol.value

    def _on_record(self, b):
        from IPython.display import display, Javascript
        if not self.is_recording:
            self.is_recording = True
            self.record_btn.description = '⏹️停止'
            self.record_btn.button_style = 'danger'
            self.record_status.value = '<span style="color:red;">●REC</span>'

            # 録音開始時刻を記録
            self.recording_start_time = datetime.now()
            start_str = self.recording_start_time.strftime('%H:%M:%S')
            self.rec_start_input.value = start_str
            print(f"🎤 録音開始: {start_str}")

            g = self.gain_slider.value
            display(Javascript(f"""(async function(){{window.audioChunks=[];const stream=await navigator.mediaDevices.getUserMedia({{audio:true}});const ctx=new AudioContext();const src=ctx.createMediaStreamSource(stream);const gn=ctx.createGain();gn.gain.value={g};const dest=ctx.createMediaStreamDestination();src.connect(gn);gn.connect(dest);const candidates=['audio/webm;codecs=opus','audio/webm','audio/ogg;codecs=opus'];let chosen='';for(const c of candidates){{if(MediaRecorder.isTypeSupported(c)){{chosen=c;break;}}}}window.__recMime=chosen||'audio/webm';window.mediaRecorder=new MediaRecorder(dest.stream,chosen?{{mimeType:chosen}}:{{}});window.originalStream=stream;window.mediaRecorder.ondataavailable=(e)=>{{if(e.data&&e.data.size>0)window.audioChunks.push(e.data);}};window.mediaRecorder.start(1000);}})();"""))
        else:
            self.is_recording = False
            self.record_btn.description = '🎤録音'
            self.record_btn.button_style = 'info'
            self.record_status.value = '<span style="color:blue;">⏳</span>'

            # 録音終了時刻を記録
            self.recording_end_time = datetime.now()
            end_str = self.recording_end_time.strftime('%H:%M:%S')
            self.rec_end_input.value = end_str
            duration = (self.recording_end_time - self.recording_start_time).total_seconds()
            print(f"⏹️ 録音停止: {end_str} (録音時間: {duration:.1f}秒)")

            display(Javascript(r"""(async function(){if(window.mediaRecorder&&window.mediaRecorder.state==='recording'){window.mediaRecorder.stop();window.mediaRecorder.onstop=async()=>{if(window.originalStream)window.originalStream.getTracks().forEach(t=>t.stop());const blob=new Blob(window.audioChunks,{type:window.__recMime||'audio/webm'});const reader=new FileReader();reader.onloadend=function(){const base64=reader.result.split(',')[1];if(typeof google!=='undefined'&&google.colab){google.colab.kernel.invokeFunction('piano.receive_audio',[base64,window.__recMime],{});}};reader.readAsDataURL(blob);};}})();"""))

    def _on_play(self, b):
        from IPython.display import display, Audio
        with self.output:
            if self.recorded_audio: display(Audio(self.recorded_audio, autoplay=True))
            elif self.audio_path: display(Audio(self.audio_path, autoplay=True))
            else: print("再生する音声がありません")

    def _save_files(self):
        import tempfile, subprocess, os
        if self.input_mode.value == 'ファイル' and self.audio_upload.value:
            name = list(self.audio_upload.value.keys())[0]
            content = self.audio_upload.value[name]['content']
            ext = '.' + name.split('.')[-1].lower() if '.' in name else '.wav'
            with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
                f.write(content)
                original_path = f.name

            # .qta, .m4a, .aac は ffmpeg で wav に変換
            if ext in ['.qta', '.m4a', '.aac']:
                print(f"🔄 {ext} → .wav 変換中...")
                wav_path = original_path.replace(ext, '.wav')
                try:
                    result = subprocess.run(
                        ['ffmpeg', '-y', '-i', original_path, '-ar', '44100', '-ac', '1', wav_path],
                        capture_output=True, text=True, timeout=60
                    )
                    if result.returncode == 0 and os.path.exists(wav_path):
                        self.audio_path = wav_path
                        print(f"  ✓ 変換完了: {wav_path}")
                    else:
                        print(f"  ⚠️ ffmpeg変換失敗: {result.stderr}")
                        self.audio_path = original_path
                except Exception as e:
                    print(f"  ⚠️ 変換エラー: {e}")
                    self.audio_path = original_path
            else:
                self.audio_path = original_path
        elif self.input_mode.value == '録音' and self.recorded_audio:
            self.audio_path = self.recorded_audio
        if self.midi_upload.value:
            name = list(self.midi_upload.value.keys())[0]
            with tempfile.NamedTemporaryFile(suffix='.mid', delete=False) as f:
                f.write(self.midi_upload.value[name]['content'])
                self.midi_path = f.name
        if self.mxl_upload.value:
            name = list(self.mxl_upload.value.keys())[0]
            ext = '.' + name.split('.')[-1] if '.' in name else '.mxl'
            with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
                f.write(self.mxl_upload.value[name]['content'])
                self.mxl_path = f.name

    def _on_run(self, b):
        with self.output:
            self.output.clear_output()
            # デバッグ情報
            print(f"[DEBUG] input_mode: {self.input_mode.value}")
            print(f"[DEBUG] audio_upload.value: {bool(self.audio_upload.value)}")
            print(f"[DEBUG] recorded_audio: {self.recorded_audio}")
            print(f"[DEBUG] audio_path: {self.audio_path}")

            has_audio = bool(self.audio_upload.value) or bool(self.recorded_audio)
            if not has_audio: print("⚠️音声をアップロードまたは録音してください"); return
            if not self.midi_upload.value: print("⚠️MIDIをアップロードしてください"); return
            if not self.mxl_upload.value: print("⚠️MXLをアップロードしてください"); return
            self._update_params()
            try:
                self._save_files()
                if not self.audio_path: print("⚠️音声ファイルがありません"); return
                parser = ReferenceParser(self.midi_path, self.mxl_path)
                self.measures, self.ref_notes = parser.parse()
                if not self.measures: print("❌参照データ解析失敗"); return
                transcriber = AudioTranscriber(use_gpu=True)
                self.perf_notes, self.audio_duration = transcriber.transcribe(self.audio_path)
                aligner = DTWAligner(self.params)
                self.measures = aligner.align(self.perf_notes, self.ref_notes, self.measures)

                # 脳波統合
                brain_parser = brain_csv_parser if (self.brain_enabled.value and brain_csv_parser.aligned_data) else None
                scorer = ScoringEngine(self.params, brain_parser)
                self.result = scorer.score_all(self.measures, self.audio_duration)

                self.visualizer.show_summary(self.result)
                self.visualizer.show_table(self.result, only_played=True)
                self.measure_sel.max = len(self.measures)
            except Exception as e:
                print(f"❌Error: {e}")
                import traceback; traceback.print_exc()

    def _on_reset_recording(self, b):
        from IPython.display import display, Javascript
        self.audio_upload.value.clear()
        self.recorded_audio = None; self.audio_path = None; self.is_recording = False
        self.record_btn.description = '🎤録音'; self.record_btn.button_style = 'info'
        self.record_status.value = '<span style="color:gray;">待機</span>'
        display(Javascript('window.audioChunks=[];'))
        with self.output: self.output.clear_output(); print("✓ 録音リセット")

    def _on_reset_all(self, b):
        from IPython.display import display, Javascript
        self.audio_upload.value.clear(); self.midi_upload.value.clear(); self.mxl_upload.value.clear()
        self.brain_csv_upload.value.clear()
        self.audio_path = self.midi_path = self.mxl_path = self.recorded_audio = self.brain_csv_path = None
        self.measures, self.ref_notes, self.perf_notes, self.result = [], [], [], None
        self.is_recording = False; self.record_btn.description = '🎤録音'; self.record_btn.button_style = 'info'
        self.record_status.value = '<span style="color:gray;">待機</span>'
        self.brain_status.value = '<span style="color:gray;">CSVをアップロード</span>'
        brain_csv_parser.data = []; brain_csv_parser.aligned_data = []
        self.output.clear_output(); self.debug_output.clear_output()
        display(Javascript('window.audioChunks=[];'))
        with self.output: print("✓ 全リセット完了")

    def _on_show_measure(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.show_measure_comparison(self.measures[idx])

    def _on_show_detail(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.show_measure_detail(self.measures[idx])

    def _on_play_ref(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.play_measure_audio(self.measures[idx], play_ref=True)

    def _on_play_perf(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.play_measure_audio(self.measures[idx], play_ref=False)

    def _on_show_score(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures or not self.mxl_path: print("採点を実行してください"); return
            print("🎼 楽譜読み込み中...")
            self.visualizer.show_score_with_click(self.mxl_path, self.measures)

    def _on_show_problems(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            self.visualizer.show_problem_measures(self.measures)

    def _on_show_brain(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            self.visualizer.show_brain_timeline(self.measures)

    def _on_show_all(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.result: print("採点を実行してください"); return
            self.visualizer.show_table(self.result, only_played=False)

    def _on_thesis_analysis(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures or not self.result: print("採点を実行してください"); return
            try:
                self.visualizer.show_thesis_analysis(self.measures, self.result)
            except Exception as e:
                print(f"分析エラー: {e}")
                import traceback; traceback.print_exc()

    def _on_export(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures or not self.result: print("採点を実行してください"); return
            try:
                self.visualizer.export_thesis_data(self.measures, self.result)
            except Exception as e:
                print(f"エクスポートエラー: {e}")
                import traceback; traceback.print_exc()

def main():
    print("🎹🧠 Piano Performance Scorer v4.4 (Offline Brain Wave Integration)")
    print("=" * 60)
    install_dependencies()
    ui = PianoScorerUI()
    ui.create_ui()
    return ui

if __name__ == "__main__":
    ui = main()

13:17:11 13:05:42

In [ ]:
from google.colab import files

# 1. 先にファイルをアップロード
print("=== ファイルをアップロード ===")
uploaded = files.upload()

# アップロードしたファイル名を確認
for name in uploaded.keys():
    print(f"  - {name}")

In [ ]:
# 2. パスを直接指定して採点
audio_path = "moonlight20260203_1.mp3"  # アップロードしたファイル名
midi_path = "Sonate_No._14_Moonlight_3rd_Movement (1).mid"              # MIDIファイル名
mxl_path = "Sonate_No._14_Moonlight_3rd_Movement (1).mxl"               # MXLファイル名
brain_csv = "mindMonitor_2026-02-03--13-05-42.csv" # 脳波CSV（オプション）

# 脳波解析
brain_csv_parser.parse_csv(brain_csv)
brain_csv_parser.set_recording_time("15:35:17", "15:46:47")  # 録音時刻
brain_csv_parser.align_to_recording()

# 採点実行
parser = ReferenceParser(midi_path, mxl_path)
measures, ref_notes = parser.parse()

transcriber = AudioTranscriber(use_gpu=True)
perf_notes, audio_duration = transcriber.transcribe(audio_path)

aligner = DTWAligner(ScoringParams())
measures = aligner.align(perf_notes, ref_notes, measures)

scorer = ScoringEngine(ScoringParams(), brain_csv_parser)
result = scorer.score_all(measures, audio_duration)

# 結果表示
visualizer = Visualizer()
visualizer.show_summary(result)
visualizer.show_table(result)

In [ ]:
from google.colab import files
import librosa

print("mp3をアップロード")
uploaded = files.upload()

for name in uploaded.keys():
    print(f"ファイル: {name}")
    print(f"サイズ: {len(uploaded[name]) / 1024:.1f} KB")

    with open(name, 'wb') as f:
        f.write(uploaded[name])

    try:
        y, sr = librosa.load(name, sr=None, duration=10)  # 最初の10秒だけ
        print(f"✓ 読み込み成功: {len(y)/sr:.1f}秒, サンプルレート: {sr}")
    except Exception as e:
        print(f"✗ 読み込み失敗: {e}")

In [ ]:
import librosa

# 1. ファイル読み込みテスト
print("1. ファイル読み込み...")
y, sr = librosa.load("moonlight20260203_1.mp3", sr=None)
duration = len(y) / sr
print(f"   ✓ 長さ: {duration:.1f}秒 ({duration/60:.1f}分)")

# 2. Transkun テスト（これが時間かかる）
print("2. Transkun 音声認識中...（数分かかります）")
import subprocess
subprocess.run(['python3', '-m', 'transkun.transcribe',
                'moonlight20260203_1.mp3', 'test_output.mid',
                '--device', 'cuda'],
               timeout=600)
print("   ✓ 完了")

In [ ]:
# ==============================================================================
# 🎹🧠 Piano Performance Scorer v4.4 - Offline Brain Wave Integration
# ==============================================================================
# Mind MonitorのCSVをアップロードして、録音と時間軸を合わせて脳波を小節に割り当て
#
# 使い方:
# 1. Mind Monitorで脳波を記録（CSVエクスポート）
# 2. 録音開始時刻と終了時刻をメモ（Mind Monitorの画面で確認）
# 3. Colabで録音 or 音声ファイルをアップロード
# 4. 脳波CSVをアップロード + 録音開始/終了時刻を入力
# 5. 採点実行 → 脳波と演奏が時間軸で統合される
# ==============================================================================

import numpy as np
import warnings
warnings.filterwarnings('ignore')

def install_dependencies():
    import subprocess, sys
    packages = [
        'pretty_midi', 'music21', 'librosa', 'matplotlib', 'pandas',
        'ipywidgets', 'transkun', 'dtw-python', 'verovio'
    ]
    for pkg in packages:
        try:
            __import__(pkg.replace('-', '_').replace('dtw-python', 'dtw'))
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    print("✓ 依存ライブラリ準備完了")

from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any
from datetime import datetime, timedelta
import threading
import time

@dataclass
class NoteEvent:
    pitch: int
    start_time: float
    end_time: float
    velocity: int = 64
    matched: bool = False
    @property
    def duration(self): return self.end_time - self.start_time
    def pitch_name(self):
        names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
        return f"{names[self.pitch % 12]}{(self.pitch // 12) - 1}"

@dataclass
class MeasureData:
    number: int
    index: int
    start_time: float
    end_time: float
    tempo: float
    time_signature: Tuple[int, int]
    ref_notes: List[NoteEvent] = field(default_factory=list)
    perf_notes: List[NoteEvent] = field(default_factory=list)
    pitch_score: float = 0.0
    rhythm_score: float = 0.0
    tempo_score: float = 0.0
    perf_start_time: float = 0.0
    perf_end_time: float = 0.0
    brain_state: str = ""
    brain_metrics: Dict = field(default_factory=dict)
    @property
    def duration(self): return self.end_time - self.start_time

@dataclass
class ScoringParams:
    pitch_tolerance: int = 0
    ignore_octave: bool = False
    chord_time_window: float = 0.05
    allow_arpeggio: bool = True
    arpeggio_max_time: float = 0.2
    rhythm_tolerance: float = 0.1
    tempo_tolerance: float = 0.1

# -----------------------------
# Brain Wave CSV Parser
# -----------------------------
class BrainWaveCSVParser:
    """Mind MonitorのCSVを解析"""

    def __init__(self):
        self.data = []
        self.start_time = None
        self.end_time = None
        self.recording_start = None  # 録音開始時刻（絶対時刻）
        self.recording_end = None    # 録音終了時刻（絶対時刻）
        self.aligned_data = []       # 録音に合わせた相対時刻データ

    def parse_csv(self, csv_path: str) -> bool:
        """Mind Monitor CSVを解析"""
        import pandas as pd

        try:
            df = pd.read_csv(csv_path)
            print(f"📊 CSV読み込み: {len(df)}行")
            print(f"   列: {list(df.columns)[:10]}...")

            # タイムスタンプ列を探す
            time_col = None
            for col in ['TimeStamp', 'Timestamp', 'timestamp', 'Time', 'time']:
                if col in df.columns:
                    time_col = col
                    break

            if time_col is None:
                print("⚠️ タイムスタンプ列が見つかりません")
                return False

            # α/β/θ列を探す
            alpha_cols = [c for c in df.columns if 'Alpha' in c or 'alpha' in c]
            beta_cols = [c for c in df.columns if 'Beta' in c or 'beta' in c]
            theta_cols = [c for c in df.columns if 'Theta' in c or 'theta' in c]

            print(f"   Alpha列: {alpha_cols[:2]}")
            print(f"   Beta列: {beta_cols[:2]}")
            print(f"   Theta列: {theta_cols[:2]}")

            self.data = []
            for _, row in df.iterrows():
                try:
                    # タイムスタンプをパース
                    ts_str = str(row[time_col])
                    # 複数のフォーマットを試す
                    ts = None
                    for fmt in ['%Y-%m-%d %H:%M:%S.%f', '%Y-%m-%d %H:%M:%S',
                               '%H:%M:%S.%f', '%H:%M:%S',
                               '%Y/%m/%d %H:%M:%S.%f', '%Y/%m/%d %H:%M:%S']:
                        try:
                            ts = datetime.strptime(ts_str, fmt)
                            break
                        except:
                            continue

                    if ts is None:
                        # Unix timestampかも
                        try:
                            ts = datetime.fromtimestamp(float(ts_str))
                        except:
                            continue

                    # α/β/θの平均を計算
                    def get_avg(cols):
                        vals = [float(row[c]) for c in cols if c in row and pd.notna(row[c])]
                        return sum(vals) / len(vals) if vals else 0.0

                    alpha = get_avg(alpha_cols)
                    beta = get_avg(beta_cols)
                    theta = get_avg(theta_cols)

                    self.data.append({
                        'timestamp': ts,
                        'alpha': alpha,
                        'beta': beta,
                        'theta': theta
                    })
                except Exception as e:
                    continue

            if not self.data:
                print("⚠️ 有効なデータがありません")
                return False

            self.start_time = self.data[0]['timestamp']
            self.end_time = self.data[-1]['timestamp']
            duration = (self.end_time - self.start_time).total_seconds()

            print(f"✓ 脳波データ: {len(self.data)}サンプル")
            print(f"   期間: {self.start_time.strftime('%H:%M:%S')} - {self.end_time.strftime('%H:%M:%S')} ({duration:.1f}秒)")

            return True

        except Exception as e:
            print(f"❌ CSV解析エラー: {e}")
            import traceback
            traceback.print_exc()
            return False

    def set_recording_time(self, start_str: str, end_str: str) -> bool:
        """録音の開始/終了時刻を設定（HH:MM:SS形式）"""
        try:
            # 今日の日付を使用
            base_date = self.start_time.date() if self.start_time else datetime.now().date()

            # 時刻をパース
            for fmt in ['%H:%M:%S.%f', '%H:%M:%S', '%H:%M']:
                try:
                    start_time = datetime.strptime(start_str, fmt).time()
                    break
                except:
                    continue
            else:
                print(f"⚠️ 開始時刻のフォーマットエラー: {start_str}")
                return False

            for fmt in ['%H:%M:%S.%f', '%H:%M:%S', '%H:%M']:
                try:
                    end_time = datetime.strptime(end_str, fmt).time()
                    break
                except:
                    continue
            else:
                print(f"⚠️ 終了時刻のフォーマットエラー: {end_str}")
                return False

            self.recording_start = datetime.combine(base_date, start_time)
            self.recording_end = datetime.combine(base_date, end_time)

            duration = (self.recording_end - self.recording_start).total_seconds()
            print(f"✓ 録音時間設定: {start_str} - {end_str} ({duration:.1f}秒)")

            return True

        except Exception as e:
            print(f"❌ 時刻設定エラー: {e}")
            return False

    def align_to_recording(self) -> bool:
        """脳波データを録音時間に合わせて相対時刻に変換（Z-score方式で状態判定）"""
        if not self.data or not self.recording_start or not self.recording_end:
            print("⚠️ データまたは録音時間が設定されていません")
            return False

        self.aligned_data = []
        rec_duration = (self.recording_end - self.recording_start).total_seconds()

        # まず録音範囲内のデータを抽出して相対値を計算
        temp_data = []
        for d in self.data:
            ts = d['timestamp']
            if self.recording_start <= ts <= self.recording_end:
                alpha, beta, theta = d['alpha'], d['beta'], d['theta']
                total = alpha + beta + theta
                if total > 0:
                    a_rel = alpha / total
                    b_rel = beta / total
                    t_rel = theta / total
                    engagement = b_rel / (a_rel + t_rel + 1e-9)
                    temp_data.append({
                        'timestamp': ts,
                        'relative_time': (ts - self.recording_start).total_seconds(),
                        'alpha': alpha, 'beta': beta, 'theta': theta,
                        'alpha_rel': a_rel, 'beta_rel': b_rel, 'theta_rel': t_rel,
                        'engagement': engagement
                    })

        if not temp_data:
            print("⚠️ 録音範囲内にデータがありません")
            return False

        # ベースライン統計を計算（Z-score用）
        import statistics
        alpha_vals = [d['alpha_rel'] for d in temp_data]
        eng_vals = [d['engagement'] for d in temp_data]

        alpha_mean = statistics.fmean(alpha_vals)
        alpha_std = statistics.pstdev(alpha_vals) or 0.01
        eng_mean = statistics.fmean(eng_vals)
        eng_std = statistics.pstdev(eng_vals) or 0.01

        print(f"   ベースライン: α相対={alpha_mean:.3f}±{alpha_std:.3f}, eng={eng_mean:.3f}±{eng_std:.3f}")

        # Z-score方式で状態判定
        focus_count = relax_count = neutral_count = 0
        for d in temp_data:
            z_alpha = (d['alpha_rel'] - alpha_mean) / alpha_std
            z_eng = (d['engagement'] - eng_mean) / eng_std

            # Z-score判定：平均からの偏差で判断
            if z_eng > 0.5 and z_alpha < 0:
                state = 'FOCUSED'
                focus_count += 1
            elif z_alpha > 0.5 and z_eng < 0:
                state = 'RELAXED'
                relax_count += 1
            else:
                state = 'NEUTRAL'
                neutral_count += 1

            self.aligned_data.append({
                'relative_time': d['relative_time'],
                'alpha': d['alpha'], 'beta': d['beta'], 'theta': d['theta'],
                'alpha_rel': d['alpha_rel'], 'beta_rel': d['beta_rel'], 'theta_rel': d['theta_rel'],
                'engagement': d['engagement'],
                'z_alpha': z_alpha, 'z_eng': z_eng,
                'state': state
            })

        print(f"✓ アライメント完了: {len(self.aligned_data)}サンプル（{rec_duration:.1f}秒間）")
        print(f"   状態分布: FOCUSED={focus_count} ({focus_count/len(self.aligned_data)*100:.1f}%), RELAXED={relax_count} ({relax_count/len(self.aligned_data)*100:.1f}%), NEUTRAL={neutral_count} ({neutral_count/len(self.aligned_data)*100:.1f}%)")
        return len(self.aligned_data) > 0

    def get_states_for_time_range(self, start_sec: float, end_sec: float) -> Dict[str, Any]:
        """指定時間範囲の脳波状態を取得"""
        if not self.aligned_data:
            return {'available': False, 'dominant_state': 'NO_DATA', 'count': 0}

        states_in_range = [
            d for d in self.aligned_data
            if start_sec <= d['relative_time'] <= end_sec
        ]

        if not states_in_range:
            return {'available': True, 'dominant_state': 'NO_DATA', 'count': 0, 'focus_ratio': 0.0}

        counts = {'FOCUSED': 0, 'RELAXED': 0, 'NEUTRAL': 0, 'NO_DATA': 0}
        alpha_sum, beta_sum, theta_sum = 0.0, 0.0, 0.0

        for s in states_in_range:
            st = s.get('state', 'NEUTRAL')
            if st in counts:
                counts[st] += 1
            alpha_sum += s.get('alpha_rel', 0)
            beta_sum += s.get('beta_rel', 0)
            theta_sum += s.get('theta_rel', 0)

        total = len(states_in_range)
        dominant = max(counts, key=counts.get)

        return {
            'available': True,
            'dominant_state': dominant,
            'count': total,
            'focus_ratio': counts['FOCUSED'] / total if total > 0 else 0.0,
            'relax_ratio': counts['RELAXED'] / total if total > 0 else 0.0,
            'avg_alpha': alpha_sum / total if total > 0 else 0,
            'avg_beta': beta_sum / total if total > 0 else 0,
            'avg_theta': theta_sum / total if total > 0 else 0,
            'state_counts': counts
        }

    def get_session_summary(self) -> Dict[str, Any]:
        """セッション全体のサマリー"""
        if not self.aligned_data:
            return {'available': False}

        counts = {'FOCUSED': 0, 'RELAXED': 0, 'NEUTRAL': 0, 'NO_DATA': 0}
        for d in self.aligned_data:
            st = d.get('state', 'NEUTRAL')
            if st in counts:
                counts[st] += 1

        total = len(self.aligned_data)
        duration = self.aligned_data[-1]['relative_time'] if self.aligned_data else 0

        return {
            'available': True,
            'total_samples': total,
            'duration': duration,
            'focus_ratio': counts['FOCUSED'] / total if total > 0 else 0.0,
            'relax_ratio': counts['RELAXED'] / total if total > 0 else 0.0,
            'neutral_ratio': counts['NEUTRAL'] / total if total > 0 else 0.0,
            'state_counts': counts
        }

brain_csv_parser = BrainWaveCSVParser()

# -----------------------------
# Reference Parser
# -----------------------------
class ReferenceParser:
    def __init__(self, midi_path, mxl_path):
        self.midi_path, self.mxl_path = midi_path, mxl_path

    def parse(self):
        print("📖 参照データを解析中...")
        midi_notes, midi_dur = self._parse_midi()
        print(f"  ✓ MIDI: {len(midi_notes)}音符, {midi_dur:.1f}秒")
        mxl_measures = self._parse_mxl()
        print(f"  ✓ MXL: {len(mxl_measures)}小節")
        measures = self._create_measures(midi_notes, mxl_measures, midi_dur)
        return measures, midi_notes

    def _parse_midi(self):
        import pretty_midi
        pm = pretty_midi.PrettyMIDI(self.midi_path)
        notes = []
        for inst in pm.instruments:
            if not inst.is_drum:
                for n in inst.notes:
                    notes.append(NoteEvent(pitch=n.pitch, start_time=n.start, end_time=n.end, velocity=n.velocity))
        notes.sort(key=lambda n: (n.start_time, n.pitch))
        return notes, pm.get_end_time()

    def _parse_mxl(self):
        from music21 import converter, tempo as m21tempo, meter
        score = converter.parse(self.mxl_path)
        try:
            score = score.expandRepeats()
        except:
            pass
        measures, current_tempo, current_ts = [], 120.0, (4, 4)
        for t in score.flatten().getElementsByClass(m21tempo.MetronomeMark):
            if t.number:
                current_tempo = float(t.number)
                break
        part = score.parts[0] if score.parts else score
        for m in part.getElementsByClass('Measure'):
            for t in m.getElementsByClass(m21tempo.MetronomeMark):
                if t.number:
                    current_tempo = float(t.number)
            for ts in m.getElementsByClass(meter.TimeSignature):
                if ts.numerator:
                    current_ts = (ts.numerator, ts.denominator)
            measures.append({
                'number': m.measureNumber,
                'tempo': current_tempo or 120,
                'time_signature': current_ts,
                'quarter_length': float(m.quarterLength) if m.quarterLength else 4.0
            })
        return measures

    def _create_measures(self, midi_notes, mxl_measures, midi_dur):
        if not mxl_measures:
            return []
        measures = []
        total_ql = sum(m['quarter_length'] for m in mxl_measures)
        midi_span = (midi_notes[-1].start_time - midi_notes[0].start_time) if midi_notes else midi_dur
        midi_offset = midi_notes[0].start_time if midi_notes else 0
        current_time = midi_offset
        for i, mxl in enumerate(mxl_measures):
            ql = mxl['quarter_length']
            dur = (ql / total_ql) * midi_span if total_ql > 0 else 2.0
            end_time = current_time + dur
            ref_notes = [
                NoteEvent(pitch=n.pitch, start_time=n.start_time, end_time=n.end_time, velocity=n.velocity)
                for n in midi_notes
                if current_time - 0.05 <= n.start_time < end_time + 0.05
            ]
            measures.append(MeasureData(
                number=mxl['number'], index=i, start_time=current_time, end_time=end_time,
                tempo=mxl['tempo'], time_signature=mxl['time_signature'], ref_notes=ref_notes
            ))
            current_time = end_time
        return measures

# -----------------------------
# Audio Transcriber
# -----------------------------
class AudioTranscriber:
    def __init__(self, use_gpu=True):
        self.use_gpu = use_gpu

    def transcribe(self, audio_path):
        import subprocess, tempfile, os, pretty_midi, librosa
        print("🎵 音声を解析中...")
        device = 'cpu'
        if self.use_gpu:
            try:
                import torch
                if torch.cuda.is_available():
                    device = 'cuda'
                    print("  ✓ CUDA使用")
            except:
                pass
        y, sr = librosa.load(audio_path, sr=None)
        duration = len(y) / sr
        print(f"  - 音声長: {duration:.1f}秒")
        with tempfile.TemporaryDirectory() as td:
            out = os.path.join(td, "out.mid")
            subprocess.run(
                ['python3', '-m', 'transkun.transcribe', audio_path, out, '--device', device],
                check=True, capture_output=True, timeout=600
            )
            pm = pretty_midi.PrettyMIDI(out)
            notes = [
                NoteEvent(pitch=n.pitch, start_time=n.start, end_time=n.end, velocity=n.velocity)
                for inst in pm.instruments if not inst.is_drum for n in inst.notes
            ]
            notes.sort(key=lambda n: (n.start_time, n.pitch))
            print(f"  ✓ {len(notes)}音符を認識")
            return notes, duration

# -----------------------------
# DTW Aligner
# -----------------------------
class DTWAligner:
    def __init__(self, params):
        self.params = params

    def align(self, perf_notes, ref_notes, measures):
        print("🔗 アライメント中...")
        if not perf_notes or not ref_notes:
            return measures
        try:
            from dtw import dtw
            pp = np.array([n.pitch for n in perf_notes]).reshape(-1, 1)
            rp = np.array([n.pitch for n in ref_notes]).reshape(-1, 1)
            def dist(x, y):
                d = abs(x[0] - y[0])
                return d/12*2 if d % 12 == 0 and d > 0 else 0 if d <= self.params.pitch_tolerance else d
            alignment = dtw(pp, rp, dist_method=dist, keep_internals=True,
                          step_pattern='symmetric2', window_type='sakoechiba',
                          window_args={'window_size': 400})
            path = list(zip(alignment.index1, alignment.index2))
        except:
            path = [(i, min(i, len(ref_notes)-1)) for i in range(len(perf_notes))]

        pairs = [(perf_notes[p].start_time, ref_notes[r].start_time)
                 for p, r in path if p < len(perf_notes) and r < len(ref_notes)]
        if not pairs:
            return measures

        pt = np.array([p[0] for p in pairs])
        rt = np.array([p[1] for p in pairs])

        def time_map(t):
            if t <= pt[0]: return rt[0]
            if t >= pt[-1]: return rt[-1]
            idx = np.searchsorted(pt, t)
            if idx == 0: return rt[0]
            ratio = (t - pt[idx-1]) / (pt[idx] - pt[idx-1]) if pt[idx] != pt[idx-1] else 0
            return rt[idx-1] + ratio * (rt[idx] - rt[idx-1])

        for note in perf_notes:
            ref_t = time_map(note.start_time)
            for m in measures:
                if m.start_time <= ref_t < m.end_time:
                    m.perf_notes.append(NoteEvent(
                        pitch=note.pitch, start_time=note.start_time,
                        end_time=note.end_time, velocity=note.velocity
                    ))
                    if not m.perf_start_time or note.start_time < m.perf_start_time:
                        m.perf_start_time = note.start_time
                    if note.start_time > m.perf_end_time:
                        m.perf_end_time = note.start_time
                    break

        print(f"  ✓ {sum(len(m.perf_notes) for m in measures)}/{len(perf_notes)}音符をマッピング")
        return measures

# -----------------------------
# Scoring Engine
# -----------------------------
class ScoringEngine:
    def __init__(self, params, brain_parser=None):
        self.params = params
        self.brain_parser = brain_parser

    def score_all(self, measures, audio_duration=None):
        print("📊 採点中...")

        all_perf = [n for m in measures for n in m.perf_notes]
        if all_perf:
            perf_start = min(n.start_time for n in all_perf)
            perf_end = max(n.end_time for n in all_perf)
            perf_duration = perf_end - perf_start
        else:
            perf_start, perf_end, perf_duration = 0, 0, 0

        brain_duration = None
        if self.brain_parser and self.brain_parser.aligned_data:
            brain_duration = self.brain_parser.aligned_data[-1]['relative_time']
            print(f"  🧠 脳波: {len(self.brain_parser.aligned_data)}サンプル, {brain_duration:.1f}秒")

        for m in measures:
            m.pitch_score = self._score_pitch(m)
            m.rhythm_score = self._score_rhythm(m)
            m.tempo_score = self._score_tempo(m)

            # 脳波割り当て
            if self.brain_parser and brain_duration and brain_duration > 0 and m.perf_notes:
                if perf_duration > 0:
                    m_start = m.perf_start_time if m.perf_start_time else min(n.start_time for n in m.perf_notes)
                    m_end = m.perf_end_time if m.perf_end_time else max(n.end_time for n in m.perf_notes)

                    # 演奏時刻を脳波時刻に変換
                    brain_start = ((m_start - perf_start) / perf_duration) * brain_duration
                    brain_end = ((m_end - perf_start) / perf_duration) * brain_duration

                    summary = self.brain_parser.get_states_for_time_range(brain_start, brain_end)
                    m.brain_state = summary.get('dominant_state', '')
                    m.brain_metrics = {
                        'focus_ratio': summary.get('focus_ratio', 0.0),
                        'relax_ratio': summary.get('relax_ratio', 0.0),
                        'count': summary.get('count', 0),
                    }

        result = self._calc_total(measures)
        print(f"  ✓ 総合スコア: {result['total_score']:.1f}")

        if self.brain_parser and self.brain_parser.aligned_data:
            result['brain_summary'] = self.brain_parser.get_session_summary()

        return result

    def _score_pitch(self, m):
        if not m.ref_notes: return 1.0 if not m.perf_notes else 0.0
        if not m.perf_notes: return 0.0
        ref_groups = self._group_by_time(m.ref_notes, self.params.chord_time_window)
        perf_groups = self._group_by_time(m.perf_notes, self.params.arpeggio_max_time if self.params.allow_arpeggio else self.params.chord_time_window)
        total = sum(len(g['pitches']) for g in ref_groups)
        pool = set(p for g in perf_groups for p in g['pitches'])
        matched = sum(1 for g in ref_groups for rp in g['pitches'] if rp in pool or any(self._match(rp, pp) for pp in pool))
        return matched / total if total > 0 else 1.0

    def _group_by_time(self, notes, window):
        if not notes: return []
        notes = sorted(notes, key=lambda n: n.start_time)
        groups = [{'time': notes[0].start_time, 'pitches': [notes[0].pitch]}]
        for n in notes[1:]:
            if n.start_time - groups[-1]['time'] <= window:
                groups[-1]['pitches'].append(n.pitch)
            else:
                groups.append({'time': n.start_time, 'pitches': [n.pitch]})
        return groups

    def _match(self, rp, pp):
        d = abs(rp - pp)
        return d == 0 or d <= self.params.pitch_tolerance or (self.params.ignore_octave and d % 12 == 0)

    def _score_rhythm(self, m):
        if not m.ref_notes or not m.perf_notes: return 1.0 if not m.ref_notes else 0.0
        def pos(notes, s, e):
            d = e - s if e > s else 0.1
            return [(n.start_time - s) / d for n in notes]
        ref_pos = pos(m.ref_notes, m.start_time, m.end_time)
        perf_pos = pos(m.perf_notes, m.perf_start_time, m.perf_end_time) if m.perf_end_time > m.perf_start_time else pos(m.perf_notes, m.perf_notes[0].start_time, m.perf_notes[-1].start_time + 0.1)
        tol = self.params.rhythm_tolerance
        return sum(1 for rp in ref_pos if any(abs(rp - pp) <= tol for pp in perf_pos)) / len(ref_pos) if ref_pos else 1.0

    def _score_tempo(self, m):
        if not m.perf_notes or len(m.perf_notes) < 2: return 1.0
        pd, rd = m.perf_end_time - m.perf_start_time, m.duration
        if pd <= 0 or rd <= 0: return 1.0
        r = pd / rd
        return 1.0 if 1 - self.params.tempo_tolerance <= r <= 1 + self.params.tempo_tolerance else max(0, 1 - (abs(r - 1) - self.params.tempo_tolerance) * 2)

    def _calc_total(self, measures):
        played = [m for m in measures if m.perf_notes]
        if not played:
            return {'total_score': 0, 'pitch_score': 0, 'rhythm_score': 0, 'tempo_score': 0, 'measures_played': 0, 'measures_total': len(measures), 'measure_details': []}
        ap, ar, at = np.mean([m.pitch_score for m in played]), np.mean([m.rhythm_score for m in played]), np.mean([m.tempo_score for m in played])
        return {
            'total_score': (ap * 0.5 + ar * 0.3 + at * 0.2) * 100, 'pitch_score': ap * 100, 'rhythm_score': ar * 100, 'tempo_score': at * 100,
            'measures_played': len(played), 'measures_total': len(measures),
            'start_measure': measures[min(m.index for m in played)].number, 'end_measure': measures[max(m.index for m in played)].number,
            'measure_details': [{'number': m.number, 'index': m.index, 'pitch': m.pitch_score*100, 'rhythm': m.rhythm_score*100, 'tempo': m.tempo_score*100, 'ref_notes': len(m.ref_notes), 'perf_notes': len(m.perf_notes), 'brain_state': m.brain_state, 'brain_metrics': m.brain_metrics} for m in measures]
        }

# -----------------------------
# Visualizer
# -----------------------------
class Visualizer:
    def show_summary(self, result):
        from IPython.display import display, HTML
        r = result
        brain_html = ""
        if 'brain_summary' in r and r['brain_summary'].get('available'):
            bs = r['brain_summary']
            brain_html = f"""<div style="margin-top:15px;padding:10px;background:rgba(255,255,255,0.1);border-radius:8px;"><div style="font-size:14px;margin-bottom:5px;">🧠 脳波 ({bs['total_samples']}サンプル, {bs.get('duration', 0):.1f}秒)</div><div style="display:flex;gap:15px;flex-wrap:wrap;"><span>🟢集中: {bs['focus_ratio']*100:.0f}%</span><span>🔵リラックス: {bs['relax_ratio']*100:.0f}%</span><span>⚪中立: {bs['neutral_ratio']*100:.0f}%</span></div></div>"""
        html = f"""<div style="background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);padding:20px;border-radius:15px;color:white;margin:10px 0;"><h2 style="margin:0 0 15px 0;">🎹 採点結果</h2><div style="display:flex;justify-content:space-around;flex-wrap:wrap;"><div style="text-align:center;padding:10px;"><div style="font-size:36px;font-weight:bold;">{r['total_score']:.1f}</div><div>総合</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['pitch_score']:.1f}</div><div>🎵ピッチ</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['rhythm_score']:.1f}</div><div>🥁リズム</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['tempo_score']:.1f}</div><div>⏱️テンポ</div></div></div><div style="margin-top:10px;">演奏: {r['measures_played']}/{r['measures_total']}小節</div>{brain_html}</div>"""
        display(HTML(html))

    def show_table(self, result, only_played=True):
        import pandas as pd
        from IPython.display import display
        df = pd.DataFrame(result.get('measure_details', []))
        if df.empty: return
        if only_played: df = df[df['perf_notes'] > 0]
        has_brain = any(d.get('brain_state') for d in result.get('measure_details', []))
        if has_brain:
            df['brain'] = df.apply(lambda r: (r['brain_state'][:3] if r.get('brain_state') else '-'), axis=1)
            df = df[['number', 'index', 'pitch', 'rhythm', 'tempo', 'ref_notes', 'perf_notes', 'brain']]
            df.columns = ['小節', '通し', 'ピッチ', 'リズム', 'テンポ', '楽譜', '演奏', '脳波']
        else:
            df = df[['number', 'index', 'pitch', 'rhythm', 'tempo', 'ref_notes', 'perf_notes']]
            df.columns = ['小節', '通し', 'ピッチ', 'リズム', 'テンポ', '楽譜', '演奏']
        def color(v):
            try:
                v = float(v)
                return 'background-color:#90EE90' if v >= 80 else 'background-color:#FFE4B5' if v >= 50 else 'background-color:#FFB6C1' if v > 0 else ''
            except: return ''
        display(df.style.applymap(color, subset=['ピッチ', 'リズム', 'テンポ']))

    def show_measure_detail(self, m):
        """小節の詳細情報をテキストで表示（音符リスト付き）"""
        from IPython.display import display, HTML
        def p2n(p):
            names = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
            return f"{names[p%12]}{p//12-1}"

        ref_pitches = sorted(set(n.pitch for n in m.ref_notes))
        perf_pitches = sorted(set(n.pitch for n in m.perf_notes))
        matched = set(ref_pitches) & set(perf_pitches)
        missing = set(ref_pitches) - set(perf_pitches)
        extra = set(perf_pitches) - set(ref_pitches)

        brain_html = ""
        if m.brain_state:
            metrics = m.brain_metrics or {}
            brain_html = f"""<div style="background:#e9ecef;padding:10px;border-radius:4px;margin:10px 0;">
                🧠 <b>{m.brain_state}</b> | 集中率: {metrics.get('focus_ratio', 0)*100:.0f}% | サンプル数: {metrics.get('count', 0)}
            </div>"""

        html = f"""<div style="border:2px solid #333;padding:15px;border-radius:8px;background:#fafafa;">
            <h3>小節 {m.number} (通し番号: {m.index})</h3>
            <div style="display:flex;gap:10px;margin:10px 0;">
                <span style="padding:5px 12px;border-radius:4px;background:{'#90EE90' if m.pitch_score>=0.8 else '#FFE4B5' if m.pitch_score>=0.5 else '#FFB6C1'};">🎵 ピッチ: {m.pitch_score*100:.1f}%</span>
                <span style="padding:5px 12px;border-radius:4px;background:{'#90EE90' if m.rhythm_score>=0.8 else '#FFE4B5' if m.rhythm_score>=0.5 else '#FFB6C1'};">🥁 リズム: {m.rhythm_score*100:.1f}%</span>
                <span style="padding:5px 12px;border-radius:4px;background:{'#90EE90' if m.tempo_score>=0.8 else '#FFE4B5' if m.tempo_score>=0.5 else '#FFB6C1'};">⏱️ テンポ: {m.tempo_score*100:.1f}%</span>
            </div>
            {brain_html}
            <div style="background:#f0f0f0;padding:10px;border-radius:5px;margin-top:10px;">
                <div style="color:green;margin:3px 0;">✓ 一致: {', '.join(p2n(p) for p in sorted(matched)) or 'なし'}</div>
                <div style="color:red;margin:3px 0;">✗ 不足: {', '.join(p2n(p) for p in sorted(missing)) or 'なし'}</div>
                <div style="color:orange;margin:3px 0;">+ 余分: {', '.join(p2n(p) for p in sorted(extra)) or 'なし'}</div>
            </div>
            <div style="margin-top:10px;font-size:12px;color:#666;">
                楽譜: {len(m.ref_notes)}音 ({m.start_time:.2f}s - {m.end_time:.2f}s) |
                演奏: {len(m.perf_notes)}音 ({m.perf_start_time:.2f}s - {m.perf_end_time:.2f}s)
            </div>
        </div>"""
        display(HTML(html))

    def play_measure_audio(self, m, play_ref=True):
        """小節の音符をWebAudioで再生（JavaScript）"""
        from IPython.display import display, Javascript
        import json

        if play_ref:
            notes = [{'pitch': n.pitch, 'start': n.start_time - m.start_time, 'dur': min(n.duration, 2.0)} for n in m.ref_notes]
            label = "楽譜"
        else:
            if not m.perf_notes:
                print("演奏データがありません")
                return
            base = m.perf_notes[0].start_time
            notes = [{'pitch': n.pitch, 'start': n.start_time - base, 'dur': min(n.duration, 2.0)} for n in m.perf_notes]
            label = "演奏"

        print(f"🔊 小節{m.number}の{label}を再生中...")

        js_code = f"""
(function() {{
    var notes = {json.dumps(notes)};
    var ctx = new (window.AudioContext || window.webkitAudioContext)();
    var real = new Float32Array([0, 1.0, 0.5, 0.35, 0.2, 0.12, 0.08, 0.05, 0.03]);
    var imag = new Float32Array(real.length);
    var pianoWave = ctx.createPeriodicWave(real, imag);

    notes.forEach(function(n) {{
        var o = ctx.createOscillator();
        var g = ctx.createGain();
        o.connect(g);
        g.connect(ctx.destination);
        o.frequency.value = 440 * Math.pow(2, (n.pitch - 69) / 12);
        o.setPeriodicWave(pianoWave);
        var s = Math.max(0, n.start);
        var d = n.dur;
        g.gain.setValueAtTime(0, ctx.currentTime + s);
        g.gain.linearRampToValueAtTime(0.3, ctx.currentTime + s + 0.01);
        g.gain.exponentialRampToValueAtTime(0.001, ctx.currentTime + s + d + 0.3);
        o.start(ctx.currentTime + s);
        o.stop(ctx.currentTime + s + d + 0.4);
    }});
}})();
"""
        display(Javascript(js_code))

    def show_brain_timeline(self, measures):
        import matplotlib.pyplot as plt
        played = [m for m in measures if m.perf_notes]
        if not played: print("演奏小節がありません"); return
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
        indices = [m.index for m in played]
        scores = [(m.pitch_score * 0.5 + m.rhythm_score * 0.3 + m.tempo_score * 0.2) * 100 for m in played]
        colors = ['#90EE90' if s >= 80 else '#FFE4B5' if s >= 50 else '#FFB6C1' for s in scores]
        ax1.bar(indices, scores, color=colors, edgecolor='gray')
        ax1.axhline(y=70, color='red', linestyle='--', alpha=0.5, label='70%')
        ax1.set_ylabel('Score (%)')
        ax1.set_ylim(0, 100)
        ax1.legend()
        ax1.set_title('Score and Brain State by Measure')
        state_colors = {'FOCUSED': '#28a745', 'RELAXED': '#007bff', 'NEUTRAL': '#6c757d', 'NO_DATA': '#dc3545'}
        brain_colors = [state_colors.get(m.brain_state, '#999') for m in played]
        ax2.bar(indices, [1]*len(indices), color=brain_colors, edgecolor='gray')
        ax2.set_ylabel('Brain')
        ax2.set_xlabel('Measure Index')
        ax2.set_yticks([])
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=c, label=s) for s, c in state_colors.items()]
        ax2.legend(handles=legend_elements, loc='upper right', ncol=4)
        plt.tight_layout()
        plt.show()

        # 統計
        state_counts = {}
        for m in played:
            st = m.brain_state or 'NO_DATA'
            state_counts[st] = state_counts.get(st, 0) + 1
        print("\n脳波状態の統計:")
        for st, cnt in sorted(state_counts.items(), key=lambda x: -x[1]):
            print(f"  {st}: {cnt}小節 ({cnt/len(played)*100:.1f}%)")

    def show_measure_comparison(self, m):
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        def plot(ax, notes, off, c, title):
            if not notes: ax.text(0.5, 0.5, 'No notes', ha='center', va='center', transform=ax.transAxes); ax.set_title(title); return
            for n in notes: ax.barh(n.pitch, n.duration, left=n.start_time - off, height=0.6, color=c, alpha=0.6)
            ax.set_ylim(min(n.pitch for n in notes) - 2, max(n.pitch for n in notes) + 2)
            ax.set_xlabel('Time (s)'); ax.set_ylabel('Pitch'); ax.set_title(title); ax.grid(True, alpha=0.3)
        plot(axes[0], m.ref_notes, m.start_time, 'blue', f'Score ({len(m.ref_notes)} notes)')
        plot(axes[1], m.perf_notes, m.perf_notes[0].start_time if m.perf_notes else 0, 'red', f'Perf ({len(m.perf_notes)} notes)')
        brain = f" | 🧠{m.brain_state}" if m.brain_state else ""
        fig.suptitle(f"Measure {m.number} | P:{m.pitch_score*100:.1f}% | R:{m.rhythm_score*100:.1f}%{brain}", y=1.02)
        plt.tight_layout(); plt.show()

    def show_problem_measures(self, measures, threshold=0.7):
        from IPython.display import display, HTML
        problems = []
        for m in measures:
            issues = []
            if m.pitch_score < threshold: issues.append(f"P:{m.pitch_score*100:.0f}%")
            if m.rhythm_score < threshold: issues.append(f"R:{m.rhythm_score*100:.0f}%")
            if issues: problems.append({'num': m.number, 'idx': m.index, 'issues': issues, 'brain': m.brain_state or '-'})
        if not problems: print("✓ No problems (all >= 70%)"); return
        html = f"<h3>⚠️ Problems ({len(problems)})</h3><table style='border-collapse:collapse;'><tr style='background:#f0f0f0;'><th style='padding:5px;border:1px solid #ddd;'>M#</th><th style='padding:5px;border:1px solid #ddd;'>Idx</th><th style='padding:5px;border:1px solid #ddd;'>Issues</th><th style='padding:5px;border:1px solid #ddd;'>Brain</th></tr>"
        for p in problems[:30]: html += f"<tr><td style='padding:5px;border:1px solid #ddd;'>{p['num']}</td><td style='padding:5px;border:1px solid #ddd;'>{p['idx']}</td><td style='padding:5px;border:1px solid #ddd;color:red;'>{', '.join(p['issues'])}</td><td style='padding:5px;border:1px solid #ddd;'>{p['brain']}</td></tr>"
        html += "</table>"
        display(HTML(html))

    def show_pitch_histogram(self, measures):
        import matplotlib.pyplot as plt
        ref_p = [n.pitch for m in measures for n in m.ref_notes]
        perf_p = [n.pitch for m in measures for n in m.perf_notes]
        if not ref_p and not perf_p: print("No data"); return
        fig, ax = plt.subplots(figsize=(12, 4))
        bins = range(min(ref_p + perf_p) - 1, max(ref_p + perf_p) + 2)
        ax.hist(ref_p, bins=bins, alpha=0.5, label=f'Score({len(ref_p)})', color='blue')
        ax.hist(perf_p, bins=bins, alpha=0.5, label=f'Perf({len(perf_p)})', color='red')
        ax.legend(); ax.grid(True, alpha=0.3); plt.show()

    # ========== 卒論用エクスポート機能 ==========

    def export_thesis_data(self, measures, result, filename_prefix="thesis_data"):
        """卒論用のデータをCSVとしてエクスポート"""
        import pandas as pd
        from datetime import datetime

        # 1. 小節ごとの詳細データ
        measure_data = []
        for m in measures:
            if not m.perf_notes:
                continue
            measure_data.append({
                'measure_number': m.number,
                'measure_index': m.index,
                'pitch_score': m.pitch_score * 100,
                'rhythm_score': m.rhythm_score * 100,
                'tempo_score': m.tempo_score * 100,
                'total_score': (m.pitch_score * 0.5 + m.rhythm_score * 0.3 + m.tempo_score * 0.2) * 100,
                'ref_notes': len(m.ref_notes),
                'perf_notes': len(m.perf_notes),
                'brain_state': m.brain_state or 'NO_DATA',
                'focus_ratio': m.brain_metrics.get('focus_ratio', 0) if m.brain_metrics else 0,
                'brain_samples': m.brain_metrics.get('count', 0) if m.brain_metrics else 0,
            })

        df_measures = pd.DataFrame(measure_data)

        # 2. サマリーデータ
        summary_data = {
            'total_score': result['total_score'],
            'pitch_score': result['pitch_score'],
            'rhythm_score': result['rhythm_score'],
            'tempo_score': result['tempo_score'],
            'measures_played': result['measures_played'],
            'measures_total': result['measures_total'],
        }
        if 'brain_summary' in result and result['brain_summary'].get('available'):
            bs = result['brain_summary']
            summary_data.update({
                'brain_samples': bs['total_samples'],
                'brain_duration': bs['duration'],
                'focus_ratio': bs['focus_ratio'],
                'relax_ratio': bs['relax_ratio'],
                'neutral_ratio': bs['neutral_ratio'],
            })

        df_summary = pd.DataFrame([summary_data])

        # 3. 脳波状態別の統計
        if df_measures['brain_state'].notna().any():
            brain_stats = df_measures.groupby('brain_state').agg({
                'pitch_score': ['mean', 'std', 'count'],
                'rhythm_score': ['mean', 'std'],
                'total_score': ['mean', 'std'],
            }).round(2)
            brain_stats.columns = ['_'.join(col) for col in brain_stats.columns]
            brain_stats = brain_stats.reset_index()
        else:
            brain_stats = pd.DataFrame()

        # ファイル保存
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        measures_file = f"{filename_prefix}_measures_{timestamp}.csv"
        summary_file = f"{filename_prefix}_summary_{timestamp}.csv"
        brain_file = f"{filename_prefix}_brain_stats_{timestamp}.csv"

        df_measures.to_csv(measures_file, index=False, encoding='utf-8-sig')
        df_summary.to_csv(summary_file, index=False, encoding='utf-8-sig')
        if not brain_stats.empty:
            brain_stats.to_csv(brain_file, index=False, encoding='utf-8-sig')

        print(f"✓ エクスポート完了:")
        print(f"  - {measures_file} (小節ごとデータ)")
        print(f"  - {summary_file} (サマリー)")
        if not brain_stats.empty:
            print(f"  - {brain_file} (脳波状態別統計)")

        return df_measures, df_summary, brain_stats

    def show_thesis_analysis(self, measures, result):
        """卒論用の統計分析を表示"""
        import pandas as pd
        import matplotlib.pyplot as plt
        from IPython.display import display, HTML

        # データフレーム作成
        data = []
        for m in measures:
            if not m.perf_notes:
                continue
            data.append({
                'measure': m.number,
                'pitch': m.pitch_score * 100,
                'rhythm': m.rhythm_score * 100,
                'tempo': m.tempo_score * 100,
                'total': (m.pitch_score * 0.5 + m.rhythm_score * 0.3 + m.tempo_score * 0.2) * 100,
                'brain': m.brain_state or 'NO_DATA',
            })

        df = pd.DataFrame(data)

        # 1. 基本統計
        html = "<h2>📊 卒論用統計分析</h2>"
        html += "<h3>1. スコア基本統計</h3>"
        stats = df[['pitch', 'rhythm', 'tempo', 'total']].describe().round(2)
        html += stats.to_html()

        # 2. 脳波状態別統計
        if df['brain'].notna().any() and len(df['brain'].unique()) > 1:
            html += "<h3>2. 脳波状態別スコア</h3>"
            brain_stats = df.groupby('brain')[['pitch', 'rhythm', 'tempo', 'total']].agg(['mean', 'std', 'count']).round(2)
            html += brain_stats.to_html()

            # 3. 統計検定（FOCUSED vs NEUTRAL）
            from scipy import stats as scipy_stats
            focused = df[df['brain'] == 'FOCUSED']['total']
            neutral = df[df['brain'] == 'NEUTRAL']['total']
            relaxed = df[df['brain'] == 'RELAXED']['total']

            html += "<h3>3. 統計検定</h3>"
            if len(focused) >= 2 and len(neutral) >= 2:
                t_stat, p_val = scipy_stats.ttest_ind(focused, neutral)
                sig = "有意 (p<0.05)" if p_val < 0.05 else "有意差なし"
                html += f"<p><b>FOCUSED vs NEUTRAL:</b> t={t_stat:.3f}, p={p_val:.4f} → {sig}</p>"
                html += f"<p>  FOCUSED: M={focused.mean():.1f}, SD={focused.std():.1f}, N={len(focused)}</p>"
                html += f"<p>  NEUTRAL: M={neutral.mean():.1f}, SD={neutral.std():.1f}, N={len(neutral)}</p>"

            if len(relaxed) >= 2 and len(neutral) >= 2:
                t_stat, p_val = scipy_stats.ttest_ind(relaxed, neutral)
                sig = "有意 (p<0.05)" if p_val < 0.05 else "有意差なし"
                html += f"<p><b>RELAXED vs NEUTRAL:</b> t={t_stat:.3f}, p={p_val:.4f} → {sig}</p>"

        display(HTML(html))

        # 4. グラフ
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        # 4-1. スコア分布
        df[['pitch', 'rhythm', 'tempo', 'total']].hist(ax=axes[0], bins=20, alpha=0.7)
        axes[0].set_title('Score Distribution')

        # 4-2. 脳波状態別boxplot
        if len(df['brain'].unique()) > 1:
            df.boxplot(column='total', by='brain', ax=axes[1])
            axes[1].set_title('Total Score by Brain State')
            axes[1].set_xlabel('Brain State')
            axes[1].set_ylabel('Total Score')
            plt.suptitle('')

        # 4-3. スコア推移
        axes[2].plot(df['measure'], df['total'], 'b-', alpha=0.7, label='Total')
        axes[2].axhline(y=70, color='red', linestyle='--', alpha=0.5)
        axes[2].set_xlabel('Measure')
        axes[2].set_ylabel('Score')
        axes[2].set_title('Score Progression')
        axes[2].legend()

        plt.tight_layout()
        plt.show()

        return df

    def show_score_with_click(self, mxl_path, measures):
        from IPython.display import display, HTML
        import verovio, re, json
        try:
            tk = verovio.toolkit()
            tk.setOptions({"pageWidth": 1800, "pageHeight": 3000, "scale": 35, "adjustPageHeight": True})
            tk.loadFile(mxl_path)
            pc = tk.getPageCount()
            amd = [{'index': m.index, 'number': m.number, 'pitch': m.pitch_score*100, 'rhythm': m.rhythm_score*100, 'ref_notes': len(m.ref_notes), 'perf_notes': len(m.perf_notes), 'brain_state': m.brain_state, 'start_time': m.start_time, 'end_time': m.end_time, 'perf_start': m.perf_notes[0].start_time if m.perf_notes else 0, 'ref_note_list': [{'pitch': n.pitch, 'start': n.start_time, 'dur': n.duration} for n in m.ref_notes], 'perf_note_list': [{'pitch': n.pitch, 'start': n.start_time, 'dur': n.duration} for n in m.perf_notes]} for m in measures]
            mc = {i: "#90EE90" if (m.pitch_score + m.rhythm_score)/2*100 >= 90 else "#FFFF99" if (m.pitch_score + m.rhythm_score)/2*100 >= 70 else "#FFE4B5" if (m.pitch_score + m.rhythm_score)/2*100 >= 50 else "#FFB6C1" for i, m in enumerate(measures)}
            svgs = []
            measure_idx = 0
            for p in range(1, pc + 1):
                svg = tk.renderToSVG(p)
                def add_attr(match):
                    nonlocal measure_idx
                    result = match.group(0)
                    if 'data-measure-idx' not in result:
                        result = result[:-1] + f' data-measure-idx="{measure_idx}" style="cursor:pointer;">'
                        measure_idx += 1
                    return result
                svg = re.sub(r'<g[^>]*class="[^"]*measure[^"]*"[^>]*>', add_attr, svg)
                svgs.append(svg)
            html = f'''<style>.score-viewer{{display:grid;grid-template-columns:1.5fr 1fr;gap:15px;height:700px;}}.score-panel{{overflow:auto;border:1px solid #ccc;padding:10px;background:white;}}.detail-panel{{border:1px solid #ccc;border-radius:8px;background:#f8f9fa;padding:15px;overflow-y:auto;}}.measure-highlight{{cursor:pointer;}}.measure-highlight:hover{{filter:brightness(0.85);}}</style><div class="score-viewer"><div class="score-panel" id="score-panel">{"".join(svgs)}</div><div class="detail-panel" id="detail-panel"><h3>Click a measure</h3></div></div><script>(function(){{var mc={json.dumps(mc)};var am={json.dumps(amd)};var pn=['C','C#','D','D#','E','F','F#','G','G#','A','A#','B'];function p2n(p){{return pn[p%12]+(Math.floor(p/12)-1);}}function createPianoWave(ctx){{var real=new Float32Array([0,1.0,0.5,0.35,0.2,0.12,0.08,0.05,0.03]);var imag=new Float32Array(real.length);return ctx.createPeriodicWave(real,imag);}}function show(i){{var m=am[i];if(!m)return;var rp=new Set(m.ref_note_list.map(n=>n.pitch));var pp=new Set(m.perf_note_list.map(n=>n.pitch));var mt=[...rp].filter(p=>pp.has(p));var ms=[...rp].filter(p=>!pp.has(p));var h='<h3>M'+m.number+'</h3><div style="margin:10px 0;"><span style="padding:4px 8px;border-radius:4px;background:'+(m.pitch>=80?'#90EE90':'#FFB6C1')+';">P:'+m.pitch.toFixed(1)+'%</span> <span style="padding:4px 8px;border-radius:4px;background:'+(m.rhythm>=80?'#87CEEB':'#FFB6C1')+';">R:'+m.rhythm.toFixed(1)+'%</span></div>';if(m.brain_state)h+='<div style="background:#e9ecef;padding:5px;border-radius:4px;margin:5px 0;">🧠'+m.brain_state+'</div>';h+='<div style="margin:10px 0;"><button onclick="playRef('+i+')" style="padding:8px 16px;margin:5px;cursor:pointer;border:none;border-radius:4px;background:#4CAF50;color:white;">🔊楽譜</button>';if(m.perf_notes>0)h+='<button onclick="playPerf('+i+')" style="padding:8px 16px;margin:5px;cursor:pointer;border:none;border-radius:4px;background:#f44336;color:white;">🔊演奏</button>';h+='</div><div style="font-size:11px;"><span style="color:green;">✓'+mt.map(p=>p2n(p)).join(',')||'none'+'</span><br><span style="color:red;">✗'+ms.map(p=>p2n(p)).join(',')||'none'+'</span></div>';document.getElementById('detail-panel').innerHTML=h;}}function playRef(i){{var m=am[i];if(!m||!m.ref_note_list.length)return;var ctx=new(window.AudioContext||window.webkitAudioContext)();var pianoWave=createPianoWave(ctx);m.ref_note_list.forEach(n=>{{var o=ctx.createOscillator();var g=ctx.createGain();o.connect(g);g.connect(ctx.destination);o.frequency.value=440*Math.pow(2,(n.pitch-69)/12);o.setPeriodicWave(pianoWave);var s=Math.max(0,n.start-m.start_time);var d=Math.min(n.dur,2.0);g.gain.setValueAtTime(0,ctx.currentTime+s);g.gain.linearRampToValueAtTime(0.35,ctx.currentTime+s+0.008);g.gain.exponentialRampToValueAtTime(0.001,ctx.currentTime+s+d+0.5);o.start(ctx.currentTime+s);o.stop(ctx.currentTime+s+d+0.6);}});}}function playPerf(i){{var m=am[i];if(!m||!m.perf_note_list.length)return;var ctx=new(window.AudioContext||window.webkitAudioContext)();var pianoWave=createPianoWave(ctx);m.perf_note_list.forEach(n=>{{var o=ctx.createOscillator();var g=ctx.createGain();o.connect(g);g.connect(ctx.destination);o.frequency.value=440*Math.pow(2,(n.pitch-69)/12);o.setPeriodicWave(pianoWave);var s=Math.max(0,n.start-m.perf_start);var d=Math.min(n.dur,2.0);g.gain.setValueAtTime(0,ctx.currentTime+s);g.gain.linearRampToValueAtTime(0.3,ctx.currentTime+s+0.008);g.gain.exponentialRampToValueAtTime(0.001,ctx.currentTime+s+d+0.5);o.start(ctx.currentTime+s);o.stop(ctx.currentTime+s+d+0.6);}});}}window.playRef=playRef;window.playPerf=playPerf;setTimeout(function(){{var panel=document.getElementById('score-panel');if(!panel)return;var els=panel.querySelectorAll('[data-measure-idx]');if(!els.length){{els=panel.querySelectorAll('g[class*="measure"]');els.forEach(function(el,idx){{el.setAttribute('data-measure-idx',idx);}});}}els.forEach(function(el){{var idx=parseInt(el.getAttribute('data-measure-idx'));var color=mc[idx]||'#FFF';el.classList.add('measure-highlight');try{{var bbox=el.getBBox();if(bbox.width>0){{var rect=document.createElementNS('http://www.w3.org/2000/svg','rect');rect.setAttribute('x',bbox.x-5);rect.setAttribute('y',bbox.y-5);rect.setAttribute('width',bbox.width+10);rect.setAttribute('height',bbox.height+10);rect.setAttribute('fill',color);rect.setAttribute('opacity','0.4');rect.style.pointerEvents='none';el.insertBefore(rect,el.firstChild);}}}}catch(e){{}}el.addEventListener('click',function(e){{e.stopPropagation();if(idx<am.length)show(idx);}});}});}},500);}})();</script>'''
            display(HTML(html))
        except Exception as e:
            print(f"楽譜表示エラー: {e}")

# -----------------------------
# UI
# -----------------------------
class PianoScorerUI:
    def __init__(self):
        self.params = ScoringParams()
        self.measures, self.ref_notes, self.perf_notes, self.result = [], [], [], None
        self.visualizer = Visualizer()
        self.audio_path = self.midi_path = self.mxl_path = self.recorded_audio = self.brain_csv_path = None
        self.audio_duration = None
        self.is_recording = False
        # 録音時刻を自動記録
        self.recording_start_time: Optional[datetime] = None
        self.recording_end_time: Optional[datetime] = None

    def _receive_audio_callback(self, base64_data, mime_type='audio/webm'):
        import base64, tempfile
        try:
            raw = base64.b64decode(base64_data)
            suffix = '.ogg' if 'ogg' in str(mime_type) else '.webm'
            with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
                f.write(raw)
                self.recorded_audio = f.name
            if self.input_mode.value == '録音':
                self.audio_path = self.recorded_audio
            self.record_status.value = '<span style="color:green;">✓完了</span>'
        except Exception as e:
            self.record_status.value = f'<span style="color:red;">Error:{e}</span>'

    def create_ui(self):
        import ipywidgets as widgets
        from IPython.display import display, HTML

        display(HTML("""<div style="background:linear-gradient(135deg,#1a1a2e 0%,#16213e 100%);padding:20px;border-radius:15px;color:white;margin-bottom:20px;">
            <h1 style="margin:0;">🎹🧠 Piano Performance Scorer v4.4</h1>
            <p style="margin:5px 0 0 0;opacity:0.8;">オフライン脳波統合版（Mind Monitor CSVアップロード）</p>
        </div>"""))

        # ---- 脳波CSV ----
        self.brain_enabled = widgets.Checkbox(value=False, description='🧠脳波CSVを使用')
        self.brain_enabled.observe(self._on_brain_toggle, names='value')
        self.brain_csv_upload = widgets.FileUpload(accept='.csv', multiple=False, description='脳波CSV')
        self.rec_start_input = widgets.Text(placeholder='HH:MM:SS', description='録音開始:', layout=widgets.Layout(width='180px'))
        self.rec_end_input = widgets.Text(placeholder='HH:MM:SS', description='録音終了:', layout=widgets.Layout(width='180px'))
        self.parse_brain_btn = widgets.Button(description='🧠CSV解析', button_style='info', layout=widgets.Layout(width='100px'))
        self.parse_brain_btn.on_click(self._on_parse_brain)
        self.brain_status = widgets.HTML(value='<span style="color:gray;">CSVをアップロード</span>')
        self.brain_controls = widgets.VBox([
            widgets.HBox([widgets.Label("脳波CSV:"), self.brain_csv_upload]),
            widgets.HBox([self.rec_start_input, self.rec_end_input, self.parse_brain_btn]),
            self.brain_status
        ])
        self.brain_controls.layout.display = 'none'
        brain_info = widgets.HTML("""<div style="background:#e8f4f8;padding:10px;border-radius:5px;margin:5px 0;font-size:12px;">
            <b>使い方:</b> Mind Monitorで録音中に脳波を記録 → CSVエクスポート → 録音開始/終了時刻を入力 → CSV解析
        </div>""")
        brain_box = widgets.VBox([widgets.HTML("<h3>🧠脳波CSV</h3>"), self.brain_enabled, self.brain_controls, brain_info])

        # ---- ファイル/録音 ----
        self.input_mode = widgets.RadioButtons(options=['ファイル', '録音'], value='ファイル', layout=widgets.Layout(width='100px'))
        self.input_mode.observe(self._on_input_mode_change, names='value')
        self.audio_upload = widgets.FileUpload(accept='.mp3,.wav,.m4a,.qta,.webm,.ogg,.aac', multiple=False)
        self.midi_upload = widgets.FileUpload(accept='.mid,.midi', multiple=False)
        self.mxl_upload = widgets.FileUpload(accept='.mxl,.xml,.musicxml', multiple=False)
        self.record_btn = widgets.Button(description='🎤録音', button_style='info', layout=widgets.Layout(width='80px'))
        self.record_btn.on_click(self._on_record)
        self.gain_slider = widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description='Gain:', layout=widgets.Layout(width='180px'))
        self.record_status = widgets.HTML(value='<span style="color:gray;">待機</span>')
        self.audio_file_box = widgets.VBox([widgets.Label("音声"), self.audio_upload])
        self.audio_record_box = widgets.VBox([widgets.HBox([self.record_btn, self.record_status]), self.gain_slider])
        self.audio_record_box.layout.display = 'none'
        upload_box = widgets.VBox([
            widgets.HTML("<h3>📁入力</h3>"),
            widgets.HBox([
                widgets.VBox([self.input_mode, self.audio_file_box, self.audio_record_box]),
                widgets.VBox([widgets.Label("MIDI"), self.midi_upload]),
                widgets.VBox([widgets.Label("MXL"), self.mxl_upload]),
            ])
        ])

        # ---- パラメータ ----
        self.pitch_tol = widgets.IntSlider(value=0, min=0, max=3, description='Pitch:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        self.ignore_oct = widgets.Checkbox(value=False, description='Oct無視')
        self.arpeggio = widgets.Checkbox(value=True, description='アルペジオ')
        self.rhythm_tol = widgets.FloatSlider(value=0.1, min=0.05, max=0.3, step=0.01, description='Rhythm:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        self.tempo_tol = widgets.FloatSlider(value=0.1, min=0.05, max=0.3, step=0.05, description='Tempo:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        params_box = widgets.VBox([
            widgets.HTML("<h3>⚙️パラメータ</h3>"),
            widgets.HBox([self.pitch_tol, self.ignore_oct, self.arpeggio]),
            widgets.HBox([self.rhythm_tol, self.tempo_tol]),
        ])

        # ---- ボタン ----
        self.run_btn = widgets.Button(description='🎵採点', button_style='success', layout=widgets.Layout(width='100px', height='40px'))
        self.run_btn.on_click(self._on_run)
        self.play_btn = widgets.Button(description='▶️再生', button_style='info', layout=widgets.Layout(width='80px'))
        self.play_btn.on_click(self._on_play)
        self.reset_rec_btn = widgets.Button(description='🔄録音リセット', button_style='warning', layout=widgets.Layout(width='100px'))
        self.reset_rec_btn.on_click(self._on_reset_recording)
        self.reset_all_btn = widgets.Button(description='🗑️全リセット', button_style='danger', layout=widgets.Layout(width='100px'))
        self.reset_all_btn.on_click(self._on_reset_all)
        btn_box = widgets.HBox([self.run_btn, self.play_btn, self.reset_rec_btn, self.reset_all_btn],
                               layout=widgets.Layout(justify_content='center', margin='15px 0'))

        self.output = widgets.Output()

        # ---- 分析 ----
        self.measure_sel = widgets.IntSlider(value=1, min=1, max=100, description='M#:', layout=widgets.Layout(width='200px'))
        self.show_btn = widgets.Button(description='📊グラフ', button_style='info', layout=widgets.Layout(width='70px'))
        self.show_btn.on_click(self._on_show_measure)
        self.show_detail_btn = widgets.Button(description='📋詳細', button_style='info', layout=widgets.Layout(width='60px'))
        self.show_detail_btn.on_click(self._on_show_detail)
        self.play_ref_btn = widgets.Button(description='🔊楽譜', button_style='success', layout=widgets.Layout(width='60px'))
        self.play_ref_btn.on_click(self._on_play_ref)
        self.play_perf_btn = widgets.Button(description='🔊演奏', button_style='danger', layout=widgets.Layout(width='60px'))
        self.play_perf_btn.on_click(self._on_play_perf)
        self.show_score_btn = widgets.Button(description='🎼楽譜全体', button_style='success', layout=widgets.Layout(width='80px'))
        self.show_score_btn.on_click(self._on_show_score)
        self.show_problems_btn = widgets.Button(description='⚠️問題', button_style='warning', layout=widgets.Layout(width='70px'))
        self.show_problems_btn.on_click(self._on_show_problems)
        self.show_brain_btn = widgets.Button(description='🧠タイムライン', button_style='info', layout=widgets.Layout(width='100px'))
        self.show_brain_btn.on_click(self._on_show_brain)
        self.show_all_btn = widgets.Button(description='📋全表', layout=widgets.Layout(width='60px'))
        self.show_all_btn.on_click(self._on_show_all)

        # 卒論用
        self.thesis_btn = widgets.Button(description='📊卒論分析', button_style='primary', layout=widgets.Layout(width='90px'))
        self.thesis_btn.on_click(self._on_thesis_analysis)
        self.export_btn = widgets.Button(description='💾CSV出力', button_style='primary', layout=widgets.Layout(width='80px'))
        self.export_btn.on_click(self._on_export)

        debug_box = widgets.VBox([
            widgets.HTML("<h3>🔍分析</h3>"),
            widgets.HBox([self.measure_sel, self.show_btn, self.show_detail_btn, self.play_ref_btn, self.play_perf_btn]),
            widgets.HBox([self.show_score_btn, self.show_problems_btn, self.show_brain_btn, self.show_all_btn, self.thesis_btn, self.export_btn]),
        ])
        self.debug_output = widgets.Output()

        try:
            from google.colab import output as colab_output
            colab_output.register_callback('piano.receive_audio', self._receive_audio_callback)
        except: pass

        display(widgets.VBox([brain_box, upload_box, params_box, btn_box, self.output, debug_box, self.debug_output]))

    def _on_brain_toggle(self, c):
        self.brain_controls.layout.display = 'block' if c['new'] else 'none'

    def _on_parse_brain(self, b):
        import tempfile
        with self.output:
            self.output.clear_output()
            if not self.brain_csv_upload.value:
                print("⚠️ 脳波CSVをアップロードしてください")
                return

            # CSVを一時ファイルに保存
            name = list(self.brain_csv_upload.value.keys())[0]
            content = self.brain_csv_upload.value[name]['content']
            with tempfile.NamedTemporaryFile(suffix='.csv', delete=False) as f:
                f.write(content)
                self.brain_csv_path = f.name

            # CSV解析
            if not brain_csv_parser.parse_csv(self.brain_csv_path):
                self.brain_status.value = '<span style="color:red;">CSV解析失敗</span>'
                return

            # 時刻設定（自動入力された値を使用）
            start_str = self.rec_start_input.value.strip()
            end_str = self.rec_end_input.value.strip()

            if not start_str or not end_str:
                # 録音時刻がない場合はCSV全体を使う提案
                csv_start = brain_csv_parser.start_time.strftime("%H:%M:%S")
                csv_end = brain_csv_parser.end_time.strftime("%H:%M:%S")
                self.brain_status.value = f'''<span style="color:orange;">
                    録音時刻を入力してください<br>
                    CSV範囲: {csv_start} - {csv_end}<br>
                    <small>録音ボタンを使うと自動入力されます</small>
                </span>'''
                return

            if not brain_csv_parser.set_recording_time(start_str, end_str):
                self.brain_status.value = '<span style="color:red;">時刻設定失敗</span>'
                return

            # アライメント
            if brain_csv_parser.align_to_recording():
                duration = (brain_csv_parser.recording_end - brain_csv_parser.recording_start).total_seconds()
                self.brain_status.value = f'''<span style="color:green;">
                    ✓ {len(brain_csv_parser.aligned_data)}サンプル準備完了<br>
                    {start_str} - {end_str} ({duration:.1f}秒)
                </span>'''
            else:
                self.brain_status.value = '<span style="color:red;">アライメント失敗（録音時間がCSV範囲外？）</span>'

    def _on_input_mode_change(self, c):
        if c['new'] == 'ファイル':
            self.audio_file_box.layout.display = 'block'
            self.audio_record_box.layout.display = 'none'
        else:
            self.audio_file_box.layout.display = 'none'
            self.audio_record_box.layout.display = 'block'

    def _update_params(self):
        self.params.pitch_tolerance = self.pitch_tol.value
        self.params.ignore_octave = self.ignore_oct.value
        self.params.allow_arpeggio = self.arpeggio.value
        self.params.rhythm_tolerance = self.rhythm_tol.value
        self.params.tempo_tolerance = self.tempo_tol.value

    def _on_record(self, b):
        from IPython.display import display, Javascript
        if not self.is_recording:
            self.is_recording = True
            self.record_btn.description = '⏹️停止'
            self.record_btn.button_style = 'danger'
            self.record_status.value = '<span style="color:red;">●REC</span>'

            # 録音開始時刻を記録
            self.recording_start_time = datetime.now()
            start_str = self.recording_start_time.strftime('%H:%M:%S')
            self.rec_start_input.value = start_str
            print(f"🎤 録音開始: {start_str}")

            g = self.gain_slider.value
            display(Javascript(f"""(async function(){{window.audioChunks=[];const stream=await navigator.mediaDevices.getUserMedia({{audio:true}});const ctx=new AudioContext();const src=ctx.createMediaStreamSource(stream);const gn=ctx.createGain();gn.gain.value={g};const dest=ctx.createMediaStreamDestination();src.connect(gn);gn.connect(dest);const candidates=['audio/webm;codecs=opus','audio/webm','audio/ogg;codecs=opus'];let chosen='';for(const c of candidates){{if(MediaRecorder.isTypeSupported(c)){{chosen=c;break;}}}}window.__recMime=chosen||'audio/webm';window.mediaRecorder=new MediaRecorder(dest.stream,chosen?{{mimeType:chosen}}:{{}});window.originalStream=stream;window.mediaRecorder.ondataavailable=(e)=>{{if(e.data&&e.data.size>0)window.audioChunks.push(e.data);}};window.mediaRecorder.start(1000);}})();"""))
        else:
            self.is_recording = False
            self.record_btn.description = '🎤録音'
            self.record_btn.button_style = 'info'
            self.record_status.value = '<span style="color:blue;">⏳</span>'

            # 録音終了時刻を記録
            self.recording_end_time = datetime.now()
            end_str = self.recording_end_time.strftime('%H:%M:%S')
            self.rec_end_input.value = end_str
            duration = (self.recording_end_time - self.recording_start_time).total_seconds()
            print(f"⏹️ 録音停止: {end_str} (録音時間: {duration:.1f}秒)")

            display(Javascript(r"""(async function(){if(window.mediaRecorder&&window.mediaRecorder.state==='recording'){window.mediaRecorder.stop();window.mediaRecorder.onstop=async()=>{if(window.originalStream)window.originalStream.getTracks().forEach(t=>t.stop());const blob=new Blob(window.audioChunks,{type:window.__recMime||'audio/webm'});const reader=new FileReader();reader.onloadend=function(){const base64=reader.result.split(',')[1];if(typeof google!=='undefined'&&google.colab){google.colab.kernel.invokeFunction('piano.receive_audio',[base64,window.__recMime],{});}};reader.readAsDataURL(blob);};}})();"""))

    def _on_play(self, b):
        from IPython.display import display, Audio
        with self.output:
            if self.recorded_audio: display(Audio(self.recorded_audio, autoplay=True))
            elif self.audio_path: display(Audio(self.audio_path, autoplay=True))
            else: print("再生する音声がありません")

    def _save_files(self):
        import tempfile
        if self.input_mode.value == 'ファイル' and self.audio_upload.value:
            name = list(self.audio_upload.value.keys())[0]
            content = self.audio_upload.value[name]['content']
            ext = '.' + name.split('.')[-1] if '.' in name else '.wav'
            with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
                f.write(content)
                self.audio_path = f.name
        elif self.input_mode.value == '録音' and self.recorded_audio:
            self.audio_path = self.recorded_audio
        if self.midi_upload.value:
            name = list(self.midi_upload.value.keys())[0]
            with tempfile.NamedTemporaryFile(suffix='.mid', delete=False) as f:
                f.write(self.midi_upload.value[name]['content'])
                self.midi_path = f.name
        if self.mxl_upload.value:
            name = list(self.mxl_upload.value.keys())[0]
            ext = '.' + name.split('.')[-1] if '.' in name else '.mxl'
            with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
                f.write(self.mxl_upload.value[name]['content'])
                self.mxl_path = f.name

    def _on_run(self, b):
        with self.output:
            self.output.clear_output()
            # デバッグ情報
            print(f"[DEBUG] input_mode: {self.input_mode.value}")
            print(f"[DEBUG] audio_upload.value: {bool(self.audio_upload.value)}")
            print(f"[DEBUG] recorded_audio: {self.recorded_audio}")
            print(f"[DEBUG] audio_path: {self.audio_path}")

            has_audio = bool(self.audio_upload.value) or bool(self.recorded_audio)
            if not has_audio: print("⚠️音声をアップロードまたは録音してください"); return
            if not self.midi_upload.value: print("⚠️MIDIをアップロードしてください"); return
            if not self.mxl_upload.value: print("⚠️MXLをアップロードしてください"); return
            self._update_params()
            try:
                self._save_files()
                if not self.audio_path: print("⚠️音声ファイルがありません"); return
                parser = ReferenceParser(self.midi_path, self.mxl_path)
                self.measures, self.ref_notes = parser.parse()
                if not self.measures: print("❌参照データ解析失敗"); return
                transcriber = AudioTranscriber(use_gpu=True)
                self.perf_notes, self.audio_duration = transcriber.transcribe(self.audio_path)
                aligner = DTWAligner(self.params)
                self.measures = aligner.align(self.perf_notes, self.ref_notes, self.measures)

                # 脳波統合
                brain_parser = brain_csv_parser if (self.brain_enabled.value and brain_csv_parser.aligned_data) else None
                scorer = ScoringEngine(self.params, brain_parser)
                self.result = scorer.score_all(self.measures, self.audio_duration)

                self.visualizer.show_summary(self.result)
                self.visualizer.show_table(self.result, only_played=True)
                self.measure_sel.max = len(self.measures)
            except Exception as e:
                print(f"❌Error: {e}")
                import traceback; traceback.print_exc()

    def _on_reset_recording(self, b):
        from IPython.display import display, Javascript
        self.audio_upload.value.clear()
        self.recorded_audio = None; self.audio_path = None; self.is_recording = False
        self.record_btn.description = '🎤録音'; self.record_btn.button_style = 'info'
        self.record_status.value = '<span style="color:gray;">待機</span>'
        display(Javascript('window.audioChunks=[];'))
        with self.output: self.output.clear_output(); print("✓ 録音リセット")

    def _on_reset_all(self, b):
        from IPython.display import display, Javascript
        self.audio_upload.value.clear(); self.midi_upload.value.clear(); self.mxl_upload.value.clear()
        self.brain_csv_upload.value.clear()
        self.audio_path = self.midi_path = self.mxl_path = self.recorded_audio = self.brain_csv_path = None
        self.measures, self.ref_notes, self.perf_notes, self.result = [], [], [], None
        self.is_recording = False; self.record_btn.description = '🎤録音'; self.record_btn.button_style = 'info'
        self.record_status.value = '<span style="color:gray;">待機</span>'
        self.brain_status.value = '<span style="color:gray;">CSVをアップロード</span>'
        brain_csv_parser.data = []; brain_csv_parser.aligned_data = []
        self.output.clear_output(); self.debug_output.clear_output()
        display(Javascript('window.audioChunks=[];'))
        with self.output: print("✓ 全リセット完了")

    def _on_show_measure(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.show_measure_comparison(self.measures[idx])

    def _on_show_detail(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.show_measure_detail(self.measures[idx])

    def _on_play_ref(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.play_measure_audio(self.measures[idx], play_ref=True)

    def _on_play_perf(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.play_measure_audio(self.measures[idx], play_ref=False)

    def _on_show_score(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures or not self.mxl_path: print("採点を実行してください"); return
            print("🎼 楽譜読み込み中...")
            self.visualizer.show_score_with_click(self.mxl_path, self.measures)

    def _on_show_problems(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            self.visualizer.show_problem_measures(self.measures)

    def _on_show_brain(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            self.visualizer.show_brain_timeline(self.measures)

    def _on_show_all(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.result: print("採点を実行してください"); return
            self.visualizer.show_table(self.result, only_played=False)

    def _on_thesis_analysis(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures or not self.result: print("採点を実行してください"); return
            try:
                self.visualizer.show_thesis_analysis(self.measures, self.result)
            except Exception as e:
                print(f"分析エラー: {e}")
                import traceback; traceback.print_exc()

    def _on_export(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures or not self.result: print("採点を実行してください"); return
            try:
                self.visualizer.export_thesis_data(self.measures, self.result)
            except Exception as e:
                print(f"エクスポートエラー: {e}")
                import traceback; traceback.print_exc()

def main():
    print("🎹🧠 Piano Performance Scorer v4.4 (Offline Brain Wave Integration)")
    print("=" * 60)
    install_dependencies()
    ui = PianoScorerUI()
    ui.create_ui()
    return ui

if __name__ == "__main__":
    ui = main()

In [ ]:
# ==============================================================================
# 🎹🧠 Piano Performance Scorer v4.4 - Offline Brain Wave Integration
# ==============================================================================
# Mind MonitorのCSVをアップロードして、録音と時間軸を合わせて脳波を小節に割り当て
#
# 使い方:
# 1. Mind Monitorで脳波を記録（CSVエクスポート）
# 2. 録音開始時刻と終了時刻をメモ（Mind Monitorの画面で確認）
# 3. Colabで録音 or 音声ファイルをアップロード
# 4. 脳波CSVをアップロード + 録音開始/終了時刻を入力
# 5. 採点実行 → 脳波と演奏が時間軸で統合される
# ==============================================================================

import numpy as np
import warnings
warnings.filterwarnings('ignore')

def install_dependencies():
    import subprocess, sys
    packages = [
        'pretty_midi', 'music21', 'librosa', 'matplotlib', 'pandas',
        'ipywidgets', 'transkun', 'dtw-python', 'verovio'
    ]
    for pkg in packages:
        try:
            __import__(pkg.replace('-', '_').replace('dtw-python', 'dtw'))
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    print("✓ 依存ライブラリ準備完了")

from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any
from datetime import datetime, timedelta
import threading
import time

@dataclass
class NoteEvent:
    pitch: int
    start_time: float
    end_time: float
    velocity: int = 64
    matched: bool = False
    @property
    def duration(self): return self.end_time - self.start_time
    def pitch_name(self):
        names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
        return f"{names[self.pitch % 12]}{(self.pitch // 12) - 1}"

@dataclass
class MeasureData:
    number: int
    index: int
    start_time: float
    end_time: float
    tempo: float
    time_signature: Tuple[int, int]
    ref_notes: List[NoteEvent] = field(default_factory=list)
    perf_notes: List[NoteEvent] = field(default_factory=list)
    pitch_score: float = 0.0
    rhythm_score: float = 0.0
    tempo_score: float = 0.0
    perf_start_time: float = 0.0
    perf_end_time: float = 0.0
    brain_state: str = ""
    brain_metrics: Dict = field(default_factory=dict)
    @property
    def duration(self): return self.end_time - self.start_time

@dataclass
class ScoringParams:
    pitch_tolerance: int = 0
    ignore_octave: bool = False
    chord_time_window: float = 0.05
    allow_arpeggio: bool = True
    arpeggio_max_time: float = 0.2
    rhythm_tolerance: float = 0.1
    tempo_tolerance: float = 0.1

# -----------------------------
# Brain Wave CSV Parser
# -----------------------------
class BrainWaveCSVParser:
    """Mind MonitorのCSVを解析"""

    def __init__(self):
        self.data = []
        self.start_time = None
        self.end_time = None
        self.recording_start = None  # 録音開始時刻（絶対時刻）
        self.recording_end = None    # 録音終了時刻（絶対時刻）
        self.aligned_data = []       # 録音に合わせた相対時刻データ

    def parse_csv(self, csv_path: str) -> bool:
        """Mind Monitor CSVを解析"""
        import pandas as pd

        try:
            df = pd.read_csv(csv_path)
            print(f"📊 CSV読み込み: {len(df)}行")
            print(f"   列: {list(df.columns)[:10]}...")

            # タイムスタンプ列を探す
            time_col = None
            for col in ['TimeStamp', 'Timestamp', 'timestamp', 'Time', 'time']:
                if col in df.columns:
                    time_col = col
                    break

            if time_col is None:
                print("⚠️ タイムスタンプ列が見つかりません")
                return False

            # α/β/θ列を探す
            alpha_cols = [c for c in df.columns if 'Alpha' in c or 'alpha' in c]
            beta_cols = [c for c in df.columns if 'Beta' in c or 'beta' in c]
            theta_cols = [c for c in df.columns if 'Theta' in c or 'theta' in c]

            print(f"   Alpha列: {alpha_cols[:2]}")
            print(f"   Beta列: {beta_cols[:2]}")
            print(f"   Theta列: {theta_cols[:2]}")

            self.data = []
            for _, row in df.iterrows():
                try:
                    # タイムスタンプをパース
                    ts_str = str(row[time_col])
                    # 複数のフォーマットを試す
                    ts = None
                    for fmt in ['%Y-%m-%d %H:%M:%S.%f', '%Y-%m-%d %H:%M:%S',
                               '%H:%M:%S.%f', '%H:%M:%S',
                               '%Y/%m/%d %H:%M:%S.%f', '%Y/%m/%d %H:%M:%S']:
                        try:
                            ts = datetime.strptime(ts_str, fmt)
                            break
                        except:
                            continue

                    if ts is None:
                        # Unix timestampかも
                        try:
                            ts = datetime.fromtimestamp(float(ts_str))
                        except:
                            continue

                    # α/β/θの平均を計算
                    def get_avg(cols):
                        vals = [float(row[c]) for c in cols if c in row and pd.notna(row[c])]
                        return sum(vals) / len(vals) if vals else 0.0

                    alpha = get_avg(alpha_cols)
                    beta = get_avg(beta_cols)
                    theta = get_avg(theta_cols)

                    self.data.append({
                        'timestamp': ts,
                        'alpha': alpha,
                        'beta': beta,
                        'theta': theta
                    })
                except Exception as e:
                    continue

            if not self.data:
                print("⚠️ 有効なデータがありません")
                return False

            self.start_time = self.data[0]['timestamp']
            self.end_time = self.data[-1]['timestamp']
            duration = (self.end_time - self.start_time).total_seconds()

            print(f"✓ 脳波データ: {len(self.data)}サンプル")
            print(f"   期間: {self.start_time.strftime('%H:%M:%S')} - {self.end_time.strftime('%H:%M:%S')} ({duration:.1f}秒)")

            return True

        except Exception as e:
            print(f"❌ CSV解析エラー: {e}")
            import traceback
            traceback.print_exc()
            return False

    def set_recording_time(self, start_str: str, end_str: str) -> bool:
        """録音の開始/終了時刻を設定（HH:MM:SS形式）"""
        try:
            # 今日の日付を使用
            base_date = self.start_time.date() if self.start_time else datetime.now().date()

            # 時刻をパース
            for fmt in ['%H:%M:%S.%f', '%H:%M:%S', '%H:%M']:
                try:
                    start_time = datetime.strptime(start_str, fmt).time()
                    break
                except:
                    continue
            else:
                print(f"⚠️ 開始時刻のフォーマットエラー: {start_str}")
                return False

            for fmt in ['%H:%M:%S.%f', '%H:%M:%S', '%H:%M']:
                try:
                    end_time = datetime.strptime(end_str, fmt).time()
                    break
                except:
                    continue
            else:
                print(f"⚠️ 終了時刻のフォーマットエラー: {end_str}")
                return False

            self.recording_start = datetime.combine(base_date, start_time)
            self.recording_end = datetime.combine(base_date, end_time)

            duration = (self.recording_end - self.recording_start).total_seconds()
            print(f"✓ 録音時間設定: {start_str} - {end_str} ({duration:.1f}秒)")

            return True

        except Exception as e:
            print(f"❌ 時刻設定エラー: {e}")
            return False

    def align_to_recording(self) -> bool:
        """脳波データを録音時間に合わせて相対時刻に変換（Z-score方式で状態判定）"""
        if not self.data or not self.recording_start or not self.recording_end:
            print("⚠️ データまたは録音時間が設定されていません")
            return False

        self.aligned_data = []
        rec_duration = (self.recording_end - self.recording_start).total_seconds()

        # まず録音範囲内のデータを抽出して相対値を計算
        temp_data = []
        for d in self.data:
            ts = d['timestamp']
            if self.recording_start <= ts <= self.recording_end:
                alpha, beta, theta = d['alpha'], d['beta'], d['theta']
                total = alpha + beta + theta
                if total > 0:
                    a_rel = alpha / total
                    b_rel = beta / total
                    t_rel = theta / total
                    engagement = b_rel / (a_rel + t_rel + 1e-9)
                    temp_data.append({
                        'timestamp': ts,
                        'relative_time': (ts - self.recording_start).total_seconds(),
                        'alpha': alpha, 'beta': beta, 'theta': theta,
                        'alpha_rel': a_rel, 'beta_rel': b_rel, 'theta_rel': t_rel,
                        'engagement': engagement
                    })

        if not temp_data:
            print("⚠️ 録音範囲内にデータがありません")
            return False

        # ベースライン統計を計算（Z-score用）
        import statistics
        alpha_vals = [d['alpha_rel'] for d in temp_data]
        eng_vals = [d['engagement'] for d in temp_data]

        alpha_mean = statistics.fmean(alpha_vals)
        alpha_std = statistics.pstdev(alpha_vals) or 0.01
        eng_mean = statistics.fmean(eng_vals)
        eng_std = statistics.pstdev(eng_vals) or 0.01

        print(f"   ベースライン: α相対={alpha_mean:.3f}±{alpha_std:.3f}, eng={eng_mean:.3f}±{eng_std:.3f}")

        # Z-score方式で状態判定
        focus_count = relax_count = neutral_count = 0
        for d in temp_data:
            z_alpha = (d['alpha_rel'] - alpha_mean) / alpha_std
            z_eng = (d['engagement'] - eng_mean) / eng_std

            # Z-score判定：平均からの偏差で判断
            if z_eng > 0.5 and z_alpha < 0:
                state = 'FOCUSED'
                focus_count += 1
            elif z_alpha > 0.5 and z_eng < 0:
                state = 'RELAXED'
                relax_count += 1
            else:
                state = 'NEUTRAL'
                neutral_count += 1

            self.aligned_data.append({
                'relative_time': d['relative_time'],
                'alpha': d['alpha'], 'beta': d['beta'], 'theta': d['theta'],
                'alpha_rel': d['alpha_rel'], 'beta_rel': d['beta_rel'], 'theta_rel': d['theta_rel'],
                'engagement': d['engagement'],
                'z_alpha': z_alpha, 'z_eng': z_eng,
                'state': state
            })

        print(f"✓ アライメント完了: {len(self.aligned_data)}サンプル（{rec_duration:.1f}秒間）")
        print(f"   状態分布: FOCUSED={focus_count} ({focus_count/len(self.aligned_data)*100:.1f}%), RELAXED={relax_count} ({relax_count/len(self.aligned_data)*100:.1f}%), NEUTRAL={neutral_count} ({neutral_count/len(self.aligned_data)*100:.1f}%)")
        return len(self.aligned_data) > 0

    def get_states_for_time_range(self, start_sec: float, end_sec: float) -> Dict[str, Any]:
        """指定時間範囲の脳波状態を取得"""
        if not self.aligned_data:
            return {'available': False, 'dominant_state': 'NO_DATA', 'count': 0}

        states_in_range = [
            d for d in self.aligned_data
            if start_sec <= d['relative_time'] <= end_sec
        ]

        if not states_in_range:
            return {'available': True, 'dominant_state': 'NO_DATA', 'count': 0, 'focus_ratio': 0.0}

        counts = {'FOCUSED': 0, 'RELAXED': 0, 'NEUTRAL': 0, 'NO_DATA': 0}
        alpha_sum, beta_sum, theta_sum = 0.0, 0.0, 0.0

        for s in states_in_range:
            st = s.get('state', 'NEUTRAL')
            if st in counts:
                counts[st] += 1
            alpha_sum += s.get('alpha_rel', 0)
            beta_sum += s.get('beta_rel', 0)
            theta_sum += s.get('theta_rel', 0)

        total = len(states_in_range)
        dominant = max(counts, key=counts.get)

        return {
            'available': True,
            'dominant_state': dominant,
            'count': total,
            'focus_ratio': counts['FOCUSED'] / total if total > 0 else 0.0,
            'relax_ratio': counts['RELAXED'] / total if total > 0 else 0.0,
            'avg_alpha': alpha_sum / total if total > 0 else 0,
            'avg_beta': beta_sum / total if total > 0 else 0,
            'avg_theta': theta_sum / total if total > 0 else 0,
            'state_counts': counts
        }

    def get_session_summary(self) -> Dict[str, Any]:
        """セッション全体のサマリー"""
        if not self.aligned_data:
            return {'available': False}

        counts = {'FOCUSED': 0, 'RELAXED': 0, 'NEUTRAL': 0, 'NO_DATA': 0}
        for d in self.aligned_data:
            st = d.get('state', 'NEUTRAL')
            if st in counts:
                counts[st] += 1

        total = len(self.aligned_data)
        duration = self.aligned_data[-1]['relative_time'] if self.aligned_data else 0

        return {
            'available': True,
            'total_samples': total,
            'duration': duration,
            'focus_ratio': counts['FOCUSED'] / total if total > 0 else 0.0,
            'relax_ratio': counts['RELAXED'] / total if total > 0 else 0.0,
            'neutral_ratio': counts['NEUTRAL'] / total if total > 0 else 0.0,
            'state_counts': counts
        }

brain_csv_parser = BrainWaveCSVParser()

# -----------------------------
# Reference Parser
# -----------------------------
class ReferenceParser:
    def __init__(self, midi_path, mxl_path):
        self.midi_path, self.mxl_path = midi_path, mxl_path

    def parse(self):
        print("📖 参照データを解析中...")
        midi_notes, midi_dur = self._parse_midi()
        print(f"  ✓ MIDI: {len(midi_notes)}音符, {midi_dur:.1f}秒")
        mxl_measures = self._parse_mxl()
        print(f"  ✓ MXL: {len(mxl_measures)}小節")
        measures = self._create_measures(midi_notes, mxl_measures, midi_dur)
        return measures, midi_notes

    def _parse_midi(self):
        import pretty_midi
        pm = pretty_midi.PrettyMIDI(self.midi_path)
        notes = []
        for inst in pm.instruments:
            if not inst.is_drum:
                for n in inst.notes:
                    notes.append(NoteEvent(pitch=n.pitch, start_time=n.start, end_time=n.end, velocity=n.velocity))
        notes.sort(key=lambda n: (n.start_time, n.pitch))
        return notes, pm.get_end_time()

    def _parse_mxl(self):
        from music21 import converter, tempo as m21tempo, meter
        score = converter.parse(self.mxl_path)
        try:
            score = score.expandRepeats()
        except:
            pass
        measures, current_tempo, current_ts = [], 120.0, (4, 4)
        for t in score.flatten().getElementsByClass(m21tempo.MetronomeMark):
            if t.number:
                current_tempo = float(t.number)
                break
        part = score.parts[0] if score.parts else score
        for m in part.getElementsByClass('Measure'):
            for t in m.getElementsByClass(m21tempo.MetronomeMark):
                if t.number:
                    current_tempo = float(t.number)
            for ts in m.getElementsByClass(meter.TimeSignature):
                if ts.numerator:
                    current_ts = (ts.numerator, ts.denominator)
            measures.append({
                'number': m.measureNumber,
                'tempo': current_tempo or 120,
                'time_signature': current_ts,
                'quarter_length': float(m.quarterLength) if m.quarterLength else 4.0
            })
        return measures

    def _create_measures(self, midi_notes, mxl_measures, midi_dur):
        if not mxl_measures:
            return []
        measures = []
        total_ql = sum(m['quarter_length'] for m in mxl_measures)
        midi_span = (midi_notes[-1].start_time - midi_notes[0].start_time) if midi_notes else midi_dur
        midi_offset = midi_notes[0].start_time if midi_notes else 0
        current_time = midi_offset
        for i, mxl in enumerate(mxl_measures):
            ql = mxl['quarter_length']
            dur = (ql / total_ql) * midi_span if total_ql > 0 else 2.0
            end_time = current_time + dur
            ref_notes = [
                NoteEvent(pitch=n.pitch, start_time=n.start_time, end_time=n.end_time, velocity=n.velocity)
                for n in midi_notes
                if current_time - 0.05 <= n.start_time < end_time + 0.05
            ]
            measures.append(MeasureData(
                number=mxl['number'], index=i, start_time=current_time, end_time=end_time,
                tempo=mxl['tempo'], time_signature=mxl['time_signature'], ref_notes=ref_notes
            ))
            current_time = end_time
        return measures

# -----------------------------
# Audio Transcriber
# -----------------------------
class AudioTranscriber:
    def __init__(self, use_gpu=True):
        self.use_gpu = use_gpu

    def transcribe(self, audio_path):
        import subprocess, tempfile, os, pretty_midi, librosa
        print("🎵 音声を解析中...")
        device = 'cpu'
        if self.use_gpu:
            try:
                import torch
                if torch.cuda.is_available():
                    device = 'cuda'
                    print("  ✓ CUDA使用")
            except:
                pass
        y, sr = librosa.load(audio_path, sr=None)
        duration = len(y) / sr
        print(f"  - 音声長: {duration:.1f}秒")
        with tempfile.TemporaryDirectory() as td:
            out = os.path.join(td, "out.mid")
            subprocess.run(
                ['python3', '-m', 'transkun.transcribe', audio_path, out, '--device', device],
                check=True, capture_output=True, timeout=600
            )
            pm = pretty_midi.PrettyMIDI(out)
            notes = [
                NoteEvent(pitch=n.pitch, start_time=n.start, end_time=n.end, velocity=n.velocity)
                for inst in pm.instruments if not inst.is_drum for n in inst.notes
            ]
            notes.sort(key=lambda n: (n.start_time, n.pitch))
            print(f"  ✓ {len(notes)}音符を認識")
            return notes, duration

# -----------------------------
# DTW Aligner
# -----------------------------
class DTWAligner:
    def __init__(self, params):
        self.params = params

    def align(self, perf_notes, ref_notes, measures):
        print("🔗 アライメント中...")
        if not perf_notes or not ref_notes:
            return measures
        try:
            from dtw import dtw
            pp = np.array([n.pitch for n in perf_notes]).reshape(-1, 1)
            rp = np.array([n.pitch for n in ref_notes]).reshape(-1, 1)
            def dist(x, y):
                d = abs(x[0] - y[0])
                return d/12*2 if d % 12 == 0 and d > 0 else 0 if d <= self.params.pitch_tolerance else d
            alignment = dtw(pp, rp, dist_method=dist, keep_internals=True,
                          step_pattern='symmetric2', window_type='sakoechiba',
                          window_args={'window_size': 400})
            path = list(zip(alignment.index1, alignment.index2))
        except:
            path = [(i, min(i, len(ref_notes)-1)) for i in range(len(perf_notes))]

        pairs = [(perf_notes[p].start_time, ref_notes[r].start_time)
                 for p, r in path if p < len(perf_notes) and r < len(ref_notes)]
        if not pairs:
            return measures

        pt = np.array([p[0] for p in pairs])
        rt = np.array([p[1] for p in pairs])

        def time_map(t):
            if t <= pt[0]: return rt[0]
            if t >= pt[-1]: return rt[-1]
            idx = np.searchsorted(pt, t)
            if idx == 0: return rt[0]
            ratio = (t - pt[idx-1]) / (pt[idx] - pt[idx-1]) if pt[idx] != pt[idx-1] else 0
            return rt[idx-1] + ratio * (rt[idx] - rt[idx-1])

        for note in perf_notes:
            ref_t = time_map(note.start_time)
            for m in measures:
                if m.start_time <= ref_t < m.end_time:
                    m.perf_notes.append(NoteEvent(
                        pitch=note.pitch, start_time=note.start_time,
                        end_time=note.end_time, velocity=note.velocity
                    ))
                    if not m.perf_start_time or note.start_time < m.perf_start_time:
                        m.perf_start_time = note.start_time
                    if note.start_time > m.perf_end_time:
                        m.perf_end_time = note.start_time
                    break

        print(f"  ✓ {sum(len(m.perf_notes) for m in measures)}/{len(perf_notes)}音符をマッピング")
        return measures

# -----------------------------
# Scoring Engine
# -----------------------------
class ScoringEngine:
    def __init__(self, params, brain_parser=None):
        self.params = params
        self.brain_parser = brain_parser

    def score_all(self, measures, audio_duration=None):
        print("📊 採点中...")

        all_perf = [n for m in measures for n in m.perf_notes]
        if all_perf:
            perf_start = min(n.start_time for n in all_perf)
            perf_end = max(n.end_time for n in all_perf)
            perf_duration = perf_end - perf_start
        else:
            perf_start, perf_end, perf_duration = 0, 0, 0

        brain_duration = None
        if self.brain_parser and self.brain_parser.aligned_data:
            brain_duration = self.brain_parser.aligned_data[-1]['relative_time']
            print(f"  🧠 脳波: {len(self.brain_parser.aligned_data)}サンプル, {brain_duration:.1f}秒")

        for m in measures:
            m.pitch_score = self._score_pitch(m)
            m.rhythm_score = self._score_rhythm(m)
            m.tempo_score = self._score_tempo(m)

            # 脳波割り当て
            if self.brain_parser and brain_duration and brain_duration > 0 and m.perf_notes:
                if perf_duration > 0:
                    m_start = m.perf_start_time if m.perf_start_time else min(n.start_time for n in m.perf_notes)
                    m_end = m.perf_end_time if m.perf_end_time else max(n.end_time for n in m.perf_notes)

                    # 演奏時刻を脳波時刻に変換
                    brain_start = ((m_start - perf_start) / perf_duration) * brain_duration
                    brain_end = ((m_end - perf_start) / perf_duration) * brain_duration

                    summary = self.brain_parser.get_states_for_time_range(brain_start, brain_end)
                    m.brain_state = summary.get('dominant_state', '')
                    m.brain_metrics = {
                        'focus_ratio': summary.get('focus_ratio', 0.0),
                        'relax_ratio': summary.get('relax_ratio', 0.0),
                        'count': summary.get('count', 0),
                    }

        result = self._calc_total(measures)
        print(f"  ✓ 総合スコア: {result['total_score']:.1f}")

        if self.brain_parser and self.brain_parser.aligned_data:
            result['brain_summary'] = self.brain_parser.get_session_summary()

        return result

    def _score_pitch(self, m):
        if not m.ref_notes: return 1.0 if not m.perf_notes else 0.0
        if not m.perf_notes: return 0.0
        ref_groups = self._group_by_time(m.ref_notes, self.params.chord_time_window)
        perf_groups = self._group_by_time(m.perf_notes, self.params.arpeggio_max_time if self.params.allow_arpeggio else self.params.chord_time_window)
        total = sum(len(g['pitches']) for g in ref_groups)
        pool = set(p for g in perf_groups for p in g['pitches'])
        matched = sum(1 for g in ref_groups for rp in g['pitches'] if rp in pool or any(self._match(rp, pp) for pp in pool))
        return matched / total if total > 0 else 1.0

    def _group_by_time(self, notes, window):
        if not notes: return []
        notes = sorted(notes, key=lambda n: n.start_time)
        groups = [{'time': notes[0].start_time, 'pitches': [notes[0].pitch]}]
        for n in notes[1:]:
            if n.start_time - groups[-1]['time'] <= window:
                groups[-1]['pitches'].append(n.pitch)
            else:
                groups.append({'time': n.start_time, 'pitches': [n.pitch]})
        return groups

    def _match(self, rp, pp):
        d = abs(rp - pp)
        return d == 0 or d <= self.params.pitch_tolerance or (self.params.ignore_octave and d % 12 == 0)

    def _score_rhythm(self, m):
        if not m.ref_notes or not m.perf_notes: return 1.0 if not m.ref_notes else 0.0
        def pos(notes, s, e):
            d = e - s if e > s else 0.1
            return [(n.start_time - s) / d for n in notes]
        ref_pos = pos(m.ref_notes, m.start_time, m.end_time)
        perf_pos = pos(m.perf_notes, m.perf_start_time, m.perf_end_time) if m.perf_end_time > m.perf_start_time else pos(m.perf_notes, m.perf_notes[0].start_time, m.perf_notes[-1].start_time + 0.1)
        tol = self.params.rhythm_tolerance
        return sum(1 for rp in ref_pos if any(abs(rp - pp) <= tol for pp in perf_pos)) / len(ref_pos) if ref_pos else 1.0

    def _score_tempo(self, m):
        if not m.perf_notes or len(m.perf_notes) < 2: return 1.0
        pd, rd = m.perf_end_time - m.perf_start_time, m.duration
        if pd <= 0 or rd <= 0: return 1.0
        r = pd / rd
        return 1.0 if 1 - self.params.tempo_tolerance <= r <= 1 + self.params.tempo_tolerance else max(0, 1 - (abs(r - 1) - self.params.tempo_tolerance) * 2)

    def _calc_total(self, measures):
        played = [m for m in measures if m.perf_notes]
        if not played:
            return {'total_score': 0, 'pitch_score': 0, 'rhythm_score': 0, 'tempo_score': 0, 'measures_played': 0, 'measures_total': len(measures), 'measure_details': []}
        ap, ar, at = np.mean([m.pitch_score for m in played]), np.mean([m.rhythm_score for m in played]), np.mean([m.tempo_score for m in played])
        return {
            'total_score': (ap * 0.5 + ar * 0.3 + at * 0.2) * 100, 'pitch_score': ap * 100, 'rhythm_score': ar * 100, 'tempo_score': at * 100,
            'measures_played': len(played), 'measures_total': len(measures),
            'start_measure': measures[min(m.index for m in played)].number, 'end_measure': measures[max(m.index for m in played)].number,
            'measure_details': [{'number': m.number, 'index': m.index, 'pitch': m.pitch_score*100, 'rhythm': m.rhythm_score*100, 'tempo': m.tempo_score*100, 'ref_notes': len(m.ref_notes), 'perf_notes': len(m.perf_notes), 'brain_state': m.brain_state, 'brain_metrics': m.brain_metrics} for m in measures]
        }

# -----------------------------
# Visualizer
# -----------------------------
class Visualizer:
    def show_summary(self, result):
        from IPython.display import display, HTML
        r = result
        brain_html = ""
        if 'brain_summary' in r and r['brain_summary'].get('available'):
            bs = r['brain_summary']
            brain_html = f"""<div style="margin-top:15px;padding:10px;background:rgba(255,255,255,0.1);border-radius:8px;"><div style="font-size:14px;margin-bottom:5px;">🧠 脳波 ({bs['total_samples']}サンプル, {bs.get('duration', 0):.1f}秒)</div><div style="display:flex;gap:15px;flex-wrap:wrap;"><span>🟢集中: {bs['focus_ratio']*100:.0f}%</span><span>🔵リラックス: {bs['relax_ratio']*100:.0f}%</span><span>⚪中立: {bs['neutral_ratio']*100:.0f}%</span></div></div>"""
        html = f"""<div style="background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);padding:20px;border-radius:15px;color:white;margin:10px 0;"><h2 style="margin:0 0 15px 0;">🎹 採点結果</h2><div style="display:flex;justify-content:space-around;flex-wrap:wrap;"><div style="text-align:center;padding:10px;"><div style="font-size:36px;font-weight:bold;">{r['total_score']:.1f}</div><div>総合</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['pitch_score']:.1f}</div><div>🎵ピッチ</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['rhythm_score']:.1f}</div><div>🥁リズム</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['tempo_score']:.1f}</div><div>⏱️テンポ</div></div></div><div style="margin-top:10px;">演奏: {r['measures_played']}/{r['measures_total']}小節</div>{brain_html}</div>"""
        display(HTML(html))

    def show_table(self, result, only_played=True):
        import pandas as pd
        from IPython.display import display
        df = pd.DataFrame(result.get('measure_details', []))
        if df.empty: return
        if only_played: df = df[df['perf_notes'] > 0]
        has_brain = any(d.get('brain_state') for d in result.get('measure_details', []))
        if has_brain:
            df['brain'] = df.apply(lambda r: (r['brain_state'][:3] if r.get('brain_state') else '-'), axis=1)
            df = df[['number', 'index', 'pitch', 'rhythm', 'tempo', 'ref_notes', 'perf_notes', 'brain']]
            df.columns = ['小節', '通し', 'ピッチ', 'リズム', 'テンポ', '楽譜', '演奏', '脳波']
        else:
            df = df[['number', 'index', 'pitch', 'rhythm', 'tempo', 'ref_notes', 'perf_notes']]
            df.columns = ['小節', '通し', 'ピッチ', 'リズム', 'テンポ', '楽譜', '演奏']
        def color(v):
            try:
                v = float(v)
                return 'background-color:#90EE90' if v >= 80 else 'background-color:#FFE4B5' if v >= 50 else 'background-color:#FFB6C1' if v > 0 else ''
            except: return ''
        display(df.style.applymap(color, subset=['ピッチ', 'リズム', 'テンポ']))

    def show_measure_detail(self, m):
        """小節の詳細情報をテキストで表示（音符リスト付き）"""
        from IPython.display import display, HTML
        def p2n(p):
            names = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
            return f"{names[p%12]}{p//12-1}"

        ref_pitches = sorted(set(n.pitch for n in m.ref_notes))
        perf_pitches = sorted(set(n.pitch for n in m.perf_notes))
        matched = set(ref_pitches) & set(perf_pitches)
        missing = set(ref_pitches) - set(perf_pitches)
        extra = set(perf_pitches) - set(ref_pitches)

        brain_html = ""
        if m.brain_state:
            metrics = m.brain_metrics or {}
            brain_html = f"""<div style="background:#e9ecef;padding:10px;border-radius:4px;margin:10px 0;">
                🧠 <b>{m.brain_state}</b> | 集中率: {metrics.get('focus_ratio', 0)*100:.0f}% | サンプル数: {metrics.get('count', 0)}
            </div>"""

        html = f"""<div style="border:2px solid #333;padding:15px;border-radius:8px;background:#fafafa;">
            <h3>小節 {m.number} (通し番号: {m.index})</h3>
            <div style="display:flex;gap:10px;margin:10px 0;">
                <span style="padding:5px 12px;border-radius:4px;background:{'#90EE90' if m.pitch_score>=0.8 else '#FFE4B5' if m.pitch_score>=0.5 else '#FFB6C1'};">🎵 ピッチ: {m.pitch_score*100:.1f}%</span>
                <span style="padding:5px 12px;border-radius:4px;background:{'#90EE90' if m.rhythm_score>=0.8 else '#FFE4B5' if m.rhythm_score>=0.5 else '#FFB6C1'};">🥁 リズム: {m.rhythm_score*100:.1f}%</span>
                <span style="padding:5px 12px;border-radius:4px;background:{'#90EE90' if m.tempo_score>=0.8 else '#FFE4B5' if m.tempo_score>=0.5 else '#FFB6C1'};">⏱️ テンポ: {m.tempo_score*100:.1f}%</span>
            </div>
            {brain_html}
            <div style="background:#f0f0f0;padding:10px;border-radius:5px;margin-top:10px;">
                <div style="color:green;margin:3px 0;">✓ 一致: {', '.join(p2n(p) for p in sorted(matched)) or 'なし'}</div>
                <div style="color:red;margin:3px 0;">✗ 不足: {', '.join(p2n(p) for p in sorted(missing)) or 'なし'}</div>
                <div style="color:orange;margin:3px 0;">+ 余分: {', '.join(p2n(p) for p in sorted(extra)) or 'なし'}</div>
            </div>
            <div style="margin-top:10px;font-size:12px;color:#666;">
                楽譜: {len(m.ref_notes)}音 ({m.start_time:.2f}s - {m.end_time:.2f}s) |
                演奏: {len(m.perf_notes)}音 ({m.perf_start_time:.2f}s - {m.perf_end_time:.2f}s)
            </div>
        </div>"""
        display(HTML(html))

    def play_measure_audio(self, m, play_ref=True):
        """小節の音符をWebAudioで再生（JavaScript）"""
        from IPython.display import display, Javascript
        import json

        if play_ref:
            notes = [{'pitch': n.pitch, 'start': n.start_time - m.start_time, 'dur': min(n.duration, 2.0)} for n in m.ref_notes]
            label = "楽譜"
        else:
            if not m.perf_notes:
                print("演奏データがありません")
                return
            base = m.perf_notes[0].start_time
            notes = [{'pitch': n.pitch, 'start': n.start_time - base, 'dur': min(n.duration, 2.0)} for n in m.perf_notes]
            label = "演奏"

        print(f"🔊 小節{m.number}の{label}を再生中...")

        js_code = f"""
(function() {{
    var notes = {json.dumps(notes)};
    var ctx = new (window.AudioContext || window.webkitAudioContext)();
    var real = new Float32Array([0, 1.0, 0.5, 0.35, 0.2, 0.12, 0.08, 0.05, 0.03]);
    var imag = new Float32Array(real.length);
    var pianoWave = ctx.createPeriodicWave(real, imag);

    notes.forEach(function(n) {{
        var o = ctx.createOscillator();
        var g = ctx.createGain();
        o.connect(g);
        g.connect(ctx.destination);
        o.frequency.value = 440 * Math.pow(2, (n.pitch - 69) / 12);
        o.setPeriodicWave(pianoWave);
        var s = Math.max(0, n.start);
        var d = n.dur;
        g.gain.setValueAtTime(0, ctx.currentTime + s);
        g.gain.linearRampToValueAtTime(0.3, ctx.currentTime + s + 0.01);
        g.gain.exponentialRampToValueAtTime(0.001, ctx.currentTime + s + d + 0.3);
        o.start(ctx.currentTime + s);
        o.stop(ctx.currentTime + s + d + 0.4);
    }});
}})();
"""
        display(Javascript(js_code))

    def show_brain_timeline(self, measures):
        import matplotlib.pyplot as plt
        played = [m for m in measures if m.perf_notes]
        if not played: print("演奏小節がありません"); return
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
        indices = [m.index for m in played]
        scores = [(m.pitch_score * 0.5 + m.rhythm_score * 0.3 + m.tempo_score * 0.2) * 100 for m in played]
        colors = ['#90EE90' if s >= 80 else '#FFE4B5' if s >= 50 else '#FFB6C1' for s in scores]
        ax1.bar(indices, scores, color=colors, edgecolor='gray')
        ax1.axhline(y=70, color='red', linestyle='--', alpha=0.5, label='70%')
        ax1.set_ylabel('Score (%)')
        ax1.set_ylim(0, 100)
        ax1.legend()
        ax1.set_title('Score and Brain State by Measure')
        state_colors = {'FOCUSED': '#28a745', 'RELAXED': '#007bff', 'NEUTRAL': '#6c757d', 'NO_DATA': '#dc3545'}
        brain_colors = [state_colors.get(m.brain_state, '#999') for m in played]
        ax2.bar(indices, [1]*len(indices), color=brain_colors, edgecolor='gray')
        ax2.set_ylabel('Brain')
        ax2.set_xlabel('Measure Index')
        ax2.set_yticks([])
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=c, label=s) for s, c in state_colors.items()]
        ax2.legend(handles=legend_elements, loc='upper right', ncol=4)
        plt.tight_layout()
        plt.show()

        # 統計
        state_counts = {}
        for m in played:
            st = m.brain_state or 'NO_DATA'
            state_counts[st] = state_counts.get(st, 0) + 1
        print("\n脳波状態の統計:")
        for st, cnt in sorted(state_counts.items(), key=lambda x: -x[1]):
            print(f"  {st}: {cnt}小節 ({cnt/len(played)*100:.1f}%)")

    def show_measure_comparison(self, m):
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        def plot(ax, notes, off, c, title):
            if not notes: ax.text(0.5, 0.5, 'No notes', ha='center', va='center', transform=ax.transAxes); ax.set_title(title); return
            for n in notes: ax.barh(n.pitch, n.duration, left=n.start_time - off, height=0.6, color=c, alpha=0.6)
            ax.set_ylim(min(n.pitch for n in notes) - 2, max(n.pitch for n in notes) + 2)
            ax.set_xlabel('Time (s)'); ax.set_ylabel('Pitch'); ax.set_title(title); ax.grid(True, alpha=0.3)
        plot(axes[0], m.ref_notes, m.start_time, 'blue', f'Score ({len(m.ref_notes)} notes)')
        plot(axes[1], m.perf_notes, m.perf_notes[0].start_time if m.perf_notes else 0, 'red', f'Perf ({len(m.perf_notes)} notes)')
        brain = f" | 🧠{m.brain_state}" if m.brain_state else ""
        fig.suptitle(f"Measure {m.number} | P:{m.pitch_score*100:.1f}% | R:{m.rhythm_score*100:.1f}%{brain}", y=1.02)
        plt.tight_layout(); plt.show()

    def show_problem_measures(self, measures, threshold=0.7):
        from IPython.display import display, HTML
        problems = []
        for m in measures:
            issues = []
            if m.pitch_score < threshold: issues.append(f"P:{m.pitch_score*100:.0f}%")
            if m.rhythm_score < threshold: issues.append(f"R:{m.rhythm_score*100:.0f}%")
            if issues: problems.append({'num': m.number, 'idx': m.index, 'issues': issues, 'brain': m.brain_state or '-'})
        if not problems: print("✓ No problems (all >= 70%)"); return
        html = f"<h3>⚠️ Problems ({len(problems)})</h3><table style='border-collapse:collapse;'><tr style='background:#f0f0f0;'><th style='padding:5px;border:1px solid #ddd;'>M#</th><th style='padding:5px;border:1px solid #ddd;'>Idx</th><th style='padding:5px;border:1px solid #ddd;'>Issues</th><th style='padding:5px;border:1px solid #ddd;'>Brain</th></tr>"
        for p in problems[:30]: html += f"<tr><td style='padding:5px;border:1px solid #ddd;'>{p['num']}</td><td style='padding:5px;border:1px solid #ddd;'>{p['idx']}</td><td style='padding:5px;border:1px solid #ddd;color:red;'>{', '.join(p['issues'])}</td><td style='padding:5px;border:1px solid #ddd;'>{p['brain']}</td></tr>"
        html += "</table>"
        display(HTML(html))

    def show_pitch_histogram(self, measures):
        import matplotlib.pyplot as plt
        ref_p = [n.pitch for m in measures for n in m.ref_notes]
        perf_p = [n.pitch for m in measures for n in m.perf_notes]
        if not ref_p and not perf_p: print("No data"); return
        fig, ax = plt.subplots(figsize=(12, 4))
        bins = range(min(ref_p + perf_p) - 1, max(ref_p + perf_p) + 2)
        ax.hist(ref_p, bins=bins, alpha=0.5, label=f'Score({len(ref_p)})', color='blue')
        ax.hist(perf_p, bins=bins, alpha=0.5, label=f'Perf({len(perf_p)})', color='red')
        ax.legend(); ax.grid(True, alpha=0.3); plt.show()

    # ========== 卒論用エクスポート機能 ==========

    def export_thesis_data(self, measures, result, filename_prefix="thesis_data"):
        """卒論用のデータをCSVとしてエクスポート"""
        import pandas as pd
        from datetime import datetime

        # 1. 小節ごとの詳細データ
        measure_data = []
        for m in measures:
            if not m.perf_notes:
                continue
            measure_data.append({
                'measure_number': m.number,
                'measure_index': m.index,
                'pitch_score': m.pitch_score * 100,
                'rhythm_score': m.rhythm_score * 100,
                'tempo_score': m.tempo_score * 100,
                'total_score': (m.pitch_score * 0.5 + m.rhythm_score * 0.3 + m.tempo_score * 0.2) * 100,
                'ref_notes': len(m.ref_notes),
                'perf_notes': len(m.perf_notes),
                'brain_state': m.brain_state or 'NO_DATA',
                'focus_ratio': m.brain_metrics.get('focus_ratio', 0) if m.brain_metrics else 0,
                'brain_samples': m.brain_metrics.get('count', 0) if m.brain_metrics else 0,
            })

        df_measures = pd.DataFrame(measure_data)

        # 2. サマリーデータ
        summary_data = {
            'total_score': result['total_score'],
            'pitch_score': result['pitch_score'],
            'rhythm_score': result['rhythm_score'],
            'tempo_score': result['tempo_score'],
            'measures_played': result['measures_played'],
            'measures_total': result['measures_total'],
        }
        if 'brain_summary' in result and result['brain_summary'].get('available'):
            bs = result['brain_summary']
            summary_data.update({
                'brain_samples': bs['total_samples'],
                'brain_duration': bs['duration'],
                'focus_ratio': bs['focus_ratio'],
                'relax_ratio': bs['relax_ratio'],
                'neutral_ratio': bs['neutral_ratio'],
            })

        df_summary = pd.DataFrame([summary_data])

        # 3. 脳波状態別の統計
        if df_measures['brain_state'].notna().any():
            brain_stats = df_measures.groupby('brain_state').agg({
                'pitch_score': ['mean', 'std', 'count'],
                'rhythm_score': ['mean', 'std'],
                'total_score': ['mean', 'std'],
            }).round(2)
            brain_stats.columns = ['_'.join(col) for col in brain_stats.columns]
            brain_stats = brain_stats.reset_index()
        else:
            brain_stats = pd.DataFrame()

        # ファイル保存
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        measures_file = f"{filename_prefix}_measures_{timestamp}.csv"
        summary_file = f"{filename_prefix}_summary_{timestamp}.csv"
        brain_file = f"{filename_prefix}_brain_stats_{timestamp}.csv"

        df_measures.to_csv(measures_file, index=False, encoding='utf-8-sig')
        df_summary.to_csv(summary_file, index=False, encoding='utf-8-sig')
        if not brain_stats.empty:
            brain_stats.to_csv(brain_file, index=False, encoding='utf-8-sig')

        print(f"✓ エクスポート完了:")
        print(f"  - {measures_file} (小節ごとデータ)")
        print(f"  - {summary_file} (サマリー)")
        if not brain_stats.empty:
            print(f"  - {brain_file} (脳波状態別統計)")

        return df_measures, df_summary, brain_stats

    def show_thesis_analysis(self, measures, result):
        """卒論用の統計分析を表示"""
        import pandas as pd
        import matplotlib.pyplot as plt
        from IPython.display import display, HTML

        # データフレーム作成
        data = []
        for m in measures:
            if not m.perf_notes:
                continue
            data.append({
                'measure': m.number,
                'pitch': m.pitch_score * 100,
                'rhythm': m.rhythm_score * 100,
                'tempo': m.tempo_score * 100,
                'total': (m.pitch_score * 0.5 + m.rhythm_score * 0.3 + m.tempo_score * 0.2) * 100,
                'brain': m.brain_state or 'NO_DATA',
            })

        df = pd.DataFrame(data)

        # 1. 基本統計
        html = "<h2>📊 卒論用統計分析</h2>"
        html += "<h3>1. スコア基本統計</h3>"
        stats = df[['pitch', 'rhythm', 'tempo', 'total']].describe().round(2)
        html += stats.to_html()

        # 2. 脳波状態別統計
        if df['brain'].notna().any() and len(df['brain'].unique()) > 1:
            html += "<h3>2. 脳波状態別スコア</h3>"
            brain_stats = df.groupby('brain')[['pitch', 'rhythm', 'tempo', 'total']].agg(['mean', 'std', 'count']).round(2)
            html += brain_stats.to_html()

            # 3. 統計検定（FOCUSED vs NEUTRAL）
            from scipy import stats as scipy_stats
            focused = df[df['brain'] == 'FOCUSED']['total']
            neutral = df[df['brain'] == 'NEUTRAL']['total']
            relaxed = df[df['brain'] == 'RELAXED']['total']

            html += "<h3>3. 統計検定</h3>"
            if len(focused) >= 2 and len(neutral) >= 2:
                t_stat, p_val = scipy_stats.ttest_ind(focused, neutral)
                sig = "有意 (p<0.05)" if p_val < 0.05 else "有意差なし"
                html += f"<p><b>FOCUSED vs NEUTRAL:</b> t={t_stat:.3f}, p={p_val:.4f} → {sig}</p>"
                html += f"<p>  FOCUSED: M={focused.mean():.1f}, SD={focused.std():.1f}, N={len(focused)}</p>"
                html += f"<p>  NEUTRAL: M={neutral.mean():.1f}, SD={neutral.std():.1f}, N={len(neutral)}</p>"

            if len(relaxed) >= 2 and len(neutral) >= 2:
                t_stat, p_val = scipy_stats.ttest_ind(relaxed, neutral)
                sig = "有意 (p<0.05)" if p_val < 0.05 else "有意差なし"
                html += f"<p><b>RELAXED vs NEUTRAL:</b> t={t_stat:.3f}, p={p_val:.4f} → {sig}</p>"

        display(HTML(html))

        # 4. グラフ
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        # 4-1. スコア分布
        df[['pitch', 'rhythm', 'tempo', 'total']].hist(ax=axes[0], bins=20, alpha=0.7)
        axes[0].set_title('Score Distribution')

        # 4-2. 脳波状態別boxplot
        if len(df['brain'].unique()) > 1:
            df.boxplot(column='total', by='brain', ax=axes[1])
            axes[1].set_title('Total Score by Brain State')
            axes[1].set_xlabel('Brain State')
            axes[1].set_ylabel('Total Score')
            plt.suptitle('')

        # 4-3. スコア推移
        axes[2].plot(df['measure'], df['total'], 'b-', alpha=0.7, label='Total')
        axes[2].axhline(y=70, color='red', linestyle='--', alpha=0.5)
        axes[2].set_xlabel('Measure')
        axes[2].set_ylabel('Score')
        axes[2].set_title('Score Progression')
        axes[2].legend()

        plt.tight_layout()
        plt.show()

        return df

    def show_score_with_click(self, mxl_path, measures):
        from IPython.display import display, HTML
        import verovio, re, json
        try:
            tk = verovio.toolkit()
            tk.setOptions({"pageWidth": 1800, "pageHeight": 3000, "scale": 35, "adjustPageHeight": True})
            tk.loadFile(mxl_path)
            pc = tk.getPageCount()
            amd = [{'index': m.index, 'number': m.number, 'pitch': m.pitch_score*100, 'rhythm': m.rhythm_score*100, 'ref_notes': len(m.ref_notes), 'perf_notes': len(m.perf_notes), 'brain_state': m.brain_state, 'start_time': m.start_time, 'end_time': m.end_time, 'perf_start': m.perf_notes[0].start_time if m.perf_notes else 0, 'ref_note_list': [{'pitch': n.pitch, 'start': n.start_time, 'dur': n.duration} for n in m.ref_notes], 'perf_note_list': [{'pitch': n.pitch, 'start': n.start_time, 'dur': n.duration} for n in m.perf_notes]} for m in measures]
            mc = {i: "#90EE90" if (m.pitch_score + m.rhythm_score)/2*100 >= 90 else "#FFFF99" if (m.pitch_score + m.rhythm_score)/2*100 >= 70 else "#FFE4B5" if (m.pitch_score + m.rhythm_score)/2*100 >= 50 else "#FFB6C1" for i, m in enumerate(measures)}
            svgs = []
            measure_idx = 0
            for p in range(1, pc + 1):
                svg = tk.renderToSVG(p)
                def add_attr(match):
                    nonlocal measure_idx
                    result = match.group(0)
                    if 'data-measure-idx' not in result:
                        result = result[:-1] + f' data-measure-idx="{measure_idx}" style="cursor:pointer;">'
                        measure_idx += 1
                    return result
                svg = re.sub(r'<g[^>]*class="[^"]*measure[^"]*"[^>]*>', add_attr, svg)
                svgs.append(svg)
            html = f'''<style>.score-viewer{{display:grid;grid-template-columns:1.5fr 1fr;gap:15px;height:700px;}}.score-panel{{overflow:auto;border:1px solid #ccc;padding:10px;background:white;}}.detail-panel{{border:1px solid #ccc;border-radius:8px;background:#f8f9fa;padding:15px;overflow-y:auto;}}.measure-highlight{{cursor:pointer;}}.measure-highlight:hover{{filter:brightness(0.85);}}</style><div class="score-viewer"><div class="score-panel" id="score-panel">{"".join(svgs)}</div><div class="detail-panel" id="detail-panel"><h3>Click a measure</h3></div></div><script>(function(){{var mc={json.dumps(mc)};var am={json.dumps(amd)};var pn=['C','C#','D','D#','E','F','F#','G','G#','A','A#','B'];function p2n(p){{return pn[p%12]+(Math.floor(p/12)-1);}}function createPianoWave(ctx){{var real=new Float32Array([0,1.0,0.5,0.35,0.2,0.12,0.08,0.05,0.03]);var imag=new Float32Array(real.length);return ctx.createPeriodicWave(real,imag);}}function show(i){{var m=am[i];if(!m)return;var rp=new Set(m.ref_note_list.map(n=>n.pitch));var pp=new Set(m.perf_note_list.map(n=>n.pitch));var mt=[...rp].filter(p=>pp.has(p));var ms=[...rp].filter(p=>!pp.has(p));var h='<h3>M'+m.number+'</h3><div style="margin:10px 0;"><span style="padding:4px 8px;border-radius:4px;background:'+(m.pitch>=80?'#90EE90':'#FFB6C1')+';">P:'+m.pitch.toFixed(1)+'%</span> <span style="padding:4px 8px;border-radius:4px;background:'+(m.rhythm>=80?'#87CEEB':'#FFB6C1')+';">R:'+m.rhythm.toFixed(1)+'%</span></div>';if(m.brain_state)h+='<div style="background:#e9ecef;padding:5px;border-radius:4px;margin:5px 0;">🧠'+m.brain_state+'</div>';h+='<div style="margin:10px 0;"><button onclick="playRef('+i+')" style="padding:8px 16px;margin:5px;cursor:pointer;border:none;border-radius:4px;background:#4CAF50;color:white;">🔊楽譜</button>';if(m.perf_notes>0)h+='<button onclick="playPerf('+i+')" style="padding:8px 16px;margin:5px;cursor:pointer;border:none;border-radius:4px;background:#f44336;color:white;">🔊演奏</button>';h+='</div><div style="font-size:11px;"><span style="color:green;">✓'+mt.map(p=>p2n(p)).join(',')||'none'+'</span><br><span style="color:red;">✗'+ms.map(p=>p2n(p)).join(',')||'none'+'</span></div>';document.getElementById('detail-panel').innerHTML=h;}}function playRef(i){{var m=am[i];if(!m||!m.ref_note_list.length)return;var ctx=new(window.AudioContext||window.webkitAudioContext)();var pianoWave=createPianoWave(ctx);m.ref_note_list.forEach(n=>{{var o=ctx.createOscillator();var g=ctx.createGain();o.connect(g);g.connect(ctx.destination);o.frequency.value=440*Math.pow(2,(n.pitch-69)/12);o.setPeriodicWave(pianoWave);var s=Math.max(0,n.start-m.start_time);var d=Math.min(n.dur,2.0);g.gain.setValueAtTime(0,ctx.currentTime+s);g.gain.linearRampToValueAtTime(0.35,ctx.currentTime+s+0.008);g.gain.exponentialRampToValueAtTime(0.001,ctx.currentTime+s+d+0.5);o.start(ctx.currentTime+s);o.stop(ctx.currentTime+s+d+0.6);}});}}function playPerf(i){{var m=am[i];if(!m||!m.perf_note_list.length)return;var ctx=new(window.AudioContext||window.webkitAudioContext)();var pianoWave=createPianoWave(ctx);m.perf_note_list.forEach(n=>{{var o=ctx.createOscillator();var g=ctx.createGain();o.connect(g);g.connect(ctx.destination);o.frequency.value=440*Math.pow(2,(n.pitch-69)/12);o.setPeriodicWave(pianoWave);var s=Math.max(0,n.start-m.perf_start);var d=Math.min(n.dur,2.0);g.gain.setValueAtTime(0,ctx.currentTime+s);g.gain.linearRampToValueAtTime(0.3,ctx.currentTime+s+0.008);g.gain.exponentialRampToValueAtTime(0.001,ctx.currentTime+s+d+0.5);o.start(ctx.currentTime+s);o.stop(ctx.currentTime+s+d+0.6);}});}}window.playRef=playRef;window.playPerf=playPerf;setTimeout(function(){{var panel=document.getElementById('score-panel');if(!panel)return;var els=panel.querySelectorAll('[data-measure-idx]');if(!els.length){{els=panel.querySelectorAll('g[class*="measure"]');els.forEach(function(el,idx){{el.setAttribute('data-measure-idx',idx);}});}}els.forEach(function(el){{var idx=parseInt(el.getAttribute('data-measure-idx'));var color=mc[idx]||'#FFF';el.classList.add('measure-highlight');try{{var bbox=el.getBBox();if(bbox.width>0){{var rect=document.createElementNS('http://www.w3.org/2000/svg','rect');rect.setAttribute('x',bbox.x-5);rect.setAttribute('y',bbox.y-5);rect.setAttribute('width',bbox.width+10);rect.setAttribute('height',bbox.height+10);rect.setAttribute('fill',color);rect.setAttribute('opacity','0.4');rect.style.pointerEvents='none';el.insertBefore(rect,el.firstChild);}}}}catch(e){{}}el.addEventListener('click',function(e){{e.stopPropagation();if(idx<am.length)show(idx);}});}});}},500);}})();</script>'''
            display(HTML(html))
        except Exception as e:
            print(f"楽譜表示エラー: {e}")

# -----------------------------
# UI
# -----------------------------
class PianoScorerUI:
    def __init__(self):
        self.params = ScoringParams()
        self.measures, self.ref_notes, self.perf_notes, self.result = [], [], [], None
        self.visualizer = Visualizer()
        self.audio_path = self.midi_path = self.mxl_path = self.recorded_audio = self.brain_csv_path = None
        self.audio_duration = None
        self.is_recording = False
        # 録音時刻を自動記録
        self.recording_start_time: Optional[datetime] = None
        self.recording_end_time: Optional[datetime] = None

    def _receive_audio_callback(self, base64_data, mime_type='audio/webm'):
        import base64, tempfile
        try:
            raw = base64.b64decode(base64_data)
            suffix = '.ogg' if 'ogg' in str(mime_type) else '.webm'
            with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
                f.write(raw)
                self.recorded_audio = f.name
            if self.input_mode.value == '録音':
                self.audio_path = self.recorded_audio
            self.record_status.value = '<span style="color:green;">✓完了</span>'
        except Exception as e:
            self.record_status.value = f'<span style="color:red;">Error:{e}</span>'

    def create_ui(self):
        import ipywidgets as widgets
        from IPython.display import display, HTML

        display(HTML("""<div style="background:linear-gradient(135deg,#1a1a2e 0%,#16213e 100%);padding:20px;border-radius:15px;color:white;margin-bottom:20px;">
            <h1 style="margin:0;">🎹🧠 Piano Performance Scorer v4.4</h1>
            <p style="margin:5px 0 0 0;opacity:0.8;">オフライン脳波統合版（Mind Monitor CSVアップロード）</p>
        </div>"""))

        # ---- 脳波CSV ----
        self.brain_enabled = widgets.Checkbox(value=False, description='🧠脳波CSVを使用')
        self.brain_enabled.observe(self._on_brain_toggle, names='value')
        self.brain_csv_upload = widgets.FileUpload(accept='.csv', multiple=False, description='脳波CSV')
        self.rec_start_input = widgets.Text(placeholder='HH:MM:SS', description='録音開始:', layout=widgets.Layout(width='180px'))
        self.rec_end_input = widgets.Text(placeholder='HH:MM:SS', description='録音終了:', layout=widgets.Layout(width='180px'))
        self.parse_brain_btn = widgets.Button(description='🧠CSV解析', button_style='info', layout=widgets.Layout(width='100px'))
        self.parse_brain_btn.on_click(self._on_parse_brain)
        self.brain_status = widgets.HTML(value='<span style="color:gray;">CSVをアップロード</span>')
        self.brain_controls = widgets.VBox([
            widgets.HBox([widgets.Label("脳波CSV:"), self.brain_csv_upload]),
            widgets.HBox([self.rec_start_input, self.rec_end_input, self.parse_brain_btn]),
            self.brain_status
        ])
        self.brain_controls.layout.display = 'none'
        brain_info = widgets.HTML("""<div style="background:#e8f4f8;padding:10px;border-radius:5px;margin:5px 0;font-size:12px;">
            <b>使い方:</b> Mind Monitorで録音中に脳波を記録 → CSVエクスポート → 録音開始/終了時刻を入力 → CSV解析
        </div>""")
        brain_box = widgets.VBox([widgets.HTML("<h3>🧠脳波CSV</h3>"), self.brain_enabled, self.brain_controls, brain_info])

        # ---- ファイル/録音 ----
        self.input_mode = widgets.RadioButtons(options=['ファイル', '録音'], value='ファイル', layout=widgets.Layout(width='100px'))
        self.input_mode.observe(self._on_input_mode_change, names='value')
        self.audio_upload = widgets.FileUpload(accept='.mp3,.wav,.m4a,.webm,.ogg', multiple=False)
        self.midi_upload = widgets.FileUpload(accept='.mid,.midi', multiple=False)
        self.mxl_upload = widgets.FileUpload(accept='.mxl,.xml,.musicxml', multiple=False)
        self.record_btn = widgets.Button(description='🎤録音', button_style='info', layout=widgets.Layout(width='80px'))
        self.record_btn.on_click(self._on_record)
        self.gain_slider = widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description='Gain:', layout=widgets.Layout(width='180px'))
        self.record_status = widgets.HTML(value='<span style="color:gray;">待機</span>')
        self.audio_file_box = widgets.VBox([widgets.Label("音声"), self.audio_upload])
        self.audio_record_box = widgets.VBox([widgets.HBox([self.record_btn, self.record_status]), self.gain_slider])
        self.audio_record_box.layout.display = 'none'
        upload_box = widgets.VBox([
            widgets.HTML("<h3>📁入力</h3>"),
            widgets.HBox([
                widgets.VBox([self.input_mode, self.audio_file_box, self.audio_record_box]),
                widgets.VBox([widgets.Label("MIDI"), self.midi_upload]),
                widgets.VBox([widgets.Label("MXL"), self.mxl_upload]),
            ])
        ])

        # ---- パラメータ ----
        self.pitch_tol = widgets.IntSlider(value=0, min=0, max=3, description='Pitch:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        self.ignore_oct = widgets.Checkbox(value=False, description='Oct無視')
        self.arpeggio = widgets.Checkbox(value=True, description='アルペジオ')
        self.rhythm_tol = widgets.FloatSlider(value=0.1, min=0.05, max=0.3, step=0.01, description='Rhythm:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        self.tempo_tol = widgets.FloatSlider(value=0.1, min=0.05, max=0.3, step=0.05, description='Tempo:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        params_box = widgets.VBox([
            widgets.HTML("<h3>⚙️パラメータ</h3>"),
            widgets.HBox([self.pitch_tol, self.ignore_oct, self.arpeggio]),
            widgets.HBox([self.rhythm_tol, self.tempo_tol]),
        ])

        # ---- ボタン ----
        self.run_btn = widgets.Button(description='🎵採点', button_style='success', layout=widgets.Layout(width='100px', height='40px'))
        self.run_btn.on_click(self._on_run)
        self.play_btn = widgets.Button(description='▶️再生', button_style='info', layout=widgets.Layout(width='80px'))
        self.play_btn.on_click(self._on_play)
        self.reset_rec_btn = widgets.Button(description='🔄録音リセット', button_style='warning', layout=widgets.Layout(width='100px'))
        self.reset_rec_btn.on_click(self._on_reset_recording)
        self.reset_all_btn = widgets.Button(description='🗑️全リセット', button_style='danger', layout=widgets.Layout(width='100px'))
        self.reset_all_btn.on_click(self._on_reset_all)
        btn_box = widgets.HBox([self.run_btn, self.play_btn, self.reset_rec_btn, self.reset_all_btn],
                               layout=widgets.Layout(justify_content='center', margin='15px 0'))

        self.output = widgets.Output()

        # ---- 分析 ----
        self.measure_sel = widgets.IntSlider(value=1, min=1, max=100, description='M#:', layout=widgets.Layout(width='200px'))
        self.show_btn = widgets.Button(description='📊グラフ', button_style='info', layout=widgets.Layout(width='70px'))
        self.show_btn.on_click(self._on_show_measure)
        self.show_detail_btn = widgets.Button(description='📋詳細', button_style='info', layout=widgets.Layout(width='60px'))
        self.show_detail_btn.on_click(self._on_show_detail)
        self.play_ref_btn = widgets.Button(description='🔊楽譜', button_style='success', layout=widgets.Layout(width='60px'))
        self.play_ref_btn.on_click(self._on_play_ref)
        self.play_perf_btn = widgets.Button(description='🔊演奏', button_style='danger', layout=widgets.Layout(width='60px'))
        self.play_perf_btn.on_click(self._on_play_perf)
        self.show_score_btn = widgets.Button(description='🎼楽譜全体', button_style='success', layout=widgets.Layout(width='80px'))
        self.show_score_btn.on_click(self._on_show_score)
        self.show_problems_btn = widgets.Button(description='⚠️問題', button_style='warning', layout=widgets.Layout(width='70px'))
        self.show_problems_btn.on_click(self._on_show_problems)
        self.show_brain_btn = widgets.Button(description='🧠タイムライン', button_style='info', layout=widgets.Layout(width='100px'))
        self.show_brain_btn.on_click(self._on_show_brain)
        self.show_all_btn = widgets.Button(description='📋全表', layout=widgets.Layout(width='60px'))
        self.show_all_btn.on_click(self._on_show_all)

        # 卒論用
        self.thesis_btn = widgets.Button(description='📊卒論分析', button_style='primary', layout=widgets.Layout(width='90px'))
        self.thesis_btn.on_click(self._on_thesis_analysis)
        self.export_btn = widgets.Button(description='💾CSV出力', button_style='primary', layout=widgets.Layout(width='80px'))
        self.export_btn.on_click(self._on_export)

        debug_box = widgets.VBox([
            widgets.HTML("<h3>🔍分析</h3>"),
            widgets.HBox([self.measure_sel, self.show_btn, self.show_detail_btn, self.play_ref_btn, self.play_perf_btn]),
            widgets.HBox([self.show_score_btn, self.show_problems_btn, self.show_brain_btn, self.show_all_btn, self.thesis_btn, self.export_btn]),
        ])
        self.debug_output = widgets.Output()

        try:
            from google.colab import output as colab_output
            colab_output.register_callback('piano.receive_audio', self._receive_audio_callback)
        except: pass

        display(widgets.VBox([brain_box, upload_box, params_box, btn_box, self.output, debug_box, self.debug_output]))

    def _on_brain_toggle(self, c):
        self.brain_controls.layout.display = 'block' if c['new'] else 'none'

    def _on_parse_brain(self, b):
        import tempfile
        with self.output:
            self.output.clear_output()
            if not self.brain_csv_upload.value:
                print("⚠️ 脳波CSVをアップロードしてください")
                return

            # CSVを一時ファイルに保存
            name = list(self.brain_csv_upload.value.keys())[0]
            content = self.brain_csv_upload.value[name]['content']
            with tempfile.NamedTemporaryFile(suffix='.csv', delete=False) as f:
                f.write(content)
                self.brain_csv_path = f.name

            # CSV解析
            if not brain_csv_parser.parse_csv(self.brain_csv_path):
                self.brain_status.value = '<span style="color:red;">CSV解析失敗</span>'
                return

            # 時刻設定（自動入力された値を使用）
            start_str = self.rec_start_input.value.strip()
            end_str = self.rec_end_input.value.strip()

            if not start_str or not end_str:
                # 録音時刻がない場合はCSV全体を使う提案
                csv_start = brain_csv_parser.start_time.strftime("%H:%M:%S")
                csv_end = brain_csv_parser.end_time.strftime("%H:%M:%S")
                self.brain_status.value = f'''<span style="color:orange;">
                    録音時刻を入力してください<br>
                    CSV範囲: {csv_start} - {csv_end}<br>
                    <small>録音ボタンを使うと自動入力されます</small>
                </span>'''
                return

            if not brain_csv_parser.set_recording_time(start_str, end_str):
                self.brain_status.value = '<span style="color:red;">時刻設定失敗</span>'
                return

            # アライメント
            if brain_csv_parser.align_to_recording():
                duration = (brain_csv_parser.recording_end - brain_csv_parser.recording_start).total_seconds()
                self.brain_status.value = f'''<span style="color:green;">
                    ✓ {len(brain_csv_parser.aligned_data)}サンプル準備完了<br>
                    {start_str} - {end_str} ({duration:.1f}秒)
                </span>'''
            else:
                self.brain_status.value = '<span style="color:red;">アライメント失敗（録音時間がCSV範囲外？）</span>'

    def _on_input_mode_change(self, c):
        if c['new'] == 'ファイル':
            self.audio_file_box.layout.display = 'block'
            self.audio_record_box.layout.display = 'none'
        else:
            self.audio_file_box.layout.display = 'none'
            self.audio_record_box.layout.display = 'block'

    def _update_params(self):
        self.params.pitch_tolerance = self.pitch_tol.value
        self.params.ignore_octave = self.ignore_oct.value
        self.params.allow_arpeggio = self.arpeggio.value
        self.params.rhythm_tolerance = self.rhythm_tol.value
        self.params.tempo_tolerance = self.tempo_tol.value

    def _on_record(self, b):
        from IPython.display import display, Javascript
        if not self.is_recording:
            self.is_recording = True
            self.record_btn.description = '⏹️停止'
            self.record_btn.button_style = 'danger'
            self.record_status.value = '<span style="color:red;">●REC</span>'

            # 録音開始時刻を記録
            self.recording_start_time = datetime.now()
            start_str = self.recording_start_time.strftime('%H:%M:%S')
            self.rec_start_input.value = start_str
            print(f"🎤 録音開始: {start_str}")

            g = self.gain_slider.value
            display(Javascript(f"""(async function(){{window.audioChunks=[];const stream=await navigator.mediaDevices.getUserMedia({{audio:true}});const ctx=new AudioContext();const src=ctx.createMediaStreamSource(stream);const gn=ctx.createGain();gn.gain.value={g};const dest=ctx.createMediaStreamDestination();src.connect(gn);gn.connect(dest);const candidates=['audio/webm;codecs=opus','audio/webm','audio/ogg;codecs=opus'];let chosen='';for(const c of candidates){{if(MediaRecorder.isTypeSupported(c)){{chosen=c;break;}}}}window.__recMime=chosen||'audio/webm';window.mediaRecorder=new MediaRecorder(dest.stream,chosen?{{mimeType:chosen}}:{{}});window.originalStream=stream;window.mediaRecorder.ondataavailable=(e)=>{{if(e.data&&e.data.size>0)window.audioChunks.push(e.data);}};window.mediaRecorder.start(1000);}})();"""))
        else:
            self.is_recording = False
            self.record_btn.description = '🎤録音'
            self.record_btn.button_style = 'info'
            self.record_status.value = '<span style="color:blue;">⏳</span>'

            # 録音終了時刻を記録
            self.recording_end_time = datetime.now()
            end_str = self.recording_end_time.strftime('%H:%M:%S')
            self.rec_end_input.value = end_str
            duration = (self.recording_end_time - self.recording_start_time).total_seconds()
            print(f"⏹️ 録音停止: {end_str} (録音時間: {duration:.1f}秒)")

            display(Javascript(r"""(async function(){if(window.mediaRecorder&&window.mediaRecorder.state==='recording'){window.mediaRecorder.stop();window.mediaRecorder.onstop=async()=>{if(window.originalStream)window.originalStream.getTracks().forEach(t=>t.stop());const blob=new Blob(window.audioChunks,{type:window.__recMime||'audio/webm'});const reader=new FileReader();reader.onloadend=function(){const base64=reader.result.split(',')[1];if(typeof google!=='undefined'&&google.colab){google.colab.kernel.invokeFunction('piano.receive_audio',[base64,window.__recMime],{});}};reader.readAsDataURL(blob);};}})();"""))

    def _on_play(self, b):
        from IPython.display import display, Audio
        with self.output:
            if self.recorded_audio: display(Audio(self.recorded_audio, autoplay=True))
            elif self.audio_path: display(Audio(self.audio_path, autoplay=True))
            else: print("再生する音声がありません")

    def _save_files(self):
        import tempfile
        if self.input_mode.value == 'ファイル' and self.audio_upload.value:
            name = list(self.audio_upload.value.keys())[0]
            content = self.audio_upload.value[name]['content']
            ext = '.' + name.split('.')[-1] if '.' in name else '.wav'
            with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
                f.write(content)
                self.audio_path = f.name
        elif self.input_mode.value == '録音' and self.recorded_audio:
            self.audio_path = self.recorded_audio
        if self.midi_upload.value:
            name = list(self.midi_upload.value.keys())[0]
            with tempfile.NamedTemporaryFile(suffix='.mid', delete=False) as f:
                f.write(self.midi_upload.value[name]['content'])
                self.midi_path = f.name
        if self.mxl_upload.value:
            name = list(self.mxl_upload.value.keys())[0]
            ext = '.' + name.split('.')[-1] if '.' in name else '.mxl'
            with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
                f.write(self.mxl_upload.value[name]['content'])
                self.mxl_path = f.name

    def _on_run(self, b):
        with self.output:
            self.output.clear_output()
            # デバッグ情報
            print(f"[DEBUG] input_mode: {self.input_mode.value}")
            print(f"[DEBUG] audio_upload.value: {bool(self.audio_upload.value)}")
            print(f"[DEBUG] recorded_audio: {self.recorded_audio}")
            print(f"[DEBUG] audio_path: {self.audio_path}")

            has_audio = bool(self.audio_upload.value) or bool(self.recorded_audio)
            if not has_audio: print("⚠️音声をアップロードまたは録音してください"); return
            if not self.midi_upload.value: print("⚠️MIDIをアップロードしてください"); return
            if not self.mxl_upload.value: print("⚠️MXLをアップロードしてください"); return
            self._update_params()
            try:
                self._save_files()
                if not self.audio_path: print("⚠️音声ファイルがありません"); return
                parser = ReferenceParser(self.midi_path, self.mxl_path)
                self.measures, self.ref_notes = parser.parse()
                if not self.measures: print("❌参照データ解析失敗"); return
                transcriber = AudioTranscriber(use_gpu=True)
                self.perf_notes, self.audio_duration = transcriber.transcribe(self.audio_path)
                aligner = DTWAligner(self.params)
                self.measures = aligner.align(self.perf_notes, self.ref_notes, self.measures)

                # 脳波統合
                brain_parser = brain_csv_parser if (self.brain_enabled.value and brain_csv_parser.aligned_data) else None
                scorer = ScoringEngine(self.params, brain_parser)
                self.result = scorer.score_all(self.measures, self.audio_duration)

                self.visualizer.show_summary(self.result)
                self.visualizer.show_table(self.result, only_played=True)
                self.measure_sel.max = len(self.measures)
            except Exception as e:
                print(f"❌Error: {e}")
                import traceback; traceback.print_exc()

    def _on_reset_recording(self, b):
        from IPython.display import display, Javascript
        self.audio_upload.value.clear()
        self.recorded_audio = None; self.audio_path = None; self.is_recording = False
        self.record_btn.description = '🎤録音'; self.record_btn.button_style = 'info'
        self.record_status.value = '<span style="color:gray;">待機</span>'
        display(Javascript('window.audioChunks=[];'))
        with self.output: self.output.clear_output(); print("✓ 録音リセット")

    def _on_reset_all(self, b):
        from IPython.display import display, Javascript
        self.audio_upload.value.clear(); self.midi_upload.value.clear(); self.mxl_upload.value.clear()
        self.brain_csv_upload.value.clear()
        self.audio_path = self.midi_path = self.mxl_path = self.recorded_audio = self.brain_csv_path = None
        self.measures, self.ref_notes, self.perf_notes, self.result = [], [], [], None
        self.is_recording = False; self.record_btn.description = '🎤録音'; self.record_btn.button_style = 'info'
        self.record_status.value = '<span style="color:gray;">待機</span>'
        self.brain_status.value = '<span style="color:gray;">CSVをアップロード</span>'
        brain_csv_parser.data = []; brain_csv_parser.aligned_data = []
        self.output.clear_output(); self.debug_output.clear_output()
        display(Javascript('window.audioChunks=[];'))
        with self.output: print("✓ 全リセット完了")

    def _on_show_measure(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.show_measure_comparison(self.measures[idx])

    def _on_show_detail(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.show_measure_detail(self.measures[idx])

    def _on_play_ref(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.play_measure_audio(self.measures[idx], play_ref=True)

    def _on_play_perf(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.play_measure_audio(self.measures[idx], play_ref=False)

    def _on_show_score(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures or not self.mxl_path: print("採点を実行してください"); return
            print("🎼 楽譜読み込み中...")
            self.visualizer.show_score_with_click(self.mxl_path, self.measures)

    def _on_show_problems(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            self.visualizer.show_problem_measures(self.measures)

    def _on_show_brain(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            self.visualizer.show_brain_timeline(self.measures)

    def _on_show_all(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.result: print("採点を実行してください"); return
            self.visualizer.show_table(self.result, only_played=False)

    def _on_thesis_analysis(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures or not self.result: print("採点を実行してください"); return
            try:
                self.visualizer.show_thesis_analysis(self.measures, self.result)
            except Exception as e:
                print(f"分析エラー: {e}")
                import traceback; traceback.print_exc()

    def _on_export(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures or not self.result: print("採点を実行してください"); return
            try:
                self.visualizer.export_thesis_data(self.measures, self.result)
            except Exception as e:
                print(f"エクスポートエラー: {e}")
                import traceback; traceback.print_exc()

def main():
    print("🎹🧠 Piano Performance Scorer v4.4 (Offline Brain Wave Integration)")
    print("=" * 60)
    install_dependencies()
    ui = PianoScorerUI()
    ui.create_ui()
    return ui

if __name__ == "__main__":
    ui = main()

In [ ]:
# ==============================================================================
# 🎹🧠 Piano Performance Scorer v4.4 - Offline Brain Wave Integration
# ==============================================================================
# Mind MonitorのCSVをアップロードして、録音と時間軸を合わせて脳波を小節に割り当て
#
# 使い方:
# 1. Mind Monitorで脳波を記録（CSVエクスポート）
# 2. 録音開始時刻と終了時刻をメモ（Mind Monitorの画面で確認）
# 3. Colabで録音 or 音声ファイルをアップロード
# 4. 脳波CSVをアップロード + 録音開始/終了時刻を入力
# 5. 採点実行 → 脳波と演奏が時間軸で統合される
# ==============================================================================

import numpy as np
import warnings
warnings.filterwarnings('ignore')

def install_dependencies():
    import subprocess, sys
    packages = [
        'pretty_midi', 'music21', 'librosa', 'matplotlib', 'pandas',
        'ipywidgets', 'transkun', 'dtw-python', 'verovio'
    ]
    for pkg in packages:
        try:
            __import__(pkg.replace('-', '_').replace('dtw-python', 'dtw'))
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    print("✓ 依存ライブラリ準備完了")

from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any
from datetime import datetime, timedelta
import threading
import time

@dataclass
class NoteEvent:
    pitch: int
    start_time: float
    end_time: float
    velocity: int = 64
    matched: bool = False
    @property
    def duration(self): return self.end_time - self.start_time
    def pitch_name(self):
        names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
        return f"{names[self.pitch % 12]}{(self.pitch // 12) - 1}"

@dataclass
class MeasureData:
    number: int
    index: int
    start_time: float
    end_time: float
    tempo: float
    time_signature: Tuple[int, int]
    ref_notes: List[NoteEvent] = field(default_factory=list)
    perf_notes: List[NoteEvent] = field(default_factory=list)
    pitch_score: float = 0.0
    rhythm_score: float = 0.0
    tempo_score: float = 0.0
    perf_start_time: float = 0.0
    perf_end_time: float = 0.0
    brain_state: str = ""
    brain_metrics: Dict = field(default_factory=dict)
    @property
    def duration(self): return self.end_time - self.start_time

@dataclass
class ScoringParams:
    pitch_tolerance: int = 0
    ignore_octave: bool = False
    chord_time_window: float = 0.05
    allow_arpeggio: bool = True
    arpeggio_max_time: float = 0.2
    rhythm_tolerance: float = 0.1
    tempo_tolerance: float = 0.1

# -----------------------------
# Brain Wave CSV Parser
# -----------------------------
class BrainWaveCSVParser:
    """Mind MonitorのCSVを解析"""

    def __init__(self):
        self.data = []
        self.start_time = None
        self.end_time = None
        self.recording_start = None  # 録音開始時刻（絶対時刻）
        self.recording_end = None    # 録音終了時刻（絶対時刻）
        self.aligned_data = []       # 録音に合わせた相対時刻データ

    def parse_csv(self, csv_path: str) -> bool:
        """Mind Monitor CSVを解析"""
        import pandas as pd

        try:
            df = pd.read_csv(csv_path)
            print(f"📊 CSV読み込み: {len(df)}行")
            print(f"   列: {list(df.columns)[:10]}...")

            # タイムスタンプ列を探す
            time_col = None
            for col in ['TimeStamp', 'Timestamp', 'timestamp', 'Time', 'time']:
                if col in df.columns:
                    time_col = col
                    break

            if time_col is None:
                print("⚠️ タイムスタンプ列が見つかりません")
                return False

            # α/β/θ列を探す
            alpha_cols = [c for c in df.columns if 'Alpha' in c or 'alpha' in c]
            beta_cols = [c for c in df.columns if 'Beta' in c or 'beta' in c]
            theta_cols = [c for c in df.columns if 'Theta' in c or 'theta' in c]

            print(f"   Alpha列: {alpha_cols[:2]}")
            print(f"   Beta列: {beta_cols[:2]}")
            print(f"   Theta列: {theta_cols[:2]}")

            self.data = []
            for _, row in df.iterrows():
                try:
                    # タイムスタンプをパース
                    ts_str = str(row[time_col])
                    # 複数のフォーマットを試す
                    ts = None
                    for fmt in ['%Y-%m-%d %H:%M:%S.%f', '%Y-%m-%d %H:%M:%S',
                               '%H:%M:%S.%f', '%H:%M:%S',
                               '%Y/%m/%d %H:%M:%S.%f', '%Y/%m/%d %H:%M:%S']:
                        try:
                            ts = datetime.strptime(ts_str, fmt)
                            break
                        except:
                            continue

                    if ts is None:
                        # Unix timestampかも
                        try:
                            ts = datetime.fromtimestamp(float(ts_str))
                        except:
                            continue

                    # α/β/θの平均を計算
                    def get_avg(cols):
                        vals = [float(row[c]) for c in cols if c in row and pd.notna(row[c])]
                        return sum(vals) / len(vals) if vals else 0.0

                    alpha = get_avg(alpha_cols)
                    beta = get_avg(beta_cols)
                    theta = get_avg(theta_cols)

                    self.data.append({
                        'timestamp': ts,
                        'alpha': alpha,
                        'beta': beta,
                        'theta': theta
                    })
                except Exception as e:
                    continue

            if not self.data:
                print("⚠️ 有効なデータがありません")
                return False

            self.start_time = self.data[0]['timestamp']
            self.end_time = self.data[-1]['timestamp']
            duration = (self.end_time - self.start_time).total_seconds()

            print(f"✓ 脳波データ: {len(self.data)}サンプル")
            print(f"   期間: {self.start_time.strftime('%H:%M:%S')} - {self.end_time.strftime('%H:%M:%S')} ({duration:.1f}秒)")

            return True

        except Exception as e:
            print(f"❌ CSV解析エラー: {e}")
            import traceback
            traceback.print_exc()
            return False

    def set_recording_time(self, start_str: str, end_str: str) -> bool:
        """録音の開始/終了時刻を設定（HH:MM:SS形式）"""
        try:
            # 今日の日付を使用
            base_date = self.start_time.date() if self.start_time else datetime.now().date()

            # 時刻をパース
            for fmt in ['%H:%M:%S.%f', '%H:%M:%S', '%H:%M']:
                try:
                    start_time = datetime.strptime(start_str, fmt).time()
                    break
                except:
                    continue
            else:
                print(f"⚠️ 開始時刻のフォーマットエラー: {start_str}")
                return False

            for fmt in ['%H:%M:%S.%f', '%H:%M:%S', '%H:%M']:
                try:
                    end_time = datetime.strptime(end_str, fmt).time()
                    break
                except:
                    continue
            else:
                print(f"⚠️ 終了時刻のフォーマットエラー: {end_str}")
                return False

            self.recording_start = datetime.combine(base_date, start_time)
            self.recording_end = datetime.combine(base_date, end_time)

            duration = (self.recording_end - self.recording_start).total_seconds()
            print(f"✓ 録音時間設定: {start_str} - {end_str} ({duration:.1f}秒)")

            return True

        except Exception as e:
            print(f"❌ 時刻設定エラー: {e}")
            return False

    def align_to_recording(self) -> bool:
        """脳波データを録音時間に合わせて相対時刻に変換（Z-score方式で状態判定）"""
        if not self.data or not self.recording_start or not self.recording_end:
            print("⚠️ データまたは録音時間が設定されていません")
            return False

        self.aligned_data = []
        rec_duration = (self.recording_end - self.recording_start).total_seconds()

        # まず録音範囲内のデータを抽出して相対値を計算
        temp_data = []
        for d in self.data:
            ts = d['timestamp']
            if self.recording_start <= ts <= self.recording_end:
                alpha, beta, theta = d['alpha'], d['beta'], d['theta']
                total = alpha + beta + theta
                if total > 0:
                    a_rel = alpha / total
                    b_rel = beta / total
                    t_rel = theta / total
                    engagement = b_rel / (a_rel + t_rel + 1e-9)
                    temp_data.append({
                        'timestamp': ts,
                        'relative_time': (ts - self.recording_start).total_seconds(),
                        'alpha': alpha, 'beta': beta, 'theta': theta,
                        'alpha_rel': a_rel, 'beta_rel': b_rel, 'theta_rel': t_rel,
                        'engagement': engagement
                    })

        if not temp_data:
            print("⚠️ 録音範囲内にデータがありません")
            return False

        # ベースライン統計を計算（Z-score用）
        import statistics
        alpha_vals = [d['alpha_rel'] for d in temp_data]
        eng_vals = [d['engagement'] for d in temp_data]

        alpha_mean = statistics.fmean(alpha_vals)
        alpha_std = statistics.pstdev(alpha_vals) or 0.01
        eng_mean = statistics.fmean(eng_vals)
        eng_std = statistics.pstdev(eng_vals) or 0.01

        print(f"   ベースライン: α相対={alpha_mean:.3f}±{alpha_std:.3f}, eng={eng_mean:.3f}±{eng_std:.3f}")

        # Z-score方式で状態判定
        focus_count = relax_count = neutral_count = 0
        for d in temp_data:
            z_alpha = (d['alpha_rel'] - alpha_mean) / alpha_std
            z_eng = (d['engagement'] - eng_mean) / eng_std

            # Z-score判定：平均からの偏差で判断
            if z_eng > 0.5 and z_alpha < 0:
                state = 'FOCUSED'
                focus_count += 1
            elif z_alpha > 0.5 and z_eng < 0:
                state = 'RELAXED'
                relax_count += 1
            else:
                state = 'NEUTRAL'
                neutral_count += 1

            self.aligned_data.append({
                'relative_time': d['relative_time'],
                'alpha': d['alpha'], 'beta': d['beta'], 'theta': d['theta'],
                'alpha_rel': d['alpha_rel'], 'beta_rel': d['beta_rel'], 'theta_rel': d['theta_rel'],
                'engagement': d['engagement'],
                'z_alpha': z_alpha, 'z_eng': z_eng,
                'state': state
            })

        print(f"✓ アライメント完了: {len(self.aligned_data)}サンプル（{rec_duration:.1f}秒間）")
        print(f"   状態分布: FOCUSED={focus_count} ({focus_count/len(self.aligned_data)*100:.1f}%), RELAXED={relax_count} ({relax_count/len(self.aligned_data)*100:.1f}%), NEUTRAL={neutral_count} ({neutral_count/len(self.aligned_data)*100:.1f}%)")
        return len(self.aligned_data) > 0

    def get_states_for_time_range(self, start_sec: float, end_sec: float) -> Dict[str, Any]:
        """指定時間範囲の脳波状態を取得"""
        if not self.aligned_data:
            return {'available': False, 'dominant_state': 'NO_DATA', 'count': 0}

        states_in_range = [
            d for d in self.aligned_data
            if start_sec <= d['relative_time'] <= end_sec
        ]

        if not states_in_range:
            return {'available': True, 'dominant_state': 'NO_DATA', 'count': 0, 'focus_ratio': 0.0}

        counts = {'FOCUSED': 0, 'RELAXED': 0, 'NEUTRAL': 0, 'NO_DATA': 0}
        alpha_sum, beta_sum, theta_sum = 0.0, 0.0, 0.0

        for s in states_in_range:
            st = s.get('state', 'NEUTRAL')
            if st in counts:
                counts[st] += 1
            alpha_sum += s.get('alpha_rel', 0)
            beta_sum += s.get('beta_rel', 0)
            theta_sum += s.get('theta_rel', 0)

        total = len(states_in_range)
        dominant = max(counts, key=counts.get)

        return {
            'available': True,
            'dominant_state': dominant,
            'count': total,
            'focus_ratio': counts['FOCUSED'] / total if total > 0 else 0.0,
            'relax_ratio': counts['RELAXED'] / total if total > 0 else 0.0,
            'avg_alpha': alpha_sum / total if total > 0 else 0,
            'avg_beta': beta_sum / total if total > 0 else 0,
            'avg_theta': theta_sum / total if total > 0 else 0,
            'state_counts': counts
        }

    def get_session_summary(self) -> Dict[str, Any]:
        """セッション全体のサマリー"""
        if not self.aligned_data:
            return {'available': False}

        counts = {'FOCUSED': 0, 'RELAXED': 0, 'NEUTRAL': 0, 'NO_DATA': 0}
        for d in self.aligned_data:
            st = d.get('state', 'NEUTRAL')
            if st in counts:
                counts[st] += 1

        total = len(self.aligned_data)
        duration = self.aligned_data[-1]['relative_time'] if self.aligned_data else 0

        return {
            'available': True,
            'total_samples': total,
            'duration': duration,
            'focus_ratio': counts['FOCUSED'] / total if total > 0 else 0.0,
            'relax_ratio': counts['RELAXED'] / total if total > 0 else 0.0,
            'neutral_ratio': counts['NEUTRAL'] / total if total > 0 else 0.0,
            'state_counts': counts
        }

brain_csv_parser = BrainWaveCSVParser()

# -----------------------------
# Reference Parser
# -----------------------------
class ReferenceParser:
    def __init__(self, midi_path, mxl_path):
        self.midi_path, self.mxl_path = midi_path, mxl_path

    def parse(self):
        print("📖 参照データを解析中...")
        midi_notes, midi_dur = self._parse_midi()
        print(f"  ✓ MIDI: {len(midi_notes)}音符, {midi_dur:.1f}秒")
        mxl_measures = self._parse_mxl()
        print(f"  ✓ MXL: {len(mxl_measures)}小節")
        measures = self._create_measures(midi_notes, mxl_measures, midi_dur)
        return measures, midi_notes

    def _parse_midi(self):
        import pretty_midi
        pm = pretty_midi.PrettyMIDI(self.midi_path)
        notes = []
        for inst in pm.instruments:
            if not inst.is_drum:
                for n in inst.notes:
                    notes.append(NoteEvent(pitch=n.pitch, start_time=n.start, end_time=n.end, velocity=n.velocity))
        notes.sort(key=lambda n: (n.start_time, n.pitch))
        return notes, pm.get_end_time()

    def _parse_mxl(self):
        from music21 import converter, tempo as m21tempo, meter
        score = converter.parse(self.mxl_path)
        try:
            score = score.expandRepeats()
        except:
            pass
        measures, current_tempo, current_ts = [], 120.0, (4, 4)
        for t in score.flatten().getElementsByClass(m21tempo.MetronomeMark):
            if t.number:
                current_tempo = float(t.number)
                break
        part = score.parts[0] if score.parts else score
        for m in part.getElementsByClass('Measure'):
            for t in m.getElementsByClass(m21tempo.MetronomeMark):
                if t.number:
                    current_tempo = float(t.number)
            for ts in m.getElementsByClass(meter.TimeSignature):
                if ts.numerator:
                    current_ts = (ts.numerator, ts.denominator)
            measures.append({
                'number': m.measureNumber,
                'tempo': current_tempo or 120,
                'time_signature': current_ts,
                'quarter_length': float(m.quarterLength) if m.quarterLength else 4.0
            })
        return measures

    def _create_measures(self, midi_notes, mxl_measures, midi_dur):
        if not mxl_measures:
            return []
        measures = []
        total_ql = sum(m['quarter_length'] for m in mxl_measures)
        midi_span = (midi_notes[-1].start_time - midi_notes[0].start_time) if midi_notes else midi_dur
        midi_offset = midi_notes[0].start_time if midi_notes else 0
        current_time = midi_offset
        for i, mxl in enumerate(mxl_measures):
            ql = mxl['quarter_length']
            dur = (ql / total_ql) * midi_span if total_ql > 0 else 2.0
            end_time = current_time + dur
            ref_notes = [
                NoteEvent(pitch=n.pitch, start_time=n.start_time, end_time=n.end_time, velocity=n.velocity)
                for n in midi_notes
                if current_time - 0.05 <= n.start_time < end_time + 0.05
            ]
            measures.append(MeasureData(
                number=mxl['number'], index=i, start_time=current_time, end_time=end_time,
                tempo=mxl['tempo'], time_signature=mxl['time_signature'], ref_notes=ref_notes
            ))
            current_time = end_time
        return measures

# -----------------------------
# Audio Transcriber
# -----------------------------
class AudioTranscriber:
    def __init__(self, use_gpu=True):
        self.use_gpu = use_gpu

    def transcribe(self, audio_path):
        import subprocess, tempfile, os, pretty_midi, librosa
        print("🎵 音声を解析中...")
        device = 'cpu'
        if self.use_gpu:
            try:
                import torch
                if torch.cuda.is_available():
                    device = 'cuda'
                    print("  ✓ CUDA使用")
            except:
                pass
        y, sr = librosa.load(audio_path, sr=None)
        duration = len(y) / sr
        print(f"  - 音声長: {duration:.1f}秒")
        with tempfile.TemporaryDirectory() as td:
            out = os.path.join(td, "out.mid")
            subprocess.run(
                ['python3', '-m', 'transkun.transcribe', audio_path, out, '--device', device],
                check=True, capture_output=True, timeout=600
            )
            pm = pretty_midi.PrettyMIDI(out)
            notes = [
                NoteEvent(pitch=n.pitch, start_time=n.start, end_time=n.end, velocity=n.velocity)
                for inst in pm.instruments if not inst.is_drum for n in inst.notes
            ]
            notes.sort(key=lambda n: (n.start_time, n.pitch))
            print(f"  ✓ {len(notes)}音符を認識")
            return notes, duration

# -----------------------------
# DTW Aligner
# -----------------------------
class DTWAligner:
    def __init__(self, params):
        self.params = params

    def align(self, perf_notes, ref_notes, measures):
        print("🔗 アライメント中...")
        if not perf_notes or not ref_notes:
            return measures
        try:
            from dtw import dtw
            pp = np.array([n.pitch for n in perf_notes]).reshape(-1, 1)
            rp = np.array([n.pitch for n in ref_notes]).reshape(-1, 1)
            def dist(x, y):
                d = abs(x[0] - y[0])
                return d/12*2 if d % 12 == 0 and d > 0 else 0 if d <= self.params.pitch_tolerance else d
            alignment = dtw(pp, rp, dist_method=dist, keep_internals=True,
                          step_pattern='symmetric2', window_type='sakoechiba',
                          window_args={'window_size': 400})
            path = list(zip(alignment.index1, alignment.index2))
        except:
            path = [(i, min(i, len(ref_notes)-1)) for i in range(len(perf_notes))]

        pairs = [(perf_notes[p].start_time, ref_notes[r].start_time)
                 for p, r in path if p < len(perf_notes) and r < len(ref_notes)]
        if not pairs:
            return measures

        pt = np.array([p[0] for p in pairs])
        rt = np.array([p[1] for p in pairs])

        def time_map(t):
            if t <= pt[0]: return rt[0]
            if t >= pt[-1]: return rt[-1]
            idx = np.searchsorted(pt, t)
            if idx == 0: return rt[0]
            ratio = (t - pt[idx-1]) / (pt[idx] - pt[idx-1]) if pt[idx] != pt[idx-1] else 0
            return rt[idx-1] + ratio * (rt[idx] - rt[idx-1])

        for note in perf_notes:
            ref_t = time_map(note.start_time)
            for m in measures:
                if m.start_time <= ref_t < m.end_time:
                    m.perf_notes.append(NoteEvent(
                        pitch=note.pitch, start_time=note.start_time,
                        end_time=note.end_time, velocity=note.velocity
                    ))
                    if not m.perf_start_time or note.start_time < m.perf_start_time:
                        m.perf_start_time = note.start_time
                    if note.start_time > m.perf_end_time:
                        m.perf_end_time = note.start_time
                    break

        print(f"  ✓ {sum(len(m.perf_notes) for m in measures)}/{len(perf_notes)}音符をマッピング")
        return measures

# -----------------------------
# Scoring Engine
# -----------------------------
class ScoringEngine:
    def __init__(self, params, brain_parser=None):
        self.params = params
        self.brain_parser = brain_parser

    def score_all(self, measures, audio_duration=None):
        print("📊 採点中...")

        all_perf = [n for m in measures for n in m.perf_notes]
        if all_perf:
            perf_start = min(n.start_time for n in all_perf)
            perf_end = max(n.end_time for n in all_perf)
            perf_duration = perf_end - perf_start
        else:
            perf_start, perf_end, perf_duration = 0, 0, 0

        brain_duration = None
        if self.brain_parser and self.brain_parser.aligned_data:
            brain_duration = self.brain_parser.aligned_data[-1]['relative_time']
            print(f"  🧠 脳波: {len(self.brain_parser.aligned_data)}サンプル, {brain_duration:.1f}秒")

        for m in measures:
            m.pitch_score = self._score_pitch(m)
            m.rhythm_score = self._score_rhythm(m)
            m.tempo_score = self._score_tempo(m)

            # 脳波割り当て
            if self.brain_parser and brain_duration and brain_duration > 0 and m.perf_notes:
                if perf_duration > 0:
                    m_start = m.perf_start_time if m.perf_start_time else min(n.start_time for n in m.perf_notes)
                    m_end = m.perf_end_time if m.perf_end_time else max(n.end_time for n in m.perf_notes)

                    # 演奏時刻を脳波時刻に変換
                    brain_start = ((m_start - perf_start) / perf_duration) * brain_duration
                    brain_end = ((m_end - perf_start) / perf_duration) * brain_duration

                    summary = self.brain_parser.get_states_for_time_range(brain_start, brain_end)
                    m.brain_state = summary.get('dominant_state', '')
                    m.brain_metrics = {
                        'focus_ratio': summary.get('focus_ratio', 0.0),
                        'relax_ratio': summary.get('relax_ratio', 0.0),
                        'count': summary.get('count', 0),
                    }

        result = self._calc_total(measures)
        print(f"  ✓ 総合スコア: {result['total_score']:.1f}")

        if self.brain_parser and self.brain_parser.aligned_data:
            result['brain_summary'] = self.brain_parser.get_session_summary()

        return result

    def _score_pitch(self, m):
        if not m.ref_notes: return 1.0 if not m.perf_notes else 0.0
        if not m.perf_notes: return 0.0
        ref_groups = self._group_by_time(m.ref_notes, self.params.chord_time_window)
        perf_groups = self._group_by_time(m.perf_notes, self.params.arpeggio_max_time if self.params.allow_arpeggio else self.params.chord_time_window)
        total = sum(len(g['pitches']) for g in ref_groups)
        pool = set(p for g in perf_groups for p in g['pitches'])
        matched = sum(1 for g in ref_groups for rp in g['pitches'] if rp in pool or any(self._match(rp, pp) for pp in pool))
        return matched / total if total > 0 else 1.0

    def _group_by_time(self, notes, window):
        if not notes: return []
        notes = sorted(notes, key=lambda n: n.start_time)
        groups = [{'time': notes[0].start_time, 'pitches': [notes[0].pitch]}]
        for n in notes[1:]:
            if n.start_time - groups[-1]['time'] <= window:
                groups[-1]['pitches'].append(n.pitch)
            else:
                groups.append({'time': n.start_time, 'pitches': [n.pitch]})
        return groups

    def _match(self, rp, pp):
        d = abs(rp - pp)
        return d == 0 or d <= self.params.pitch_tolerance or (self.params.ignore_octave and d % 12 == 0)

    def _score_rhythm(self, m):
        if not m.ref_notes or not m.perf_notes: return 1.0 if not m.ref_notes else 0.0
        def pos(notes, s, e):
            d = e - s if e > s else 0.1
            return [(n.start_time - s) / d for n in notes]
        ref_pos = pos(m.ref_notes, m.start_time, m.end_time)
        perf_pos = pos(m.perf_notes, m.perf_start_time, m.perf_end_time) if m.perf_end_time > m.perf_start_time else pos(m.perf_notes, m.perf_notes[0].start_time, m.perf_notes[-1].start_time + 0.1)
        tol = self.params.rhythm_tolerance
        return sum(1 for rp in ref_pos if any(abs(rp - pp) <= tol for pp in perf_pos)) / len(ref_pos) if ref_pos else 1.0

    def _score_tempo(self, m):
        if not m.perf_notes or len(m.perf_notes) < 2: return 1.0
        pd, rd = m.perf_end_time - m.perf_start_time, m.duration
        if pd <= 0 or rd <= 0: return 1.0
        r = pd / rd
        return 1.0 if 1 - self.params.tempo_tolerance <= r <= 1 + self.params.tempo_tolerance else max(0, 1 - (abs(r - 1) - self.params.tempo_tolerance) * 2)

    def _calc_total(self, measures):
        played = [m for m in measures if m.perf_notes]
        if not played:
            return {'total_score': 0, 'pitch_score': 0, 'rhythm_score': 0, 'tempo_score': 0, 'measures_played': 0, 'measures_total': len(measures), 'measure_details': []}
        ap, ar, at = np.mean([m.pitch_score for m in played]), np.mean([m.rhythm_score for m in played]), np.mean([m.tempo_score for m in played])
        return {
            'total_score': (ap * 0.5 + ar * 0.3 + at * 0.2) * 100, 'pitch_score': ap * 100, 'rhythm_score': ar * 100, 'tempo_score': at * 100,
            'measures_played': len(played), 'measures_total': len(measures),
            'start_measure': measures[min(m.index for m in played)].number, 'end_measure': measures[max(m.index for m in played)].number,
            'measure_details': [{'number': m.number, 'index': m.index, 'pitch': m.pitch_score*100, 'rhythm': m.rhythm_score*100, 'tempo': m.tempo_score*100, 'ref_notes': len(m.ref_notes), 'perf_notes': len(m.perf_notes), 'brain_state': m.brain_state, 'brain_metrics': m.brain_metrics} for m in measures]
        }

# -----------------------------
# Visualizer
# -----------------------------
class Visualizer:
    def show_summary(self, result):
        from IPython.display import display, HTML
        r = result
        brain_html = ""
        if 'brain_summary' in r and r['brain_summary'].get('available'):
            bs = r['brain_summary']
            brain_html = f"""<div style="margin-top:15px;padding:10px;background:rgba(255,255,255,0.1);border-radius:8px;"><div style="font-size:14px;margin-bottom:5px;">🧠 脳波 ({bs['total_samples']}サンプル, {bs.get('duration', 0):.1f}秒)</div><div style="display:flex;gap:15px;flex-wrap:wrap;"><span>🟢集中: {bs['focus_ratio']*100:.0f}%</span><span>🔵リラックス: {bs['relax_ratio']*100:.0f}%</span><span>⚪中立: {bs['neutral_ratio']*100:.0f}%</span></div></div>"""
        html = f"""<div style="background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);padding:20px;border-radius:15px;color:white;margin:10px 0;"><h2 style="margin:0 0 15px 0;">🎹 採点結果</h2><div style="display:flex;justify-content:space-around;flex-wrap:wrap;"><div style="text-align:center;padding:10px;"><div style="font-size:36px;font-weight:bold;">{r['total_score']:.1f}</div><div>総合</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['pitch_score']:.1f}</div><div>🎵ピッチ</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['rhythm_score']:.1f}</div><div>🥁リズム</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['tempo_score']:.1f}</div><div>⏱️テンポ</div></div></div><div style="margin-top:10px;">演奏: {r['measures_played']}/{r['measures_total']}小節</div>{brain_html}</div>"""
        display(HTML(html))

    def show_table(self, result, only_played=True):
        import pandas as pd
        from IPython.display import display
        df = pd.DataFrame(result.get('measure_details', []))
        if df.empty: return
        if only_played: df = df[df['perf_notes'] > 0]
        has_brain = any(d.get('brain_state') for d in result.get('measure_details', []))
        if has_brain:
            df['brain'] = df.apply(lambda r: (r['brain_state'][:3] if r.get('brain_state') else '-'), axis=1)
            df = df[['number', 'index', 'pitch', 'rhythm', 'tempo', 'ref_notes', 'perf_notes', 'brain']]
            df.columns = ['小節', '通し', 'ピッチ', 'リズム', 'テンポ', '楽譜', '演奏', '脳波']
        else:
            df = df[['number', 'index', 'pitch', 'rhythm', 'tempo', 'ref_notes', 'perf_notes']]
            df.columns = ['小節', '通し', 'ピッチ', 'リズム', 'テンポ', '楽譜', '演奏']
        def color(v):
            try:
                v = float(v)
                return 'background-color:#90EE90' if v >= 80 else 'background-color:#FFE4B5' if v >= 50 else 'background-color:#FFB6C1' if v > 0 else ''
            except: return ''
        display(df.style.applymap(color, subset=['ピッチ', 'リズム', 'テンポ']))

    def show_measure_detail(self, m):
        """小節の詳細情報をテキストで表示（音符リスト付き）"""
        from IPython.display import display, HTML
        def p2n(p):
            names = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
            return f"{names[p%12]}{p//12-1}"

        ref_pitches = sorted(set(n.pitch for n in m.ref_notes))
        perf_pitches = sorted(set(n.pitch for n in m.perf_notes))
        matched = set(ref_pitches) & set(perf_pitches)
        missing = set(ref_pitches) - set(perf_pitches)
        extra = set(perf_pitches) - set(ref_pitches)

        brain_html = ""
        if m.brain_state:
            metrics = m.brain_metrics or {}
            brain_html = f"""<div style="background:#e9ecef;padding:10px;border-radius:4px;margin:10px 0;">
                🧠 <b>{m.brain_state}</b> | 集中率: {metrics.get('focus_ratio', 0)*100:.0f}% | サンプル数: {metrics.get('count', 0)}
            </div>"""

        html = f"""<div style="border:2px solid #333;padding:15px;border-radius:8px;background:#fafafa;">
            <h3>小節 {m.number} (通し番号: {m.index})</h3>
            <div style="display:flex;gap:10px;margin:10px 0;">
                <span style="padding:5px 12px;border-radius:4px;background:{'#90EE90' if m.pitch_score>=0.8 else '#FFE4B5' if m.pitch_score>=0.5 else '#FFB6C1'};">🎵 ピッチ: {m.pitch_score*100:.1f}%</span>
                <span style="padding:5px 12px;border-radius:4px;background:{'#90EE90' if m.rhythm_score>=0.8 else '#FFE4B5' if m.rhythm_score>=0.5 else '#FFB6C1'};">🥁 リズム: {m.rhythm_score*100:.1f}%</span>
                <span style="padding:5px 12px;border-radius:4px;background:{'#90EE90' if m.tempo_score>=0.8 else '#FFE4B5' if m.tempo_score>=0.5 else '#FFB6C1'};">⏱️ テンポ: {m.tempo_score*100:.1f}%</span>
            </div>
            {brain_html}
            <div style="background:#f0f0f0;padding:10px;border-radius:5px;margin-top:10px;">
                <div style="color:green;margin:3px 0;">✓ 一致: {', '.join(p2n(p) for p in sorted(matched)) or 'なし'}</div>
                <div style="color:red;margin:3px 0;">✗ 不足: {', '.join(p2n(p) for p in sorted(missing)) or 'なし'}</div>
                <div style="color:orange;margin:3px 0;">+ 余分: {', '.join(p2n(p) for p in sorted(extra)) or 'なし'}</div>
            </div>
            <div style="margin-top:10px;font-size:12px;color:#666;">
                楽譜: {len(m.ref_notes)}音 ({m.start_time:.2f}s - {m.end_time:.2f}s) |
                演奏: {len(m.perf_notes)}音 ({m.perf_start_time:.2f}s - {m.perf_end_time:.2f}s)
            </div>
        </div>"""
        display(HTML(html))

    def play_measure_audio(self, m, play_ref=True):
        """小節の音符をWebAudioで再生（JavaScript）"""
        from IPython.display import display, Javascript
        import json

        if play_ref:
            notes = [{'pitch': n.pitch, 'start': n.start_time - m.start_time, 'dur': min(n.duration, 2.0)} for n in m.ref_notes]
            label = "楽譜"
        else:
            if not m.perf_notes:
                print("演奏データがありません")
                return
            base = m.perf_notes[0].start_time
            notes = [{'pitch': n.pitch, 'start': n.start_time - base, 'dur': min(n.duration, 2.0)} for n in m.perf_notes]
            label = "演奏"

        print(f"🔊 小節{m.number}の{label}を再生中...")

        js_code = f"""
(function() {{
    var notes = {json.dumps(notes)};
    var ctx = new (window.AudioContext || window.webkitAudioContext)();
    var real = new Float32Array([0, 1.0, 0.5, 0.35, 0.2, 0.12, 0.08, 0.05, 0.03]);
    var imag = new Float32Array(real.length);
    var pianoWave = ctx.createPeriodicWave(real, imag);

    notes.forEach(function(n) {{
        var o = ctx.createOscillator();
        var g = ctx.createGain();
        o.connect(g);
        g.connect(ctx.destination);
        o.frequency.value = 440 * Math.pow(2, (n.pitch - 69) / 12);
        o.setPeriodicWave(pianoWave);
        var s = Math.max(0, n.start);
        var d = n.dur;
        g.gain.setValueAtTime(0, ctx.currentTime + s);
        g.gain.linearRampToValueAtTime(0.3, ctx.currentTime + s + 0.01);
        g.gain.exponentialRampToValueAtTime(0.001, ctx.currentTime + s + d + 0.3);
        o.start(ctx.currentTime + s);
        o.stop(ctx.currentTime + s + d + 0.4);
    }});
}})();
"""
        display(Javascript(js_code))

    def show_brain_timeline(self, measures):
        import matplotlib.pyplot as plt
        played = [m for m in measures if m.perf_notes]
        if not played: print("演奏小節がありません"); return
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
        indices = [m.index for m in played]
        scores = [(m.pitch_score * 0.5 + m.rhythm_score * 0.3 + m.tempo_score * 0.2) * 100 for m in played]
        colors = ['#90EE90' if s >= 80 else '#FFE4B5' if s >= 50 else '#FFB6C1' for s in scores]
        ax1.bar(indices, scores, color=colors, edgecolor='gray')
        ax1.axhline(y=70, color='red', linestyle='--', alpha=0.5, label='70%')
        ax1.set_ylabel('Score (%)')
        ax1.set_ylim(0, 100)
        ax1.legend()
        ax1.set_title('Score and Brain State by Measure')
        state_colors = {'FOCUSED': '#28a745', 'RELAXED': '#007bff', 'NEUTRAL': '#6c757d', 'NO_DATA': '#dc3545'}
        brain_colors = [state_colors.get(m.brain_state, '#999') for m in played]
        ax2.bar(indices, [1]*len(indices), color=brain_colors, edgecolor='gray')
        ax2.set_ylabel('Brain')
        ax2.set_xlabel('Measure Index')
        ax2.set_yticks([])
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=c, label=s) for s, c in state_colors.items()]
        ax2.legend(handles=legend_elements, loc='upper right', ncol=4)
        plt.tight_layout()
        plt.show()

        # 統計
        state_counts = {}
        for m in played:
            st = m.brain_state or 'NO_DATA'
            state_counts[st] = state_counts.get(st, 0) + 1
        print("\n脳波状態の統計:")
        for st, cnt in sorted(state_counts.items(), key=lambda x: -x[1]):
            print(f"  {st}: {cnt}小節 ({cnt/len(played)*100:.1f}%)")

    def show_measure_comparison(self, m):
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        def plot(ax, notes, off, c, title):
            if not notes: ax.text(0.5, 0.5, 'No notes', ha='center', va='center', transform=ax.transAxes); ax.set_title(title); return
            for n in notes: ax.barh(n.pitch, n.duration, left=n.start_time - off, height=0.6, color=c, alpha=0.6)
            ax.set_ylim(min(n.pitch for n in notes) - 2, max(n.pitch for n in notes) + 2)
            ax.set_xlabel('Time (s)'); ax.set_ylabel('Pitch'); ax.set_title(title); ax.grid(True, alpha=0.3)
        plot(axes[0], m.ref_notes, m.start_time, 'blue', f'Score ({len(m.ref_notes)} notes)')
        plot(axes[1], m.perf_notes, m.perf_notes[0].start_time if m.perf_notes else 0, 'red', f'Perf ({len(m.perf_notes)} notes)')
        brain = f" | 🧠{m.brain_state}" if m.brain_state else ""
        fig.suptitle(f"Measure {m.number} | P:{m.pitch_score*100:.1f}% | R:{m.rhythm_score*100:.1f}%{brain}", y=1.02)
        plt.tight_layout(); plt.show()

    def show_problem_measures(self, measures, threshold=0.7):
        from IPython.display import display, HTML
        problems = []
        for m in measures:
            issues = []
            if m.pitch_score < threshold: issues.append(f"P:{m.pitch_score*100:.0f}%")
            if m.rhythm_score < threshold: issues.append(f"R:{m.rhythm_score*100:.0f}%")
            if issues: problems.append({'num': m.number, 'idx': m.index, 'issues': issues, 'brain': m.brain_state or '-'})
        if not problems: print("✓ No problems (all >= 70%)"); return
        html = f"<h3>⚠️ Problems ({len(problems)})</h3><table style='border-collapse:collapse;'><tr style='background:#f0f0f0;'><th style='padding:5px;border:1px solid #ddd;'>M#</th><th style='padding:5px;border:1px solid #ddd;'>Idx</th><th style='padding:5px;border:1px solid #ddd;'>Issues</th><th style='padding:5px;border:1px solid #ddd;'>Brain</th></tr>"
        for p in problems[:30]: html += f"<tr><td style='padding:5px;border:1px solid #ddd;'>{p['num']}</td><td style='padding:5px;border:1px solid #ddd;'>{p['idx']}</td><td style='padding:5px;border:1px solid #ddd;color:red;'>{', '.join(p['issues'])}</td><td style='padding:5px;border:1px solid #ddd;'>{p['brain']}</td></tr>"
        html += "</table>"
        display(HTML(html))

    def show_pitch_histogram(self, measures):
        import matplotlib.pyplot as plt
        ref_p = [n.pitch for m in measures for n in m.ref_notes]
        perf_p = [n.pitch for m in measures for n in m.perf_notes]
        if not ref_p and not perf_p: print("No data"); return
        fig, ax = plt.subplots(figsize=(12, 4))
        bins = range(min(ref_p + perf_p) - 1, max(ref_p + perf_p) + 2)
        ax.hist(ref_p, bins=bins, alpha=0.5, label=f'Score({len(ref_p)})', color='blue')
        ax.hist(perf_p, bins=bins, alpha=0.5, label=f'Perf({len(perf_p)})', color='red')
        ax.legend(); ax.grid(True, alpha=0.3); plt.show()

    # ========== 卒論用エクスポート機能 ==========

    def export_thesis_data(self, measures, result, filename_prefix="thesis_data"):
        """卒論用のデータをCSVとしてエクスポート"""
        import pandas as pd
        from datetime import datetime

        # 1. 小節ごとの詳細データ
        measure_data = []
        for m in measures:
            if not m.perf_notes:
                continue
            measure_data.append({
                'measure_number': m.number,
                'measure_index': m.index,
                'pitch_score': m.pitch_score * 100,
                'rhythm_score': m.rhythm_score * 100,
                'tempo_score': m.tempo_score * 100,
                'total_score': (m.pitch_score * 0.5 + m.rhythm_score * 0.3 + m.tempo_score * 0.2) * 100,
                'ref_notes': len(m.ref_notes),
                'perf_notes': len(m.perf_notes),
                'brain_state': m.brain_state or 'NO_DATA',
                'focus_ratio': m.brain_metrics.get('focus_ratio', 0) if m.brain_metrics else 0,
                'brain_samples': m.brain_metrics.get('count', 0) if m.brain_metrics else 0,
            })

        df_measures = pd.DataFrame(measure_data)

        # 2. サマリーデータ
        summary_data = {
            'total_score': result['total_score'],
            'pitch_score': result['pitch_score'],
            'rhythm_score': result['rhythm_score'],
            'tempo_score': result['tempo_score'],
            'measures_played': result['measures_played'],
            'measures_total': result['measures_total'],
        }
        if 'brain_summary' in result and result['brain_summary'].get('available'):
            bs = result['brain_summary']
            summary_data.update({
                'brain_samples': bs['total_samples'],
                'brain_duration': bs['duration'],
                'focus_ratio': bs['focus_ratio'],
                'relax_ratio': bs['relax_ratio'],
                'neutral_ratio': bs['neutral_ratio'],
            })

        df_summary = pd.DataFrame([summary_data])

        # 3. 脳波状態別の統計
        if df_measures['brain_state'].notna().any():
            brain_stats = df_measures.groupby('brain_state').agg({
                'pitch_score': ['mean', 'std', 'count'],
                'rhythm_score': ['mean', 'std'],
                'total_score': ['mean', 'std'],
            }).round(2)
            brain_stats.columns = ['_'.join(col) for col in brain_stats.columns]
            brain_stats = brain_stats.reset_index()
        else:
            brain_stats = pd.DataFrame()

        # ファイル保存
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        measures_file = f"{filename_prefix}_measures_{timestamp}.csv"
        summary_file = f"{filename_prefix}_summary_{timestamp}.csv"
        brain_file = f"{filename_prefix}_brain_stats_{timestamp}.csv"

        df_measures.to_csv(measures_file, index=False, encoding='utf-8-sig')
        df_summary.to_csv(summary_file, index=False, encoding='utf-8-sig')
        if not brain_stats.empty:
            brain_stats.to_csv(brain_file, index=False, encoding='utf-8-sig')

        print(f"✓ エクスポート完了:")
        print(f"  - {measures_file} (小節ごとデータ)")
        print(f"  - {summary_file} (サマリー)")
        if not brain_stats.empty:
            print(f"  - {brain_file} (脳波状態別統計)")

        return df_measures, df_summary, brain_stats

    def show_thesis_analysis(self, measures, result):
        """卒論用の統計分析を表示"""
        import pandas as pd
        import matplotlib.pyplot as plt
        from IPython.display import display, HTML

        # データフレーム作成
        data = []
        for m in measures:
            if not m.perf_notes:
                continue
            data.append({
                'measure': m.number,
                'pitch': m.pitch_score * 100,
                'rhythm': m.rhythm_score * 100,
                'tempo': m.tempo_score * 100,
                'total': (m.pitch_score * 0.5 + m.rhythm_score * 0.3 + m.tempo_score * 0.2) * 100,
                'brain': m.brain_state or 'NO_DATA',
            })

        df = pd.DataFrame(data)

        # 1. 基本統計
        html = "<h2>📊 卒論用統計分析</h2>"
        html += "<h3>1. スコア基本統計</h3>"
        stats = df[['pitch', 'rhythm', 'tempo', 'total']].describe().round(2)
        html += stats.to_html()

        # 2. 脳波状態別統計
        if df['brain'].notna().any() and len(df['brain'].unique()) > 1:
            html += "<h3>2. 脳波状態別スコア</h3>"
            brain_stats = df.groupby('brain')[['pitch', 'rhythm', 'tempo', 'total']].agg(['mean', 'std', 'count']).round(2)
            html += brain_stats.to_html()

            # 3. 統計検定（FOCUSED vs NEUTRAL）
            from scipy import stats as scipy_stats
            focused = df[df['brain'] == 'FOCUSED']['total']
            neutral = df[df['brain'] == 'NEUTRAL']['total']
            relaxed = df[df['brain'] == 'RELAXED']['total']

            html += "<h3>3. 統計検定</h3>"
            if len(focused) >= 2 and len(neutral) >= 2:
                t_stat, p_val = scipy_stats.ttest_ind(focused, neutral)
                sig = "有意 (p<0.05)" if p_val < 0.05 else "有意差なし"
                html += f"<p><b>FOCUSED vs NEUTRAL:</b> t={t_stat:.3f}, p={p_val:.4f} → {sig}</p>"
                html += f"<p>  FOCUSED: M={focused.mean():.1f}, SD={focused.std():.1f}, N={len(focused)}</p>"
                html += f"<p>  NEUTRAL: M={neutral.mean():.1f}, SD={neutral.std():.1f}, N={len(neutral)}</p>"

            if len(relaxed) >= 2 and len(neutral) >= 2:
                t_stat, p_val = scipy_stats.ttest_ind(relaxed, neutral)
                sig = "有意 (p<0.05)" if p_val < 0.05 else "有意差なし"
                html += f"<p><b>RELAXED vs NEUTRAL:</b> t={t_stat:.3f}, p={p_val:.4f} → {sig}</p>"

        display(HTML(html))

        # 4. グラフ
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        # 4-1. スコア分布
        df[['pitch', 'rhythm', 'tempo', 'total']].hist(ax=axes[0], bins=20, alpha=0.7)
        axes[0].set_title('Score Distribution')

        # 4-2. 脳波状態別boxplot
        if len(df['brain'].unique()) > 1:
            df.boxplot(column='total', by='brain', ax=axes[1])
            axes[1].set_title('Total Score by Brain State')
            axes[1].set_xlabel('Brain State')
            axes[1].set_ylabel('Total Score')
            plt.suptitle('')

        # 4-3. スコア推移
        axes[2].plot(df['measure'], df['total'], 'b-', alpha=0.7, label='Total')
        axes[2].axhline(y=70, color='red', linestyle='--', alpha=0.5)
        axes[2].set_xlabel('Measure')
        axes[2].set_ylabel('Score')
        axes[2].set_title('Score Progression')
        axes[2].legend()

        plt.tight_layout()
        plt.show()

        return df

    def show_score_with_click(self, mxl_path, measures):
        from IPython.display import display, HTML
        import verovio, re, json
        try:
            tk = verovio.toolkit()
            tk.setOptions({"pageWidth": 1800, "pageHeight": 3000, "scale": 35, "adjustPageHeight": True})
            tk.loadFile(mxl_path)
            pc = tk.getPageCount()
            amd = [{'index': m.index, 'number': m.number, 'pitch': m.pitch_score*100, 'rhythm': m.rhythm_score*100, 'ref_notes': len(m.ref_notes), 'perf_notes': len(m.perf_notes), 'brain_state': m.brain_state, 'start_time': m.start_time, 'end_time': m.end_time, 'perf_start': m.perf_notes[0].start_time if m.perf_notes else 0, 'ref_note_list': [{'pitch': n.pitch, 'start': n.start_time, 'dur': n.duration} for n in m.ref_notes], 'perf_note_list': [{'pitch': n.pitch, 'start': n.start_time, 'dur': n.duration} for n in m.perf_notes]} for m in measures]
            mc = {i: "#90EE90" if (m.pitch_score + m.rhythm_score)/2*100 >= 90 else "#FFFF99" if (m.pitch_score + m.rhythm_score)/2*100 >= 70 else "#FFE4B5" if (m.pitch_score + m.rhythm_score)/2*100 >= 50 else "#FFB6C1" for i, m in enumerate(measures)}
            svgs = []
            measure_idx = 0
            for p in range(1, pc + 1):
                svg = tk.renderToSVG(p)
                def add_attr(match):
                    nonlocal measure_idx
                    result = match.group(0)
                    if 'data-measure-idx' not in result:
                        result = result[:-1] + f' data-measure-idx="{measure_idx}" style="cursor:pointer;">'
                        measure_idx += 1
                    return result
                svg = re.sub(r'<g[^>]*class="[^"]*measure[^"]*"[^>]*>', add_attr, svg)
                svgs.append(svg)
            html = f'''<style>.score-viewer{{display:grid;grid-template-columns:1.5fr 1fr;gap:15px;height:700px;}}.score-panel{{overflow:auto;border:1px solid #ccc;padding:10px;background:white;}}.detail-panel{{border:1px solid #ccc;border-radius:8px;background:#f8f9fa;padding:15px;overflow-y:auto;}}.measure-highlight{{cursor:pointer;}}.measure-highlight:hover{{filter:brightness(0.85);}}</style><div class="score-viewer"><div class="score-panel" id="score-panel">{"".join(svgs)}</div><div class="detail-panel" id="detail-panel"><h3>Click a measure</h3></div></div><script>(function(){{var mc={json.dumps(mc)};var am={json.dumps(amd)};var pn=['C','C#','D','D#','E','F','F#','G','G#','A','A#','B'];function p2n(p){{return pn[p%12]+(Math.floor(p/12)-1);}}function createPianoWave(ctx){{var real=new Float32Array([0,1.0,0.5,0.35,0.2,0.12,0.08,0.05,0.03]);var imag=new Float32Array(real.length);return ctx.createPeriodicWave(real,imag);}}function show(i){{var m=am[i];if(!m)return;var rp=new Set(m.ref_note_list.map(n=>n.pitch));var pp=new Set(m.perf_note_list.map(n=>n.pitch));var mt=[...rp].filter(p=>pp.has(p));var ms=[...rp].filter(p=>!pp.has(p));var h='<h3>M'+m.number+'</h3><div style="margin:10px 0;"><span style="padding:4px 8px;border-radius:4px;background:'+(m.pitch>=80?'#90EE90':'#FFB6C1')+';">P:'+m.pitch.toFixed(1)+'%</span> <span style="padding:4px 8px;border-radius:4px;background:'+(m.rhythm>=80?'#87CEEB':'#FFB6C1')+';">R:'+m.rhythm.toFixed(1)+'%</span></div>';if(m.brain_state)h+='<div style="background:#e9ecef;padding:5px;border-radius:4px;margin:5px 0;">🧠'+m.brain_state+'</div>';h+='<div style="margin:10px 0;"><button onclick="playRef('+i+')" style="padding:8px 16px;margin:5px;cursor:pointer;border:none;border-radius:4px;background:#4CAF50;color:white;">🔊楽譜</button>';if(m.perf_notes>0)h+='<button onclick="playPerf('+i+')" style="padding:8px 16px;margin:5px;cursor:pointer;border:none;border-radius:4px;background:#f44336;color:white;">🔊演奏</button>';h+='</div><div style="font-size:11px;"><span style="color:green;">✓'+mt.map(p=>p2n(p)).join(',')||'none'+'</span><br><span style="color:red;">✗'+ms.map(p=>p2n(p)).join(',')||'none'+'</span></div>';document.getElementById('detail-panel').innerHTML=h;}}function playRef(i){{var m=am[i];if(!m||!m.ref_note_list.length)return;var ctx=new(window.AudioContext||window.webkitAudioContext)();var pianoWave=createPianoWave(ctx);m.ref_note_list.forEach(n=>{{var o=ctx.createOscillator();var g=ctx.createGain();o.connect(g);g.connect(ctx.destination);o.frequency.value=440*Math.pow(2,(n.pitch-69)/12);o.setPeriodicWave(pianoWave);var s=Math.max(0,n.start-m.start_time);var d=Math.min(n.dur,2.0);g.gain.setValueAtTime(0,ctx.currentTime+s);g.gain.linearRampToValueAtTime(0.35,ctx.currentTime+s+0.008);g.gain.exponentialRampToValueAtTime(0.001,ctx.currentTime+s+d+0.5);o.start(ctx.currentTime+s);o.stop(ctx.currentTime+s+d+0.6);}});}}function playPerf(i){{var m=am[i];if(!m||!m.perf_note_list.length)return;var ctx=new(window.AudioContext||window.webkitAudioContext)();var pianoWave=createPianoWave(ctx);m.perf_note_list.forEach(n=>{{var o=ctx.createOscillator();var g=ctx.createGain();o.connect(g);g.connect(ctx.destination);o.frequency.value=440*Math.pow(2,(n.pitch-69)/12);o.setPeriodicWave(pianoWave);var s=Math.max(0,n.start-m.perf_start);var d=Math.min(n.dur,2.0);g.gain.setValueAtTime(0,ctx.currentTime+s);g.gain.linearRampToValueAtTime(0.3,ctx.currentTime+s+0.008);g.gain.exponentialRampToValueAtTime(0.001,ctx.currentTime+s+d+0.5);o.start(ctx.currentTime+s);o.stop(ctx.currentTime+s+d+0.6);}});}}window.playRef=playRef;window.playPerf=playPerf;setTimeout(function(){{var panel=document.getElementById('score-panel');if(!panel)return;var els=panel.querySelectorAll('[data-measure-idx]');if(!els.length){{els=panel.querySelectorAll('g[class*="measure"]');els.forEach(function(el,idx){{el.setAttribute('data-measure-idx',idx);}});}}els.forEach(function(el){{var idx=parseInt(el.getAttribute('data-measure-idx'));var color=mc[idx]||'#FFF';el.classList.add('measure-highlight');try{{var bbox=el.getBBox();if(bbox.width>0){{var rect=document.createElementNS('http://www.w3.org/2000/svg','rect');rect.setAttribute('x',bbox.x-5);rect.setAttribute('y',bbox.y-5);rect.setAttribute('width',bbox.width+10);rect.setAttribute('height',bbox.height+10);rect.setAttribute('fill',color);rect.setAttribute('opacity','0.4');rect.style.pointerEvents='none';el.insertBefore(rect,el.firstChild);}}}}catch(e){{}}el.addEventListener('click',function(e){{e.stopPropagation();if(idx<am.length)show(idx);}});}});}},500);}})();</script>'''
            display(HTML(html))
        except Exception as e:
            print(f"楽譜表示エラー: {e}")

# -----------------------------
# UI
# -----------------------------
class PianoScorerUI:
    def __init__(self):
        self.params = ScoringParams()
        self.measures, self.ref_notes, self.perf_notes, self.result = [], [], [], None
        self.visualizer = Visualizer()
        self.audio_path = self.midi_path = self.mxl_path = self.recorded_audio = self.brain_csv_path = None
        self.audio_duration = None
        self.is_recording = False
        # 録音時刻を自動記録
        self.recording_start_time: Optional[datetime] = None
        self.recording_end_time: Optional[datetime] = None

    def _receive_audio_callback(self, base64_data, mime_type='audio/webm'):
        import base64, tempfile
        try:
            raw = base64.b64decode(base64_data)
            suffix = '.ogg' if 'ogg' in str(mime_type) else '.webm'
            with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
                f.write(raw)
                self.recorded_audio = f.name
            if self.input_mode.value == '録音':
                self.audio_path = self.recorded_audio
            self.record_status.value = '<span style="color:green;">✓完了</span>'
        except Exception as e:
            self.record_status.value = f'<span style="color:red;">Error:{e}</span>'

    def create_ui(self):
        import ipywidgets as widgets
        from IPython.display import display, HTML

        display(HTML("""<div style="background:linear-gradient(135deg,#1a1a2e 0%,#16213e 100%);padding:20px;border-radius:15px;color:white;margin-bottom:20px;">
            <h1 style="margin:0;">🎹🧠 Piano Performance Scorer v4.4</h1>
            <p style="margin:5px 0 0 0;opacity:0.8;">オフライン脳波統合版（Mind Monitor CSVアップロード）</p>
        </div>"""))

        # ---- 脳波CSV ----
        self.brain_enabled = widgets.Checkbox(value=False, description='🧠脳波CSVを使用')
        self.brain_enabled.observe(self._on_brain_toggle, names='value')
        self.brain_csv_upload = widgets.FileUpload(accept='.csv', multiple=False, description='脳波CSV')
        self.rec_start_input = widgets.Text(placeholder='HH:MM:SS', description='録音開始:', layout=widgets.Layout(width='180px'))
        self.rec_end_input = widgets.Text(placeholder='HH:MM:SS', description='録音終了:', layout=widgets.Layout(width='180px'))
        self.parse_brain_btn = widgets.Button(description='🧠CSV解析', button_style='info', layout=widgets.Layout(width='100px'))
        self.parse_brain_btn.on_click(self._on_parse_brain)
        self.brain_status = widgets.HTML(value='<span style="color:gray;">CSVをアップロード</span>')
        self.brain_controls = widgets.VBox([
            widgets.HBox([widgets.Label("脳波CSV:"), self.brain_csv_upload]),
            widgets.HBox([self.rec_start_input, self.rec_end_input, self.parse_brain_btn]),
            self.brain_status
        ])
        self.brain_controls.layout.display = 'none'
        brain_info = widgets.HTML("""<div style="background:#e8f4f8;padding:10px;border-radius:5px;margin:5px 0;font-size:12px;">
            <b>使い方:</b> Mind Monitorで録音中に脳波を記録 → CSVエクスポート → 録音開始/終了時刻を入力 → CSV解析
        </div>""")
        brain_box = widgets.VBox([widgets.HTML("<h3>🧠脳波CSV</h3>"), self.brain_enabled, self.brain_controls, brain_info])

        # ---- ファイル/録音 ----
        self.input_mode = widgets.RadioButtons(options=['ファイル', '録音'], value='ファイル', layout=widgets.Layout(width='100px'))
        self.input_mode.observe(self._on_input_mode_change, names='value')
        self.audio_upload = widgets.FileUpload(accept='.mp3,.wav,.m4a,.webm,.ogg', multiple=False)
        self.midi_upload = widgets.FileUpload(accept='.mid,.midi', multiple=False)
        self.mxl_upload = widgets.FileUpload(accept='.mxl,.xml,.musicxml', multiple=False)
        self.record_btn = widgets.Button(description='🎤録音', button_style='info', layout=widgets.Layout(width='80px'))
        self.record_btn.on_click(self._on_record)
        self.gain_slider = widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description='Gain:', layout=widgets.Layout(width='180px'))
        self.record_status = widgets.HTML(value='<span style="color:gray;">待機</span>')
        self.audio_file_box = widgets.VBox([widgets.Label("音声"), self.audio_upload])
        self.audio_record_box = widgets.VBox([widgets.HBox([self.record_btn, self.record_status]), self.gain_slider])
        self.audio_record_box.layout.display = 'none'
        upload_box = widgets.VBox([
            widgets.HTML("<h3>📁入力</h3>"),
            widgets.HBox([
                widgets.VBox([self.input_mode, self.audio_file_box, self.audio_record_box]),
                widgets.VBox([widgets.Label("MIDI"), self.midi_upload]),
                widgets.VBox([widgets.Label("MXL"), self.mxl_upload]),
            ])
        ])

        # ---- パラメータ ----
        self.pitch_tol = widgets.IntSlider(value=0, min=0, max=3, description='Pitch:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        self.ignore_oct = widgets.Checkbox(value=False, description='Oct無視')
        self.arpeggio = widgets.Checkbox(value=True, description='アルペジオ')
        self.rhythm_tol = widgets.FloatSlider(value=0.1, min=0.05, max=0.3, step=0.01, description='Rhythm:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        self.tempo_tol = widgets.FloatSlider(value=0.1, min=0.05, max=0.3, step=0.05, description='Tempo:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        params_box = widgets.VBox([
            widgets.HTML("<h3>⚙️パラメータ</h3>"),
            widgets.HBox([self.pitch_tol, self.ignore_oct, self.arpeggio]),
            widgets.HBox([self.rhythm_tol, self.tempo_tol]),
        ])

        # ---- ボタン ----
        self.run_btn = widgets.Button(description='🎵採点', button_style='success', layout=widgets.Layout(width='100px', height='40px'))
        self.run_btn.on_click(self._on_run)
        self.play_btn = widgets.Button(description='▶️再生', button_style='info', layout=widgets.Layout(width='80px'))
        self.play_btn.on_click(self._on_play)
        self.reset_rec_btn = widgets.Button(description='🔄録音リセット', button_style='warning', layout=widgets.Layout(width='100px'))
        self.reset_rec_btn.on_click(self._on_reset_recording)
        self.reset_all_btn = widgets.Button(description='🗑️全リセット', button_style='danger', layout=widgets.Layout(width='100px'))
        self.reset_all_btn.on_click(self._on_reset_all)
        btn_box = widgets.HBox([self.run_btn, self.play_btn, self.reset_rec_btn, self.reset_all_btn],
                               layout=widgets.Layout(justify_content='center', margin='15px 0'))

        self.output = widgets.Output()

        # ---- 分析 ----
        self.measure_sel = widgets.IntSlider(value=1, min=1, max=100, description='M#:', layout=widgets.Layout(width='200px'))
        self.show_btn = widgets.Button(description='📊グラフ', button_style='info', layout=widgets.Layout(width='70px'))
        self.show_btn.on_click(self._on_show_measure)
        self.show_detail_btn = widgets.Button(description='📋詳細', button_style='info', layout=widgets.Layout(width='60px'))
        self.show_detail_btn.on_click(self._on_show_detail)
        self.play_ref_btn = widgets.Button(description='🔊楽譜', button_style='success', layout=widgets.Layout(width='60px'))
        self.play_ref_btn.on_click(self._on_play_ref)
        self.play_perf_btn = widgets.Button(description='🔊演奏', button_style='danger', layout=widgets.Layout(width='60px'))
        self.play_perf_btn.on_click(self._on_play_perf)
        self.show_score_btn = widgets.Button(description='🎼楽譜全体', button_style='success', layout=widgets.Layout(width='80px'))
        self.show_score_btn.on_click(self._on_show_score)
        self.show_problems_btn = widgets.Button(description='⚠️問題', button_style='warning', layout=widgets.Layout(width='70px'))
        self.show_problems_btn.on_click(self._on_show_problems)
        self.show_brain_btn = widgets.Button(description='🧠タイムライン', button_style='info', layout=widgets.Layout(width='100px'))
        self.show_brain_btn.on_click(self._on_show_brain)
        self.show_all_btn = widgets.Button(description='📋全表', layout=widgets.Layout(width='60px'))
        self.show_all_btn.on_click(self._on_show_all)

        # 卒論用
        self.thesis_btn = widgets.Button(description='📊卒論分析', button_style='primary', layout=widgets.Layout(width='90px'))
        self.thesis_btn.on_click(self._on_thesis_analysis)
        self.export_btn = widgets.Button(description='💾CSV出力', button_style='primary', layout=widgets.Layout(width='80px'))
        self.export_btn.on_click(self._on_export)

        debug_box = widgets.VBox([
            widgets.HTML("<h3>🔍分析</h3>"),
            widgets.HBox([self.measure_sel, self.show_btn, self.show_detail_btn, self.play_ref_btn, self.play_perf_btn]),
            widgets.HBox([self.show_score_btn, self.show_problems_btn, self.show_brain_btn, self.show_all_btn, self.thesis_btn, self.export_btn]),
        ])
        self.debug_output = widgets.Output()

        try:
            from google.colab import output as colab_output
            colab_output.register_callback('piano.receive_audio', self._receive_audio_callback)
        except: pass

        display(widgets.VBox([brain_box, upload_box, params_box, btn_box, self.output, debug_box, self.debug_output]))

    def _on_brain_toggle(self, c):
        self.brain_controls.layout.display = 'block' if c['new'] else 'none'

    def _on_parse_brain(self, b):
        import tempfile
        with self.output:
            self.output.clear_output()
            if not self.brain_csv_upload.value:
                print("⚠️ 脳波CSVをアップロードしてください")
                return

            # CSVを一時ファイルに保存
            name = list(self.brain_csv_upload.value.keys())[0]
            content = self.brain_csv_upload.value[name]['content']
            with tempfile.NamedTemporaryFile(suffix='.csv', delete=False) as f:
                f.write(content)
                self.brain_csv_path = f.name

            # CSV解析
            if not brain_csv_parser.parse_csv(self.brain_csv_path):
                self.brain_status.value = '<span style="color:red;">CSV解析失敗</span>'
                return

            # 時刻設定（自動入力された値を使用）
            start_str = self.rec_start_input.value.strip()
            end_str = self.rec_end_input.value.strip()

            if not start_str or not end_str:
                # 録音時刻がない場合はCSV全体を使う提案
                csv_start = brain_csv_parser.start_time.strftime("%H:%M:%S")
                csv_end = brain_csv_parser.end_time.strftime("%H:%M:%S")
                self.brain_status.value = f'''<span style="color:orange;">
                    録音時刻を入力してください<br>
                    CSV範囲: {csv_start} - {csv_end}<br>
                    <small>録音ボタンを使うと自動入力されます</small>
                </span>'''
                return

            if not brain_csv_parser.set_recording_time(start_str, end_str):
                self.brain_status.value = '<span style="color:red;">時刻設定失敗</span>'
                return

            # アライメント
            if brain_csv_parser.align_to_recording():
                duration = (brain_csv_parser.recording_end - brain_csv_parser.recording_start).total_seconds()
                self.brain_status.value = f'''<span style="color:green;">
                    ✓ {len(brain_csv_parser.aligned_data)}サンプル準備完了<br>
                    {start_str} - {end_str} ({duration:.1f}秒)
                </span>'''
            else:
                self.brain_status.value = '<span style="color:red;">アライメント失敗（録音時間がCSV範囲外？）</span>'

    def _on_input_mode_change(self, c):
        if c['new'] == 'ファイル':
            self.audio_file_box.layout.display = 'block'
            self.audio_record_box.layout.display = 'none'
        else:
            self.audio_file_box.layout.display = 'none'
            self.audio_record_box.layout.display = 'block'

    def _update_params(self):
        self.params.pitch_tolerance = self.pitch_tol.value
        self.params.ignore_octave = self.ignore_oct.value
        self.params.allow_arpeggio = self.arpeggio.value
        self.params.rhythm_tolerance = self.rhythm_tol.value
        self.params.tempo_tolerance = self.tempo_tol.value

    def _on_record(self, b):
        from IPython.display import display, Javascript
        if not self.is_recording:
            self.is_recording = True
            self.record_btn.description = '⏹️停止'
            self.record_btn.button_style = 'danger'
            self.record_status.value = '<span style="color:red;">●REC</span>'

            # 録音開始時刻を記録
            self.recording_start_time = datetime.now()
            start_str = self.recording_start_time.strftime('%H:%M:%S')
            self.rec_start_input.value = start_str
            print(f"🎤 録音開始: {start_str}")

            g = self.gain_slider.value
            display(Javascript(f"""(async function(){{window.audioChunks=[];const stream=await navigator.mediaDevices.getUserMedia({{audio:true}});const ctx=new AudioContext();const src=ctx.createMediaStreamSource(stream);const gn=ctx.createGain();gn.gain.value={g};const dest=ctx.createMediaStreamDestination();src.connect(gn);gn.connect(dest);const candidates=['audio/webm;codecs=opus','audio/webm','audio/ogg;codecs=opus'];let chosen='';for(const c of candidates){{if(MediaRecorder.isTypeSupported(c)){{chosen=c;break;}}}}window.__recMime=chosen||'audio/webm';window.mediaRecorder=new MediaRecorder(dest.stream,chosen?{{mimeType:chosen}}:{{}});window.originalStream=stream;window.mediaRecorder.ondataavailable=(e)=>{{if(e.data&&e.data.size>0)window.audioChunks.push(e.data);}};window.mediaRecorder.start(1000);}})();"""))
        else:
            self.is_recording = False
            self.record_btn.description = '🎤録音'
            self.record_btn.button_style = 'info'
            self.record_status.value = '<span style="color:blue;">⏳</span>'

            # 録音終了時刻を記録
            self.recording_end_time = datetime.now()
            end_str = self.recording_end_time.strftime('%H:%M:%S')
            self.rec_end_input.value = end_str
            duration = (self.recording_end_time - self.recording_start_time).total_seconds()
            print(f"⏹️ 録音停止: {end_str} (録音時間: {duration:.1f}秒)")

            display(Javascript(r"""(async function(){if(window.mediaRecorder&&window.mediaRecorder.state==='recording'){window.mediaRecorder.stop();window.mediaRecorder.onstop=async()=>{if(window.originalStream)window.originalStream.getTracks().forEach(t=>t.stop());const blob=new Blob(window.audioChunks,{type:window.__recMime||'audio/webm'});const reader=new FileReader();reader.onloadend=function(){const base64=reader.result.split(',')[1];if(typeof google!=='undefined'&&google.colab){google.colab.kernel.invokeFunction('piano.receive_audio',[base64,window.__recMime],{});}};reader.readAsDataURL(blob);};}})();"""))

    def _on_play(self, b):
        from IPython.display import display, Audio
        with self.output:
            if self.recorded_audio: display(Audio(self.recorded_audio, autoplay=True))
            elif self.audio_path: display(Audio(self.audio_path, autoplay=True))
            else: print("再生する音声がありません")

    def _save_files(self):
        import tempfile
        if self.input_mode.value == 'ファイル' and self.audio_upload.value:
            name = list(self.audio_upload.value.keys())[0]
            content = self.audio_upload.value[name]['content']
            ext = '.' + name.split('.')[-1] if '.' in name else '.wav'
            with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
                f.write(content)
                self.audio_path = f.name
        elif self.input_mode.value == '録音' and self.recorded_audio:
            self.audio_path = self.recorded_audio
        if self.midi_upload.value:
            name = list(self.midi_upload.value.keys())[0]
            with tempfile.NamedTemporaryFile(suffix='.mid', delete=False) as f:
                f.write(self.midi_upload.value[name]['content'])
                self.midi_path = f.name
        if self.mxl_upload.value:
            name = list(self.mxl_upload.value.keys())[0]
            ext = '.' + name.split('.')[-1] if '.' in name else '.mxl'
            with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
                f.write(self.mxl_upload.value[name]['content'])
                self.mxl_path = f.name

    def _on_run(self, b):
        with self.output:
            self.output.clear_output()
            # デバッグ情報
            print(f"[DEBUG] input_mode: {self.input_mode.value}")
            print(f"[DEBUG] audio_upload.value: {bool(self.audio_upload.value)}")
            print(f"[DEBUG] recorded_audio: {self.recorded_audio}")
            print(f"[DEBUG] audio_path: {self.audio_path}")

            has_audio = bool(self.audio_upload.value) or bool(self.recorded_audio)
            if not has_audio: print("⚠️音声をアップロードまたは録音してください"); return
            if not self.midi_upload.value: print("⚠️MIDIをアップロードしてください"); return
            if not self.mxl_upload.value: print("⚠️MXLをアップロードしてください"); return
            self._update_params()
            try:
                self._save_files()
                if not self.audio_path: print("⚠️音声ファイルがありません"); return
                parser = ReferenceParser(self.midi_path, self.mxl_path)
                self.measures, self.ref_notes = parser.parse()
                if not self.measures: print("❌参照データ解析失敗"); return
                transcriber = AudioTranscriber(use_gpu=True)
                self.perf_notes, self.audio_duration = transcriber.transcribe(self.audio_path)
                aligner = DTWAligner(self.params)
                self.measures = aligner.align(self.perf_notes, self.ref_notes, self.measures)

                # 脳波統合
                brain_parser = brain_csv_parser if (self.brain_enabled.value and brain_csv_parser.aligned_data) else None
                scorer = ScoringEngine(self.params, brain_parser)
                self.result = scorer.score_all(self.measures, self.audio_duration)

                self.visualizer.show_summary(self.result)
                self.visualizer.show_table(self.result, only_played=True)
                self.measure_sel.max = len(self.measures)
            except Exception as e:
                print(f"❌Error: {e}")
                import traceback; traceback.print_exc()

    def _on_reset_recording(self, b):
        from IPython.display import display, Javascript
        self.audio_upload.value.clear()
        self.recorded_audio = None; self.audio_path = None; self.is_recording = False
        self.record_btn.description = '🎤録音'; self.record_btn.button_style = 'info'
        self.record_status.value = '<span style="color:gray;">待機</span>'
        display(Javascript('window.audioChunks=[];'))
        with self.output: self.output.clear_output(); print("✓ 録音リセット")

    def _on_reset_all(self, b):
        from IPython.display import display, Javascript
        self.audio_upload.value.clear(); self.midi_upload.value.clear(); self.mxl_upload.value.clear()
        self.brain_csv_upload.value.clear()
        self.audio_path = self.midi_path = self.mxl_path = self.recorded_audio = self.brain_csv_path = None
        self.measures, self.ref_notes, self.perf_notes, self.result = [], [], [], None
        self.is_recording = False; self.record_btn.description = '🎤録音'; self.record_btn.button_style = 'info'
        self.record_status.value = '<span style="color:gray;">待機</span>'
        self.brain_status.value = '<span style="color:gray;">CSVをアップロード</span>'
        brain_csv_parser.data = []; brain_csv_parser.aligned_data = []
        self.output.clear_output(); self.debug_output.clear_output()
        display(Javascript('window.audioChunks=[];'))
        with self.output: print("✓ 全リセット完了")

    def _on_show_measure(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.show_measure_comparison(self.measures[idx])

    def _on_show_detail(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.show_measure_detail(self.measures[idx])

    def _on_play_ref(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.play_measure_audio(self.measures[idx], play_ref=True)

    def _on_play_perf(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.play_measure_audio(self.measures[idx], play_ref=False)

    def _on_show_score(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures or not self.mxl_path: print("採点を実行してください"); return
            print("🎼 楽譜読み込み中...")
            self.visualizer.show_score_with_click(self.mxl_path, self.measures)

    def _on_show_problems(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            self.visualizer.show_problem_measures(self.measures)

    def _on_show_brain(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            self.visualizer.show_brain_timeline(self.measures)

    def _on_show_all(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.result: print("採点を実行してください"); return
            self.visualizer.show_table(self.result, only_played=False)

    def _on_thesis_analysis(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures or not self.result: print("採点を実行してください"); return
            try:
                self.visualizer.show_thesis_analysis(self.measures, self.result)
            except Exception as e:
                print(f"分析エラー: {e}")
                import traceback; traceback.print_exc()

    def _on_export(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures or not self.result: print("採点を実行してください"); return
            try:
                self.visualizer.export_thesis_data(self.measures, self.result)
            except Exception as e:
                print(f"エクスポートエラー: {e}")
                import traceback; traceback.print_exc()

def main():
    print("🎹🧠 Piano Performance Scorer v4.4 (Offline Brain Wave Integration)")
    print("=" * 60)
    install_dependencies()
    ui = PianoScorerUI()
    ui.create_ui()
    return ui

if __name__ == "__main__":
    ui = main()

In [ ]:
# ==============================================================================
# 🎹🧠 Piano Performance Scorer v4.4 - Offline Brain Wave Integration
# ==============================================================================
# Mind MonitorのCSVをアップロードして、録音と時間軸を合わせて脳波を小節に割り当て
#
# 使い方:
# 1. Mind Monitorで脳波を記録（CSVエクスポート）
# 2. 録音開始時刻と終了時刻をメモ（Mind Monitorの画面で確認）
# 3. Colabで録音 or 音声ファイルをアップロード
# 4. 脳波CSVをアップロード + 録音開始/終了時刻を入力
# 5. 採点実行 → 脳波と演奏が時間軸で統合される
# ==============================================================================

import numpy as np
import warnings
warnings.filterwarnings('ignore')

def install_dependencies():
    import subprocess, sys
    packages = [
        'pretty_midi', 'music21', 'librosa', 'matplotlib', 'pandas',
        'ipywidgets', 'transkun', 'dtw-python', 'verovio'
    ]
    for pkg in packages:
        try:
            __import__(pkg.replace('-', '_').replace('dtw-python', 'dtw'))
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    print("✓ 依存ライブラリ準備完了")

from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any
from datetime import datetime, timedelta
import threading
import time

@dataclass
class NoteEvent:
    pitch: int
    start_time: float
    end_time: float
    velocity: int = 64
    matched: bool = False
    @property
    def duration(self): return self.end_time - self.start_time
    def pitch_name(self):
        names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
        return f"{names[self.pitch % 12]}{(self.pitch // 12) - 1}"

@dataclass
class MeasureData:
    number: int
    index: int
    start_time: float
    end_time: float
    tempo: float
    time_signature: Tuple[int, int]
    ref_notes: List[NoteEvent] = field(default_factory=list)
    perf_notes: List[NoteEvent] = field(default_factory=list)
    pitch_score: float = 0.0
    rhythm_score: float = 0.0
    tempo_score: float = 0.0
    perf_start_time: float = 0.0
    perf_end_time: float = 0.0
    brain_state: str = ""
    brain_metrics: Dict = field(default_factory=dict)
    @property
    def duration(self): return self.end_time - self.start_time

@dataclass
class ScoringParams:
    pitch_tolerance: int = 0
    ignore_octave: bool = False
    chord_time_window: float = 0.05
    allow_arpeggio: bool = True
    arpeggio_max_time: float = 0.2
    rhythm_tolerance: float = 0.1
    tempo_tolerance: float = 0.1

# -----------------------------
# Brain Wave CSV Parser
# -----------------------------
class BrainWaveCSVParser:
    """Mind MonitorのCSVを解析"""

    def __init__(self):
        self.data = []
        self.start_time = None
        self.end_time = None
        self.recording_start = None  # 録音開始時刻（絶対時刻）
        self.recording_end = None    # 録音終了時刻（絶対時刻）
        self.aligned_data = []       # 録音に合わせた相対時刻データ

    def parse_csv(self, csv_path: str) -> bool:
        """Mind Monitor CSVを解析"""
        import pandas as pd

        try:
            df = pd.read_csv(csv_path)
            print(f"📊 CSV読み込み: {len(df)}行")
            print(f"   列: {list(df.columns)[:10]}...")

            # タイムスタンプ列を探す
            time_col = None
            for col in ['TimeStamp', 'Timestamp', 'timestamp', 'Time', 'time']:
                if col in df.columns:
                    time_col = col
                    break

            if time_col is None:
                print("⚠️ タイムスタンプ列が見つかりません")
                return False

            # α/β/θ列を探す
            alpha_cols = [c for c in df.columns if 'Alpha' in c or 'alpha' in c]
            beta_cols = [c for c in df.columns if 'Beta' in c or 'beta' in c]
            theta_cols = [c for c in df.columns if 'Theta' in c or 'theta' in c]

            print(f"   Alpha列: {alpha_cols[:2]}")
            print(f"   Beta列: {beta_cols[:2]}")
            print(f"   Theta列: {theta_cols[:2]}")

            self.data = []
            for _, row in df.iterrows():
                try:
                    # タイムスタンプをパース
                    ts_str = str(row[time_col])
                    # 複数のフォーマットを試す
                    ts = None
                    for fmt in ['%Y-%m-%d %H:%M:%S.%f', '%Y-%m-%d %H:%M:%S',
                               '%H:%M:%S.%f', '%H:%M:%S',
                               '%Y/%m/%d %H:%M:%S.%f', '%Y/%m/%d %H:%M:%S']:
                        try:
                            ts = datetime.strptime(ts_str, fmt)
                            break
                        except:
                            continue

                    if ts is None:
                        # Unix timestampかも
                        try:
                            ts = datetime.fromtimestamp(float(ts_str))
                        except:
                            continue

                    # α/β/θの平均を計算
                    def get_avg(cols):
                        vals = [float(row[c]) for c in cols if c in row and pd.notna(row[c])]
                        return sum(vals) / len(vals) if vals else 0.0

                    alpha = get_avg(alpha_cols)
                    beta = get_avg(beta_cols)
                    theta = get_avg(theta_cols)

                    self.data.append({
                        'timestamp': ts,
                        'alpha': alpha,
                        'beta': beta,
                        'theta': theta
                    })
                except Exception as e:
                    continue

            if not self.data:
                print("⚠️ 有効なデータがありません")
                return False

            self.start_time = self.data[0]['timestamp']
            self.end_time = self.data[-1]['timestamp']
            duration = (self.end_time - self.start_time).total_seconds()

            print(f"✓ 脳波データ: {len(self.data)}サンプル")
            print(f"   期間: {self.start_time.strftime('%H:%M:%S')} - {self.end_time.strftime('%H:%M:%S')} ({duration:.1f}秒)")

            return True

        except Exception as e:
            print(f"❌ CSV解析エラー: {e}")
            import traceback
            traceback.print_exc()
            return False

    def set_recording_time(self, start_str: str, end_str: str) -> bool:
        """録音の開始/終了時刻を設定（HH:MM:SS形式）"""
        try:
            # 今日の日付を使用
            base_date = self.start_time.date() if self.start_time else datetime.now().date()

            # 時刻をパース
            for fmt in ['%H:%M:%S.%f', '%H:%M:%S', '%H:%M']:
                try:
                    start_time = datetime.strptime(start_str, fmt).time()
                    break
                except:
                    continue
            else:
                print(f"⚠️ 開始時刻のフォーマットエラー: {start_str}")
                return False

            for fmt in ['%H:%M:%S.%f', '%H:%M:%S', '%H:%M']:
                try:
                    end_time = datetime.strptime(end_str, fmt).time()
                    break
                except:
                    continue
            else:
                print(f"⚠️ 終了時刻のフォーマットエラー: {end_str}")
                return False

            self.recording_start = datetime.combine(base_date, start_time)
            self.recording_end = datetime.combine(base_date, end_time)

            duration = (self.recording_end - self.recording_start).total_seconds()
            print(f"✓ 録音時間設定: {start_str} - {end_str} ({duration:.1f}秒)")

            return True

        except Exception as e:
            print(f"❌ 時刻設定エラー: {e}")
            return False

    def align_to_recording(self) -> bool:
        """脳波データを録音時間に合わせて相対時刻に変換（Z-score方式で状態判定）"""
        if not self.data or not self.recording_start or not self.recording_end:
            print("⚠️ データまたは録音時間が設定されていません")
            return False

        self.aligned_data = []
        rec_duration = (self.recording_end - self.recording_start).total_seconds()

        # まず録音範囲内のデータを抽出して相対値を計算
        temp_data = []
        for d in self.data:
            ts = d['timestamp']
            if self.recording_start <= ts <= self.recording_end:
                alpha, beta, theta = d['alpha'], d['beta'], d['theta']
                total = alpha + beta + theta
                if total > 0:
                    a_rel = alpha / total
                    b_rel = beta / total
                    t_rel = theta / total
                    engagement = b_rel / (a_rel + t_rel + 1e-9)
                    temp_data.append({
                        'timestamp': ts,
                        'relative_time': (ts - self.recording_start).total_seconds(),
                        'alpha': alpha, 'beta': beta, 'theta': theta,
                        'alpha_rel': a_rel, 'beta_rel': b_rel, 'theta_rel': t_rel,
                        'engagement': engagement
                    })

        if not temp_data:
            print("⚠️ 録音範囲内にデータがありません")
            return False

        # ベースライン統計を計算（Z-score用）
        import statistics
        alpha_vals = [d['alpha_rel'] for d in temp_data]
        eng_vals = [d['engagement'] for d in temp_data]

        alpha_mean = statistics.fmean(alpha_vals)
        alpha_std = statistics.pstdev(alpha_vals) or 0.01
        eng_mean = statistics.fmean(eng_vals)
        eng_std = statistics.pstdev(eng_vals) or 0.01

        print(f"   ベースライン: α相対={alpha_mean:.3f}±{alpha_std:.3f}, eng={eng_mean:.3f}±{eng_std:.3f}")

        # Z-score方式で状態判定
        focus_count = relax_count = neutral_count = 0
        for d in temp_data:
            z_alpha = (d['alpha_rel'] - alpha_mean) / alpha_std
            z_eng = (d['engagement'] - eng_mean) / eng_std

            # Z-score判定：平均からの偏差で判断
            if z_eng > 0.5 and z_alpha < 0:
                state = 'FOCUSED'
                focus_count += 1
            elif z_alpha > 0.5 and z_eng < 0:
                state = 'RELAXED'
                relax_count += 1
            else:
                state = 'NEUTRAL'
                neutral_count += 1

            self.aligned_data.append({
                'relative_time': d['relative_time'],
                'alpha': d['alpha'], 'beta': d['beta'], 'theta': d['theta'],
                'alpha_rel': d['alpha_rel'], 'beta_rel': d['beta_rel'], 'theta_rel': d['theta_rel'],
                'engagement': d['engagement'],
                'z_alpha': z_alpha, 'z_eng': z_eng,
                'state': state
            })

        print(f"✓ アライメント完了: {len(self.aligned_data)}サンプル（{rec_duration:.1f}秒間）")
        print(f"   状態分布: FOCUSED={focus_count} ({focus_count/len(self.aligned_data)*100:.1f}%), RELAXED={relax_count} ({relax_count/len(self.aligned_data)*100:.1f}%), NEUTRAL={neutral_count} ({neutral_count/len(self.aligned_data)*100:.1f}%)")
        return len(self.aligned_data) > 0

    def get_states_for_time_range(self, start_sec: float, end_sec: float) -> Dict[str, Any]:
        """指定時間範囲の脳波状態を取得"""
        if not self.aligned_data:
            return {'available': False, 'dominant_state': 'NO_DATA', 'count': 0}

        states_in_range = [
            d for d in self.aligned_data
            if start_sec <= d['relative_time'] <= end_sec
        ]

        if not states_in_range:
            return {'available': True, 'dominant_state': 'NO_DATA', 'count': 0, 'focus_ratio': 0.0}

        counts = {'FOCUSED': 0, 'RELAXED': 0, 'NEUTRAL': 0, 'NO_DATA': 0}
        alpha_sum, beta_sum, theta_sum = 0.0, 0.0, 0.0

        for s in states_in_range:
            st = s.get('state', 'NEUTRAL')
            if st in counts:
                counts[st] += 1
            alpha_sum += s.get('alpha_rel', 0)
            beta_sum += s.get('beta_rel', 0)
            theta_sum += s.get('theta_rel', 0)

        total = len(states_in_range)
        dominant = max(counts, key=counts.get)

        return {
            'available': True,
            'dominant_state': dominant,
            'count': total,
            'focus_ratio': counts['FOCUSED'] / total if total > 0 else 0.0,
            'relax_ratio': counts['RELAXED'] / total if total > 0 else 0.0,
            'avg_alpha': alpha_sum / total if total > 0 else 0,
            'avg_beta': beta_sum / total if total > 0 else 0,
            'avg_theta': theta_sum / total if total > 0 else 0,
            'state_counts': counts
        }

    def get_session_summary(self) -> Dict[str, Any]:
        """セッション全体のサマリー"""
        if not self.aligned_data:
            return {'available': False}

        counts = {'FOCUSED': 0, 'RELAXED': 0, 'NEUTRAL': 0, 'NO_DATA': 0}
        for d in self.aligned_data:
            st = d.get('state', 'NEUTRAL')
            if st in counts:
                counts[st] += 1

        total = len(self.aligned_data)
        duration = self.aligned_data[-1]['relative_time'] if self.aligned_data else 0

        return {
            'available': True,
            'total_samples': total,
            'duration': duration,
            'focus_ratio': counts['FOCUSED'] / total if total > 0 else 0.0,
            'relax_ratio': counts['RELAXED'] / total if total > 0 else 0.0,
            'neutral_ratio': counts['NEUTRAL'] / total if total > 0 else 0.0,
            'state_counts': counts
        }

brain_csv_parser = BrainWaveCSVParser()

# -----------------------------
# Reference Parser
# -----------------------------
class ReferenceParser:
    def __init__(self, midi_path, mxl_path):
        self.midi_path, self.mxl_path = midi_path, mxl_path

    def parse(self):
        print("📖 参照データを解析中...")
        midi_notes, midi_dur = self._parse_midi()
        print(f"  ✓ MIDI: {len(midi_notes)}音符, {midi_dur:.1f}秒")
        mxl_measures = self._parse_mxl()
        print(f"  ✓ MXL: {len(mxl_measures)}小節")
        measures = self._create_measures(midi_notes, mxl_measures, midi_dur)
        return measures, midi_notes

    def _parse_midi(self):
        import pretty_midi
        pm = pretty_midi.PrettyMIDI(self.midi_path)
        notes = []
        for inst in pm.instruments:
            if not inst.is_drum:
                for n in inst.notes:
                    notes.append(NoteEvent(pitch=n.pitch, start_time=n.start, end_time=n.end, velocity=n.velocity))
        notes.sort(key=lambda n: (n.start_time, n.pitch))
        return notes, pm.get_end_time()

    def _parse_mxl(self):
        from music21 import converter, tempo as m21tempo, meter
        score = converter.parse(self.mxl_path)
        try:
            score = score.expandRepeats()
        except:
            pass
        measures, current_tempo, current_ts = [], 120.0, (4, 4)
        for t in score.flatten().getElementsByClass(m21tempo.MetronomeMark):
            if t.number:
                current_tempo = float(t.number)
                break
        part = score.parts[0] if score.parts else score
        for m in part.getElementsByClass('Measure'):
            for t in m.getElementsByClass(m21tempo.MetronomeMark):
                if t.number:
                    current_tempo = float(t.number)
            for ts in m.getElementsByClass(meter.TimeSignature):
                if ts.numerator:
                    current_ts = (ts.numerator, ts.denominator)
            measures.append({
                'number': m.measureNumber,
                'tempo': current_tempo or 120,
                'time_signature': current_ts,
                'quarter_length': float(m.quarterLength) if m.quarterLength else 4.0
            })
        return measures

    def _create_measures(self, midi_notes, mxl_measures, midi_dur):
        if not mxl_measures:
            return []
        measures = []
        total_ql = sum(m['quarter_length'] for m in mxl_measures)
        midi_span = (midi_notes[-1].start_time - midi_notes[0].start_time) if midi_notes else midi_dur
        midi_offset = midi_notes[0].start_time if midi_notes else 0
        current_time = midi_offset
        for i, mxl in enumerate(mxl_measures):
            ql = mxl['quarter_length']
            dur = (ql / total_ql) * midi_span if total_ql > 0 else 2.0
            end_time = current_time + dur
            ref_notes = [
                NoteEvent(pitch=n.pitch, start_time=n.start_time, end_time=n.end_time, velocity=n.velocity)
                for n in midi_notes
                if current_time - 0.05 <= n.start_time < end_time + 0.05
            ]
            measures.append(MeasureData(
                number=mxl['number'], index=i, start_time=current_time, end_time=end_time,
                tempo=mxl['tempo'], time_signature=mxl['time_signature'], ref_notes=ref_notes
            ))
            current_time = end_time
        return measures

# -----------------------------
# Audio Transcriber
# -----------------------------
class AudioTranscriber:
    def __init__(self, use_gpu=True):
        self.use_gpu = use_gpu

    def transcribe(self, audio_path):
        import subprocess, tempfile, os, pretty_midi, librosa
        print("🎵 音声を解析中...")
        device = 'cpu'
        if self.use_gpu:
            try:
                import torch
                if torch.cuda.is_available():
                    device = 'cuda'
                    print("  ✓ CUDA使用")
            except:
                pass
        y, sr = librosa.load(audio_path, sr=None)
        duration = len(y) / sr
        print(f"  - 音声長: {duration:.1f}秒")
        with tempfile.TemporaryDirectory() as td:
            out = os.path.join(td, "out.mid")
            subprocess.run(
                ['python3', '-m', 'transkun.transcribe', audio_path, out, '--device', device],
                check=True, capture_output=True, timeout=600
            )
            pm = pretty_midi.PrettyMIDI(out)
            notes = [
                NoteEvent(pitch=n.pitch, start_time=n.start, end_time=n.end, velocity=n.velocity)
                for inst in pm.instruments if not inst.is_drum for n in inst.notes
            ]
            notes.sort(key=lambda n: (n.start_time, n.pitch))
            print(f"  ✓ {len(notes)}音符を認識")
            return notes, duration

# -----------------------------
# DTW Aligner
# -----------------------------
class DTWAligner:
    def __init__(self, params):
        self.params = params

    def align(self, perf_notes, ref_notes, measures):
        print("🔗 アライメント中...")
        if not perf_notes or not ref_notes:
            return measures
        try:
            from dtw import dtw
            pp = np.array([n.pitch for n in perf_notes]).reshape(-1, 1)
            rp = np.array([n.pitch for n in ref_notes]).reshape(-1, 1)
            def dist(x, y):
                d = abs(x[0] - y[0])
                return d/12*2 if d % 12 == 0 and d > 0 else 0 if d <= self.params.pitch_tolerance else d
            alignment = dtw(pp, rp, dist_method=dist, keep_internals=True,
                          step_pattern='symmetric2', window_type='sakoechiba',
                          window_args={'window_size': 400})
            path = list(zip(alignment.index1, alignment.index2))
        except:
            path = [(i, min(i, len(ref_notes)-1)) for i in range(len(perf_notes))]

        pairs = [(perf_notes[p].start_time, ref_notes[r].start_time)
                 for p, r in path if p < len(perf_notes) and r < len(ref_notes)]
        if not pairs:
            return measures

        pt = np.array([p[0] for p in pairs])
        rt = np.array([p[1] for p in pairs])

        def time_map(t):
            if t <= pt[0]: return rt[0]
            if t >= pt[-1]: return rt[-1]
            idx = np.searchsorted(pt, t)
            if idx == 0: return rt[0]
            ratio = (t - pt[idx-1]) / (pt[idx] - pt[idx-1]) if pt[idx] != pt[idx-1] else 0
            return rt[idx-1] + ratio * (rt[idx] - rt[idx-1])

        for note in perf_notes:
            ref_t = time_map(note.start_time)
            for m in measures:
                if m.start_time <= ref_t < m.end_time:
                    m.perf_notes.append(NoteEvent(
                        pitch=note.pitch, start_time=note.start_time,
                        end_time=note.end_time, velocity=note.velocity
                    ))
                    if not m.perf_start_time or note.start_time < m.perf_start_time:
                        m.perf_start_time = note.start_time
                    if note.start_time > m.perf_end_time:
                        m.perf_end_time = note.start_time
                    break

        print(f"  ✓ {sum(len(m.perf_notes) for m in measures)}/{len(perf_notes)}音符をマッピング")
        return measures

# -----------------------------
# Scoring Engine
# -----------------------------
class ScoringEngine:
    def __init__(self, params, brain_parser=None):
        self.params = params
        self.brain_parser = brain_parser

    def score_all(self, measures, audio_duration=None):
        print("📊 採点中...")

        all_perf = [n for m in measures for n in m.perf_notes]
        if all_perf:
            perf_start = min(n.start_time for n in all_perf)
            perf_end = max(n.end_time for n in all_perf)
            perf_duration = perf_end - perf_start
        else:
            perf_start, perf_end, perf_duration = 0, 0, 0

        brain_duration = None
        if self.brain_parser and self.brain_parser.aligned_data:
            brain_duration = self.brain_parser.aligned_data[-1]['relative_time']
            print(f"  🧠 脳波: {len(self.brain_parser.aligned_data)}サンプル, {brain_duration:.1f}秒")

        for m in measures:
            m.pitch_score = self._score_pitch(m)
            m.rhythm_score = self._score_rhythm(m)
            m.tempo_score = self._score_tempo(m)

            # 脳波割り当て
            if self.brain_parser and brain_duration and brain_duration > 0 and m.perf_notes:
                if perf_duration > 0:
                    m_start = m.perf_start_time if m.perf_start_time else min(n.start_time for n in m.perf_notes)
                    m_end = m.perf_end_time if m.perf_end_time else max(n.end_time for n in m.perf_notes)

                    # 演奏時刻を脳波時刻に変換
                    brain_start = ((m_start - perf_start) / perf_duration) * brain_duration
                    brain_end = ((m_end - perf_start) / perf_duration) * brain_duration

                    summary = self.brain_parser.get_states_for_time_range(brain_start, brain_end)
                    m.brain_state = summary.get('dominant_state', '')
                    m.brain_metrics = {
                        'focus_ratio': summary.get('focus_ratio', 0.0),
                        'relax_ratio': summary.get('relax_ratio', 0.0),
                        'count': summary.get('count', 0),
                    }

        result = self._calc_total(measures)
        print(f"  ✓ 総合スコア: {result['total_score']:.1f}")

        if self.brain_parser and self.brain_parser.aligned_data:
            result['brain_summary'] = self.brain_parser.get_session_summary()

        return result

    def _score_pitch(self, m):
        if not m.ref_notes: return 1.0 if not m.perf_notes else 0.0
        if not m.perf_notes: return 0.0
        ref_groups = self._group_by_time(m.ref_notes, self.params.chord_time_window)
        perf_groups = self._group_by_time(m.perf_notes, self.params.arpeggio_max_time if self.params.allow_arpeggio else self.params.chord_time_window)
        total = sum(len(g['pitches']) for g in ref_groups)
        pool = set(p for g in perf_groups for p in g['pitches'])
        matched = sum(1 for g in ref_groups for rp in g['pitches'] if rp in pool or any(self._match(rp, pp) for pp in pool))
        return matched / total if total > 0 else 1.0

    def _group_by_time(self, notes, window):
        if not notes: return []
        notes = sorted(notes, key=lambda n: n.start_time)
        groups = [{'time': notes[0].start_time, 'pitches': [notes[0].pitch]}]
        for n in notes[1:]:
            if n.start_time - groups[-1]['time'] <= window:
                groups[-1]['pitches'].append(n.pitch)
            else:
                groups.append({'time': n.start_time, 'pitches': [n.pitch]})
        return groups

    def _match(self, rp, pp):
        d = abs(rp - pp)
        return d == 0 or d <= self.params.pitch_tolerance or (self.params.ignore_octave and d % 12 == 0)

    def _score_rhythm(self, m):
        if not m.ref_notes or not m.perf_notes: return 1.0 if not m.ref_notes else 0.0
        def pos(notes, s, e):
            d = e - s if e > s else 0.1
            return [(n.start_time - s) / d for n in notes]
        ref_pos = pos(m.ref_notes, m.start_time, m.end_time)
        perf_pos = pos(m.perf_notes, m.perf_start_time, m.perf_end_time) if m.perf_end_time > m.perf_start_time else pos(m.perf_notes, m.perf_notes[0].start_time, m.perf_notes[-1].start_time + 0.1)
        tol = self.params.rhythm_tolerance
        return sum(1 for rp in ref_pos if any(abs(rp - pp) <= tol for pp in perf_pos)) / len(ref_pos) if ref_pos else 1.0

    def _score_tempo(self, m):
        if not m.perf_notes or len(m.perf_notes) < 2: return 1.0
        pd, rd = m.perf_end_time - m.perf_start_time, m.duration
        if pd <= 0 or rd <= 0: return 1.0
        r = pd / rd
        return 1.0 if 1 - self.params.tempo_tolerance <= r <= 1 + self.params.tempo_tolerance else max(0, 1 - (abs(r - 1) - self.params.tempo_tolerance) * 2)

    def _calc_total(self, measures):
        played = [m for m in measures if m.perf_notes]
        if not played:
            return {'total_score': 0, 'pitch_score': 0, 'rhythm_score': 0, 'tempo_score': 0, 'measures_played': 0, 'measures_total': len(measures), 'measure_details': []}
        ap, ar, at = np.mean([m.pitch_score for m in played]), np.mean([m.rhythm_score for m in played]), np.mean([m.tempo_score for m in played])
        return {
            'total_score': (ap * 0.5 + ar * 0.3 + at * 0.2) * 100, 'pitch_score': ap * 100, 'rhythm_score': ar * 100, 'tempo_score': at * 100,
            'measures_played': len(played), 'measures_total': len(measures),
            'start_measure': measures[min(m.index for m in played)].number, 'end_measure': measures[max(m.index for m in played)].number,
            'measure_details': [{'number': m.number, 'index': m.index, 'pitch': m.pitch_score*100, 'rhythm': m.rhythm_score*100, 'tempo': m.tempo_score*100, 'ref_notes': len(m.ref_notes), 'perf_notes': len(m.perf_notes), 'brain_state': m.brain_state, 'brain_metrics': m.brain_metrics} for m in measures]
        }

# -----------------------------
# Visualizer
# -----------------------------
class Visualizer:
    def show_summary(self, result):
        from IPython.display import display, HTML
        r = result
        brain_html = ""
        if 'brain_summary' in r and r['brain_summary'].get('available'):
            bs = r['brain_summary']
            brain_html = f"""<div style="margin-top:15px;padding:10px;background:rgba(255,255,255,0.1);border-radius:8px;"><div style="font-size:14px;margin-bottom:5px;">🧠 脳波 ({bs['total_samples']}サンプル, {bs.get('duration', 0):.1f}秒)</div><div style="display:flex;gap:15px;flex-wrap:wrap;"><span>🟢集中: {bs['focus_ratio']*100:.0f}%</span><span>🔵リラックス: {bs['relax_ratio']*100:.0f}%</span><span>⚪中立: {bs['neutral_ratio']*100:.0f}%</span></div></div>"""
        html = f"""<div style="background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);padding:20px;border-radius:15px;color:white;margin:10px 0;"><h2 style="margin:0 0 15px 0;">🎹 採点結果</h2><div style="display:flex;justify-content:space-around;flex-wrap:wrap;"><div style="text-align:center;padding:10px;"><div style="font-size:36px;font-weight:bold;">{r['total_score']:.1f}</div><div>総合</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['pitch_score']:.1f}</div><div>🎵ピッチ</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['rhythm_score']:.1f}</div><div>🥁リズム</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['tempo_score']:.1f}</div><div>⏱️テンポ</div></div></div><div style="margin-top:10px;">演奏: {r['measures_played']}/{r['measures_total']}小節</div>{brain_html}</div>"""
        display(HTML(html))

    def show_table(self, result, only_played=True):
        import pandas as pd
        from IPython.display import display
        df = pd.DataFrame(result.get('measure_details', []))
        if df.empty: return
        if only_played: df = df[df['perf_notes'] > 0]
        has_brain = any(d.get('brain_state') for d in result.get('measure_details', []))
        if has_brain:
            df['brain'] = df.apply(lambda r: (r['brain_state'][:3] if r.get('brain_state') else '-'), axis=1)
            df = df[['number', 'index', 'pitch', 'rhythm', 'tempo', 'ref_notes', 'perf_notes', 'brain']]
            df.columns = ['小節', '通し', 'ピッチ', 'リズム', 'テンポ', '楽譜', '演奏', '脳波']
        else:
            df = df[['number', 'index', 'pitch', 'rhythm', 'tempo', 'ref_notes', 'perf_notes']]
            df.columns = ['小節', '通し', 'ピッチ', 'リズム', 'テンポ', '楽譜', '演奏']
        def color(v):
            try:
                v = float(v)
                return 'background-color:#90EE90' if v >= 80 else 'background-color:#FFE4B5' if v >= 50 else 'background-color:#FFB6C1' if v > 0 else ''
            except: return ''
        display(df.style.applymap(color, subset=['ピッチ', 'リズム', 'テンポ']))

    def show_brain_timeline(self, measures):
        import matplotlib.pyplot as plt
        played = [m for m in measures if m.perf_notes]
        if not played: print("演奏小節がありません"); return
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
        indices = [m.index for m in played]
        scores = [(m.pitch_score * 0.5 + m.rhythm_score * 0.3 + m.tempo_score * 0.2) * 100 for m in played]
        colors = ['#90EE90' if s >= 80 else '#FFE4B5' if s >= 50 else '#FFB6C1' for s in scores]
        ax1.bar(indices, scores, color=colors, edgecolor='gray')
        ax1.axhline(y=70, color='red', linestyle='--', alpha=0.5, label='70%')
        ax1.set_ylabel('Score (%)')
        ax1.set_ylim(0, 100)
        ax1.legend()
        ax1.set_title('Score and Brain State by Measure')
        state_colors = {'FOCUSED': '#28a745', 'RELAXED': '#007bff', 'NEUTRAL': '#6c757d', 'NO_DATA': '#dc3545'}
        brain_colors = [state_colors.get(m.brain_state, '#999') for m in played]
        ax2.bar(indices, [1]*len(indices), color=brain_colors, edgecolor='gray')
        ax2.set_ylabel('Brain')
        ax2.set_xlabel('Measure Index')
        ax2.set_yticks([])
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=c, label=s) for s, c in state_colors.items()]
        ax2.legend(handles=legend_elements, loc='upper right', ncol=4)
        plt.tight_layout()
        plt.show()

        # 統計
        state_counts = {}
        for m in played:
            st = m.brain_state or 'UNKNOWN'
            state_counts[st] = state_counts.get(st, 0) + 1
        print("\n脳波状態の統計:")
        for st, cnt in sorted(state_counts.items(), key=lambda x: -x[1]):
            print(f"  {st}: {cnt}小節 ({cnt/len(played)*100:.1f}%)")

    def show_measure_comparison(self, m):
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        def plot(ax, notes, off, c, title):
            if not notes: ax.text(0.5, 0.5, 'No notes', ha='center', va='center', transform=ax.transAxes); ax.set_title(title); return
            for n in notes: ax.barh(n.pitch, n.duration, left=n.start_time - off, height=0.6, color=c, alpha=0.6)
            ax.set_ylim(min(n.pitch for n in notes) - 2, max(n.pitch for n in notes) + 2)
            ax.set_xlabel('Time (s)'); ax.set_ylabel('Pitch'); ax.set_title(title); ax.grid(True, alpha=0.3)
        plot(axes[0], m.ref_notes, m.start_time, 'blue', f'Score ({len(m.ref_notes)} notes)')
        plot(axes[1], m.perf_notes, m.perf_notes[0].start_time if m.perf_notes else 0, 'red', f'Perf ({len(m.perf_notes)} notes)')
        brain = f" | 🧠{m.brain_state}" if m.brain_state else ""
        fig.suptitle(f"Measure {m.number} | P:{m.pitch_score*100:.1f}% | R:{m.rhythm_score*100:.1f}%{brain}", y=1.02)
        plt.tight_layout(); plt.show()

    def show_problem_measures(self, measures, threshold=0.7):
        from IPython.display import display, HTML
        problems = []
        for m in measures:
            issues = []
            if m.pitch_score < threshold: issues.append(f"P:{m.pitch_score*100:.0f}%")
            if m.rhythm_score < threshold: issues.append(f"R:{m.rhythm_score*100:.0f}%")
            if issues: problems.append({'num': m.number, 'idx': m.index, 'issues': issues, 'brain': m.brain_state or '-'})
        if not problems: print("✓ No problems (all >= 70%)"); return
        html = f"<h3>⚠️ Problems ({len(problems)})</h3><table style='border-collapse:collapse;'><tr style='background:#f0f0f0;'><th style='padding:5px;border:1px solid #ddd;'>M#</th><th style='padding:5px;border:1px solid #ddd;'>Idx</th><th style='padding:5px;border:1px solid #ddd;'>Issues</th><th style='padding:5px;border:1px solid #ddd;'>Brain</th></tr>"
        for p in problems[:30]: html += f"<tr><td style='padding:5px;border:1px solid #ddd;'>{p['num']}</td><td style='padding:5px;border:1px solid #ddd;'>{p['idx']}</td><td style='padding:5px;border:1px solid #ddd;color:red;'>{', '.join(p['issues'])}</td><td style='padding:5px;border:1px solid #ddd;'>{p['brain']}</td></tr>"
        html += "</table>"
        display(HTML(html))

    def show_pitch_histogram(self, measures):
        import matplotlib.pyplot as plt
        ref_p = [n.pitch for m in measures for n in m.ref_notes]
        perf_p = [n.pitch for m in measures for n in m.perf_notes]
        if not ref_p and not perf_p: print("No data"); return
        fig, ax = plt.subplots(figsize=(12, 4))
        bins = range(min(ref_p + perf_p) - 1, max(ref_p + perf_p) + 2)
        ax.hist(ref_p, bins=bins, alpha=0.5, label=f'Score({len(ref_p)})', color='blue')
        ax.hist(perf_p, bins=bins, alpha=0.5, label=f'Perf({len(perf_p)})', color='red')
        ax.legend(); ax.grid(True, alpha=0.3); plt.show()

    def show_score_with_click(self, mxl_path, measures):
        from IPython.display import display, HTML
        import verovio, re, json
        try:
            tk = verovio.toolkit()
            tk.setOptions({"pageWidth": 1800, "pageHeight": 3000, "scale": 35, "adjustPageHeight": True})
            tk.loadFile(mxl_path)
            pc = tk.getPageCount()
            amd = [{'index': m.index, 'number': m.number, 'pitch': m.pitch_score*100, 'rhythm': m.rhythm_score*100, 'ref_notes': len(m.ref_notes), 'perf_notes': len(m.perf_notes), 'brain_state': m.brain_state, 'start_time': m.start_time, 'end_time': m.end_time, 'perf_start': m.perf_notes[0].start_time if m.perf_notes else 0, 'ref_note_list': [{'pitch': n.pitch, 'start': n.start_time, 'dur': n.duration} for n in m.ref_notes], 'perf_note_list': [{'pitch': n.pitch, 'start': n.start_time, 'dur': n.duration} for n in m.perf_notes]} for m in measures]
            mc = {i: "#90EE90" if (m.pitch_score + m.rhythm_score)/2*100 >= 90 else "#FFFF99" if (m.pitch_score + m.rhythm_score)/2*100 >= 70 else "#FFE4B5" if (m.pitch_score + m.rhythm_score)/2*100 >= 50 else "#FFB6C1" for i, m in enumerate(measures)}
            svgs = []
            measure_idx = 0
            for p in range(1, pc + 1):
                svg = tk.renderToSVG(p)
                def add_attr(match):
                    nonlocal measure_idx
                    result = match.group(0)
                    if 'data-measure-idx' not in result:
                        result = result[:-1] + f' data-measure-idx="{measure_idx}" style="cursor:pointer;">'
                        measure_idx += 1
                    return result
                svg = re.sub(r'<g[^>]*class="[^"]*measure[^"]*"[^>]*>', add_attr, svg)
                svgs.append(svg)
            html = f'''<style>.score-viewer{{display:grid;grid-template-columns:1.5fr 1fr;gap:15px;height:700px;}}.score-panel{{overflow:auto;border:1px solid #ccc;padding:10px;background:white;}}.detail-panel{{border:1px solid #ccc;border-radius:8px;background:#f8f9fa;padding:15px;overflow-y:auto;}}.measure-highlight{{cursor:pointer;}}.measure-highlight:hover{{filter:brightness(0.85);}}</style><div class="score-viewer"><div class="score-panel" id="score-panel">{"".join(svgs)}</div><div class="detail-panel" id="detail-panel"><h3>Click a measure</h3></div></div><script>(function(){{var mc={json.dumps(mc)};var am={json.dumps(amd)};var pn=['C','C#','D','D#','E','F','F#','G','G#','A','A#','B'];function p2n(p){{return pn[p%12]+(Math.floor(p/12)-1);}}function createPianoWave(ctx){{var real=new Float32Array([0,1.0,0.5,0.35,0.2,0.12,0.08,0.05,0.03]);var imag=new Float32Array(real.length);return ctx.createPeriodicWave(real,imag);}}function show(i){{var m=am[i];if(!m)return;var rp=new Set(m.ref_note_list.map(n=>n.pitch));var pp=new Set(m.perf_note_list.map(n=>n.pitch));var mt=[...rp].filter(p=>pp.has(p));var ms=[...rp].filter(p=>!pp.has(p));var h='<h3>M'+m.number+'</h3><div style="margin:10px 0;"><span style="padding:4px 8px;border-radius:4px;background:'+(m.pitch>=80?'#90EE90':'#FFB6C1')+';">P:'+m.pitch.toFixed(1)+'%</span> <span style="padding:4px 8px;border-radius:4px;background:'+(m.rhythm>=80?'#87CEEB':'#FFB6C1')+';">R:'+m.rhythm.toFixed(1)+'%</span></div>';if(m.brain_state)h+='<div style="background:#e9ecef;padding:5px;border-radius:4px;margin:5px 0;">🧠'+m.brain_state+'</div>';h+='<div style="margin:10px 0;"><button onclick="playRef('+i+')" style="padding:8px 16px;margin:5px;cursor:pointer;border:none;border-radius:4px;background:#4CAF50;color:white;">🔊楽譜</button>';if(m.perf_notes>0)h+='<button onclick="playPerf('+i+')" style="padding:8px 16px;margin:5px;cursor:pointer;border:none;border-radius:4px;background:#f44336;color:white;">🔊演奏</button>';h+='</div><div style="font-size:11px;"><span style="color:green;">✓'+mt.map(p=>p2n(p)).join(',')||'none'+'</span><br><span style="color:red;">✗'+ms.map(p=>p2n(p)).join(',')||'none'+'</span></div>';document.getElementById('detail-panel').innerHTML=h;}}function playRef(i){{var m=am[i];if(!m||!m.ref_note_list.length)return;var ctx=new(window.AudioContext||window.webkitAudioContext)();var pianoWave=createPianoWave(ctx);m.ref_note_list.forEach(n=>{{var o=ctx.createOscillator();var g=ctx.createGain();o.connect(g);g.connect(ctx.destination);o.frequency.value=440*Math.pow(2,(n.pitch-69)/12);o.setPeriodicWave(pianoWave);var s=Math.max(0,n.start-m.start_time);var d=Math.min(n.dur,2.0);g.gain.setValueAtTime(0,ctx.currentTime+s);g.gain.linearRampToValueAtTime(0.35,ctx.currentTime+s+0.008);g.gain.exponentialRampToValueAtTime(0.001,ctx.currentTime+s+d+0.5);o.start(ctx.currentTime+s);o.stop(ctx.currentTime+s+d+0.6);}});}}function playPerf(i){{var m=am[i];if(!m||!m.perf_note_list.length)return;var ctx=new(window.AudioContext||window.webkitAudioContext)();var pianoWave=createPianoWave(ctx);m.perf_note_list.forEach(n=>{{var o=ctx.createOscillator();var g=ctx.createGain();o.connect(g);g.connect(ctx.destination);o.frequency.value=440*Math.pow(2,(n.pitch-69)/12);o.setPeriodicWave(pianoWave);var s=Math.max(0,n.start-m.perf_start);var d=Math.min(n.dur,2.0);g.gain.setValueAtTime(0,ctx.currentTime+s);g.gain.linearRampToValueAtTime(0.3,ctx.currentTime+s+0.008);g.gain.exponentialRampToValueAtTime(0.001,ctx.currentTime+s+d+0.5);o.start(ctx.currentTime+s);o.stop(ctx.currentTime+s+d+0.6);}});}}window.playRef=playRef;window.playPerf=playPerf;setTimeout(function(){{var panel=document.getElementById('score-panel');if(!panel)return;var els=panel.querySelectorAll('[data-measure-idx]');if(!els.length){{els=panel.querySelectorAll('g[class*="measure"]');els.forEach(function(el,idx){{el.setAttribute('data-measure-idx',idx);}});}}els.forEach(function(el){{var idx=parseInt(el.getAttribute('data-measure-idx'));var color=mc[idx]||'#FFF';el.classList.add('measure-highlight');try{{var bbox=el.getBBox();if(bbox.width>0){{var rect=document.createElementNS('http://www.w3.org/2000/svg','rect');rect.setAttribute('x',bbox.x-5);rect.setAttribute('y',bbox.y-5);rect.setAttribute('width',bbox.width+10);rect.setAttribute('height',bbox.height+10);rect.setAttribute('fill',color);rect.setAttribute('opacity','0.4');rect.style.pointerEvents='none';el.insertBefore(rect,el.firstChild);}}}}catch(e){{}}el.addEventListener('click',function(e){{e.stopPropagation();if(idx<am.length)show(idx);}});}});}},500);}})();</script>'''
            display(HTML(html))
        except Exception as e:
            print(f"楽譜表示エラー: {e}")

# -----------------------------
# UI
# -----------------------------
class PianoScorerUI:
    def __init__(self):
        self.params = ScoringParams()
        self.measures, self.ref_notes, self.perf_notes, self.result = [], [], [], None
        self.visualizer = Visualizer()
        self.audio_path = self.midi_path = self.mxl_path = self.recorded_audio = self.brain_csv_path = None
        self.audio_duration = None
        self.is_recording = False

    def _receive_audio_callback(self, base64_data, mime_type='audio/webm'):
        import base64, tempfile
        try:
            raw = base64.b64decode(base64_data)
            suffix = '.ogg' if 'ogg' in str(mime_type) else '.webm'
            with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
                f.write(raw)
                self.recorded_audio = f.name
            if self.input_mode.value == '録音':
                self.audio_path = self.recorded_audio
            self.record_status.value = '<span style="color:green;">✓完了</span>'
        except Exception as e:
            self.record_status.value = f'<span style="color:red;">Error:{e}</span>'

    def create_ui(self):
        import ipywidgets as widgets
        from IPython.display import display, HTML

        display(HTML("""<div style="background:linear-gradient(135deg,#1a1a2e 0%,#16213e 100%);padding:20px;border-radius:15px;color:white;margin-bottom:20px;">
            <h1 style="margin:0;">🎹🧠 Piano Performance Scorer v4.4</h1>
            <p style="margin:5px 0 0 0;opacity:0.8;">オフライン脳波統合版（Mind Monitor CSVアップロード）</p>
        </div>"""))

        # ---- 脳波CSV ----
        self.brain_enabled = widgets.Checkbox(value=False, description='🧠脳波CSVを使用')
        self.brain_enabled.observe(self._on_brain_toggle, names='value')
        self.brain_csv_upload = widgets.FileUpload(accept='.csv', multiple=False, description='脳波CSV')
        self.rec_start_input = widgets.Text(placeholder='HH:MM:SS', description='録音開始:', layout=widgets.Layout(width='180px'))
        self.rec_end_input = widgets.Text(placeholder='HH:MM:SS', description='録音終了:', layout=widgets.Layout(width='180px'))
        self.parse_brain_btn = widgets.Button(description='🧠CSV解析', button_style='info', layout=widgets.Layout(width='100px'))
        self.parse_brain_btn.on_click(self._on_parse_brain)
        self.brain_status = widgets.HTML(value='<span style="color:gray;">CSVをアップロード</span>')
        self.brain_controls = widgets.VBox([
            widgets.HBox([widgets.Label("脳波CSV:"), self.brain_csv_upload]),
            widgets.HBox([self.rec_start_input, self.rec_end_input, self.parse_brain_btn]),
            self.brain_status
        ])
        self.brain_controls.layout.display = 'none'
        brain_info = widgets.HTML("""<div style="background:#e8f4f8;padding:10px;border-radius:5px;margin:5px 0;font-size:12px;">
            <b>使い方:</b> Mind Monitorで録音中に脳波を記録 → CSVエクスポート → 録音開始/終了時刻を入力 → CSV解析
        </div>""")
        brain_box = widgets.VBox([widgets.HTML("<h3>🧠脳波CSV</h3>"), self.brain_enabled, self.brain_controls, brain_info])

        # ---- ファイル/録音 ----
        self.input_mode = widgets.RadioButtons(options=['ファイル', '録音'], value='ファイル', layout=widgets.Layout(width='100px'))
        self.input_mode.observe(self._on_input_mode_change, names='value')
        self.audio_upload = widgets.FileUpload(accept='.mp3,.wav,.m4a,.webm,.ogg', multiple=False)
        self.midi_upload = widgets.FileUpload(accept='.mid,.midi', multiple=False)
        self.mxl_upload = widgets.FileUpload(accept='.mxl,.xml,.musicxml', multiple=False)
        self.record_btn = widgets.Button(description='🎤録音', button_style='info', layout=widgets.Layout(width='80px'))
        self.record_btn.on_click(self._on_record)
        self.gain_slider = widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description='Gain:', layout=widgets.Layout(width='180px'))
        self.record_status = widgets.HTML(value='<span style="color:gray;">待機</span>')
        self.audio_file_box = widgets.VBox([widgets.Label("音声"), self.audio_upload])
        self.audio_record_box = widgets.VBox([widgets.HBox([self.record_btn, self.record_status]), self.gain_slider])
        self.audio_record_box.layout.display = 'none'
        upload_box = widgets.VBox([
            widgets.HTML("<h3>📁入力</h3>"),
            widgets.HBox([
                widgets.VBox([self.input_mode, self.audio_file_box, self.audio_record_box]),
                widgets.VBox([widgets.Label("MIDI"), self.midi_upload]),
                widgets.VBox([widgets.Label("MXL"), self.mxl_upload]),
            ])
        ])

        # ---- パラメータ ----
        self.pitch_tol = widgets.IntSlider(value=0, min=0, max=3, description='Pitch:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        self.ignore_oct = widgets.Checkbox(value=False, description='Oct無視')
        self.arpeggio = widgets.Checkbox(value=True, description='アルペジオ')
        self.rhythm_tol = widgets.FloatSlider(value=0.1, min=0.05, max=0.3, step=0.01, description='Rhythm:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        self.tempo_tol = widgets.FloatSlider(value=0.1, min=0.05, max=0.3, step=0.05, description='Tempo:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        params_box = widgets.VBox([
            widgets.HTML("<h3>⚙️パラメータ</h3>"),
            widgets.HBox([self.pitch_tol, self.ignore_oct, self.arpeggio]),
            widgets.HBox([self.rhythm_tol, self.tempo_tol]),
        ])

        # ---- ボタン ----
        self.run_btn = widgets.Button(description='🎵採点', button_style='success', layout=widgets.Layout(width='100px', height='40px'))
        self.run_btn.on_click(self._on_run)
        self.play_btn = widgets.Button(description='▶️再生', button_style='info', layout=widgets.Layout(width='80px'))
        self.play_btn.on_click(self._on_play)
        self.reset_rec_btn = widgets.Button(description='🔄録音リセット', button_style='warning', layout=widgets.Layout(width='100px'))
        self.reset_rec_btn.on_click(self._on_reset_recording)
        self.reset_all_btn = widgets.Button(description='🗑️全リセット', button_style='danger', layout=widgets.Layout(width='100px'))
        self.reset_all_btn.on_click(self._on_reset_all)
        btn_box = widgets.HBox([self.run_btn, self.play_btn, self.reset_rec_btn, self.reset_all_btn],
                               layout=widgets.Layout(justify_content='center', margin='15px 0'))

        self.output = widgets.Output()

        # ---- 分析 ----
        self.measure_sel = widgets.IntSlider(value=1, min=1, max=100, description='M#:', layout=widgets.Layout(width='200px'))
        self.show_btn = widgets.Button(description='📊グラフ', button_style='info', layout=widgets.Layout(width='70px'))
        self.show_btn.on_click(self._on_show_measure)
        self.show_score_btn = widgets.Button(description='🎼楽譜', button_style='success', layout=widgets.Layout(width='70px'))
        self.show_score_btn.on_click(self._on_show_score)
        self.show_problems_btn = widgets.Button(description='⚠️問題', button_style='warning', layout=widgets.Layout(width='70px'))
        self.show_problems_btn.on_click(self._on_show_problems)
        self.show_brain_btn = widgets.Button(description='🧠タイムライン', button_style='info', layout=widgets.Layout(width='100px'))
        self.show_brain_btn.on_click(self._on_show_brain)
        self.show_all_btn = widgets.Button(description='📋全表', layout=widgets.Layout(width='60px'))
        self.show_all_btn.on_click(self._on_show_all)
        debug_box = widgets.VBox([
            widgets.HTML("<h3>🔍分析</h3>"),
            widgets.HBox([self.measure_sel, self.show_btn, self.show_score_btn, self.show_problems_btn, self.show_brain_btn, self.show_all_btn]),
        ])
        self.debug_output = widgets.Output()

        try:
            from google.colab import output as colab_output
            colab_output.register_callback('piano.receive_audio', self._receive_audio_callback)
        except: pass

        display(widgets.VBox([brain_box, upload_box, params_box, btn_box, self.output, debug_box, self.debug_output]))

    def _on_brain_toggle(self, c):
        self.brain_controls.layout.display = 'block' if c['new'] else 'none'

    def _on_parse_brain(self, b):
        import tempfile
        with self.output:
            self.output.clear_output()
            if not self.brain_csv_upload.value:
                print("⚠️ 脳波CSVをアップロードしてください")
                return

            # CSVを一時ファイルに保存
            name = list(self.brain_csv_upload.value.keys())[0]
            content = self.brain_csv_upload.value[name]['content']
            with tempfile.NamedTemporaryFile(suffix='.csv', delete=False) as f:
                f.write(content)
                self.brain_csv_path = f.name

            # CSV解析
            if not brain_csv_parser.parse_csv(self.brain_csv_path):
                self.brain_status.value = '<span style="color:red;">CSV解析失敗</span>'
                return

            # 時刻設定
            start_str = self.rec_start_input.value.strip()
            end_str = self.rec_end_input.value.strip()

            if not start_str or not end_str:
                self.brain_status.value = f'<span style="color:orange;">録音時刻を入力してください<br>CSV範囲: {brain_csv_parser.start_time.strftime("%H:%M:%S")} - {brain_csv_parser.end_time.strftime("%H:%M:%S")}</span>'
                return

            if not brain_csv_parser.set_recording_time(start_str, end_str):
                self.brain_status.value = '<span style="color:red;">時刻設定失敗</span>'
                return

            # アライメント
            if brain_csv_parser.align_to_recording():
                self.brain_status.value = f'<span style="color:green;">✓ {len(brain_csv_parser.aligned_data)}サンプル準備完了</span>'
            else:
                self.brain_status.value = '<span style="color:red;">アライメント失敗</span>'

    def _on_input_mode_change(self, c):
        if c['new'] == 'ファイル':
            self.audio_file_box.layout.display = 'block'
            self.audio_record_box.layout.display = 'none'
        else:
            self.audio_file_box.layout.display = 'none'
            self.audio_record_box.layout.display = 'block'

    def _update_params(self):
        self.params.pitch_tolerance = self.pitch_tol.value
        self.params.ignore_octave = self.ignore_oct.value
        self.params.allow_arpeggio = self.arpeggio.value
        self.params.rhythm_tolerance = self.rhythm_tol.value
        self.params.tempo_tolerance = self.tempo_tol.value

    def _on_record(self, b):
        from IPython.display import display, Javascript
        if not self.is_recording:
            self.is_recording = True
            self.record_btn.description = '⏹️停止'
            self.record_btn.button_style = 'danger'
            self.record_status.value = '<span style="color:red;">●REC</span>'
            g = self.gain_slider.value
            display(Javascript(f"""(async function(){{window.audioChunks=[];const stream=await navigator.mediaDevices.getUserMedia({{audio:true}});const ctx=new AudioContext();const src=ctx.createMediaStreamSource(stream);const gn=ctx.createGain();gn.gain.value={g};const dest=ctx.createMediaStreamDestination();src.connect(gn);gn.connect(dest);const candidates=['audio/webm;codecs=opus','audio/webm','audio/ogg;codecs=opus'];let chosen='';for(const c of candidates){{if(MediaRecorder.isTypeSupported(c)){{chosen=c;break;}}}}window.__recMime=chosen||'audio/webm';window.mediaRecorder=new MediaRecorder(dest.stream,chosen?{{mimeType:chosen}}:{{}});window.originalStream=stream;window.mediaRecorder.ondataavailable=(e)=>{{if(e.data&&e.data.size>0)window.audioChunks.push(e.data);}};window.mediaRecorder.start(1000);}})();"""))
        else:
            self.is_recording = False
            self.record_btn.description = '🎤録音'
            self.record_btn.button_style = 'info'
            self.record_status.value = '<span style="color:blue;">⏳</span>'
            display(Javascript(r"""(async function(){if(window.mediaRecorder&&window.mediaRecorder.state==='recording'){window.mediaRecorder.stop();window.mediaRecorder.onstop=async()=>{if(window.originalStream)window.originalStream.getTracks().forEach(t=>t.stop());const blob=new Blob(window.audioChunks,{type:window.__recMime||'audio/webm'});const reader=new FileReader();reader.onloadend=function(){const base64=reader.result.split(',')[1];if(typeof google!=='undefined'&&google.colab){google.colab.kernel.invokeFunction('piano.receive_audio',[base64,window.__recMime],{});}};reader.readAsDataURL(blob);};}})();"""))

    def _on_play(self, b):
        from IPython.display import display, Audio
        with self.output:
            if self.recorded_audio: display(Audio(self.recorded_audio, autoplay=True))
            elif self.audio_path: display(Audio(self.audio_path, autoplay=True))
            else: print("再生する音声がありません")

    def _save_files(self):
        import tempfile
        if self.input_mode.value == 'ファイル' and self.audio_upload.value:
            name = list(self.audio_upload.value.keys())[0]
            content = self.audio_upload.value[name]['content']
            ext = '.' + name.split('.')[-1] if '.' in name else '.wav'
            with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
                f.write(content)
                self.audio_path = f.name
        elif self.input_mode.value == '録音' and self.recorded_audio:
            self.audio_path = self.recorded_audio
        if self.midi_upload.value:
            name = list(self.midi_upload.value.keys())[0]
            with tempfile.NamedTemporaryFile(suffix='.mid', delete=False) as f:
                f.write(self.midi_upload.value[name]['content'])
                self.midi_path = f.name
        if self.mxl_upload.value:
            name = list(self.mxl_upload.value.keys())[0]
            ext = '.' + name.split('.')[-1] if '.' in name else '.mxl'
            with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
                f.write(self.mxl_upload.value[name]['content'])
                self.mxl_path = f.name

    def _on_run(self, b):
        with self.output:
            self.output.clear_output()
            # デバッグ情報
            print(f"[DEBUG] input_mode: {self.input_mode.value}")
            print(f"[DEBUG] audio_upload.value: {bool(self.audio_upload.value)}")
            print(f"[DEBUG] recorded_audio: {self.recorded_audio}")
            print(f"[DEBUG] audio_path: {self.audio_path}")

            has_audio = bool(self.audio_upload.value) or bool(self.recorded_audio)
            if not has_audio: print("⚠️音声をアップロードまたは録音してください"); return
            if not self.midi_upload.value: print("⚠️MIDIをアップロードしてください"); return
            if not self.mxl_upload.value: print("⚠️MXLをアップロードしてください"); return
            self._update_params()
            try:
                self._save_files()
                if not self.audio_path: print("⚠️音声ファイルがありません"); return
                parser = ReferenceParser(self.midi_path, self.mxl_path)
                self.measures, self.ref_notes = parser.parse()
                if not self.measures: print("❌参照データ解析失敗"); return
                transcriber = AudioTranscriber(use_gpu=True)
                self.perf_notes, self.audio_duration = transcriber.transcribe(self.audio_path)
                aligner = DTWAligner(self.params)
                self.measures = aligner.align(self.perf_notes, self.ref_notes, self.measures)

                # 脳波統合
                brain_parser = brain_csv_parser if (self.brain_enabled.value and brain_csv_parser.aligned_data) else None
                scorer = ScoringEngine(self.params, brain_parser)
                self.result = scorer.score_all(self.measures, self.audio_duration)

                self.visualizer.show_summary(self.result)
                self.visualizer.show_table(self.result, only_played=True)
                self.measure_sel.max = len(self.measures)
            except Exception as e:
                print(f"❌Error: {e}")
                import traceback; traceback.print_exc()

    def _on_reset_recording(self, b):
        from IPython.display import display, Javascript
        self.audio_upload.value.clear()
        self.recorded_audio = None; self.audio_path = None; self.is_recording = False
        self.record_btn.description = '🎤録音'; self.record_btn.button_style = 'info'
        self.record_status.value = '<span style="color:gray;">待機</span>'
        display(Javascript('window.audioChunks=[];'))
        with self.output: self.output.clear_output(); print("✓ 録音リセット")

    def _on_reset_all(self, b):
        from IPython.display import display, Javascript
        self.audio_upload.value.clear(); self.midi_upload.value.clear(); self.mxl_upload.value.clear()
        self.brain_csv_upload.value.clear()
        self.audio_path = self.midi_path = self.mxl_path = self.recorded_audio = self.brain_csv_path = None
        self.measures, self.ref_notes, self.perf_notes, self.result = [], [], [], None
        self.is_recording = False; self.record_btn.description = '🎤録音'; self.record_btn.button_style = 'info'
        self.record_status.value = '<span style="color:gray;">待機</span>'
        self.brain_status.value = '<span style="color:gray;">CSVをアップロード</span>'
        brain_csv_parser.data = []; brain_csv_parser.aligned_data = []
        self.output.clear_output(); self.debug_output.clear_output()
        display(Javascript('window.audioChunks=[];'))
        with self.output: print("✓ 全リセット完了")

    def _on_show_measure(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.show_measure_comparison(self.measures[idx])

    def _on_show_score(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures or not self.mxl_path: print("採点を実行してください"); return
            print("🎼 楽譜読み込み中...")
            self.visualizer.show_score_with_click(self.mxl_path, self.measures)

    def _on_show_problems(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            self.visualizer.show_problem_measures(self.measures)

    def _on_show_brain(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            self.visualizer.show_brain_timeline(self.measures)

    def _on_show_all(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.result: print("採点を実行してください"); return
            self.visualizer.show_table(self.result, only_played=False)

def main():
    print("🎹🧠 Piano Performance Scorer v4.4 (Offline Brain Wave Integration)")
    print("=" * 60)
    install_dependencies()
    ui = PianoScorerUI()
    ui.create_ui()
    return ui

if __name__ == "__main__":
    ui = main()

In [ ]:
# ==============================================================================
# 🎹🧠 Piano Performance Scorer v4.4 - Offline Brain Wave Integration
# ==============================================================================
# Mind MonitorのCSVをアップロードして、録音と時間軸を合わせて脳波を小節に割り当て
#
# 使い方:
# 1. Mind Monitorで脳波を記録（CSVエクスポート）
# 2. 録音開始時刻と終了時刻をメモ（Mind Monitorの画面で確認）
# 3. Colabで録音 or 音声ファイルをアップロード
# 4. 脳波CSVをアップロード + 録音開始/終了時刻を入力
# 5. 採点実行 → 脳波と演奏が時間軸で統合される
# ==============================================================================

import numpy as np
import warnings
warnings.filterwarnings('ignore')

def install_dependencies():
    import subprocess, sys
    packages = [
        'pretty_midi', 'music21', 'librosa', 'matplotlib', 'pandas',
        'ipywidgets', 'transkun', 'dtw-python', 'verovio'
    ]
    for pkg in packages:
        try:
            __import__(pkg.replace('-', '_').replace('dtw-python', 'dtw'))
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    print("✓ 依存ライブラリ準備完了")

from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any
from datetime import datetime, timedelta
import threading
import time

@dataclass
class NoteEvent:
    pitch: int
    start_time: float
    end_time: float
    velocity: int = 64
    matched: bool = False
    @property
    def duration(self): return self.end_time - self.start_time
    def pitch_name(self):
        names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
        return f"{names[self.pitch % 12]}{(self.pitch // 12) - 1}"

@dataclass
class MeasureData:
    number: int
    index: int
    start_time: float
    end_time: float
    tempo: float
    time_signature: Tuple[int, int]
    ref_notes: List[NoteEvent] = field(default_factory=list)
    perf_notes: List[NoteEvent] = field(default_factory=list)
    pitch_score: float = 0.0
    rhythm_score: float = 0.0
    tempo_score: float = 0.0
    perf_start_time: float = 0.0
    perf_end_time: float = 0.0
    brain_state: str = ""
    brain_metrics: Dict = field(default_factory=dict)
    @property
    def duration(self): return self.end_time - self.start_time

@dataclass
class ScoringParams:
    pitch_tolerance: int = 0
    ignore_octave: bool = False
    chord_time_window: float = 0.05
    allow_arpeggio: bool = True
    arpeggio_max_time: float = 0.2
    rhythm_tolerance: float = 0.1
    tempo_tolerance: float = 0.1

# -----------------------------
# Brain Wave CSV Parser
# -----------------------------
class BrainWaveCSVParser:
    """Mind MonitorのCSVを解析"""

    def __init__(self):
        self.data = []
        self.start_time = None
        self.end_time = None
        self.recording_start = None  # 録音開始時刻（絶対時刻）
        self.recording_end = None    # 録音終了時刻（絶対時刻）
        self.aligned_data = []       # 録音に合わせた相対時刻データ

    def parse_csv(self, csv_path: str) -> bool:
        """Mind Monitor CSVを解析"""
        import pandas as pd

        try:
            df = pd.read_csv(csv_path)
            print(f"📊 CSV読み込み: {len(df)}行")
            print(f"   列: {list(df.columns)[:10]}...")

            # タイムスタンプ列を探す
            time_col = None
            for col in ['TimeStamp', 'Timestamp', 'timestamp', 'Time', 'time']:
                if col in df.columns:
                    time_col = col
                    break

            if time_col is None:
                print("⚠️ タイムスタンプ列が見つかりません")
                return False

            # α/β/θ列を探す
            alpha_cols = [c for c in df.columns if 'Alpha' in c or 'alpha' in c]
            beta_cols = [c for c in df.columns if 'Beta' in c or 'beta' in c]
            theta_cols = [c for c in df.columns if 'Theta' in c or 'theta' in c]

            print(f"   Alpha列: {alpha_cols[:2]}")
            print(f"   Beta列: {beta_cols[:2]}")
            print(f"   Theta列: {theta_cols[:2]}")

            self.data = []
            for _, row in df.iterrows():
                try:
                    # タイムスタンプをパース
                    ts_str = str(row[time_col])
                    # 複数のフォーマットを試す
                    ts = None
                    for fmt in ['%Y-%m-%d %H:%M:%S.%f', '%Y-%m-%d %H:%M:%S',
                               '%H:%M:%S.%f', '%H:%M:%S',
                               '%Y/%m/%d %H:%M:%S.%f', '%Y/%m/%d %H:%M:%S']:
                        try:
                            ts = datetime.strptime(ts_str, fmt)
                            break
                        except:
                            continue

                    if ts is None:
                        # Unix timestampかも
                        try:
                            ts = datetime.fromtimestamp(float(ts_str))
                        except:
                            continue

                    # α/β/θの平均を計算
                    def get_avg(cols):
                        vals = [float(row[c]) for c in cols if c in row and pd.notna(row[c])]
                        return sum(vals) / len(vals) if vals else 0.0

                    alpha = get_avg(alpha_cols)
                    beta = get_avg(beta_cols)
                    theta = get_avg(theta_cols)

                    self.data.append({
                        'timestamp': ts,
                        'alpha': alpha,
                        'beta': beta,
                        'theta': theta
                    })
                except Exception as e:
                    continue

            if not self.data:
                print("⚠️ 有効なデータがありません")
                return False

            self.start_time = self.data[0]['timestamp']
            self.end_time = self.data[-1]['timestamp']
            duration = (self.end_time - self.start_time).total_seconds()

            print(f"✓ 脳波データ: {len(self.data)}サンプル")
            print(f"   期間: {self.start_time.strftime('%H:%M:%S')} - {self.end_time.strftime('%H:%M:%S')} ({duration:.1f}秒)")

            return True

        except Exception as e:
            print(f"❌ CSV解析エラー: {e}")
            import traceback
            traceback.print_exc()
            return False

    def set_recording_time(self, start_str: str, end_str: str) -> bool:
        """録音の開始/終了時刻を設定（HH:MM:SS形式）"""
        try:
            # 今日の日付を使用
            base_date = self.start_time.date() if self.start_time else datetime.now().date()

            # 時刻をパース
            for fmt in ['%H:%M:%S.%f', '%H:%M:%S', '%H:%M']:
                try:
                    start_time = datetime.strptime(start_str, fmt).time()
                    break
                except:
                    continue
            else:
                print(f"⚠️ 開始時刻のフォーマットエラー: {start_str}")
                return False

            for fmt in ['%H:%M:%S.%f', '%H:%M:%S', '%H:%M']:
                try:
                    end_time = datetime.strptime(end_str, fmt).time()
                    break
                except:
                    continue
            else:
                print(f"⚠️ 終了時刻のフォーマットエラー: {end_str}")
                return False

            self.recording_start = datetime.combine(base_date, start_time)
            self.recording_end = datetime.combine(base_date, end_time)

            duration = (self.recording_end - self.recording_start).total_seconds()
            print(f"✓ 録音時間設定: {start_str} - {end_str} ({duration:.1f}秒)")

            return True

        except Exception as e:
            print(f"❌ 時刻設定エラー: {e}")
            return False

    def align_to_recording(self) -> bool:
        """脳波データを録音時間に合わせて相対時刻に変換"""
        if not self.data or not self.recording_start or not self.recording_end:
            print("⚠️ データまたは録音時間が設定されていません")
            return False

        self.aligned_data = []
        rec_duration = (self.recording_end - self.recording_start).total_seconds()

        for d in self.data:
            ts = d['timestamp']
            if self.recording_start <= ts <= self.recording_end:
                relative_time = (ts - self.recording_start).total_seconds()

                # 状態判定
                alpha, beta, theta = d['alpha'], d['beta'], d['theta']
                total = alpha + beta + theta
                if total > 0:
                    a_rel = alpha / total
                    b_rel = beta / total
                    t_rel = theta / total
                    engagement = b_rel / (a_rel + t_rel + 1e-9)

                    if engagement > 1.1 and a_rel < 0.4:
                        state = 'FOCUSED'
                    elif a_rel > 0.45:
                        state = 'RELAXED'
                    else:
                        state = 'NEUTRAL'
                else:
                    state = 'NO_DATA'
                    a_rel = b_rel = t_rel = 0

                self.aligned_data.append({
                    'relative_time': relative_time,
                    'alpha': alpha,
                    'beta': beta,
                    'theta': theta,
                    'alpha_rel': a_rel,
                    'beta_rel': b_rel,
                    'theta_rel': t_rel,
                    'state': state
                })

        print(f"✓ アライメント完了: {len(self.aligned_data)}サンプル（{rec_duration:.1f}秒間）")
        return len(self.aligned_data) > 0

    def get_states_for_time_range(self, start_sec: float, end_sec: float) -> Dict[str, Any]:
        """指定時間範囲の脳波状態を取得"""
        if not self.aligned_data:
            return {'available': False, 'dominant_state': 'NO_DATA', 'count': 0}

        states_in_range = [
            d for d in self.aligned_data
            if start_sec <= d['relative_time'] <= end_sec
        ]

        if not states_in_range:
            return {'available': True, 'dominant_state': 'NO_DATA', 'count': 0, 'focus_ratio': 0.0}

        counts = {'FOCUSED': 0, 'RELAXED': 0, 'NEUTRAL': 0, 'NO_DATA': 0}
        alpha_sum, beta_sum, theta_sum = 0.0, 0.0, 0.0

        for s in states_in_range:
            st = s.get('state', 'NEUTRAL')
            if st in counts:
                counts[st] += 1
            alpha_sum += s.get('alpha_rel', 0)
            beta_sum += s.get('beta_rel', 0)
            theta_sum += s.get('theta_rel', 0)

        total = len(states_in_range)
        dominant = max(counts, key=counts.get)

        return {
            'available': True,
            'dominant_state': dominant,
            'count': total,
            'focus_ratio': counts['FOCUSED'] / total if total > 0 else 0.0,
            'relax_ratio': counts['RELAXED'] / total if total > 0 else 0.0,
            'avg_alpha': alpha_sum / total if total > 0 else 0,
            'avg_beta': beta_sum / total if total > 0 else 0,
            'avg_theta': theta_sum / total if total > 0 else 0,
            'state_counts': counts
        }

    def get_session_summary(self) -> Dict[str, Any]:
        """セッション全体のサマリー"""
        if not self.aligned_data:
            return {'available': False}

        counts = {'FOCUSED': 0, 'RELAXED': 0, 'NEUTRAL': 0, 'NO_DATA': 0}
        for d in self.aligned_data:
            st = d.get('state', 'NEUTRAL')
            if st in counts:
                counts[st] += 1

        total = len(self.aligned_data)
        duration = self.aligned_data[-1]['relative_time'] if self.aligned_data else 0

        return {
            'available': True,
            'total_samples': total,
            'duration': duration,
            'focus_ratio': counts['FOCUSED'] / total if total > 0 else 0.0,
            'relax_ratio': counts['RELAXED'] / total if total > 0 else 0.0,
            'neutral_ratio': counts['NEUTRAL'] / total if total > 0 else 0.0,
            'state_counts': counts
        }

brain_csv_parser = BrainWaveCSVParser()

# -----------------------------
# Reference Parser
# -----------------------------
class ReferenceParser:
    def __init__(self, midi_path, mxl_path):
        self.midi_path, self.mxl_path = midi_path, mxl_path

    def parse(self):
        print("📖 参照データを解析中...")
        midi_notes, midi_dur = self._parse_midi()
        print(f"  ✓ MIDI: {len(midi_notes)}音符, {midi_dur:.1f}秒")
        mxl_measures = self._parse_mxl()
        print(f"  ✓ MXL: {len(mxl_measures)}小節")
        measures = self._create_measures(midi_notes, mxl_measures, midi_dur)
        return measures, midi_notes

    def _parse_midi(self):
        import pretty_midi
        pm = pretty_midi.PrettyMIDI(self.midi_path)
        notes = []
        for inst in pm.instruments:
            if not inst.is_drum:
                for n in inst.notes:
                    notes.append(NoteEvent(pitch=n.pitch, start_time=n.start, end_time=n.end, velocity=n.velocity))
        notes.sort(key=lambda n: (n.start_time, n.pitch))
        return notes, pm.get_end_time()

    def _parse_mxl(self):
        from music21 import converter, tempo as m21tempo, meter
        score = converter.parse(self.mxl_path)
        try:
            score = score.expandRepeats()
        except:
            pass
        measures, current_tempo, current_ts = [], 120.0, (4, 4)
        for t in score.flatten().getElementsByClass(m21tempo.MetronomeMark):
            if t.number:
                current_tempo = float(t.number)
                break
        part = score.parts[0] if score.parts else score
        for m in part.getElementsByClass('Measure'):
            for t in m.getElementsByClass(m21tempo.MetronomeMark):
                if t.number:
                    current_tempo = float(t.number)
            for ts in m.getElementsByClass(meter.TimeSignature):
                if ts.numerator:
                    current_ts = (ts.numerator, ts.denominator)
            measures.append({
                'number': m.measureNumber,
                'tempo': current_tempo or 120,
                'time_signature': current_ts,
                'quarter_length': float(m.quarterLength) if m.quarterLength else 4.0
            })
        return measures

    def _create_measures(self, midi_notes, mxl_measures, midi_dur):
        if not mxl_measures:
            return []
        measures = []
        total_ql = sum(m['quarter_length'] for m in mxl_measures)
        midi_span = (midi_notes[-1].start_time - midi_notes[0].start_time) if midi_notes else midi_dur
        midi_offset = midi_notes[0].start_time if midi_notes else 0
        current_time = midi_offset
        for i, mxl in enumerate(mxl_measures):
            ql = mxl['quarter_length']
            dur = (ql / total_ql) * midi_span if total_ql > 0 else 2.0
            end_time = current_time + dur
            ref_notes = [
                NoteEvent(pitch=n.pitch, start_time=n.start_time, end_time=n.end_time, velocity=n.velocity)
                for n in midi_notes
                if current_time - 0.05 <= n.start_time < end_time + 0.05
            ]
            measures.append(MeasureData(
                number=mxl['number'], index=i, start_time=current_time, end_time=end_time,
                tempo=mxl['tempo'], time_signature=mxl['time_signature'], ref_notes=ref_notes
            ))
            current_time = end_time
        return measures

# -----------------------------
# Audio Transcriber
# -----------------------------
class AudioTranscriber:
    def __init__(self, use_gpu=True):
        self.use_gpu = use_gpu

    def transcribe(self, audio_path):
        import subprocess, tempfile, os, pretty_midi, librosa
        print("🎵 音声を解析中...")
        device = 'cpu'
        if self.use_gpu:
            try:
                import torch
                if torch.cuda.is_available():
                    device = 'cuda'
                    print("  ✓ CUDA使用")
            except:
                pass
        y, sr = librosa.load(audio_path, sr=None)
        duration = len(y) / sr
        print(f"  - 音声長: {duration:.1f}秒")
        with tempfile.TemporaryDirectory() as td:
            out = os.path.join(td, "out.mid")
            subprocess.run(
                ['python3', '-m', 'transkun.transcribe', audio_path, out, '--device', device],
                check=True, capture_output=True, timeout=600
            )
            pm = pretty_midi.PrettyMIDI(out)
            notes = [
                NoteEvent(pitch=n.pitch, start_time=n.start, end_time=n.end, velocity=n.velocity)
                for inst in pm.instruments if not inst.is_drum for n in inst.notes
            ]
            notes.sort(key=lambda n: (n.start_time, n.pitch))
            print(f"  ✓ {len(notes)}音符を認識")
            return notes, duration

# -----------------------------
# DTW Aligner
# -----------------------------
class DTWAligner:
    def __init__(self, params):
        self.params = params

    def align(self, perf_notes, ref_notes, measures):
        print("🔗 アライメント中...")
        if not perf_notes or not ref_notes:
            return measures
        try:
            from dtw import dtw
            pp = np.array([n.pitch for n in perf_notes]).reshape(-1, 1)
            rp = np.array([n.pitch for n in ref_notes]).reshape(-1, 1)
            def dist(x, y):
                d = abs(x[0] - y[0])
                return d/12*2 if d % 12 == 0 and d > 0 else 0 if d <= self.params.pitch_tolerance else d
            alignment = dtw(pp, rp, dist_method=dist, keep_internals=True,
                          step_pattern='symmetric2', window_type='sakoechiba',
                          window_args={'window_size': 400})
            path = list(zip(alignment.index1, alignment.index2))
        except:
            path = [(i, min(i, len(ref_notes)-1)) for i in range(len(perf_notes))]

        pairs = [(perf_notes[p].start_time, ref_notes[r].start_time)
                 for p, r in path if p < len(perf_notes) and r < len(ref_notes)]
        if not pairs:
            return measures

        pt = np.array([p[0] for p in pairs])
        rt = np.array([p[1] for p in pairs])

        def time_map(t):
            if t <= pt[0]: return rt[0]
            if t >= pt[-1]: return rt[-1]
            idx = np.searchsorted(pt, t)
            if idx == 0: return rt[0]
            ratio = (t - pt[idx-1]) / (pt[idx] - pt[idx-1]) if pt[idx] != pt[idx-1] else 0
            return rt[idx-1] + ratio * (rt[idx] - rt[idx-1])

        for note in perf_notes:
            ref_t = time_map(note.start_time)
            for m in measures:
                if m.start_time <= ref_t < m.end_time:
                    m.perf_notes.append(NoteEvent(
                        pitch=note.pitch, start_time=note.start_time,
                        end_time=note.end_time, velocity=note.velocity
                    ))
                    if not m.perf_start_time or note.start_time < m.perf_start_time:
                        m.perf_start_time = note.start_time
                    if note.start_time > m.perf_end_time:
                        m.perf_end_time = note.start_time
                    break

        print(f"  ✓ {sum(len(m.perf_notes) for m in measures)}/{len(perf_notes)}音符をマッピング")
        return measures

# -----------------------------
# Scoring Engine
# -----------------------------
class ScoringEngine:
    def __init__(self, params, brain_parser=None):
        self.params = params
        self.brain_parser = brain_parser

    def score_all(self, measures, audio_duration=None):
        print("📊 採点中...")

        all_perf = [n for m in measures for n in m.perf_notes]
        if all_perf:
            perf_start = min(n.start_time for n in all_perf)
            perf_end = max(n.end_time for n in all_perf)
            perf_duration = perf_end - perf_start
        else:
            perf_start, perf_end, perf_duration = 0, 0, 0

        brain_duration = None
        if self.brain_parser and self.brain_parser.aligned_data:
            brain_duration = self.brain_parser.aligned_data[-1]['relative_time']
            print(f"  🧠 脳波: {len(self.brain_parser.aligned_data)}サンプル, {brain_duration:.1f}秒")

        for m in measures:
            m.pitch_score = self._score_pitch(m)
            m.rhythm_score = self._score_rhythm(m)
            m.tempo_score = self._score_tempo(m)

            # 脳波割り当て
            if self.brain_parser and brain_duration and brain_duration > 0 and m.perf_notes:
                if perf_duration > 0:
                    m_start = m.perf_start_time if m.perf_start_time else min(n.start_time for n in m.perf_notes)
                    m_end = m.perf_end_time if m.perf_end_time else max(n.end_time for n in m.perf_notes)

                    # 演奏時刻を脳波時刻に変換
                    brain_start = ((m_start - perf_start) / perf_duration) * brain_duration
                    brain_end = ((m_end - perf_start) / perf_duration) * brain_duration

                    summary = self.brain_parser.get_states_for_time_range(brain_start, brain_end)
                    m.brain_state = summary.get('dominant_state', '')
                    m.brain_metrics = {
                        'focus_ratio': summary.get('focus_ratio', 0.0),
                        'relax_ratio': summary.get('relax_ratio', 0.0),
                        'count': summary.get('count', 0),
                    }

        result = self._calc_total(measures)
        print(f"  ✓ 総合スコア: {result['total_score']:.1f}")

        if self.brain_parser and self.brain_parser.aligned_data:
            result['brain_summary'] = self.brain_parser.get_session_summary()

        return result

    def _score_pitch(self, m):
        if not m.ref_notes: return 1.0 if not m.perf_notes else 0.0
        if not m.perf_notes: return 0.0
        ref_groups = self._group_by_time(m.ref_notes, self.params.chord_time_window)
        perf_groups = self._group_by_time(m.perf_notes, self.params.arpeggio_max_time if self.params.allow_arpeggio else self.params.chord_time_window)
        total = sum(len(g['pitches']) for g in ref_groups)
        pool = set(p for g in perf_groups for p in g['pitches'])
        matched = sum(1 for g in ref_groups for rp in g['pitches'] if rp in pool or any(self._match(rp, pp) for pp in pool))
        return matched / total if total > 0 else 1.0

    def _group_by_time(self, notes, window):
        if not notes: return []
        notes = sorted(notes, key=lambda n: n.start_time)
        groups = [{'time': notes[0].start_time, 'pitches': [notes[0].pitch]}]
        for n in notes[1:]:
            if n.start_time - groups[-1]['time'] <= window:
                groups[-1]['pitches'].append(n.pitch)
            else:
                groups.append({'time': n.start_time, 'pitches': [n.pitch]})
        return groups

    def _match(self, rp, pp):
        d = abs(rp - pp)
        return d == 0 or d <= self.params.pitch_tolerance or (self.params.ignore_octave and d % 12 == 0)

    def _score_rhythm(self, m):
        if not m.ref_notes or not m.perf_notes: return 1.0 if not m.ref_notes else 0.0
        def pos(notes, s, e):
            d = e - s if e > s else 0.1
            return [(n.start_time - s) / d for n in notes]
        ref_pos = pos(m.ref_notes, m.start_time, m.end_time)
        perf_pos = pos(m.perf_notes, m.perf_start_time, m.perf_end_time) if m.perf_end_time > m.perf_start_time else pos(m.perf_notes, m.perf_notes[0].start_time, m.perf_notes[-1].start_time + 0.1)
        tol = self.params.rhythm_tolerance
        return sum(1 for rp in ref_pos if any(abs(rp - pp) <= tol for pp in perf_pos)) / len(ref_pos) if ref_pos else 1.0

    def _score_tempo(self, m):
        if not m.perf_notes or len(m.perf_notes) < 2: return 1.0
        pd, rd = m.perf_end_time - m.perf_start_time, m.duration
        if pd <= 0 or rd <= 0: return 1.0
        r = pd / rd
        return 1.0 if 1 - self.params.tempo_tolerance <= r <= 1 + self.params.tempo_tolerance else max(0, 1 - (abs(r - 1) - self.params.tempo_tolerance) * 2)

    def _calc_total(self, measures):
        played = [m for m in measures if m.perf_notes]
        if not played:
            return {'total_score': 0, 'pitch_score': 0, 'rhythm_score': 0, 'tempo_score': 0, 'measures_played': 0, 'measures_total': len(measures), 'measure_details': []}
        ap, ar, at = np.mean([m.pitch_score for m in played]), np.mean([m.rhythm_score for m in played]), np.mean([m.tempo_score for m in played])
        return {
            'total_score': (ap * 0.5 + ar * 0.3 + at * 0.2) * 100, 'pitch_score': ap * 100, 'rhythm_score': ar * 100, 'tempo_score': at * 100,
            'measures_played': len(played), 'measures_total': len(measures),
            'start_measure': measures[min(m.index for m in played)].number, 'end_measure': measures[max(m.index for m in played)].number,
            'measure_details': [{'number': m.number, 'index': m.index, 'pitch': m.pitch_score*100, 'rhythm': m.rhythm_score*100, 'tempo': m.tempo_score*100, 'ref_notes': len(m.ref_notes), 'perf_notes': len(m.perf_notes), 'brain_state': m.brain_state, 'brain_metrics': m.brain_metrics} for m in measures]
        }

# -----------------------------
# Visualizer
# -----------------------------
class Visualizer:
    def show_summary(self, result):
        from IPython.display import display, HTML
        r = result
        brain_html = ""
        if 'brain_summary' in r and r['brain_summary'].get('available'):
            bs = r['brain_summary']
            brain_html = f"""<div style="margin-top:15px;padding:10px;background:rgba(255,255,255,0.1);border-radius:8px;"><div style="font-size:14px;margin-bottom:5px;">🧠 脳波 ({bs['total_samples']}サンプル, {bs.get('duration', 0):.1f}秒)</div><div style="display:flex;gap:15px;flex-wrap:wrap;"><span>🟢集中: {bs['focus_ratio']*100:.0f}%</span><span>🔵リラックス: {bs['relax_ratio']*100:.0f}%</span><span>⚪中立: {bs['neutral_ratio']*100:.0f}%</span></div></div>"""
        html = f"""<div style="background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);padding:20px;border-radius:15px;color:white;margin:10px 0;"><h2 style="margin:0 0 15px 0;">🎹 採点結果</h2><div style="display:flex;justify-content:space-around;flex-wrap:wrap;"><div style="text-align:center;padding:10px;"><div style="font-size:36px;font-weight:bold;">{r['total_score']:.1f}</div><div>総合</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['pitch_score']:.1f}</div><div>🎵ピッチ</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['rhythm_score']:.1f}</div><div>🥁リズム</div></div><div style="text-align:center;padding:10px;"><div style="font-size:28px;">{r['tempo_score']:.1f}</div><div>⏱️テンポ</div></div></div><div style="margin-top:10px;">演奏: {r['measures_played']}/{r['measures_total']}小節</div>{brain_html}</div>"""
        display(HTML(html))

    def show_table(self, result, only_played=True):
        import pandas as pd
        from IPython.display import display
        df = pd.DataFrame(result.get('measure_details', []))
        if df.empty: return
        if only_played: df = df[df['perf_notes'] > 0]
        has_brain = any(d.get('brain_state') for d in result.get('measure_details', []))
        if has_brain:
            df['brain'] = df.apply(lambda r: (r['brain_state'][:3] if r.get('brain_state') else '-'), axis=1)
            df = df[['number', 'index', 'pitch', 'rhythm', 'tempo', 'ref_notes', 'perf_notes', 'brain']]
            df.columns = ['小節', '通し', 'ピッチ', 'リズム', 'テンポ', '楽譜', '演奏', '脳波']
        else:
            df = df[['number', 'index', 'pitch', 'rhythm', 'tempo', 'ref_notes', 'perf_notes']]
            df.columns = ['小節', '通し', 'ピッチ', 'リズム', 'テンポ', '楽譜', '演奏']
        def color(v):
            try:
                v = float(v)
                return 'background-color:#90EE90' if v >= 80 else 'background-color:#FFE4B5' if v >= 50 else 'background-color:#FFB6C1' if v > 0 else ''
            except: return ''
        display(df.style.applymap(color, subset=['ピッチ', 'リズム', 'テンポ']))

    def show_brain_timeline(self, measures):
        import matplotlib.pyplot as plt
        played = [m for m in measures if m.perf_notes]
        if not played: print("演奏小節がありません"); return
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
        indices = [m.index for m in played]
        scores = [(m.pitch_score * 0.5 + m.rhythm_score * 0.3 + m.tempo_score * 0.2) * 100 for m in played]
        colors = ['#90EE90' if s >= 80 else '#FFE4B5' if s >= 50 else '#FFB6C1' for s in scores]
        ax1.bar(indices, scores, color=colors, edgecolor='gray')
        ax1.axhline(y=70, color='red', linestyle='--', alpha=0.5, label='70%')
        ax1.set_ylabel('Score (%)')
        ax1.set_ylim(0, 100)
        ax1.legend()
        ax1.set_title('Score and Brain State by Measure')
        state_colors = {'FOCUSED': '#28a745', 'RELAXED': '#007bff', 'NEUTRAL': '#6c757d', 'NO_DATA': '#dc3545'}
        brain_colors = [state_colors.get(m.brain_state, '#999') for m in played]
        ax2.bar(indices, [1]*len(indices), color=brain_colors, edgecolor='gray')
        ax2.set_ylabel('Brain')
        ax2.set_xlabel('Measure Index')
        ax2.set_yticks([])
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=c, label=s) for s, c in state_colors.items()]
        ax2.legend(handles=legend_elements, loc='upper right', ncol=4)
        plt.tight_layout()
        plt.show()

        # 統計
        state_counts = {}
        for m in played:
            st = m.brain_state or 'UNKNOWN'
            state_counts[st] = state_counts.get(st, 0) + 1
        print("\n脳波状態の統計:")
        for st, cnt in sorted(state_counts.items(), key=lambda x: -x[1]):
            print(f"  {st}: {cnt}小節 ({cnt/len(played)*100:.1f}%)")

    def show_measure_comparison(self, m):
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        def plot(ax, notes, off, c, title):
            if not notes: ax.text(0.5, 0.5, 'No notes', ha='center', va='center', transform=ax.transAxes); ax.set_title(title); return
            for n in notes: ax.barh(n.pitch, n.duration, left=n.start_time - off, height=0.6, color=c, alpha=0.6)
            ax.set_ylim(min(n.pitch for n in notes) - 2, max(n.pitch for n in notes) + 2)
            ax.set_xlabel('Time (s)'); ax.set_ylabel('Pitch'); ax.set_title(title); ax.grid(True, alpha=0.3)
        plot(axes[0], m.ref_notes, m.start_time, 'blue', f'Score ({len(m.ref_notes)} notes)')
        plot(axes[1], m.perf_notes, m.perf_notes[0].start_time if m.perf_notes else 0, 'red', f'Perf ({len(m.perf_notes)} notes)')
        brain = f" | 🧠{m.brain_state}" if m.brain_state else ""
        fig.suptitle(f"Measure {m.number} | P:{m.pitch_score*100:.1f}% | R:{m.rhythm_score*100:.1f}%{brain}", y=1.02)
        plt.tight_layout(); plt.show()

    def show_problem_measures(self, measures, threshold=0.7):
        from IPython.display import display, HTML
        problems = []
        for m in measures:
            issues = []
            if m.pitch_score < threshold: issues.append(f"P:{m.pitch_score*100:.0f}%")
            if m.rhythm_score < threshold: issues.append(f"R:{m.rhythm_score*100:.0f}%")
            if issues: problems.append({'num': m.number, 'idx': m.index, 'issues': issues, 'brain': m.brain_state or '-'})
        if not problems: print("✓ No problems (all >= 70%)"); return
        html = f"<h3>⚠️ Problems ({len(problems)})</h3><table style='border-collapse:collapse;'><tr style='background:#f0f0f0;'><th style='padding:5px;border:1px solid #ddd;'>M#</th><th style='padding:5px;border:1px solid #ddd;'>Idx</th><th style='padding:5px;border:1px solid #ddd;'>Issues</th><th style='padding:5px;border:1px solid #ddd;'>Brain</th></tr>"
        for p in problems[:30]: html += f"<tr><td style='padding:5px;border:1px solid #ddd;'>{p['num']}</td><td style='padding:5px;border:1px solid #ddd;'>{p['idx']}</td><td style='padding:5px;border:1px solid #ddd;color:red;'>{', '.join(p['issues'])}</td><td style='padding:5px;border:1px solid #ddd;'>{p['brain']}</td></tr>"
        html += "</table>"
        display(HTML(html))

    def show_pitch_histogram(self, measures):
        import matplotlib.pyplot as plt
        ref_p = [n.pitch for m in measures for n in m.ref_notes]
        perf_p = [n.pitch for m in measures for n in m.perf_notes]
        if not ref_p and not perf_p: print("No data"); return
        fig, ax = plt.subplots(figsize=(12, 4))
        bins = range(min(ref_p + perf_p) - 1, max(ref_p + perf_p) + 2)
        ax.hist(ref_p, bins=bins, alpha=0.5, label=f'Score({len(ref_p)})', color='blue')
        ax.hist(perf_p, bins=bins, alpha=0.5, label=f'Perf({len(perf_p)})', color='red')
        ax.legend(); ax.grid(True, alpha=0.3); plt.show()

    def show_score_with_click(self, mxl_path, measures):
        from IPython.display import display, HTML
        import verovio, re, json
        try:
            tk = verovio.toolkit()
            tk.setOptions({"pageWidth": 1800, "pageHeight": 3000, "scale": 35, "adjustPageHeight": True})
            tk.loadFile(mxl_path)
            pc = tk.getPageCount()
            amd = [{'index': m.index, 'number': m.number, 'pitch': m.pitch_score*100, 'rhythm': m.rhythm_score*100, 'ref_notes': len(m.ref_notes), 'perf_notes': len(m.perf_notes), 'brain_state': m.brain_state, 'start_time': m.start_time, 'end_time': m.end_time, 'perf_start': m.perf_notes[0].start_time if m.perf_notes else 0, 'ref_note_list': [{'pitch': n.pitch, 'start': n.start_time, 'dur': n.duration} for n in m.ref_notes], 'perf_note_list': [{'pitch': n.pitch, 'start': n.start_time, 'dur': n.duration} for n in m.perf_notes]} for m in measures]
            mc = {i: "#90EE90" if (m.pitch_score + m.rhythm_score)/2*100 >= 90 else "#FFFF99" if (m.pitch_score + m.rhythm_score)/2*100 >= 70 else "#FFE4B5" if (m.pitch_score + m.rhythm_score)/2*100 >= 50 else "#FFB6C1" for i, m in enumerate(measures)}
            svgs = []
            measure_idx = 0
            for p in range(1, pc + 1):
                svg = tk.renderToSVG(p)
                def add_attr(match):
                    nonlocal measure_idx
                    result = match.group(0)
                    if 'data-measure-idx' not in result:
                        result = result[:-1] + f' data-measure-idx="{measure_idx}" style="cursor:pointer;">'
                        measure_idx += 1
                    return result
                svg = re.sub(r'<g[^>]*class="[^"]*measure[^"]*"[^>]*>', add_attr, svg)
                svgs.append(svg)
            html = f'''<style>.score-viewer{{display:grid;grid-template-columns:1.5fr 1fr;gap:15px;height:700px;}}.score-panel{{overflow:auto;border:1px solid #ccc;padding:10px;background:white;}}.detail-panel{{border:1px solid #ccc;border-radius:8px;background:#f8f9fa;padding:15px;overflow-y:auto;}}.measure-highlight{{cursor:pointer;}}.measure-highlight:hover{{filter:brightness(0.85);}}</style><div class="score-viewer"><div class="score-panel" id="score-panel">{"".join(svgs)}</div><div class="detail-panel" id="detail-panel"><h3>Click a measure</h3></div></div><script>(function(){{var mc={json.dumps(mc)};var am={json.dumps(amd)};var pn=['C','C#','D','D#','E','F','F#','G','G#','A','A#','B'];function p2n(p){{return pn[p%12]+(Math.floor(p/12)-1);}}function createPianoWave(ctx){{var real=new Float32Array([0,1.0,0.5,0.35,0.2,0.12,0.08,0.05,0.03]);var imag=new Float32Array(real.length);return ctx.createPeriodicWave(real,imag);}}function show(i){{var m=am[i];if(!m)return;var rp=new Set(m.ref_note_list.map(n=>n.pitch));var pp=new Set(m.perf_note_list.map(n=>n.pitch));var mt=[...rp].filter(p=>pp.has(p));var ms=[...rp].filter(p=>!pp.has(p));var h='<h3>M'+m.number+'</h3><div style="margin:10px 0;"><span style="padding:4px 8px;border-radius:4px;background:'+(m.pitch>=80?'#90EE90':'#FFB6C1')+';">P:'+m.pitch.toFixed(1)+'%</span> <span style="padding:4px 8px;border-radius:4px;background:'+(m.rhythm>=80?'#87CEEB':'#FFB6C1')+';">R:'+m.rhythm.toFixed(1)+'%</span></div>';if(m.brain_state)h+='<div style="background:#e9ecef;padding:5px;border-radius:4px;margin:5px 0;">🧠'+m.brain_state+'</div>';h+='<div style="margin:10px 0;"><button onclick="playRef('+i+')" style="padding:8px 16px;margin:5px;cursor:pointer;border:none;border-radius:4px;background:#4CAF50;color:white;">🔊楽譜</button>';if(m.perf_notes>0)h+='<button onclick="playPerf('+i+')" style="padding:8px 16px;margin:5px;cursor:pointer;border:none;border-radius:4px;background:#f44336;color:white;">🔊演奏</button>';h+='</div><div style="font-size:11px;"><span style="color:green;">✓'+mt.map(p=>p2n(p)).join(',')||'none'+'</span><br><span style="color:red;">✗'+ms.map(p=>p2n(p)).join(',')||'none'+'</span></div>';document.getElementById('detail-panel').innerHTML=h;}}function playRef(i){{var m=am[i];if(!m||!m.ref_note_list.length)return;var ctx=new(window.AudioContext||window.webkitAudioContext)();var pianoWave=createPianoWave(ctx);m.ref_note_list.forEach(n=>{{var o=ctx.createOscillator();var g=ctx.createGain();o.connect(g);g.connect(ctx.destination);o.frequency.value=440*Math.pow(2,(n.pitch-69)/12);o.setPeriodicWave(pianoWave);var s=Math.max(0,n.start-m.start_time);var d=Math.min(n.dur,2.0);g.gain.setValueAtTime(0,ctx.currentTime+s);g.gain.linearRampToValueAtTime(0.35,ctx.currentTime+s+0.008);g.gain.exponentialRampToValueAtTime(0.001,ctx.currentTime+s+d+0.5);o.start(ctx.currentTime+s);o.stop(ctx.currentTime+s+d+0.6);}});}}function playPerf(i){{var m=am[i];if(!m||!m.perf_note_list.length)return;var ctx=new(window.AudioContext||window.webkitAudioContext)();var pianoWave=createPianoWave(ctx);m.perf_note_list.forEach(n=>{{var o=ctx.createOscillator();var g=ctx.createGain();o.connect(g);g.connect(ctx.destination);o.frequency.value=440*Math.pow(2,(n.pitch-69)/12);o.setPeriodicWave(pianoWave);var s=Math.max(0,n.start-m.perf_start);var d=Math.min(n.dur,2.0);g.gain.setValueAtTime(0,ctx.currentTime+s);g.gain.linearRampToValueAtTime(0.3,ctx.currentTime+s+0.008);g.gain.exponentialRampToValueAtTime(0.001,ctx.currentTime+s+d+0.5);o.start(ctx.currentTime+s);o.stop(ctx.currentTime+s+d+0.6);}});}}window.playRef=playRef;window.playPerf=playPerf;setTimeout(function(){{var panel=document.getElementById('score-panel');if(!panel)return;var els=panel.querySelectorAll('[data-measure-idx]');if(!els.length){{els=panel.querySelectorAll('g[class*="measure"]');els.forEach(function(el,idx){{el.setAttribute('data-measure-idx',idx);}});}}els.forEach(function(el){{var idx=parseInt(el.getAttribute('data-measure-idx'));var color=mc[idx]||'#FFF';el.classList.add('measure-highlight');try{{var bbox=el.getBBox();if(bbox.width>0){{var rect=document.createElementNS('http://www.w3.org/2000/svg','rect');rect.setAttribute('x',bbox.x-5);rect.setAttribute('y',bbox.y-5);rect.setAttribute('width',bbox.width+10);rect.setAttribute('height',bbox.height+10);rect.setAttribute('fill',color);rect.setAttribute('opacity','0.4');rect.style.pointerEvents='none';el.insertBefore(rect,el.firstChild);}}}}catch(e){{}}el.addEventListener('click',function(e){{e.stopPropagation();if(idx<am.length)show(idx);}});}});}},500);}})();</script>'''
            display(HTML(html))
        except Exception as e:
            print(f"楽譜表示エラー: {e}")

# -----------------------------
# UI
# -----------------------------
class PianoScorerUI:
    def __init__(self):
        self.params = ScoringParams()
        self.measures, self.ref_notes, self.perf_notes, self.result = [], [], [], None
        self.visualizer = Visualizer()
        self.audio_path = self.midi_path = self.mxl_path = self.recorded_audio = self.brain_csv_path = None
        self.audio_duration = None
        self.is_recording = False

    def _receive_audio_callback(self, base64_data, mime_type='audio/webm'):
        import base64, tempfile
        try:
            raw = base64.b64decode(base64_data)
            suffix = '.ogg' if 'ogg' in str(mime_type) else '.webm'
            with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
                f.write(raw)
                self.recorded_audio = f.name
            if self.input_mode.value == '録音':
                self.audio_path = self.recorded_audio
            self.record_status.value = '<span style="color:green;">✓完了</span>'
        except Exception as e:
            self.record_status.value = f'<span style="color:red;">Error:{e}</span>'

    def create_ui(self):
        import ipywidgets as widgets
        from IPython.display import display, HTML

        display(HTML("""<div style="background:linear-gradient(135deg,#1a1a2e 0%,#16213e 100%);padding:20px;border-radius:15px;color:white;margin-bottom:20px;">
            <h1 style="margin:0;">🎹🧠 Piano Performance Scorer v4.4</h1>
            <p style="margin:5px 0 0 0;opacity:0.8;">オフライン脳波統合版（Mind Monitor CSVアップロード）</p>
        </div>"""))

        # ---- 脳波CSV ----
        self.brain_enabled = widgets.Checkbox(value=False, description='🧠脳波CSVを使用')
        self.brain_enabled.observe(self._on_brain_toggle, names='value')
        self.brain_csv_upload = widgets.FileUpload(accept='.csv', multiple=False, description='脳波CSV')
        self.rec_start_input = widgets.Text(placeholder='HH:MM:SS', description='録音開始:', layout=widgets.Layout(width='180px'))
        self.rec_end_input = widgets.Text(placeholder='HH:MM:SS', description='録音終了:', layout=widgets.Layout(width='180px'))
        self.parse_brain_btn = widgets.Button(description='🧠CSV解析', button_style='info', layout=widgets.Layout(width='100px'))
        self.parse_brain_btn.on_click(self._on_parse_brain)
        self.brain_status = widgets.HTML(value='<span style="color:gray;">CSVをアップロード</span>')
        self.brain_controls = widgets.VBox([
            widgets.HBox([widgets.Label("脳波CSV:"), self.brain_csv_upload]),
            widgets.HBox([self.rec_start_input, self.rec_end_input, self.parse_brain_btn]),
            self.brain_status
        ])
        self.brain_controls.layout.display = 'none'
        brain_info = widgets.HTML("""<div style="background:#e8f4f8;padding:10px;border-radius:5px;margin:5px 0;font-size:12px;">
            <b>使い方:</b> Mind Monitorで録音中に脳波を記録 → CSVエクスポート → 録音開始/終了時刻を入力 → CSV解析
        </div>""")
        brain_box = widgets.VBox([widgets.HTML("<h3>🧠脳波CSV</h3>"), self.brain_enabled, self.brain_controls, brain_info])

        # ---- ファイル/録音 ----
        self.input_mode = widgets.RadioButtons(options=['ファイル', '録音'], value='ファイル', layout=widgets.Layout(width='100px'))
        self.input_mode.observe(self._on_input_mode_change, names='value')
        self.audio_upload = widgets.FileUpload(accept='.mp3,.wav,.m4a,.webm,.ogg', multiple=False)
        self.midi_upload = widgets.FileUpload(accept='.mid,.midi', multiple=False)
        self.mxl_upload = widgets.FileUpload(accept='.mxl,.xml,.musicxml', multiple=False)
        self.record_btn = widgets.Button(description='🎤録音', button_style='info', layout=widgets.Layout(width='80px'))
        self.record_btn.on_click(self._on_record)
        self.gain_slider = widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description='Gain:', layout=widgets.Layout(width='180px'))
        self.record_status = widgets.HTML(value='<span style="color:gray;">待機</span>')
        self.audio_file_box = widgets.VBox([widgets.Label("音声"), self.audio_upload])
        self.audio_record_box = widgets.VBox([widgets.HBox([self.record_btn, self.record_status]), self.gain_slider])
        self.audio_record_box.layout.display = 'none'
        upload_box = widgets.VBox([
            widgets.HTML("<h3>📁入力</h3>"),
            widgets.HBox([
                widgets.VBox([self.input_mode, self.audio_file_box, self.audio_record_box]),
                widgets.VBox([widgets.Label("MIDI"), self.midi_upload]),
                widgets.VBox([widgets.Label("MXL"), self.mxl_upload]),
            ])
        ])

        # ---- パラメータ ----
        self.pitch_tol = widgets.IntSlider(value=0, min=0, max=3, description='Pitch:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        self.ignore_oct = widgets.Checkbox(value=False, description='Oct無視')
        self.arpeggio = widgets.Checkbox(value=True, description='アルペジオ')
        self.rhythm_tol = widgets.FloatSlider(value=0.1, min=0.05, max=0.3, step=0.01, description='Rhythm:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        self.tempo_tol = widgets.FloatSlider(value=0.1, min=0.05, max=0.3, step=0.05, description='Tempo:', style={'description_width': '50px'}, layout=widgets.Layout(width='200px'))
        params_box = widgets.VBox([
            widgets.HTML("<h3>⚙️パラメータ</h3>"),
            widgets.HBox([self.pitch_tol, self.ignore_oct, self.arpeggio]),
            widgets.HBox([self.rhythm_tol, self.tempo_tol]),
        ])

        # ---- ボタン ----
        self.run_btn = widgets.Button(description='🎵採点', button_style='success', layout=widgets.Layout(width='100px', height='40px'))
        self.run_btn.on_click(self._on_run)
        self.play_btn = widgets.Button(description='▶️再生', button_style='info', layout=widgets.Layout(width='80px'))
        self.play_btn.on_click(self._on_play)
        self.reset_rec_btn = widgets.Button(description='🔄録音リセット', button_style='warning', layout=widgets.Layout(width='100px'))
        self.reset_rec_btn.on_click(self._on_reset_recording)
        self.reset_all_btn = widgets.Button(description='🗑️全リセット', button_style='danger', layout=widgets.Layout(width='100px'))
        self.reset_all_btn.on_click(self._on_reset_all)
        btn_box = widgets.HBox([self.run_btn, self.play_btn, self.reset_rec_btn, self.reset_all_btn],
                               layout=widgets.Layout(justify_content='center', margin='15px 0'))

        self.output = widgets.Output()

        # ---- 分析 ----
        self.measure_sel = widgets.IntSlider(value=1, min=1, max=100, description='M#:', layout=widgets.Layout(width='200px'))
        self.show_btn = widgets.Button(description='📊グラフ', button_style='info', layout=widgets.Layout(width='70px'))
        self.show_btn.on_click(self._on_show_measure)
        self.show_score_btn = widgets.Button(description='🎼楽譜', button_style='success', layout=widgets.Layout(width='70px'))
        self.show_score_btn.on_click(self._on_show_score)
        self.show_problems_btn = widgets.Button(description='⚠️問題', button_style='warning', layout=widgets.Layout(width='70px'))
        self.show_problems_btn.on_click(self._on_show_problems)
        self.show_brain_btn = widgets.Button(description='🧠タイムライン', button_style='info', layout=widgets.Layout(width='100px'))
        self.show_brain_btn.on_click(self._on_show_brain)
        self.show_all_btn = widgets.Button(description='📋全表', layout=widgets.Layout(width='60px'))
        self.show_all_btn.on_click(self._on_show_all)
        debug_box = widgets.VBox([
            widgets.HTML("<h3>🔍分析</h3>"),
            widgets.HBox([self.measure_sel, self.show_btn, self.show_score_btn, self.show_problems_btn, self.show_brain_btn, self.show_all_btn]),
        ])
        self.debug_output = widgets.Output()

        try:
            from google.colab import output as colab_output
            colab_output.register_callback('piano.receive_audio', self._receive_audio_callback)
        except: pass

        display(widgets.VBox([brain_box, upload_box, params_box, btn_box, self.output, debug_box, self.debug_output]))

    def _on_brain_toggle(self, c):
        self.brain_controls.layout.display = 'block' if c['new'] else 'none'

    def _on_parse_brain(self, b):
        import tempfile
        with self.output:
            self.output.clear_output()
            if not self.brain_csv_upload.value:
                print("⚠️ 脳波CSVをアップロードしてください")
                return

            # CSVを一時ファイルに保存
            name = list(self.brain_csv_upload.value.keys())[0]
            content = self.brain_csv_upload.value[name]['content']
            with tempfile.NamedTemporaryFile(suffix='.csv', delete=False) as f:
                f.write(content)
                self.brain_csv_path = f.name

            # CSV解析
            if not brain_csv_parser.parse_csv(self.brain_csv_path):
                self.brain_status.value = '<span style="color:red;">CSV解析失敗</span>'
                return

            # 時刻設定
            start_str = self.rec_start_input.value.strip()
            end_str = self.rec_end_input.value.strip()

            if not start_str or not end_str:
                self.brain_status.value = f'<span style="color:orange;">録音時刻を入力してください<br>CSV範囲: {brain_csv_parser.start_time.strftime("%H:%M:%S")} - {brain_csv_parser.end_time.strftime("%H:%M:%S")}</span>'
                return

            if not brain_csv_parser.set_recording_time(start_str, end_str):
                self.brain_status.value = '<span style="color:red;">時刻設定失敗</span>'
                return

            # アライメント
            if brain_csv_parser.align_to_recording():
                self.brain_status.value = f'<span style="color:green;">✓ {len(brain_csv_parser.aligned_data)}サンプル準備完了</span>'
            else:
                self.brain_status.value = '<span style="color:red;">アライメント失敗</span>'

    def _on_input_mode_change(self, c):
        if c['new'] == 'ファイル':
            self.audio_file_box.layout.display = 'block'
            self.audio_record_box.layout.display = 'none'
        else:
            self.audio_file_box.layout.display = 'none'
            self.audio_record_box.layout.display = 'block'

    def _update_params(self):
        self.params.pitch_tolerance = self.pitch_tol.value
        self.params.ignore_octave = self.ignore_oct.value
        self.params.allow_arpeggio = self.arpeggio.value
        self.params.rhythm_tolerance = self.rhythm_tol.value
        self.params.tempo_tolerance = self.tempo_tol.value

    def _on_record(self, b):
        from IPython.display import display, Javascript
        if not self.is_recording:
            self.is_recording = True
            self.record_btn.description = '⏹️停止'
            self.record_btn.button_style = 'danger'
            self.record_status.value = '<span style="color:red;">●REC</span>'
            g = self.gain_slider.value
            display(Javascript(f"""(async function(){{window.audioChunks=[];const stream=await navigator.mediaDevices.getUserMedia({{audio:true}});const ctx=new AudioContext();const src=ctx.createMediaStreamSource(stream);const gn=ctx.createGain();gn.gain.value={g};const dest=ctx.createMediaStreamDestination();src.connect(gn);gn.connect(dest);const candidates=['audio/webm;codecs=opus','audio/webm','audio/ogg;codecs=opus'];let chosen='';for(const c of candidates){{if(MediaRecorder.isTypeSupported(c)){{chosen=c;break;}}}}window.__recMime=chosen||'audio/webm';window.mediaRecorder=new MediaRecorder(dest.stream,chosen?{{mimeType:chosen}}:{{}});window.originalStream=stream;window.mediaRecorder.ondataavailable=(e)=>{{if(e.data&&e.data.size>0)window.audioChunks.push(e.data);}};window.mediaRecorder.start(1000);}})();"""))
        else:
            self.is_recording = False
            self.record_btn.description = '🎤録音'
            self.record_btn.button_style = 'info'
            self.record_status.value = '<span style="color:blue;">⏳</span>'
            display(Javascript(r"""(async function(){if(window.mediaRecorder&&window.mediaRecorder.state==='recording'){window.mediaRecorder.stop();window.mediaRecorder.onstop=async()=>{if(window.originalStream)window.originalStream.getTracks().forEach(t=>t.stop());const blob=new Blob(window.audioChunks,{type:window.__recMime||'audio/webm'});const reader=new FileReader();reader.onloadend=function(){const base64=reader.result.split(',')[1];if(typeof google!=='undefined'&&google.colab){google.colab.kernel.invokeFunction('piano.receive_audio',[base64,window.__recMime],{});}};reader.readAsDataURL(blob);};}})();"""))

    def _on_play(self, b):
        from IPython.display import display, Audio
        with self.output:
            if self.recorded_audio: display(Audio(self.recorded_audio, autoplay=True))
            elif self.audio_path: display(Audio(self.audio_path, autoplay=True))
            else: print("再生する音声がありません")

    def _save_files(self):
        import tempfile
        if self.input_mode.value == 'ファイル' and self.audio_upload.value:
            name = list(self.audio_upload.value.keys())[0]
            content = self.audio_upload.value[name]['content']
            ext = '.' + name.split('.')[-1] if '.' in name else '.wav'
            with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
                f.write(content)
                self.audio_path = f.name
        elif self.input_mode.value == '録音' and self.recorded_audio:
            self.audio_path = self.recorded_audio
        if self.midi_upload.value:
            name = list(self.midi_upload.value.keys())[0]
            with tempfile.NamedTemporaryFile(suffix='.mid', delete=False) as f:
                f.write(self.midi_upload.value[name]['content'])
                self.midi_path = f.name
        if self.mxl_upload.value:
            name = list(self.mxl_upload.value.keys())[0]
            ext = '.' + name.split('.')[-1] if '.' in name else '.mxl'
            with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
                f.write(self.mxl_upload.value[name]['content'])
                self.mxl_path = f.name

    def _on_run(self, b):
        with self.output:
            self.output.clear_output()
            has_audio = bool(self.audio_upload.value) or bool(self.recorded_audio)
            if not has_audio: print("⚠️音声をアップロードまたは録音してください"); return
            if not self.midi_upload.value: print("⚠️MIDIをアップロードしてください"); return
            if not self.mxl_upload.value: print("⚠️MXLをアップロードしてください"); return
            self._update_params()
            try:
                self._save_files()
                if not self.audio_path: print("⚠️音声ファイルがありません"); return
                parser = ReferenceParser(self.midi_path, self.mxl_path)
                self.measures, self.ref_notes = parser.parse()
                if not self.measures: print("❌参照データ解析失敗"); return
                transcriber = AudioTranscriber(use_gpu=True)
                self.perf_notes, self.audio_duration = transcriber.transcribe(self.audio_path)
                aligner = DTWAligner(self.params)
                self.measures = aligner.align(self.perf_notes, self.ref_notes, self.measures)

                # 脳波統合
                brain_parser = brain_csv_parser if (self.brain_enabled.value and brain_csv_parser.aligned_data) else None
                scorer = ScoringEngine(self.params, brain_parser)
                self.result = scorer.score_all(self.measures, self.audio_duration)

                self.visualizer.show_summary(self.result)
                self.visualizer.show_table(self.result, only_played=True)
                self.measure_sel.max = len(self.measures)
            except Exception as e:
                print(f"❌Error: {e}")
                import traceback; traceback.print_exc()

    def _on_reset_recording(self, b):
        from IPython.display import display, Javascript
        self.audio_upload.value.clear()
        self.recorded_audio = None; self.audio_path = None; self.is_recording = False
        self.record_btn.description = '🎤録音'; self.record_btn.button_style = 'info'
        self.record_status.value = '<span style="color:gray;">待機</span>'
        display(Javascript('window.audioChunks=[];'))
        with self.output: self.output.clear_output(); print("✓ 録音リセット")

    def _on_reset_all(self, b):
        from IPython.display import display, Javascript
        self.audio_upload.value.clear(); self.midi_upload.value.clear(); self.mxl_upload.value.clear()
        self.brain_csv_upload.value.clear()
        self.audio_path = self.midi_path = self.mxl_path = self.recorded_audio = self.brain_csv_path = None
        self.measures, self.ref_notes, self.perf_notes, self.result = [], [], [], None
        self.is_recording = False; self.record_btn.description = '🎤録音'; self.record_btn.button_style = 'info'
        self.record_status.value = '<span style="color:gray;">待機</span>'
        self.brain_status.value = '<span style="color:gray;">CSVをアップロード</span>'
        brain_csv_parser.data = []; brain_csv_parser.aligned_data = []
        self.output.clear_output(); self.debug_output.clear_output()
        display(Javascript('window.audioChunks=[];'))
        with self.output: print("✓ 全リセット完了")

    def _on_show_measure(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            idx = self.measure_sel.value - 1
            if 0 <= idx < len(self.measures): self.visualizer.show_measure_comparison(self.measures[idx])

    def _on_show_score(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures or not self.mxl_path: print("採点を実行してください"); return
            print("🎼 楽譜読み込み中...")
            self.visualizer.show_score_with_click(self.mxl_path, self.measures)

    def _on_show_problems(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            self.visualizer.show_problem_measures(self.measures)

    def _on_show_brain(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.measures: print("採点を実行してください"); return
            self.visualizer.show_brain_timeline(self.measures)

    def _on_show_all(self, b):
        with self.debug_output:
            self.debug_output.clear_output()
            if not self.result: print("採点を実行してください"); return
            self.visualizer.show_table(self.result, only_played=False)

def main():
    print("🎹🧠 Piano Performance Scorer v4.4 (Offline Brain Wave Integration)")
    print("=" * 60)
    install_dependencies()
    ui = PianoScorerUI()
    ui.create_ui()
    return ui

if __name__ == "__main__":
    ui = main()